In [ ]:
import pandas as pd
import os
from pathlib import Path

data_dir = Path(r"Z:\0_Thesis_Monzurul\Coldwater MI\raw_videos\tracks_pixel")

# List all CSV files
csv_files = sorted(data_dir.glob("*.csv"))
print(f"Total CSV files found: {len(csv_files)}")
print()

# Show all filenames
for f in csv_files:
    print(f.name)

In [ ]:
# Peek at the first CSV
df_sample = pd.read_csv(csv_files[0])

print("Shape:", df_sample.shape)
print()
print("Columns:", df_sample.columns.tolist())
print()
print("First 5 rows:")
print(df_sample.head())
print()
print("Dtypes:")
print(df_sample.dtypes)
print()
print("Null counts:")
print(df_sample.isnull().sum())

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

data_dir = Path(r"Z:\0_Thesis_Monzurul\Coldwater MI\raw_videos\tracks_pixel")

csv_files = list(data_dir.glob("*.csv"))

# Parse datetime from filename
def parse_datetime_from_filename(filepath):
    name = filepath.stem  # remove .csv
    # Format: Coldwater_MI_2025-Sep-20_01-11-24_PM
    # Extract the date+time+ampm part → last 3 underscore-separated tokens
    parts = name.split("_")
    # parts[-1] = AM or PM
    # parts[-2] = HH-MM-SS
    # parts[-3] = YYYY-Mon-DD
    date_str = parts[-3]   # 2025-Aug-09
    time_str = parts[-2]   # 11-42-38
    ampm_str = parts[-1]   # AM or PM
    
    dt = datetime.strptime(f"{date_str} {time_str} {ampm_str}", "%Y-%b-%d %I-%M-%S %p")
    return dt

# Sort files chronologically
csv_files_sorted = sorted(csv_files, key=parse_datetime_from_filename)

print("Files in chronological order:")
for f in csv_files_sorted:
    dt = parse_datetime_from_filename(f)
    print(f"  {dt.strftime('%Y-%m-%d %I:%M:%S %p')}  →  {f.name}")

In [ ]:
print(f"Loading {len(csv_files_sorted)} files...\n")

dfs = []
for f in csv_files_sorted:
    dt = parse_datetime_from_filename(f)
    df_temp = pd.read_csv(f)
    df_temp["file_datetime"] = dt          # attach parsed datetime for reference
    dfs.append(df_temp)
    print(f"  {dt.strftime('%I:%M:%S %p')}  →  {len(df_temp):,} rows")

df_raw = pd.concat(dfs, ignore_index=True)

print(f"\n{'='*50}")
print(f"Total rows after merge : {len(df_raw):,}")
print(f"Total unique video_ids : {df_raw['video_id'].nunique()}")
print(f"Date range             : {df_raw['file_datetime'].min()} → {df_raw['file_datetime'].max()}")

In [ ]:
print("=== Null counts ===")
print(df_raw.isnull().sum())

print("\n=== Duplicate rows (video_id, track_id, frame_idx) ===")
dups = df_raw.duplicated(subset=["video_id", "track_id", "frame_idx"]).sum()
print(f"Duplicates: {dups:,}")

print("\n=== Class distribution ===")
print(df_raw["class"].value_counts())

print("\n=== Rows per video (chronological) ===")
rows_per_video = (
    df_raw.groupby("video_id")
          .agg(row_count=("frame_idx", "count"),
               file_datetime=("file_datetime", "first"))
          .sort_values("file_datetime")
          .reset_index()
)
print(rows_per_video[["file_datetime", "video_id", "row_count"]].to_string())

In [ ]:
valid_classes = {"car", "person", "truck", "bus", "motorcycle", "bicycle"}

df_raw = df_raw[df_raw["class"].isin(valid_classes)].copy()

print("=== Rows after class filter ===")
print(f"Total rows: {len(df_raw):,}")

print("\n=== Class distribution after filter ===")
print(df_raw["class"].value_counts())

print("\n=== Agent type mapping ===")
# Label agent type: vehicle vs pedestrian
# This will be used later for heterogeneous graph construction
vehicle_classes = {"car", "truck", "bus", "motorcycle", "bicycle"}
ped_classes = {"person"}

df_raw["agent_type"] = df_raw["class"].apply(
    lambda x: "vehicle" if x in vehicle_classes else "pedestrian"
)

print(df_raw["agent_type"].value_counts())

print("\n=== Rows per video after filter ===")
# rows_per_video = (
#     df_raw.groupby("video_id")
#           .agg(row_count=("frame_idx", "count"),
#                file_datetime=("file_datetime", "first"))
#           .sort_values("file_datetime")
#           .reset_index()
# )
# print(rows_per_video[["file_datetime", "video_id", "row_count"]].to_string())

In [ ]:
# Bottom-center is the correct ground plane anchor for homography
# u, v = bbox center → bottom-center = (u, v + h/2)

df_raw["x_anchor_px"] = df_raw["u"]
df_raw["y_anchor_px"] = df_raw["v"] + df_raw["h"] / 2.0

print("=== Anchor point sample ===")
print(df_raw[["u", "v", "w", "h", "x_anchor_px", "y_anchor_px"]].head(10))

print("\n=== Anchor point summary ===")
print(df_raw[["x_anchor_px", "y_anchor_px"]].describe())

In [ ]:
import json
import numpy as np

# Load zone config
with open("zone_config.json", "r") as f:
    zone_cfg = json.load(f)

H       = np.array(zone_cfg["homography_px_to_gps"], dtype=float)
roi_px  = np.array(zone_cfg["roi_px_TLTRBRBL"], dtype=float)
roi_gps = np.array(zone_cfg["roi_gps_TLTRBRBL"], dtype=float)

print("Homography matrix:\n", H)
print("\nROI pixel corners:\n", roi_px)
print("\nROI GPS corners:\n", roi_gps)

In [ ]:
def point_in_polygon(x, y, polygon):
    inside = False
    n = len(polygon)
    j = n - 1
    for i in range(n):
        xi, yi = polygon[i]
        xj, yj = polygon[j]
        intersects = ((yi > y) != (yj > y)) and (
            x < (xj - xi) * (y - yi) / ((yj - yi) + 1e-12) + xi
        )
        if intersects:
            inside = not inside
        j = i
    return inside

print("Applying ROI filter...")
df_raw["inside_roi"] = df_raw.apply(
    lambda r: point_in_polygon(r["x_anchor_px"], r["y_anchor_px"], roi_px),
    axis=1
)

print("\n=== ROI membership ===")
print(df_raw["inside_roi"].value_counts())

print("\n=== ROI membership by agent type ===")
print(df_raw.groupby(["agent_type", "inside_roi"]).size().unstack(fill_value=0))

df_raw = df_raw[df_raw["inside_roi"]].copy()
print(f"\nRows after ROI filter: {len(df_raw):,}")

In [ ]:
!pip freeze > requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Reload first CSV for visual check
df_check = pd.read_csv(csv_files_sorted[0])

df_check["x_anchor_px"] = df_check["u"]
df_check["y_anchor_px"] = df_check["v"] + df_check["h"] / 2.0

df_check["inside_roi"] = df_check.apply(
    lambda r: point_in_polygon(r["x_anchor_px"], r["y_anchor_px"], roi_px),
    axis=1
)

inside  = df_check[df_check["inside_roi"]]
outside = df_check[~df_check["inside_roi"]]

fig, ax = plt.subplots(figsize=(12, 8))

ax.scatter(
    outside["x_anchor_px"], outside["y_anchor_px"],
    s=1, alpha=0.1, c="red",
    label=f"Outside ROI ({len(outside):,})"
)

ax.scatter(
    inside["x_anchor_px"], inside["y_anchor_px"],
    s=1, alpha=0.3, c="blue",
    label=f"Inside ROI ({len(inside):,})"
)

# Draw ROI polygon
roi_closed = np.vstack([roi_px, roi_px[0]])
ax.plot(
    roi_closed[:, 0], roi_closed[:, 1],
    "g-", linewidth=3,
    label="ROI boundary"
)

# -------------------------------
# Dynamic plot limits
# -------------------------------
all_x = np.concatenate([
    df_check["x_anchor_px"].to_numpy(),
    roi_px[:, 0]
])

all_y = np.concatenate([
    df_check["y_anchor_px"].to_numpy(),
    roi_px[:, 1]
])

x_min, x_max = np.nanmin(all_x), np.nanmax(all_x)
y_min, y_max = np.nanmin(all_y), np.nanmax(all_y)

x_pad = 0.05 * (x_max - x_min)
y_pad = 0.05 * (y_max - y_min)

ax.set_xlim(x_min - x_pad, x_max + x_pad)

# Because image pixel coordinates increase downward
ax.set_ylim(y_max + y_pad, y_min - y_pad)

ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
ax.set_title(f"ROI check — {csv_files_sorted[0].name}")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Inside : {len(inside):,}  ({len(inside)/len(df_check):.1%})")
print(f"Outside: {len(outside):,}  ({len(outside)/len(df_check):.1%})")

In [ ]:
# Check how many inside-ROI points are near the left boundary
inside_check = df_check[df_check["inside_roi"]]
outside_check = df_check[~df_check["inside_roi"]]

# Points just outside the ROI boundary — potential edge clipping
near_boundary = outside_check[
    (outside_check["x_anchor_px"] < 300) &
    (outside_check["y_anchor_px"] > 350) &
    (outside_check["y_anchor_px"] < 600)
]

print(f"Points just outside left boundary: {len(near_boundary):,}")
print(f"\nClass distribution of near-boundary points:")
print(near_boundary["class"].value_counts())

# # Also check what % of each video is inside ROI
# print("\n=== Inside ROI ratio per video ===")
# inside_ratios = (
#     df_raw.groupby("video_id")
#           .size()
#           .reset_index(name="inside_count")
# )
# print(inside_ratios.to_string())

# Step 6 — Pixel to GPS via Homography


In [ ]:
def apply_homography(points_xy, H):
    ones = np.ones((points_xy.shape[0], 1), dtype=float)
    pts_h = np.hstack([points_xy, ones])
    out_h = pts_h @ H.T
    out = out_h[:, :2] / out_h[:, 2:3]
    return out

px_points = df_raw[["x_anchor_px", "y_anchor_px"]].to_numpy()
gps_points = apply_homography(px_points, H)

df_raw["lat"] = gps_points[:, 0]
df_raw["lon"] = gps_points[:, 1]

print("=== GPS coordinate summary ===")
print(df_raw[["lat", "lon"]].describe())

print("\n=== Sanity check vs ROI GPS corners ===")
print(f"ROI GPS lat range: {roi_gps[:,0].min():.6f} → {roi_gps[:,0].max():.6f}")
print(f"ROI GPS lon range: {roi_gps[:,1].min():.6f} → {roi_gps[:,1].max():.6f}")
print(f"Data lat range   : {df_raw['lat'].min():.6f} → {df_raw['lat'].max():.6f}")
print(f"Data lon range   : {df_raw['lon'].min():.6f} → {df_raw['lon'].max():.6f}")

# Step 7 — GPS to Local Metric XY

In [ ]:
def latlon_to_local_xy(lat, lon, lat0, lon0):
    R = 6378137.0
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    lat0_rad = np.radians(lat0)
    x = np.radians(lon - lon0) * R * np.cos(lat0_rad)
    y = np.radians(lat - lat0) * R
    return x, y

# Reference point: centroid of ROI GPS corners
lat0 = np.mean(roi_gps[:, 0])
lon0 = np.mean(roi_gps[:, 1])

print(f"Reference point: lat0={lat0:.8f}, lon0={lon0:.8f}")

df_raw["x_local_m"], df_raw["y_local_m"] = latlon_to_local_xy(
    df_raw["lat"].values,
    df_raw["lon"].values,
    lat0, lon0
)

print("\n=== Local metric coordinate summary ===")
print(df_raw[["x_local_m", "y_local_m"]].describe())

print("\n=== Physical extent of intersection (meters) ===")
x_range = df_raw["x_local_m"].max() - df_raw["x_local_m"].min()
y_range = df_raw["y_local_m"].max() - df_raw["y_local_m"].min()
print(f"X span: {x_range:.1f} m")
print(f"Y span: {y_range:.1f} m")

In [ ]:
# Sort chronologically within each track
df_raw = df_raw.sort_values(
    ["video_id", "track_id", "frame_idx"]
).reset_index(drop=True)

# Drop duplicates just in case
before = len(df_raw)
df_raw = df_raw.drop_duplicates(
    subset=["video_id", "track_id", "frame_idx"]
).copy()
after = len(df_raw)

print(f"Duplicates dropped: {before - after:,}")
print(f"Rows remaining    : {after:,}")

print("\n=== Track length distribution ===")
track_len = df_raw.groupby(["video_id", "track_id"]).size()
print(track_len.describe())

print(f"\nTracks with only 1 frame  : {(track_len == 1).sum():,}")
print(f"Tracks with < 3 frames    : {(track_len < 3).sum():,}")
print(f"Tracks with >= 3 frames   : {(track_len >= 3).sum():,}")
print(f"Total unique tracks       : {len(track_len):,}")

In [ ]:
# Compute track length per (video_id, track_id)
track_len = (
    df_raw.groupby(["video_id", "track_id"])
          .size()
          .rename("track_len")
          .reset_index()
)

df_raw = df_raw.merge(track_len, on=["video_id", "track_id"], how="left")

# Compute max frame gap per track — large gaps mean tracker lost the object
frame_gap = (
    df_raw.sort_values(["video_id", "track_id", "frame_idx"])
          .groupby(["video_id", "track_id"])["frame_idx"]
          .diff()
          .rename("frame_gap")
)
df_raw["frame_gap"] = frame_gap

max_frame_gap = (
    df_raw.groupby(["video_id", "track_id"])["frame_gap"]
          .max()
          .rename("max_frame_gap")
          .reset_index()
)

df_raw = df_raw.merge(max_frame_gap, on=["video_id", "track_id"], how="left")
df_raw["max_frame_gap"] = df_raw["max_frame_gap"].fillna(0)

print("=== Max frame gap distribution ===")
print(df_raw["max_frame_gap"].describe())
print(f"\nTracks with gap > 5  : {(df_raw.groupby(['video_id','track_id'])['max_frame_gap'].first() > 5).sum():,}")
print(f"Tracks with gap > 10 : {(df_raw.groupby(['video_id','track_id'])['max_frame_gap'].first() > 10).sum():,}")
print(f"Tracks with gap > 30 : {(df_raw.groupby(['video_id','track_id'])['max_frame_gap'].first() > 30).sum():,}")

# Apply filters
before = len(df_raw)
df_raw = df_raw[
    (df_raw["track_len"] >= 3) &
    (df_raw["max_frame_gap"] <= 5)
].copy()
after = len(df_raw)

print(f"\nRows before filter : {before:,}")
print(f"Rows after filter  : {after:,}")
print(f"Rows dropped       : {before - after:,}")

print("\n=== Track length after filter ===")
print(df_raw.groupby(["video_id", "track_id"]).size().describe())

print("\n=== Agent type distribution after filter ===")
print(df_raw["agent_type"].value_counts())

print("\n=== Class distribution after filter ===")
print(df_raw["class"].value_counts())

In [ ]:
df_raw.head()

In [ ]:
# Estimate FPS from the data
sample_video = df_raw["video_id"].iloc[0]
sample_track = (
    df_raw[df_raw["video_id"] == sample_video]
    .groupby("track_id")
    .filter(lambda x: len(x) > 30)
    .groupby("track_id")
    .first()
    .index[0]
)

track_sample = df_raw[
    (df_raw["video_id"] == sample_video) &
    (df_raw["track_id"] == sample_track)
].sort_values("frame_idx").head(30)

print("=== Frame index and timestamp sample ===")
print(track_sample[["frame_idx", "t"]].to_string())

print(f"\nEstimated FPS: {1 / track_sample['t'].diff().median():.1f}")
print(f"Frame interval (seconds): {track_sample['t'].diff().median():.4f}")

In [ ]:
# Re-merge track metadata on original pre-filter df
# First reload clean starting point
df_filtered = df_raw.copy()

# Check how many tracks/rows we recover with gap <= 30
before_strict = len(df_raw[df_raw["max_frame_gap"] <= 5])
before_relaxed = len(df_raw[df_raw["max_frame_gap"] <= 30])
total = len(df_raw)

print("=== Gap threshold comparison ===")
print(f"Gap <= 5  (0.17s) : {before_strict:,} rows  ({before_strict/total:.1%})")
print(f"Gap <= 15 (0.50s) : {len(df_raw[df_raw['max_frame_gap'] <= 15]):,} rows  ({len(df_raw[df_raw['max_frame_gap'] <= 15])/total:.1%})")
print(f"Gap <= 30 (1.00s) : {before_relaxed:,} rows  ({before_relaxed/total:.1%})")
print(f"Gap <= 60 (2.00s) : {len(df_raw[df_raw['max_frame_gap'] <= 60]):,} rows  ({len(df_raw[df_raw['max_frame_gap'] <= 60])/total:.1%})")
print(f"No gap filter     : {total:,} rows  (100%)")

print("\n=== Track count per threshold ===")
for gap in [5, 15, 30, 60]:
    n_tracks = df_raw[df_raw["max_frame_gap"] <= gap].groupby(
        ["video_id", "track_id"]
    ).ngroups
    print(f"Gap <= {gap:2d} : {n_tracks:,} tracks")

# Step 10 — Trajectory Smoothing (Savitzky-Golay)


In [ ]:
from scipy.signal import savgol_filter

def choose_savgol_window(n, preferred=7, polyorder=2):
    if n < 3:
        return None
    w = min(preferred, n)
    if w % 2 == 0:
        w -= 1
    if w <= polyorder:
        w = polyorder + 1
        if w % 2 == 0:
            w += 1
    if w > n:
        w = n if n % 2 == 1 else n - 1
    if w < 3 or w <= polyorder:
        return None
    return w

def smooth_track(group, preferred_window=7, polyorder=2):
    group = group.sort_values("t").copy()
    n = len(group)
    w = choose_savgol_window(n, preferred=preferred_window, polyorder=polyorder)
    if w is not None:
        try:
            group["x_local_m"] = savgol_filter(
                group["x_local_m"].to_numpy(), w, polyorder, mode="interp"
            )
            group["y_local_m"] = savgol_filter(
                group["y_local_m"].to_numpy(), w, polyorder, mode="interp"
            )
        except Exception:
            group["x_local_m"] = group["x_local_m"].rolling(5, center=True, min_periods=1).mean()
            group["y_local_m"] = group["y_local_m"].rolling(5, center=True, min_periods=1).mean()
    else:
        group["x_local_m"] = group["x_local_m"].rolling(3, center=True, min_periods=1).mean()
        group["y_local_m"] = group["y_local_m"].rolling(3, center=True, min_periods=1).mean()
    return group

print("Smoothing trajectories...")
df_raw = (
    df_raw.groupby(["video_id", "track_id"], group_keys=False)
          .apply(smooth_track)
          .reset_index(drop=True)
)

print("Done.")
print(f"Rows after smoothing: {len(df_raw):,}")
print("\n=== Smoothed coordinate summary ===")
print(df_raw[["x_local_m", "y_local_m"]].describe())

In [ ]:
import matplotlib.pyplot as plt

# Sample for visualization performance
sample = df_raw.sample(50000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: all agent types together ---
colors = {"vehicle": "steelblue", "pedestrian": "tomato"}
for atype, grp in sample.groupby("agent_type"):
    axes[0].scatter(
        grp["x_local_m"], grp["y_local_m"],
        s=1, alpha=0.3, c=colors[atype], label=atype
    )
axes[0].set_title("All agents — spatial distribution (smoothed)")
axes[0].set_xlabel("x_local_m")
axes[0].set_ylabel("y_local_m")
axes[0].axis("equal")
axes[0].legend(markerscale=5)

# --- Right: trajectory lines for a sample of tracks ---
# Pick 30 random tracks with track_len >= 30
long_tracks = (
    df_raw[df_raw["track_len"] >= 30]
    .groupby(["video_id", "track_id"])
    .first()
    .reset_index()[["video_id", "track_id"]]
    .sample(30, random_state=42)
)

axes[1].set_title("Sample trajectories (30 tracks, length >= 30 frames)")
axes[1].set_xlabel("x_local_m")
axes[1].set_ylabel("y_local_m")

for _, row in long_tracks.iterrows():
    traj = df_raw[
        (df_raw["video_id"] == row["video_id"]) &
        (df_raw["track_id"] == row["track_id"])
    ].sort_values("frame_idx")

    atype = traj["agent_type"].iloc[0]
    axes[1].plot(
        traj["x_local_m"], traj["y_local_m"],
        linewidth=0.8, alpha=0.7,
        color=colors[atype]
    )

axes[1].axis("equal")

# Add legend manually
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="steelblue", linewidth=1.5, label="vehicle"),
    Line2D([0], [0], color="tomato",    linewidth=1.5, label="pedestrian")
]
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.show()

Step 11 — Compute Motion Features (Velocity + Acceleration)


In [ ]:
def remove_position_jumps(group, max_jump_m=5.0):
    """
    Remove frames where position jumps more than max_jump_m meters
    from the previous frame. At 30fps, 5m/frame = 150 m/s — impossible.
    """
    group = group.sort_values("t").copy()
    
    dx = group["x_local_m"].diff().abs()
    dy = group["y_local_m"].diff().abs()
    jump = np.sqrt(dx**2 + dy**2)
    
    # First frame has no diff — always keep
    bad = (jump > max_jump_m) & jump.notna()
    
    return group[~bad]

print("Removing intra-track position jumps...")
before = len(df_raw)

df_raw = (
    df_raw.groupby(["video_id", "track_id"], group_keys=False)
          .apply(remove_position_jumps, max_jump_m=5.0)
          .reset_index(drop=True)
)

after = len(df_raw)
print(f"Rows before : {before:,}")
print(f"Rows after  : {after:,}")
print(f"Rows dropped: {before - after:,}  ({(before-after)/before:.1%})")

print("\n=== Position summary after jump removal ===")
print(df_raw[["x_local_m", "y_local_m"]].describe())

print("\n=== Track length after jump removal ===")
track_len_new = df_raw.groupby(["video_id", "track_id"]).size()
print(track_len_new.describe())
print(f"\nTracks with < 3 frames: {(track_len_new < 3).sum():,}")

In [ ]:
def compute_motion_features_sg(group, preferred_window=11, polyorder=3):
    group = group.sort_values("t").copy()
    n = len(group)
    dt = 1.0 / 30.0

    x = group["x_local_m"].to_numpy()
    y = group["y_local_m"].to_numpy()

    # Not enough points — fill with NaN and return
    if n < 3:
        group["dt"]    = dt
        group["vx"]    = np.nan
        group["vy"]    = np.nan
        group["speed"] = np.nan
        group["ax"]    = np.nan
        group["ay"]    = np.nan
        group["accel"] = np.nan
        return group

    # Choose valid window
    w = preferred_window
    if w > n: w = n if n % 2 == 1 else n - 1
    if w % 2 == 0: w -= 1
    if w < polyorder + 1:
        w = polyorder + 1
        if w % 2 == 0: w += 1
    # Final validity check
    if w > n or w < 3:
        w = 3  # minimum fallback

    # Reduce polyorder if window too small
    po = min(polyorder, w - 1)

    try:
        vx = savgol_filter(x, w, po, deriv=1, delta=dt, mode="interp")
        vy = savgol_filter(y, w, po, deriv=1, delta=dt, mode="interp")
        ax = savgol_filter(x, w, po, deriv=2, delta=dt, mode="interp")
        ay = savgol_filter(y, w, po, deriv=2, delta=dt, mode="interp")
    except Exception:
        # Last resort: simple finite difference
        vx = np.gradient(x, dt) if len(x) >= 2 else np.full_like(x, np.nan)
        vy = np.gradient(y, dt) if len(y) >= 2 else np.full_like(y, np.nan)
        ax = np.gradient(vx, dt) if len(vx) >= 2 else np.full_like(vx, np.nan)
        ay = np.gradient(vy, dt) if len(vy) >= 2 else np.full_like(vy, np.nan)

    group["dt"]    = dt
    group["vx"]    = vx
    group["vy"]    = vy
    group["speed"] = np.sqrt(vx**2 + vy**2)
    group["ax"]    = ax
    group["ay"]    = ay
    group["accel"] = np.sqrt(ax**2 + ay**2)

    return group

print("Computing motion features...")
df_raw = (
    df_raw.groupby(["video_id", "track_id"], group_keys=False)
          .apply(compute_motion_features_sg)
          .reset_index(drop=True)
)

df_raw = df_raw.replace([np.inf, -np.inf], np.nan)

print("Done.")
print("\n=== Speed summary (m/s) ===")
print(df_raw["speed"].describe())
print("\n=== Acceleration summary (m/s²) ===")
print(df_raw["accel"].describe())
print("\n=== Speed by agent type ===")
print(df_raw.groupby("agent_type")["speed"].describe())
print("\n=== Acceleration by agent type ===")
print(df_raw.groupby("agent_type")["accel"].describe())

In [ ]:
# Define physically realistic thresholds
MAX_SPEED_VEHICLE  = 20.0   # m/s = 72 km/h, reasonable for intersection
MAX_SPEED_PED      = 3.0    # m/s = 10.8 km/h, fast walk/jog
MAX_ACCEL_VEHICLE  = 9.0    # m/s² = hard braking
MAX_ACCEL_PED      = 4.0    # m/s²

print("=== Rows exceeding thresholds (before filter) ===")
veh = df_raw[df_raw["agent_type"] == "vehicle"]
ped = df_raw[df_raw["agent_type"] == "pedestrian"]

print(f"Vehicle speed  > {MAX_SPEED_VEHICLE} m/s : {(veh['speed'] > MAX_SPEED_VEHICLE).sum():,}  ({(veh['speed'] > MAX_SPEED_VEHICLE).mean():.1%})")
print(f"Vehicle accel  > {MAX_ACCEL_VEHICLE} m/s²: {(veh['accel'] > MAX_ACCEL_VEHICLE).sum():,}  ({(veh['accel'] > MAX_ACCEL_VEHICLE).mean():.1%})")
print(f"Pedestrian speed > {MAX_SPEED_PED} m/s  : {(ped['speed'] > MAX_SPEED_PED).sum():,}  ({(ped['speed'] > MAX_SPEED_PED).mean():.1%})")
print(f"Pedestrian accel > {MAX_ACCEL_PED} m/s² : {(ped['accel'] > MAX_ACCEL_PED).sum():,}  ({(ped['accel'] > MAX_ACCEL_PED).mean():.1%})")

# Flag bad rows
df_raw["speed_ok"] = (
    ((df_raw["agent_type"] == "vehicle")    & (df_raw["speed"] <= MAX_SPEED_VEHICLE)) |
    ((df_raw["agent_type"] == "pedestrian") & (df_raw["speed"] <= MAX_SPEED_PED))
)
df_raw["accel_ok"] = (
    ((df_raw["agent_type"] == "vehicle")    & (df_raw["accel"] <= MAX_ACCEL_VEHICLE)) |
    ((df_raw["agent_type"] == "pedestrian") & (df_raw["accel"] <= MAX_ACCEL_PED))
)

# NaN rows (first/second frame of each track) are acceptable — keep them
df_raw["row_ok"] = (
    (df_raw["speed_ok"] | df_raw["speed"].isna()) &
    (df_raw["accel_ok"] | df_raw["accel"].isna())
)

before = len(df_raw)
df_raw = df_raw[df_raw["row_ok"]].copy()
after  = len(df_raw)

print(f"\nRows before filter : {before:,}")
print(f"Rows after filter  : {after:,}")
print(f"Rows dropped       : {before - after:,}  ({(before-after)/before:.1%})")

print("\n=== Speed summary after filter ===")
print(df_raw.groupby("agent_type")["speed"].describe())

print("\n=== Acceleration summary after filter ===")
print(df_raw.groupby("agent_type")["accel"].describe())

In [ ]:
print("Columns:", df_raw.columns.tolist())
print("Shape:", df_raw.shape)
print("\nSample:")
print(df_raw.head(3))

In [ ]:
# ── CLEAN REBUILD FROM RAW CSVs ────────────────────────────

valid_classes    = {"car", "person", "truck", "bus", "motorcycle", "bicycle"}
vehicle_classes  = {"car", "truck", "bus", "motorcycle", "bicycle"}

# STEP 1-3: Load, merge, class filter
print("Loading CSVs...")
dfs = []
for f in csv_files_sorted:
    dt = parse_datetime_from_filename(f)
    df_temp = pd.read_csv(f)
    df_temp["file_datetime"] = dt
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
df = df[df["class"].isin(valid_classes)].copy()
df["agent_type"] = df["class"].apply(
    lambda x: "vehicle" if x in vehicle_classes else "pedestrian"
)
print(f"After class filter: {len(df):,}")

# STEP 4: Ground anchor
df["x_anchor_px"] = df["u"]
df["y_anchor_px"] = df["v"] + df["h"] / 2.0

# STEP 5: ROI filter
print("Applying ROI filter...")
df["inside_roi"] = df.apply(
    lambda r: point_in_polygon(r["x_anchor_px"], r["y_anchor_px"], roi_px),
    axis=1
)
df = df[df["inside_roi"]].copy()
print(f"After ROI filter: {len(df):,}")

# STEP 6-7: Homography + local metric XY
px_points = df[["x_anchor_px", "y_anchor_px"]].to_numpy()
gps_points = apply_homography(px_points, H)
df["lat"] = gps_points[:, 0]
df["lon"] = gps_points[:, 1]

lat0 = np.mean(roi_gps[:, 0])
lon0 = np.mean(roi_gps[:, 1])
df["x_local_m"], df["y_local_m"] = latlon_to_local_xy(
    df["lat"].values, df["lon"].values, lat0, lon0
)
print(f"After coordinate transform: {len(df):,}")

# STEP 8: Sort + track metadata
df = df.sort_values(["video_id", "track_id", "frame_idx"]).reset_index(drop=True)

track_len = (
    df.groupby(["video_id", "track_id"])
      .size()
      .rename("track_len")
      .reset_index()
)
df = df.merge(track_len, on=["video_id", "track_id"], how="left")

frame_gap = (
    df.groupby(["video_id", "track_id"])["frame_idx"]
      .diff()
      .rename("frame_gap")
)
df["frame_gap"] = frame_gap

max_frame_gap = (
    df.groupby(["video_id", "track_id"])["frame_gap"]
      .max()
      .rename("max_frame_gap")
      .reset_index()
)
df = df.merge(max_frame_gap, on=["video_id", "track_id"], how="left")
df["max_frame_gap"] = df["max_frame_gap"].fillna(0)

# STEP 9: Track quality filter
df = df[(df["track_len"] >= 3) & (df["max_frame_gap"] <= 5)].copy()
print(f"After track quality filter: {len(df):,}")

# STEP 10: Remove intra-track position jumps
def remove_position_jumps(group, max_jump_m=5.0):
    group = group.sort_values("t").copy()
    dx = group["x_local_m"].diff().abs()
    dy = group["y_local_m"].diff().abs()
    jump = np.sqrt(dx**2 + dy**2)
    bad = (jump > max_jump_m) & jump.notna()
    return group[~bad]

before = len(df)
df = (
    df.groupby(["video_id", "track_id"], group_keys=False)
      .apply(remove_position_jumps, max_jump_m=5.0)
      .reset_index(drop=True)
)
print(f"After jump removal: {len(df):,}  (dropped {before-len(df):,})")

# STEP 11: Smooth positions
print("Smoothing positions...")
df = (
    df.groupby(["video_id", "track_id"], group_keys=False)
      .apply(smooth_track)
      .reset_index(drop=True)
)
print(f"After smoothing: {len(df):,}")

# STEP 12: Compute motion via SG derivatives
print("Computing motion features...")
df = (
    df.groupby(["video_id", "track_id"], group_keys=False)
      .apply(compute_motion_features_sg)
      .reset_index(drop=True)
)
df = df.replace([np.inf, -np.inf], np.nan)
print(f"After motion computation: {len(df):,}")

print("\n=== Speed summary ===")
print(df["speed"].describe())
print("\n=== Acceleration summary ===")
print(df["accel"].describe())
print("\n=== Speed by agent type ===")
print(df.groupby("agent_type")["speed"].describe())

In [ ]:
# STEP 13: Final motion quality filter
MAX_SPEED_VEH  = 20.0   # m/s = 72 km/h
MAX_SPEED_PED  = 4.0    # m/s = fast jog
MAX_ACCEL_VEH  = 9.0    # m/s²
MAX_ACCEL_PED  = 4.0    # m/s²

before = len(df)

df = df[
    (
        ((df["agent_type"] == "vehicle")    & (df["speed"] <= MAX_SPEED_VEH) & (df["accel"] <= MAX_ACCEL_VEH)) |
        ((df["agent_type"] == "pedestrian") & (df["speed"] <= MAX_SPEED_PED) & (df["accel"] <= MAX_ACCEL_PED)) |
        (df["speed"].isna()) |
        (df["accel"].isna())
    )
].copy()

after = len(df)

print(f"Rows before : {before:,}")
print(f"Rows after  : {after:,}")
print(f"Rows dropped: {before - after:,}  ({(before-after)/before:.1%})")

print("\n=== Speed by agent type after filter ===")
print(df.groupby("agent_type")["speed"].describe())

print("\n=== Acceleration by agent type after filter ===")
print(df.groupby("agent_type")["accel"].describe())

print("\n=== Track count after filter ===")
print(f"Unique tracks: {df.groupby(['video_id','track_id']).ngroups:,}")

In [ ]:
import pickle

checkpoint_path = Path(r"Z:\0_Thesis_Monzurul\Coldwater MI")

# Save as parquet — faster and smaller than CSV for large dataframes
df.to_parquet(checkpoint_path / "df_clean.parquet", index=False)

print(f"Saved: {checkpoint_path / 'df_clean.parquet'}")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
def compute_ssms_per_frame(group_df, 
                            vv_threshold=30.0,
                            vp_threshold=20.0,
                            pp_threshold=10.0):
    """
    For one (video_id, frame_idx) group, compute pairwise SSMs
    across all agent combinations (vv, vp, pp).
    """
    nodes = group_df.reset_index(drop=True)
    N = len(nodes)
    
    if N < 2:
        return pd.DataFrame()
    
    records = []
    
    for i in range(N):
        for j in range(i + 1, N):
            ni = nodes.iloc[i]
            nj = nodes.iloc[j]
            
            # Interaction type
            ti = ni["agent_type"]
            tj = nj["agent_type"]
            
            if ti == "vehicle" and tj == "vehicle":
                itype     = "vv"
                threshold = vv_threshold
            elif (ti == "vehicle" and tj == "pedestrian") or \
                 (ti == "pedestrian" and tj == "vehicle"):
                itype     = "vp"
                threshold = vp_threshold
            else:
                itype     = "pp"
                threshold = pp_threshold
            
            # Distance
            dx   = ni["x_local_m"] - nj["x_local_m"]
            dy   = ni["y_local_m"] - nj["y_local_m"]
            dist = np.sqrt(dx**2 + dy**2)
            
            if dist > threshold:
                continue
            
            # Relative velocity
            dvx       = ni["vx"] - nj["vx"]
            dvy       = ni["vy"] - nj["vy"]
            rel_speed = np.sqrt(dvx**2 + dvy**2)
            
            # TTC
            rel_pos    = np.array([dx, dy])
            rel_vel    = np.array([dvx, dvy])
            rel_spd_sq = dvx**2 + dvy**2
            
            if rel_spd_sq > 1e-6:
                ttc_raw = -np.dot(rel_pos, rel_vel) / rel_spd_sq
                ttc     = ttc_raw if ttc_raw > 0 else np.nan
            else:
                ttc = np.nan
            
            # DRAC
            if dist > 1e-3 and rel_speed > 1e-3:
                drac = (rel_speed**2) / (2.0 * dist)
            else:
                drac = np.nan
            
            records.append({
                "video_id"        : ni["video_id"],
                "frame_idx"       : ni["frame_idx"],
                "t"               : ni["t"],
                "track_id_i"      : ni["track_id"],
                "track_id_j"      : nj["track_id"],
                "class_i"         : ni["class"],
                "class_j"         : nj["class"],
                "agent_type_i"    : ti,
                "agent_type_j"    : tj,
                "interaction_type": itype,
                "x_i"             : ni["x_local_m"],
                "y_i"             : ni["y_local_m"],
                "x_j"             : nj["x_local_m"],
                "y_j"             : nj["y_local_m"],
                "vx_i"            : ni["vx"],
                "vy_i"            : ni["vy"],
                "vx_j"            : nj["vx"],
                "vy_j"            : nj["vy"],
                "distance"        : dist,
                "rel_speed"       : rel_speed,
                "ttc"             : ttc,
                "drac"            : drac,
            })
    
    return pd.DataFrame(records)


print("Computing pairwise SSMs per frame...")
print("This may take several minutes given 1.6M rows...\n")

pairs = (
    df.groupby(["video_id", "frame_idx"], group_keys=False)
      .apply(compute_ssms_per_frame)
      .reset_index(drop=True)
)

print(f"Total interaction pairs: {len(pairs):,}")

print("\n=== Interaction type distribution ===")
print(pairs["interaction_type"].value_counts())

print("\n=== TTC summary ===")
print(pairs["ttc"].describe())

print("\n=== DRAC summary ===")
print(pairs["drac"].describe())

print("\n=== Distance summary ===")
print(pairs["distance"].describe())

In [ ]:
# ── SSM bounds ─────────────────────────────────────────────
MAX_TTC   = 10.0   # seconds — beyond this, not a meaningful conflict
MAX_DRAC  = 15.0   # m/s² — physical upper bound for emergency braking
MIN_DIST  = 0.5    # meters — below this, likely duplicate detection

# ── Filter ─────────────────────────────────────────────────
before = len(pairs)

pairs = pairs[pairs["distance"] >= MIN_DIST].copy()

# Cap TTC — don't drop, just NaN out meaningless values
pairs.loc[pairs["ttc"] > MAX_TTC, "ttc"] = np.nan

# Cap DRAC
pairs.loc[pairs["drac"] > MAX_DRAC, "drac"] = np.nan

after = len(pairs)
print(f"Rows before distance filter : {before:,}")
print(f"Rows after distance filter  : {after:,}")
print(f"Dropped                     : {before - after:,}")

# ── dTTC/dt per pair ───────────────────────────────────────
# Sort by (video_id, track_id_i, track_id_j, frame_idx)
# then compute finite difference of TTC over time
print("\nComputing dTTC/dt...")

pairs = pairs.sort_values(
    ["video_id", "track_id_i", "track_id_j", "frame_idx"]
).copy()

pairs["dttc_dt"] = (
    pairs.groupby(["video_id", "track_id_i", "track_id_j"])["ttc"]
         .diff() / (1.0 / 30.0)
)

# dTTC/dt should be negative for approaching pairs
# Cap extreme values
pairs.loc[pairs["dttc_dt"].abs() > 100, "dttc_dt"] = np.nan

print("Done.")
print("\n=== TTC summary after cap ===")
print(pairs["ttc"].describe())

print("\n=== DRAC summary after cap ===")
print(pairs["drac"].describe())

print("\n=== dTTC/dt summary ===")
print(pairs["dttc_dt"].describe())

print("\n=== Valid SSM counts ===")
print(f"Valid TTC    : {pairs['ttc'].notna().sum():,}  ({pairs['ttc'].notna().mean():.1%})")
print(f"Valid DRAC   : {pairs['drac'].notna().sum():,}  ({pairs['drac'].notna().mean():.1%})")
print(f"Valid dTTC/dt: {pairs['dttc_dt'].notna().sum():,}  ({pairs['dttc_dt'].notna().mean():.1%})")

In [ ]:
# ── Conflict thresholds ────────────────────────────────────
# Based on literature: TTC < 1.5s = serious, TTC < 3.0s = general
# We use TTC < 3.0s as primary label with distance constraint
# DRAC > 3.0 m/s² as secondary confirmation

TTC_SERIOUS  = 1.5   # s
TTC_GENERAL  = 3.0   # s
DRAC_THRESH  = 3.0   # m/s²
DIST_THRESH  = 15.0  # m — must be physically close

# ── Per-interaction-type labeling ──────────────────────────
# vp conflicts use tighter distance threshold (pedestrian safety)
VP_DIST_THRESH = 10.0

def assign_conflict_label(row):
    ttc  = row["ttc"]
    drac = row["drac"]
    dist = row["distance"]
    itype = row["interaction_type"]

    d_thresh = VP_DIST_THRESH if itype == "vp" else DIST_THRESH

    if pd.isna(ttc):
        return 0

    if dist > d_thresh:
        return 0

    if ttc < TTC_SERIOUS and drac >= DRAC_THRESH:
        return 2   # serious conflict
    elif ttc < TTC_GENERAL:
        return 1   # general conflict
    else:
        return 0   # no conflict

pairs["conflict_label"] = pairs.apply(assign_conflict_label, axis=1)

print("=== Conflict label distribution ===")
print(pairs["conflict_label"].value_counts().sort_index())
print(f"\nTotal pairs          : {len(pairs):,}")
print(f"No conflict (0)      : {(pairs['conflict_label']==0).sum():,}  ({(pairs['conflict_label']==0).mean():.1%})")
print(f"General conflict (1) : {(pairs['conflict_label']==1).sum():,}  ({(pairs['conflict_label']==1).mean():.1%})")
print(f"Serious conflict (2) : {(pairs['conflict_label']==2).sum():,}  ({(pairs['conflict_label']==2).mean():.1%})")

print("\n=== Conflict by interaction type ===")
print(pairs.groupby(["interaction_type", "conflict_label"]).size().unstack(fill_value=0))

print("\n=== TTC distribution for conflict pairs ===")
conflict_pairs = pairs[pairs["conflict_label"] > 0]
print(conflict_pairs["ttc"].describe())
print(f"\nTotal conflict interactions: {len(conflict_pairs):,}")

In [ ]:
pairs.to_parquet(checkpoint_path / "pairs_with_ssm.parquet", index=False)
print(f"Saved: {checkpoint_path / 'pairs_with_ssm.parquet'}")
print(f"Shape: {pairs.shape}")
print(f"Columns: {pairs.columns.tolist()}")

In [ ]:
import matplotlib.pyplot as plt

conflict = pairs[pairs["conflict_label"] > 0].copy()
conflict["x_mid"] = (conflict["x_i"] + conflict["x_j"]) / 2
conflict["y_mid"] = (conflict["y_i"] + conflict["y_j"]) / 2

no_conflict = pairs[pairs["conflict_label"] == 0].copy()
no_conflict["x_mid"] = (no_conflict["x_i"] + no_conflict["x_j"]) / 2
no_conflict["y_mid"] = (no_conflict["y_i"] + no_conflict["y_j"]) / 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Left: all vs conflict ──
axes[0].scatter(no_conflict["x_mid"], no_conflict["y_mid"],
                s=1, alpha=0.05, c="steelblue", label="No conflict")
axes[0].scatter(conflict["x_mid"], conflict["y_mid"],
                s=2, alpha=0.3, c="tomato", label="Conflict")
axes[0].set_title("All vs Conflict interactions")
axes[0].set_xlabel("x_local_m")
axes[0].set_ylabel("y_local_m")
axes[0].axis("equal")
axes[0].legend(markerscale=4)

# ── Middle: conflict by type ──
colors_type = {"vv": "steelblue", "vp": "tomato", "pp": "green"}
for itype, grp in conflict.groupby("interaction_type"):
    axes[1].scatter(grp["x_mid"], grp["y_mid"],
                    s=2, alpha=0.3,
                    c=colors_type[itype], label=itype)
axes[1].set_title("Conflict by interaction type")
axes[1].set_xlabel("x_local_m")
axes[1].axis("equal")
axes[1].legend(markerscale=4)

# ── Right: hotspot hexbin ──
serious = pairs[pairs["conflict_label"] == 2].copy()
serious["x_mid"] = (serious["x_i"] + serious["x_j"]) / 2
serious["y_mid"] = (serious["y_i"] + serious["y_j"]) / 2

hb = axes[2].hexbin(serious["x_mid"], serious["y_mid"],
                     gridsize=35, mincnt=1, cmap="YlOrRd")
plt.colorbar(hb, ax=axes[2], label="Serious conflict density")
axes[2].set_title("Serious conflict hotspots")
axes[2].set_xlabel("x_local_m")
axes[2].axis("equal")

plt.tight_layout()
plt.show()

In [ ]:
# Recompute conflict labels with minimum relative speed gate
MIN_REL_SPEED_VV = 1.0   # m/s — vehicles must be approaching at > 1 m/s
MIN_REL_SPEED_VP = 0.5   # m/s — lower threshold for pedestrian safety
MIN_REL_SPEED_PP = 0.3   # m/s

def assign_conflict_label_v2(row):
    ttc   = row["ttc"]
    drac  = row["drac"]
    dist  = row["distance"]
    itype = row["interaction_type"]
    rspd  = row["rel_speed"]

    d_thresh  = VP_DIST_THRESH if itype == "vp" else DIST_THRESH
    rs_thresh = (MIN_REL_SPEED_VP if itype == "vp"
                 else MIN_REL_SPEED_PP if itype == "pp"
                 else MIN_REL_SPEED_VV)

    if pd.isna(ttc):
        return 0
    if dist > d_thresh:
        return 0
    if rspd < rs_thresh:        # ← key gate: must be genuinely approaching
        return 0
    if ttc < TTC_SERIOUS and drac >= DRAC_THRESH:
        return 2
    elif ttc < TTC_GENERAL:
        return 1
    else:
        return 0

pairs["conflict_label"] = pairs.apply(assign_conflict_label_v2, axis=1)

print("=== Conflict label distribution after rel_speed gate ===")
print(pairs["conflict_label"].value_counts().sort_index())
print(f"\nNo conflict (0)      : {(pairs['conflict_label']==0).sum():,}  ({(pairs['conflict_label']==0).mean():.1%})")
print(f"General conflict (1) : {(pairs['conflict_label']==1).sum():,}  ({(pairs['conflict_label']==1).mean():.1%})")
print(f"Serious conflict (2) : {(pairs['conflict_label']==2).sum():,}  ({(pairs['conflict_label']==2).mean():.1%})")

print("\n=== Conflict by interaction type ===")
print(pairs.groupby(["interaction_type","conflict_label"]).size().unstack(fill_value=0))

In [ ]:
# Examine the serious conflict pairs more carefully
serious = pairs[pairs["conflict_label"] == 2].copy()

print("=== Serious conflict — rel_speed distribution ===")
print(serious["rel_speed"].describe())

print("\n=== Serious conflict — distance distribution ===")
print(serious["distance"].describe())

print("\n=== Serious conflict — TTC distribution ===")
print(serious["ttc"].describe())

print("\n=== Serious conflict — DRAC distribution ===")
print(serious["drac"].describe())

print("\n=== Serious conflict — speed of agent i ===")
# Join back to df to get individual speeds
serious_with_speed = serious.merge(
    df[["video_id","frame_idx","track_id","speed","agent_type"]],
    left_on=["video_id","frame_idx","track_id_i"],
    right_on=["video_id","frame_idx","track_id"],
    how="left"
).rename(columns={"speed":"speed_i"})

serious_with_speed = serious_with_speed.merge(
    df[["video_id","frame_idx","track_id","speed"]],
    left_on=["video_id","frame_idx","track_id_j"],
    right_on=["video_id","frame_idx","track_id"],
    how="left"
).rename(columns={"speed":"speed_j"})

print("Speed of agent i:")
print(serious_with_speed["speed_i"].describe())
print("\nSpeed of agent j:")
print(serious_with_speed["speed_j"].describe())

# How many serious conflicts involve at least one stopped vehicle?
stopped = (
    (serious_with_speed["speed_i"] < 0.5) | 
    (serious_with_speed["speed_j"] < 0.5)
)
print(f"\nSerious conflicts with at least one stopped agent: {stopped.sum():,} ({stopped.mean():.1%})")

# Sample 10 serious conflict pairs for manual inspection
print("\n=== Sample serious conflict pairs ===")
print(serious[["distance","rel_speed","ttc","drac","interaction_type"]].sample(10, random_state=42).to_string())

In [ ]:
import matplotlib.pyplot as plt

# Conflict rate over time of day
pairs["hour"] = pd.to_datetime(
    pairs["video_id"].str.extract(
        r'(\d{4}-\w{3}-\d{2}_\d{2}-\d{2}-\d{2}_(?:AM|PM))'
    )[0],
    format="%Y-%b-%d_%I-%M-%S_%p"
).dt.hour

conflict_by_hour = (
    pairs.groupby("hour")
         .agg(
             total=("conflict_label", "count"),
             conflicts=("conflict_label", lambda x: (x > 0).sum())
         )
)
conflict_by_hour["conflict_rate"] = conflict_by_hour["conflicts"] / conflict_by_hour["total"]

print("=== Conflict rate by hour ===")
print(conflict_by_hour)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conflict rate by hour
axes[0].bar(conflict_by_hour.index, conflict_by_hour["conflict_rate"])
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Conflict rate")
axes[0].set_title("Conflict rate by hour of day")
axes[0].yaxis.set_major_formatter(
    plt.matplotlib.ticker.PercentFormatter(xmax=1)
)

# Volume by hour
axes[1].bar(conflict_by_hour.index, conflict_by_hour["total"], color="steelblue")
axes[1].bar(conflict_by_hour.index, conflict_by_hour["conflicts"], color="tomato", label="Conflicts")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Interaction count")
axes[1].set_title("Total vs conflict interactions by hour")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Compute heading angle for each agent
df["heading"] = np.degrees(np.arctan2(df["vy"], df["vx"]))

# Add headings to pairs
pairs = pairs.merge(
    df[["video_id","frame_idx","track_id","heading"]],
    left_on=["video_id","frame_idx","track_id_i"],
    right_on=["video_id","frame_idx","track_id"],
    how="left"
).rename(columns={"heading":"heading_i"}).drop(columns="track_id")

pairs = pairs.merge(
    df[["video_id","frame_idx","track_id","heading"]],
    left_on=["video_id","frame_idx","track_id_j"],
    right_on=["video_id","frame_idx","track_id"],
    how="left"
).rename(columns={"heading":"heading_j"}).drop(columns="track_id")

# Compute heading difference
def heading_diff(h1, h2):
    diff = abs(h1 - h2) % 360
    if diff > 180:
        diff = 360 - diff
    return diff

pairs["heading_diff"] = pairs.apply(
    lambda r: heading_diff(r["heading_i"], r["heading_j"]), axis=1
)

print("=== Heading difference distribution for serious conflicts ===")
serious = pairs[pairs["conflict_label"] == 2]
print(serious["heading_diff"].describe())

print("\n=== Heading difference bins for serious conflicts ===")
bins = [0, 30, 60, 90, 120, 150, 180]
labels = ["0-30°", "30-60°", "60-90°", "90-120°", "120-150°", "150-180°"]
print(pd.cut(serious["heading_diff"], bins=bins, labels=labels).value_counts().sort_index())

print("\n=== Heading difference bins for all conflicts ===")
conflict_all = pairs[pairs["conflict_label"] > 0]
print(pd.cut(conflict_all["heading_diff"], bins=bins, labels=labels).value_counts().sort_index())

In [ ]:
def assign_conflict_label_v3(row):
    ttc   = row["ttc"]
    drac  = row["drac"]
    dist  = row["distance"]
    itype = row["interaction_type"]
    rspd  = row["rel_speed"]
    hdiff = row["heading_diff"]

    d_thresh  = VP_DIST_THRESH if itype == "vp" else DIST_THRESH
    rs_thresh = (MIN_REL_SPEED_VP if itype == "vp"
                 else MIN_REL_SPEED_PP if itype == "pp"
                 else MIN_REL_SPEED_VV)

    if pd.isna(ttc):           return 0
    if dist > d_thresh:        return 0
    if rspd < rs_thresh:       return 0

    # Heading-based filter for vv only
    if itype == "vv":
        # Opposite direction passing — require very close lateral proximity
        if hdiff > 120 and dist > 5.0:
            return 0

    if ttc < TTC_SERIOUS and drac >= DRAC_THRESH:
        return 2
    elif ttc < TTC_GENERAL:
        return 1
    else:
        return 0

pairs["conflict_label"] = pairs.apply(assign_conflict_label_v3, axis=1)

print("=== Conflict label distribution after heading filter ===")
print(pairs["conflict_label"].value_counts().sort_index())
print(f"\nNo conflict (0)      : {(pairs['conflict_label']==0).sum():,}  ({(pairs['conflict_label']==0).mean():.1%})")
print(f"General conflict (1) : {(pairs['conflict_label']==1).sum():,}  ({(pairs['conflict_label']==1).mean():.1%})")
print(f"Serious conflict (2) : {(pairs['conflict_label']==2).sum():,}  ({(pairs['conflict_label']==2).mean():.1%})")

print("\n=== Conflict by interaction type ===")
print(pairs.groupby(["interaction_type","conflict_label"]).size().unstack(fill_value=0))

print("\n=== Heading diff distribution for remaining serious conflicts ===")
serious = pairs[pairs["conflict_label"] == 2]
bins   = [0, 30, 60, 90, 120, 150, 180]
labels = ["0-30°","30-60°","60-90°","90-120°","120-150°","150-180°"]
print(pd.cut(serious["heading_diff"], bins=bins, labels=labels).value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

conflict  = pairs[pairs["conflict_label"] > 0].copy()
serious   = pairs[pairs["conflict_label"] == 2].copy()
no_conf   = pairs[pairs["conflict_label"] == 0].copy()

for p in [conflict, serious, no_conf]:
    p["x_mid"] = (p["x_i"] + p["x_j"]) / 2
    p["y_mid"] = (p["y_i"] + p["y_j"]) / 2

# Left: all vs conflict
axes[0].scatter(no_conf["x_mid"], no_conf["y_mid"],
                s=1, alpha=0.05, c="steelblue", label="No conflict")
axes[0].scatter(conflict["x_mid"], conflict["y_mid"],
                s=2, alpha=0.3, c="tomato", label="Conflict")
axes[0].set_title("All vs Conflict interactions")
axes[0].set_xlabel("x_local_m")
axes[0].set_ylabel("y_local_m")
axes[0].axis("equal")
axes[0].legend(markerscale=4)

# Middle: conflict by interaction type
colors_type = {"vv": "steelblue", "vp": "tomato", "pp": "green"}
for itype, grp in conflict.groupby("interaction_type"):
    axes[1].scatter(grp["x_mid"], grp["y_mid"],
                    s=2, alpha=0.3,
                    c=colors_type[itype], label=itype)
axes[1].set_title("Conflict by interaction type")
axes[1].set_xlabel("x_local_m")
axes[1].axis("equal")
axes[1].legend(markerscale=4)

# Right: serious conflict hotspot
hb = axes[2].hexbin(serious["x_mid"], serious["y_mid"],
                     gridsize=35, mincnt=1, cmap="YlOrRd")
plt.colorbar(hb, ax=axes[2], label="Serious conflict density")
axes[2].set_title("Serious conflict hotspots")
axes[2].set_xlabel("x_local_m")
axes[2].axis("equal")

plt.tight_layout()
plt.show()

# Temporal check
conflict_by_hour = (
    pairs.groupby("hour")
         .agg(
             total=("conflict_label","count"),
             conflicts=("conflict_label", lambda x: (x > 0).sum())
         )
)
conflict_by_hour["conflict_rate"] = (
    conflict_by_hour["conflicts"] / conflict_by_hour["total"]
)
print("=== Conflict rate by hour ===")
print(conflict_by_hour)

In [ ]:
# For each unique pair (track_id_i, track_id_j) per video,
# find the minimum distance ever reached
min_dist_per_pair = (
    pairs.groupby(["video_id", "track_id_i", "track_id_j"])["distance"]
         .min()
         .reset_index(name="min_dist")
)

print("=== Minimum distance per pair ===")
print(min_dist_per_pair["min_dist"].describe())

print("\n=== Pairs that came within 3m ===")
close = min_dist_per_pair[min_dist_per_pair["min_dist"] < 3.0]
print(f"Count: {len(close):,}  ({len(close)/len(min_dist_per_pair):.1%} of all pairs)")

print("\n=== Pairs that came within 5m ===")
close5 = min_dist_per_pair[min_dist_per_pair["min_dist"] < 5.0]
print(f"Count: {len(close5):,}  ({len(close5)/len(min_dist_per_pair):.1%} of all pairs)")

print("\n=== Distribution of minimum distance ===")
bins = [0, 1, 2, 3, 5, 8, 10, 15, 20, 30]
print(pd.cut(min_dist_per_pair["min_dist"], bins=bins).value_counts().sort_index())

In [ ]:
# Build proximity-gated conflict labels
# First get min distance per pair
min_dist_per_pair = (
    pairs.groupby(["video_id", "track_id_i", "track_id_j"])["distance"]
         .min()
         .reset_index(name="min_dist")
)

# Get min TTC per pair
min_ttc_per_pair = (
    pairs[pairs["ttc"].notna()]
         .groupby(["video_id", "track_id_i", "track_id_j"])["ttc"]
         .min()
         .reset_index(name="min_ttc")
)

# Merge
pair_summary = min_dist_per_pair.merge(
    min_ttc_per_pair,
    on=["video_id", "track_id_i", "track_id_j"],
    how="left"
)

# Add interaction type
pair_type = (
    pairs.groupby(["video_id", "track_id_i", "track_id_j"])["interaction_type"]
         .first()
         .reset_index()
)
pair_summary = pair_summary.merge(
    pair_type,
    on=["video_id", "track_id_i", "track_id_j"],
    how="left"
)

# Label
def pair_conflict_label(row):
    d    = row["min_dist"]
    ttc  = row["min_ttc"]
    itype = row["interaction_type"]

    # vp uses tighter distance
    d_serious = 2.0 if itype == "vp" else 3.0
    d_general = 3.0 if itype == "vp" else 5.0

    if pd.isna(ttc):
        # No valid TTC — use distance only as fallback
        if d < d_serious:
            return 2
        elif d < d_general:
            return 1
        return 0

    if d < d_serious and ttc < 1.5:
        return 2   # serious
    elif d < d_general and ttc < 3.0:
        return 1   # general
    return 0

pair_summary["pair_conflict_label"] = pair_summary.apply(
    pair_conflict_label, axis=1
)

print("=== Pair-level conflict distribution ===")
print(pair_summary["pair_conflict_label"].value_counts().sort_index())
print(f"\nTotal unique pairs   : {len(pair_summary):,}")
print(f"No conflict (0)      : {(pair_summary['pair_conflict_label']==0).sum():,}  ({(pair_summary['pair_conflict_label']==0).mean():.1%})")
print(f"General conflict (1) : {(pair_summary['pair_conflict_label']==1).sum():,}  ({(pair_summary['pair_conflict_label']==1).mean():.1%})")
print(f"Serious conflict (2) : {(pair_summary['pair_conflict_label']==2).sum():,}  ({(pair_summary['pair_conflict_label']==2).mean():.1%})")

print("\n=== By interaction type ===")
print(pair_summary.groupby(["interaction_type","pair_conflict_label"]).size().unstack(fill_value=0))

In [ ]:
# Fix: remove distance-only fallback
# Require BOTH proximity AND valid TTC for a conflict label
# If TTC is NaN, it means agents are not approaching — no conflict

def pair_conflict_label_v2(row):
    d     = row["min_dist"]
    ttc   = row["min_ttc"]
    itype = row["interaction_type"]

    # No valid TTC = not approaching = no conflict
    if pd.isna(ttc):
        return 0

    d_serious = 2.0 if itype == "vp" else 3.0
    d_general = 3.0 if itype == "vp" else 5.0

    if d < d_serious and ttc < 1.5:
        return 2
    elif d < d_general and ttc < 3.0:
        return 1
    return 0

pair_summary["pair_conflict_label"] = pair_summary.apply(
    pair_conflict_label_v2, axis=1
)

print("=== Pair-level conflict distribution (v2) ===")
print(pair_summary["pair_conflict_label"].value_counts().sort_index())
print(f"\nTotal unique pairs   : {len(pair_summary):,}")
print(f"No conflict (0)      : {(pair_summary['pair_conflict_label']==0).sum():,}  ({(pair_summary['pair_conflict_label']==0).mean():.1%})")
print(f"General conflict (1) : {(pair_summary['pair_conflict_label']==1).sum():,}  ({(pair_summary['pair_conflict_label']==1).mean():.1%})")
print(f"Serious conflict (2) : {(pair_summary['pair_conflict_label']==2).sum():,}  ({(pair_summary['pair_conflict_label']==2).mean():.1%})")

print("\n=== By interaction type ===")
print(pair_summary.groupby(
    ["interaction_type","pair_conflict_label"]
).size().unstack(fill_value=0))

print("\n=== Min TTC availability by interaction type ===")
print(pairs[pairs["ttc"].notna()].groupby("interaction_type").size())
print("\nTotal pairs per type:")
print(pairs.groupby("interaction_type").size())

In [ ]:
pp_with_ttc = pair_summary[
    (pair_summary["interaction_type"] == "pp") &
    (pair_summary["min_ttc"].notna())
]

print("=== PP pairs with valid TTC ===")
print(f"Count: {len(pp_with_ttc):,}")
print("\nMin distance distribution:")
print(pp_with_ttc["min_dist"].describe())
print("\nMin TTC distribution:")
print(pp_with_ttc["min_ttc"].describe())
print("\nConflict labels:")
print(pp_with_ttc["pair_conflict_label"].value_counts().sort_index())

In [ ]:
def pair_conflict_label_v3(row):
    d     = row["min_dist"]
    ttc   = row["min_ttc"]
    itype = row["interaction_type"]

    # PP conflicts not meaningful from YOLO tracking
    if itype == "pp":
        return 0

    # No valid TTC = not approaching = no conflict
    if pd.isna(ttc):
        return 0

    d_serious = 2.0 if itype == "vp" else 3.0
    d_general = 3.0 if itype == "vp" else 5.0

    if d < d_serious and ttc < 1.5:
        return 2
    elif d < d_general and ttc < 3.0:
        return 1
    return 0

pair_summary["pair_conflict_label"] = pair_summary.apply(
    pair_conflict_label_v3, axis=1
)

print("=== Final pair-level conflict distribution ===")
print(pair_summary["pair_conflict_label"].value_counts().sort_index())
print(f"\nTotal unique pairs   : {len(pair_summary):,}")
print(f"No conflict (0)      : {(pair_summary['pair_conflict_label']==0).sum():,}  ({(pair_summary['pair_conflict_label']==0).mean():.1%})")
print(f"General conflict (1) : {(pair_summary['pair_conflict_label']==1).sum():,}  ({(pair_summary['pair_conflict_label']==1).mean():.1%})")
print(f"Serious conflict (2) : {(pair_summary['pair_conflict_label']==2).sum():,}  ({(pair_summary['pair_conflict_label']==2).mean():.1%})")

print("\n=== By interaction type ===")
print(pair_summary.groupby(
    ["interaction_type","pair_conflict_label"]
).size().unstack(fill_value=0))

# Merge pair-level labels back onto frame-level pairs
pairs = pairs.merge(
    pair_summary[["video_id","track_id_i","track_id_j","pair_conflict_label"]],
    on=["video_id","track_id_i","track_id_j"],
    how="left"
)
pairs["pair_conflict_label"] = pairs["pair_conflict_label"].fillna(0).astype(int)

print("\n=== Frame-level pairs with pair label ===")
print(f"Total frame-level pairs: {len(pairs):,}")
print(pairs["pair_conflict_label"].value_counts().sort_index())

In [ ]:
# Save
pairs.to_parquet(checkpoint_path / "pairs_with_ssm_labeled.parquet", index=False)
pair_summary.to_parquet(checkpoint_path / "pair_summary.parquet", index=False)

print(f"Saved pairs_with_ssm_labeled.parquet: {pairs.shape}")
print(f"Saved pair_summary.parquet          : {pair_summary.shape}")
print(f"\nColumns in pairs: {pairs.columns.tolist()}")

In [ ]:
pairs['ttc'].value_counts(bins=[0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, np.inf])

In [ ]:
# def build_scene_graphs_fast(df_agents, df_pairs):
#     """
#     Vectorized scene graph construction — no per-frame pandas filtering.
#     """
#     print("Pre-indexing pairs by (video_id, frame_idx)...")
    
#     # Pre-group pairs into a dict keyed by (video_id, frame_idx)
#     pairs_grouped = {}
#     for (vid, fidx), grp in df_pairs.groupby(["video_id", "frame_idx"]):
#         pairs_grouped[(vid, fidx)] = grp
    
#     print(f"Pair groups indexed: {len(pairs_grouped):,}")
#     print("Building scene graphs...")

#     scene_graphs = []
    
#     for (video_id, frame_idx), agents in df_agents.groupby(
#         ["video_id", "frame_idx"]
#     ):
#         # Nodes
#         nodes = agents[[
#             "track_id", "class", "agent_type",
#             "x_local_m", "y_local_m", "vx", "vy", "speed", "heading"
#         ]].rename(columns={
#             "x_local_m": "x",
#             "y_local_m": "y"
#         }).to_dict("records")

#         # Edges — use pre-indexed dict
#         frame_edges = pairs_grouped.get((video_id, frame_idx), None)
        
#         edges = []
#         if frame_edges is not None:
#             for row in frame_edges.itertuples(index=False):
#                 edges.append({
#                     "i"               : row.track_id_i,
#                     "j"               : row.track_id_j,
#                     "interaction_type": row.interaction_type,
#                     "distance"        : row.distance,
#                     "rel_speed"       : row.rel_speed,
#                     "ttc"             : row.ttc     if pd.notna(row.ttc)     else 0.0,
#                     "drac"            : row.drac    if pd.notna(row.drac)    else 0.0,
#                     "dttc_dt"         : row.dttc_dt if pd.notna(row.dttc_dt) else 0.0,
#                     "heading_diff"    : row.heading_diff,
#                     "conflict_label"  : row.pair_conflict_label
#                 })

#         frame_label = 1 if any(e["conflict_label"] > 0 for e in edges) else 0

#         scene_graphs.append({
#             "video_id"           : video_id,
#             "frame_idx"          : frame_idx,
#             "nodes"              : nodes,
#             "edges"              : edges,
#             "label"              : frame_label,
#             "num_nodes"          : len(nodes),
#             "num_edges"          : len(edges),
#             "num_conflict_edges" : sum(e["conflict_label"] > 0 for e in edges)
#         })

#     return scene_graphs


# # Add heading if missing
# if "heading" not in df.columns:
#     df["heading"] = np.degrees(np.arctan2(df["vy"], df["vx"]))

# scene_graphs = build_scene_graphs_fast(df, pairs)

# print(f"\nTotal scene graphs : {len(scene_graphs):,}")

# labels   = [g["label"]              for g in scene_graphs]
# num_nodes = [g["num_nodes"]         for g in scene_graphs]
# num_edges = [g["num_edges"]         for g in scene_graphs]
# num_conf  = [g["num_conflict_edges"] for g in scene_graphs]

# print(f"Frames with conflict    : {sum(labels):,}  ({sum(labels)/len(labels):.1%})")
# print(f"Frames without conflict : {len(labels)-sum(labels):,}")
# print(f"\nNodes per frame — mean: {np.mean(num_nodes):.1f}  max: {max(num_nodes)}")
# print(f"Edges per frame — mean: {np.mean(num_edges):.1f}  max: {max(num_edges)}")
# print(f"Conflict edges  — mean: {np.mean(num_conf):.2f}  max: {max(num_conf)}")

In [ ]:
# Rebuild scene graphs with enriched features
def build_scene_graphs_fast_v2(df_agents, df_pairs):
    """
    Enriched scene graphs:
    - Nodes: x, y, vx, vy, speed, accel, heading, agent_type
    - Edges: distance, rel_speed, drac, heading_diff, rel_vx, rel_vy
    TTC and dTTC/dt excluded — both derived from TTC used for labeling.
    """
    print("Pre-indexing pairs...")
    pairs_grouped = {}
    for (vid, fidx), grp in df_pairs.groupby(["video_id", "frame_idx"]):
        pairs_grouped[(vid, fidx)] = grp

    print(f"Pair groups indexed: {len(pairs_grouped):,}")
    print("Building enriched scene graphs...")

    scene_graphs = []

    for (video_id, frame_idx), agents in df_agents.groupby(
        ["video_id", "frame_idx"]
    ):
        # Nodes — add accel and heading
        nodes = agents[[
            "track_id", "class", "agent_type",
            "x_local_m", "y_local_m",
            "vx", "vy", "speed", "accel", "heading"
        ]].rename(columns={
            "x_local_m": "x",
            "y_local_m": "y"
        }).to_dict("records")

        # Edges
        frame_edges = pairs_grouped.get((video_id, frame_idx), None)

        edges = []
        if frame_edges is not None:
            for row in frame_edges.itertuples(index=False):
                # Relative velocity components
                rel_vx = row.vx_i - row.vx_j
                rel_vy = row.vy_i - row.vy_j

                edges.append({
                    "i"               : row.track_id_i,
                    "j"               : row.track_id_j,
                    "interaction_type": row.interaction_type,
                    "distance"        : row.distance,
                    "rel_speed"       : row.rel_speed,
                    "drac"            : row.drac    if pd.notna(row.drac)    else 0.0,
                    "heading_diff"    : row.heading_diff,
                    "rel_vx"          : rel_vx,
                    "rel_vy"          : rel_vy,
                    "conflict_label"  : row.pair_conflict_label
                })

        frame_label = 1 if any(e["conflict_label"] > 0 for e in edges) else 0

        scene_graphs.append({
            "video_id"           : video_id,
            "frame_idx"          : frame_idx,
            "nodes"              : nodes,
            "edges"              : edges,
            "label"              : frame_label,
            "num_nodes"          : len(nodes),
            "num_edges"          : len(edges),
            "num_conflict_edges" : sum(e["conflict_label"] > 0 for e in edges)
        })

    return scene_graphs


scene_graphs = build_scene_graphs_fast_v2(df, pairs)

print(f"\nTotal scene graphs : {len(scene_graphs):,}")

labels    = [g["label"]    for g in scene_graphs]
num_nodes = [g["num_nodes"] for g in scene_graphs]
num_edges = [g["num_edges"] for g in scene_graphs]

print(f"Frames with conflict    : {sum(labels):,}  ({sum(labels)/len(labels):.1%})")
print(f"Nodes per frame — mean  : {np.mean(num_nodes):.1f}  max: {max(num_nodes)}")
print(f"Edges per frame — mean  : {np.mean(num_edges):.1f}  max: {max(num_edges)}")

# Check new node and edge fields
sample = scene_graphs[100]
print(f"\nNode fields : {list(sample['nodes'][0].keys())}")
if sample['edges']:
    print(f"Edge fields : {list(sample['edges'][0].keys())}")

In [ ]:
# Fix 1: Check edge fields on a graph that has edges
sample_with_edges = next(g for g in scene_graphs if len(g["edges"]) > 0)
print(f"Edge fields: {list(sample_with_edges['edges'][0].keys())}")

# Fix 2: Reapply instantaneous conflict labeling
def compute_frame_label(frame_edges):
    for e in frame_edges:
        itype = e["interaction_type"]
        dist  = e["distance"]
        drac  = e["drac"]

        if itype == "pp":
            continue

        d_thresh = 3.0 if itype == "vp" else 5.0

        # Use DRAC as conflict indicator — no TTC dependency
        # DRAC > 3.0 m/s² AND within distance threshold
        if dist < d_thresh and drac > 3.0:
            return 1

    return 0

# Reapply labels
for g in scene_graphs:
    g["label"] = compute_frame_label(g["edges"])

labels = [g["label"] for g in scene_graphs]
print(f"\nFrame-level conflict rate: {sum(labels)/len(labels):.1%}")
print(f"Conflict frames : {sum(labels):,}")
print(f"Safe frames     : {len(labels)-sum(labels):,}")

In [ ]:
# Option B: TTC-based frame labels, kinematic edge features
def compute_frame_label_ttc(frame_edges):
    """
    Frame is conflict if any edge has instantaneous TTC < 3.0s
    AND distance < threshold AND not PP.
    TTC is label definition only — not used as feature.
    """
    for e in frame_edges:
        itype          = e["interaction_type"]
        dist           = e["distance"]
        conflict_label = e["conflict_label"]  # pair-level TTC-based label

        if itype == "pp":
            continue

        if conflict_label > 0:
            return 1

    return 0

# Reapply
for g in scene_graphs:
    g["label"] = compute_frame_label_ttc(g["edges"])

labels = [g["label"] for g in scene_graphs]
print(f"Frame-level conflict rate : {sum(labels)/len(labels):.1%}")
print(f"Conflict frames           : {sum(labels):,}")
print(f"Safe frames               : {len(labels)-sum(labels):,}")

In [ ]:
# The pair_conflict_label in edges is pair-level (propagated)
# We need instantaneous frame-level labeling
# Pull instantaneous TTC values back from pairs dataframe

print("Re-indexing pairs with instantaneous TTC values...")

# Build lookup: (video_id, frame_idx, track_id_i, track_id_j) -> instantaneous ttc
pairs_ttc_lookup = {}
for row in pairs[["video_id","frame_idx","track_id_i","track_id_j",
                   "ttc","distance","interaction_type"]].itertuples(index=False):
    key = (row.video_id, row.frame_idx, row.track_id_i, row.track_id_j)
    pairs_ttc_lookup[key] = {
        "ttc"             : row.ttc,
        "distance"        : row.distance,
        "interaction_type": row.interaction_type
    }

print(f"Lookup entries: {len(pairs_ttc_lookup):,}")

def compute_frame_label_instantaneous(graph):
    """
    Label frame as conflict based on instantaneous TTC in this frame only.
    """
    video_id  = graph["video_id"]
    frame_idx = graph["frame_idx"]

    for e in graph["edges"]:
        itype = e["interaction_type"]
        if itype == "pp":
            continue

        key  = (video_id, frame_idx, e["i"], e["j"])
        data = pairs_ttc_lookup.get(key)

        if data is None:
            # Try reverse order
            key  = (video_id, frame_idx, e["j"], e["i"])
            data = pairs_ttc_lookup.get(key)

        if data is None:
            continue

        ttc  = data["ttc"]
        dist = data["distance"]

        if pd.isna(ttc):
            continue

        d_thresh = 3.0 if itype == "vp" else 5.0

        if dist < d_thresh and ttc < 3.0:
            return 1

    return 0

print("Recomputing instantaneous frame labels...")
for g in scene_graphs:
    g["label"] = compute_frame_label_instantaneous(g)

labels = [g["label"] for g in scene_graphs]
print(f"\nFrame-level conflict rate : {sum(labels)/len(labels):.1%}")
print(f"Conflict frames           : {sum(labels):,}")
print(f"Safe frames               : {len(labels)-sum(labels):,}")

In [ ]:
def build_sequences_strided(graphs, seq_len=8, stride=8, max_frame_gap=5):
    """
    Strided sequences — non-overlapping by default (stride=seq_len).
    Reduces conflict inflation from overlapping windows.
    """
    sequences  = []
    seq_labels = []
    n = len(graphs)

    for i in range(0, n - seq_len + 1, stride):
        seq = graphs[i:i + seq_len]

        # Same video constraint
        if len(set(g["video_id"] for g in seq)) > 1:
            continue

        # Temporal continuity
        frame_gaps = [
            seq[k+1]["frame_idx"] - seq[k]["frame_idx"]
            for k in range(seq_len - 1)
        ]
        if max(frame_gaps) > max_frame_gap:
            continue

        label = 1 if any(g["label"] == 1 for g in seq) else 0

        sequences.append(seq)
        seq_labels.append(label)

    return sequences, seq_labels

In [ ]:
SEQ_LEN = 8

In [ ]:
# Filter sparse frames
scene_graphs_filtered = [g for g in scene_graphs if g["num_nodes"] >= 2]
print(f"Scene graphs after node filter: {len(scene_graphs_filtered):,}")

# Sort chronologically
scene_graphs_filtered = sorted(
    scene_graphs_filtered,
    key=lambda x: (x["video_id"], x["frame_idx"])
)

# Rebuild sequences with stride=4
sequences, seq_labels = build_sequences_strided(
    scene_graphs_filtered,
    seq_len=SEQ_LEN,
    stride=4,
    max_frame_gap=5
)

seq_labels_arr = np.array(seq_labels)

print(f"\nTotal sequences        : {len(sequences):,}")
print(f"Conflict sequences (1) : {seq_labels_arr.sum():,}  ({seq_labels_arr.mean():.1%})")
print(f"Safe sequences    (0)  : {(seq_labels_arr==0).sum():,}  ({(seq_labels_arr==0).mean():.1%})")

# Rebuild split
video_stats = {}
for seq, label in zip(sequences, seq_labels):
    vid = seq[0]["video_id"]
    if vid not in video_stats:
        video_stats[vid] = {"total": 0, "conflict": 0}
    video_stats[vid]["total"]   += 1
    video_stats[vid]["conflict"] += label

video_df = pd.DataFrame([
    {
        "video_id"     : vid,
        "total_seqs"   : stats["total"],
        "conflict_seqs": stats["conflict"],
        "conflict_rate": stats["conflict"] / stats["total"],
        "file_datetime": parse_datetime_from_filename(
            next(f for f in csv_files_sorted
                 if f.stem == vid.replace(".mp4",""))
        )
    }
    for vid, stats in video_stats.items()
]).sort_values("file_datetime").reset_index(drop=True)

# Split by cumulative sequence count
video_df["cumulative_seqs"] = video_df["total_seqs"].cumsum()
total_seqs    = video_df["total_seqs"].sum()
train_cutoff  = total_seqs * 0.70

train_videos  = set(video_df[
    video_df["cumulative_seqs"] <= train_cutoff
]["video_id"])

remaining  = video_df[~video_df["video_id"].isin(train_videos)]
val_videos = set()
val_count  = 0
for _, row in remaining.iterrows():
    if val_count < total_seqs * 0.15:
        val_videos.add(row["video_id"])
        val_count += row["total_seqs"]
    else:
        break

test_videos = set(video_df["video_id"]) - train_videos - val_videos

# Split sequences
train_seqs = [(s,l) for s,l in zip(sequences, seq_labels)
              if s[0]["video_id"] in train_videos]
val_seqs   = [(s,l) for s,l in zip(sequences, seq_labels)
              if s[0]["video_id"] in val_videos]
test_seqs  = [(s,l) for s,l in zip(sequences, seq_labels)
              if s[0]["video_id"] in test_videos]

train_seqs, train_labels = zip(*train_seqs)
val_seqs,   val_labels   = zip(*val_seqs)
test_seqs,  test_labels  = zip(*test_seqs)

train_labels = np.array(train_labels)
val_labels   = np.array(val_labels)
test_labels  = np.array(test_labels)

print(f"\nTrain: {len(train_seqs):,}  ({len(train_seqs)/total_seqs:.1%})  conflict: {train_labels.mean():.1%}")
print(f"Val  : {len(val_seqs):,}  ({len(val_seqs)/total_seqs:.1%})  conflict: {val_labels.mean():.1%}")
print(f"Test : {len(test_seqs):,}  ({len(test_seqs)/total_seqs:.1%})  conflict: {test_labels.mean():.1%}")
print(f"Imbalance ratio (train): {(train_labels==0).sum()/train_labels.sum():.1f}:1")

In [ ]:
seq_labels_arr = np.array(seq_labels)

with open(checkpoint_path / "sequences_stride4.pkl", "wb") as f:
    pickle.dump((sequences, seq_labels), f)

print(f"Saved sequences_stride4.pkl")
print(f"Total sequences  : {len(sequences):,}")
print(f"Conflict (1)     : {seq_labels_arr.sum():,}  ({seq_labels_arr.mean():.1%})")
print(f"Safe (0)         : {(seq_labels_arr==0).sum():,}  ({(seq_labels_arr==0).mean():.1%})")
print(f"\nClass imbalance ratio: {(seq_labels_arr==0).sum() / seq_labels_arr.sum():.1f}:1")

In [ ]:
import pickle

split = {
    "train_seqs"  : train_seqs,
    "train_labels": train_labels,
    "val_seqs"    : val_seqs,
    "val_labels"  : val_labels,
    "test_seqs"   : test_seqs,
    "test_labels" : test_labels,
    "train_videos": train_videos,
    "val_videos"  : val_videos,
    "test_videos" : test_videos
}

with open(checkpoint_path / "data_split.pkl", "wb") as f:
    pickle.dump(split, f)

print("Saved data_split.pkl")
print(f"\nFinal split summary:")
print(f"Train: {len(train_seqs):,} sequences  conflict: {train_labels.mean():.1%}")
print(f"Val  : {len(val_seqs):,} sequences  conflict: {val_labels.mean():.1%}")
print(f"Test : {len(test_seqs):,} sequences  conflict: {test_labels.mean():.1%}")
print(f"Imbalance ratio (train): {(train_labels==0).sum()/train_labels.sum():.1f}:1")

In [ ]:
# Must run AFTER sequences and split are rebuilt

def extract_node_features(sequences):
    feats = []
    for seq in sequences:
        for g in seq:
            for n in g["nodes"]:
                feats.append([
                    n["x"], n["y"],
                    n["vx"], n["vy"],
                    n["speed"],
                    n["accel"],
                    n["heading"]
                ])
    return np.array(feats, dtype=np.float32)

def extract_edge_features(sequences):
    """
    Final edge features — TTC and dTTC/dt excluded.
    distance, rel_speed, drac, heading_diff, rel_vx, rel_vy
    """
    feats = []
    for seq in sequences:
        for g in seq:
            for e in g["edges"]:
                feats.append([
                    e["distance"],
                    e["rel_speed"],
                    e["drac"],
                    e["heading_diff"],
                    e["rel_vx"],
                    e["rel_vy"]
                ])
    return np.array(feats, dtype=np.float32)

print("Extracting train node features...")
train_node_feats = extract_node_features(train_seqs)
print(f"Node feature matrix: {train_node_feats.shape}")

print("Extracting train edge features...")
train_edge_feats = extract_edge_features(train_seqs)
print(f"Edge feature matrix: {train_edge_feats.shape}")

# Compute stats from train only
node_mean = train_node_feats.mean(axis=0)
node_std  = train_node_feats.std(axis=0)
node_std[node_std < 1e-6] = 1.0

edge_mean = train_edge_feats.mean(axis=0)
edge_std  = train_edge_feats.std(axis=0)
edge_std[edge_std < 1e-6] = 1.0

print("\n=== Node normalization stats ===")
node_feat_names = ["x", "y", "vx", "vy", "speed", "accel", "heading"]
for i, name in enumerate(node_feat_names):
    print(f"  {name:10s}  mean={node_mean[i]:8.3f}  std={node_std[i]:8.3f}")

print("\n=== Edge normalization stats ===")
edge_feat_names = ["distance", "rel_speed", "drac", "heading_diff", "rel_vx", "rel_vy"]
for i, name in enumerate(edge_feat_names):
    print(f"  {name:12s}  mean={edge_mean[i]:8.3f}  std={edge_std[i]:8.3f}")

# Save
norm_stats = {
    "node_mean": node_mean,
    "node_std" : node_std,
    "edge_mean": edge_mean,
    "edge_std" : edge_std
}
with open(checkpoint_path / "norm_stats.pkl", "wb") as f:
    pickle.dump(norm_stats, f)

print("\nSaved norm_stats.pkl")
print(f"NODE_DIM = 7 continuous + 2 one-hot = 9")
print(f"EDGE_DIM = 6")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

pos_weight = torch.tensor(
    [(train_labels == 0).sum() / (train_labels == 1).sum()],
    dtype=torch.float32
).to(device)
print(f"pos_weight: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
# ── Final constants ────────────────────────────────────────
NODE_DIM  = 9   # 7 normalized + 2 one-hot (is_vehicle, is_pedestrian)
EDGE_DIM  = 6   # distance, rel_speed, drac, heading_diff, rel_vx, rel_vy
MAX_NODES = 9
SEQ_LEN   = 8

def scene_graph_to_tensors(graph, max_nodes,
                            node_mean, node_std,
                            edge_mean, edge_std):
    nodes  = graph["nodes"]
    edges  = graph["edges"]
    N      = len(nodes)
    id2idx = {n["track_id"]: i for i, n in enumerate(nodes)}

    # Node features [N, 9]
    X = []
    for n in nodes:
        raw    = np.array([
            n["x"], n["y"], n["vx"], n["vy"],
            n["speed"], n["accel"], n["heading"]
        ], dtype=np.float32)
        normed = (raw - node_mean) / node_std
        is_veh = 1.0 if n["agent_type"] == "vehicle"    else 0.0
        is_ped = 1.0 if n["agent_type"] == "pedestrian" else 0.0
        X.append(np.append(normed, [is_veh, is_ped]))
    X = np.array(X, dtype=np.float32)

    # Adjacency
    A_vv = np.zeros((N, N), dtype=np.float32)
    A_vp = np.zeros((N, N), dtype=np.float32)
    A_pp = np.zeros((N, N), dtype=np.float32)

    # Edge features [N, N, 6]
    E = np.zeros((N, N, EDGE_DIM), dtype=np.float32)

    for e in edges:
        i = id2idx.get(e["i"])
        j = id2idx.get(e["j"])
        if i is None or j is None:
            continue

        raw_ef = np.array([
            e["distance"],
            e["rel_speed"],
            e["drac"],
            e["heading_diff"],
            e["rel_vx"],
            e["rel_vy"]
        ], dtype=np.float32)
        normed_ef = (raw_ef - edge_mean) / edge_std

        E[i, j] = normed_ef
        E[j, i] = normed_ef

        itype = e["interaction_type"]
        if itype == "vv":
            A_vv[i,j] = A_vv[j,i] = 1
        elif itype == "vp":
            A_vp[i,j] = A_vp[j,i] = 1
        else:
            A_pp[i,j] = A_pp[j,i] = 1

    # Pad
    M        = max_nodes
    X_pad    = np.zeros((M, NODE_DIM),    dtype=np.float32)
    A_vv_pad = np.zeros((M, M),           dtype=np.float32)
    A_vp_pad = np.zeros((M, M),           dtype=np.float32)
    A_pp_pad = np.zeros((M, M),           dtype=np.float32)
    E_pad    = np.zeros((M, M, EDGE_DIM), dtype=np.float32)

    X_pad[:N]       = X
    A_vv_pad[:N,:N] = A_vv
    A_vp_pad[:N,:N] = A_vp
    A_pp_pad[:N,:N] = A_pp
    E_pad[:N,:N]    = E

    return X_pad, A_vv_pad, A_vp_pad, A_pp_pad, E_pad


class SceneGraphDataset(Dataset):
    def __init__(self, sequences, labels,
                 node_mean, node_std,
                 edge_mean, edge_std,
                 max_nodes=MAX_NODES):
        self.sequences = sequences
        self.labels    = labels
        self.max_nodes = max_nodes
        self.node_mean = node_mean
        self.node_std  = node_std
        self.edge_mean = edge_mean
        self.edge_std  = edge_std

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq   = self.sequences[idx]
        label = self.labels[idx]

        X_seq, Avv_seq, Avp_seq, App_seq, E_seq = [], [], [], [], []

        for g in seq:
            X, Avv, Avp, App, E = scene_graph_to_tensors(
                g, self.max_nodes,
                self.node_mean, self.node_std,
                self.edge_mean, self.edge_std
            )
            X_seq.append(X)
            Avv_seq.append(Avv)
            Avp_seq.append(Avp)
            App_seq.append(App)
            E_seq.append(E)

        return (
            torch.tensor(np.stack(X_seq),   dtype=torch.float32),
            torch.tensor(np.stack(Avv_seq), dtype=torch.float32),
            torch.tensor(np.stack(Avp_seq), dtype=torch.float32),
            torch.tensor(np.stack(App_seq), dtype=torch.float32),
            torch.tensor(np.stack(E_seq),   dtype=torch.float32),
            torch.tensor(label,             dtype=torch.float32)
        )


# ── Datasets ───────────────────────────────────────────────
ds_train = SceneGraphDataset(
    train_seqs, train_labels,
    node_mean, node_std, edge_mean, edge_std
)
ds_val = SceneGraphDataset(
    val_seqs, val_labels,
    node_mean, node_std, edge_mean, edge_std
)
ds_test = SceneGraphDataset(
    test_seqs, test_labels,
    node_mean, node_std, edge_mean, edge_std
)

# ── Model ──────────────────────────────────────────────────
class HeteroSSMGNN(nn.Module):
    def __init__(self, node_dim, hidden_dim, edge_dim):
        super().__init__()
        self.W_vv     = nn.Linear(node_dim, hidden_dim)
        self.W_vp     = nn.Linear(node_dim, hidden_dim)
        self.W_pp     = nn.Linear(node_dim, hidden_dim)
        self.edge_enc = nn.Linear(edge_dim, hidden_dim)
        self.attn_vv  = nn.Linear(2 * hidden_dim, 1)
        self.attn_vp  = nn.Linear(2 * hidden_dim, 1)
        self.attn_pp  = nn.Linear(2 * hidden_dim, 1)
        self.fusion   = nn.Linear(node_dim + 3 * hidden_dim, hidden_dim)
        self.norm     = nn.LayerNorm(hidden_dim)
        self.leaky    = nn.LeakyReLU(0.2)

    def relation_aggregate(self, X, A, W, attn_layer, E):
        B, N, _ = X.shape
        H        = W(X)
        E_enc    = self.edge_enc(E)
        ssm_bias = E_enc.mean(dim=-1)
        Hi       = H.unsqueeze(2).expand(-1, -1, N, -1)
        Hj       = H.unsqueeze(1).expand(-1, N, -1, -1)
        scores   = self.leaky(
            attn_layer(torch.cat([Hi, Hj], dim=-1)).squeeze(-1)
        )
        scores   = scores + ssm_bias
        scores   = scores.masked_fill(A == 0, -1e9)
        weights  = torch.softmax(scores, dim=-1)
        return torch.bmm(weights, H)

    def forward(self, X, A_vv, A_vp, A_pp, E):
        msg_vv   = self.relation_aggregate(X, A_vv, self.W_vv, self.attn_vv, E)
        msg_vp   = self.relation_aggregate(X, A_vp, self.W_vp, self.attn_vp, E)
        msg_pp   = self.relation_aggregate(X, A_pp, self.W_pp, self.attn_pp, E)
        combined = torch.cat([X, msg_vv, msg_vp, msg_pp], dim=-1)
        return self.norm(F.relu(self.fusion(combined)))


class SceneRiskModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.gnn     = HeteroSSMGNN(node_dim, hidden_dim, edge_dim)
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim,
                               num_layers=lstm_layers,
                               batch_first=True,
                               dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []

        for t in range(T):
            X   = X_seq[:,t]
            Avv = A_vv_seq[:,t]
            Avp = A_vp_seq[:,t]
            App = A_pp_seq[:,t]
            E   = E_seq[:,t]

            H = self.gnn(X, Avv, Avp, App, E)

            # DRAC-weighted pooling (index 2 in edge features)
            drac_scores  = E_seq[:,t,:,:,2].max(dim=-1).values
            risk_weights = torch.softmax(drac_scores, dim=-1).unsqueeze(-1)
            H_pool       = (H * risk_weights).sum(dim=1)

            frame_embeds.append(H_pool)

        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# ── Sanity check ───────────────────────────────────────────
model = SceneRiskModel().to(device)
X, Avv, Avp, App, E, y = ds_train[0]
out = model(
    X.unsqueeze(0).to(device),
    Avv.unsqueeze(0).to(device),
    Avp.unsqueeze(0).to(device),
    App.unsqueeze(0).to(device),
    E.unsqueeze(0).to(device)
)
print(f"X shape          : {X.shape}   ← [T, N, 9]")
print(f"E shape          : {E.shape}  ← [T, N, N, 6]")
print(f"Output shape     : {out.shape}")
print(f"Output value     : {out.item():.4f}")
print(f"Total parameters : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Verify split is still in memory
print("=== Split verification ===")
print(f"train_seqs  : {len(train_seqs):,}")
print(f"val_seqs    : {len(val_seqs):,}")
print(f"test_seqs   : {len(test_seqs):,}")

print(f"\ntrain_labels: {len(train_labels):,}  conflict: {train_labels.mean():.1%}")
print(f"val_labels  : {len(val_labels):,}  conflict: {val_labels.mean():.1%}")
print(f"test_labels : {len(test_labels):,}  conflict: {test_labels.mean():.1%}")

print(f"\nTrain videos: {len(train_videos)}")
print(f"Val videos  : {len(val_videos)}")
print(f"Test videos : {len(test_videos)}")

print(f"\nds_train: {len(ds_train):,}")
print(f"ds_val  : {len(ds_val):,}")
print(f"ds_test : {len(ds_test):,}")

# Verify node and edge dims in dataset
X, Avv, Avp, App, E, y = ds_train[0]
print(f"\nX shape : {X.shape}  ← should be [8, 9, 9]")
print(f"E shape : {E.shape}  ← should be [8, 9, 9, 6]")

In [ ]:
print("=== Normalization stats in memory ===")
node_feat_names = ["x", "y", "vx", "vy", "speed", "accel", "heading"]
edge_feat_names = ["distance", "rel_speed", "drac", "heading_diff", "rel_vx", "rel_vy"]

print("\nNode stats:")
for i, name in enumerate(node_feat_names):
    print(f"  {name:10s}  mean={node_mean[i]:8.3f}  std={node_std[i]:8.3f}")

print("\nEdge stats:")
for i, name in enumerate(edge_feat_names):
    print(f"  {name:12s}  mean={edge_mean[i]:8.3f}  std={edge_std[i]:8.3f}")

# Verify normalization is applied in dataset
X, Avv, Avp, App, E, y = ds_train[0]
print(f"\nNode feature mean (continuous only, should be ~0): {X[:,:7].mean():.4f}")
print(f"Node feature std  (continuous only, should be ~1): {X[:,:7].std():.4f}")
print(f"Edge feature mean (should be ~0): {E.mean():.4f}")
print(f"Edge feature std  (should be ~1): {E.std():.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import time

# ── run_epoch with full metrics ────────────────────────────
def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    y_true, y_score = [], []

    with torch.set_grad_enabled(is_train):
        for batch in loader:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = model(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(y)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    preds    = [1 if p >= 0.5 else 0 for p in y_score]
    auc      = roc_auc_score(y_true, y_score)
    f1       = f1_score(y_true, preds,      zero_division=0)
    acc      = accuracy_score(y_true, preds)

    return avg_loss, auc, f1, acc


# ── Batch size test ────────────────────────────────────────
def quick_batch_test(batch_size, n_epochs=3):
    loader_tr = DataLoader(ds_train, batch_size=batch_size,
                           shuffle=True,  num_workers=0)
    loader_vl = DataLoader(ds_val,   batch_size=batch_size,
                           shuffle=False, num_workers=0)

    m   = SceneRiskModel().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(n_epochs):
        m.train()
        for batch in loader_tr:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = m(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

    m.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader_vl:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = m(X, Avv, Avp, App, E)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    return roc_auc_score(y_true, y_score)

print("=== Batch size sensitivity ===")
print(f"{'Batch':>6}  {'Val AUC (3 epochs)':>18}")
print("-" * 28)

for bs in [8, 16, 32, 64, 128]:
    auc = quick_batch_test(bs, n_epochs=3)
    print(f"{bs:>6}  {auc:>18.4f}")

In [ ]:
import time

BATCH_SIZE = 16
EPOCHS     = 100

loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

# Fresh model
model     = SceneRiskModel(hidden_dim=64, lstm_layers=2, dropout=0.3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

# Recompute pos_weight from current train labels
pos_weight = torch.tensor(
    [(train_labels == 0).sum() / (train_labels == 1).sum()],
    dtype=torch.float32
).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val_auc = 0.0
best_epoch   = 0
patience     = 5
patience_ctr = 0
history      = []

print(f"Batch size       : {BATCH_SIZE}")
print(f"Max epochs       : {EPOCHS}")
print(f"Early stopping   : patience={patience}")
print(f"pos_weight       : {pos_weight.item():.2f}")
print(f"Device           : {device}")
print(f"Parameters       : {sum(p.numel() for p in model.parameters()):,}")
print()
print(f"{'Ep':>3}  {'TrLoss':>7}  {'TrAUC':>6}  {'TrF1':>5}  {'TrAcc':>5}  "
      f"{'VaLoss':>7}  {'VaAUC':>6}  {'VaF1':>5}  {'VaAcc':>5}  "
      f"{'LR':>7}  {'Time':>5}")
print("-" * 85)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_auc, tr_f1, tr_acc = run_epoch(
        model, loader_train, criterion, optimizer, device
    )
    va_loss, va_auc, va_f1, va_acc = run_epoch(
        model, loader_val, criterion, None, device
    )

    scheduler.step(va_auc)
    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch"     : epoch,
        "train_loss": tr_loss, "train_auc": tr_auc,
        "train_f1"  : tr_f1,  "train_acc": tr_acc,
        "val_loss"  : va_loss, "val_auc"  : va_auc,
        "val_f1"    : va_f1,  "val_acc"  : va_acc
    })

    print(f"{epoch:>3}  {tr_loss:>7.4f}  {tr_auc:>6.4f}  {tr_f1:>5.4f}  "
          f"{tr_acc:>5.4f}  {va_loss:>7.4f}  {va_auc:>6.4f}  {va_f1:>5.4f}  "
          f"{va_acc:>5.4f}  {lr_now:>7.6f}  {elapsed:>4.1f}s")

    if va_auc > best_val_auc:
        best_val_auc = va_auc
        best_epoch   = epoch
        patience_ctr = 0
        torch.save(model.state_dict(),
                   checkpoint_path / "best_model.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nBest val AUC : {best_val_auc:.4f} at epoch {best_epoch}")

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix,
    classification_report, roc_curve,
    matthews_corrcoef, cohen_kappa_score,
    brier_score_loss
)

def evaluate_model(model, loader, device, threshold=0.5, split_name="Test"):
    model.eval()
    y_true, y_score = [], []

    with torch.no_grad():
        for batch in loader:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = model(X, Avv, Avp, App, E)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    y_true  = np.array(y_true)
    y_score = np.array(y_score)
    y_pred  = (y_score >= threshold).astype(int)

    # Core metrics
    auc_roc  = roc_auc_score(y_true, y_score)
    auc_pr   = average_precision_score(y_true, y_score)
    f1       = f1_score(y_true, y_pred,       zero_division=0)
    prec     = precision_score(y_true, y_pred, zero_division=0)
    rec      = recall_score(y_true, y_pred,    zero_division=0)
    acc      = accuracy_score(y_true, y_pred)
    mcc      = matthews_corrcoef(y_true, y_pred)
    kappa    = cohen_kappa_score(y_true, y_pred)
    brier    = brier_score_loss(y_true, y_score)
    cm       = confusion_matrix(y_true, y_pred)

    # Specificity
    tn, fp, fn, tp = cm.ravel()
    specificity    = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Sensitivity at fixed FAR
    fpr, tpr, _ = roc_curve(y_true, y_score)
    sens_5far   = float(tpr[np.searchsorted(fpr, 0.05)])
    sens_10far  = float(tpr[np.searchsorted(fpr, 0.10)])

    print(f"\n{'='*55}")
    print(f"  {split_name} Set Evaluation")
    print(f"{'='*55}")
    print(f"  AUC-ROC              : {auc_roc:.4f}")
    print(f"  AUC-PR               : {auc_pr:.4f}")
    print(f"  Accuracy             : {acc:.4f}")
    print(f"  F1 Score             : {f1:.4f}")
    print(f"  Precision            : {prec:.4f}")
    print(f"  Recall               : {rec:.4f}")
    print(f"  Specificity          : {specificity:.4f}")
    print(f"  MCC                  : {mcc:.4f}")
    print(f"  Cohen Kappa          : {kappa:.4f}")
    print(f"  Brier Score          : {brier:.4f}")
    print(f"  Sensitivity @ 5% FAR : {sens_5far:.4f}")
    print(f"  Sensitivity @10% FAR : {sens_10far:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"    TN={tn:,}  FP={fp:,}")
    print(f"    FN={fn:,}  TP={tp:,}")
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    return {
        "auc_roc"    : auc_roc,   "auc_pr"     : auc_pr,
        "accuracy"   : acc,       "f1"         : f1,
        "precision"  : prec,      "recall"     : rec,
        "specificity": specificity,"mcc"        : mcc,
        "kappa"      : kappa,     "brier"      : brier,
        "sens_5far"  : sens_5far, "sens_10far" : sens_10far,
        "y_true"     : y_true,    "y_score"    : y_score,
        "y_pred"     : y_pred,    "cm"         : cm
    }


# Load best model
model.load_state_dict(torch.load(checkpoint_path / "best_model.pt"))

# Evaluate on all splits
print("Loading best model from epoch 35...")
val_results  = evaluate_model(model, loader_val,   device, split_name="Validation")
test_results = evaluate_model(model, loader_test,  device, split_name="Test")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

history_df = pd.DataFrame(history)

# ── Top left: Loss curves ──
axes[0,0].plot(history_df["epoch"], history_df["train_loss"], label="Train")
axes[0,0].plot(history_df["epoch"], history_df["val_loss"],   label="Val")
axes[0,0].axvline(x=35, color="red", linestyle="--", alpha=0.5, label="Best epoch")
axes[0,0].set_xlabel("Epoch")
axes[0,0].set_ylabel("Loss")
axes[0,0].set_title("Training vs Validation Loss")
axes[0,0].legend()

# ── Top right: AUC curves ──
axes[0,1].plot(history_df["epoch"], history_df["train_auc"], label="Train AUC")
axes[0,1].plot(history_df["epoch"], history_df["val_auc"],   label="Val AUC")
axes[0,1].axvline(x=35, color="red", linestyle="--", alpha=0.5, label="Best epoch")
axes[0,1].set_xlabel("Epoch")
axes[0,1].set_ylabel("AUC-ROC")
axes[0,1].set_title("Training vs Validation AUC")
axes[0,1].legend()

# ── Bottom left: ROC curve ──
fpr_v, tpr_v, _ = roc_curve(val_results["y_true"],  val_results["y_score"])
fpr_t, tpr_t, _ = roc_curve(test_results["y_true"], test_results["y_score"])

axes[1,0].plot(fpr_v, tpr_v,
               label=f"Val  AUC={val_results['auc_roc']:.4f}")
axes[1,0].plot(fpr_t, tpr_t,
               label=f"Test AUC={test_results['auc_roc']:.4f}")
axes[1,0].plot([0,1],[0,1], "k--", alpha=0.3)
axes[1,0].axvline(x=0.05, color="red", linestyle="--",
                  alpha=0.4, label="5% FAR")
axes[1,0].axvline(x=0.10, color="orange", linestyle="--",
                  alpha=0.4, label="10% FAR")
axes[1,0].set_xlabel("False Positive Rate")
axes[1,0].set_ylabel("True Positive Rate")
axes[1,0].set_title("ROC Curve")
axes[1,0].legend()

# ── Bottom right: PR curve ──
prec_v, rec_v, _ = precision_recall_curve(
    val_results["y_true"],  val_results["y_score"]
)
prec_t, rec_t, _ = precision_recall_curve(
    test_results["y_true"], test_results["y_score"]
)

axes[1,1].plot(rec_v, prec_v,
               label=f"Val  AUC-PR={val_results['auc_pr']:.4f}")
axes[1,1].plot(rec_t, prec_t,
               label=f"Test AUC-PR={test_results['auc_pr']:.4f}")
axes[1,1].axhline(
    y=train_labels.mean(), color="k", linestyle="--",
    alpha=0.3, label="Baseline (conflict rate)"
)
axes[1,1].set_xlabel("Recall")
axes[1,1].set_ylabel("Precision")
axes[1,1].set_title("Precision-Recall Curve")
axes[1,1].legend()

plt.suptitle("SSM-Embedded HeteroGNN + LSTM — Training & Evaluation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(checkpoint_path / "training_evaluation.png", dpi=150,
            bbox_inches="tight")
plt.show()

print("Saved: training_evaluation.png")

In [ ]:
import pickle

# Save results
results = {
    "graph_model": {
        "val" : val_results,
        "test": test_results
    },
    "history"    : history,
    "best_epoch" : 35,
    "batch_size" : 16,
    "model_params": 82436
}

with open(checkpoint_path / "graph_model_results.pkl", "wb") as f:
    pickle.dump(results, f)

# Save history as CSV for easy reference
pd.DataFrame(history).to_csv(
    checkpoint_path / "training_history.csv", index=False
)

print("Saved graph_model_results.pkl")
print("Saved training_history.csv")

print("\n=== Graph Model Final Results Summary ===")
print(f"{'Metric':<25} {'Val':>8} {'Test':>8}")
print("-" * 43)
metrics = ["auc_roc","auc_pr","accuracy","f1",
           "precision","recall","specificity",
           "mcc","kappa","brier","sens_5far","sens_10far"]
for m in metrics:
    print(f"{m:<25} {val_results[m]:>8.4f} {test_results[m]:>8.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, results, split_name in zip(
    axes,
    [val_results, test_results],
    ["Validation", "Test"]
):
    cm = results["cm"]
    
    # Normalize for percentages
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    # Annotation: count + percentage
    annot = np.array([
        [f"{cm[i,j]:,}\n({cm_norm[i,j]:.1%})" 
         for j in range(2)] 
        for i in range(2)
    ])
    
    sns.heatmap(
        cm_norm,
        ax=ax,
        annot=annot,
        fmt="",
        cmap="Blues",
        xticklabels=["Pred Safe", "Pred Conflict"],
        yticklabels=["True Safe", "True Conflict"],
        vmin=0, vmax=1,
        linewidths=0.5,
        cbar_kws={"label": "Proportion"}
    )
    
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(
        f"{split_name} Set — Confusion Matrix\n"
        f"AUC={results['auc_roc']:.4f}  "
        f"F1={results['f1']:.4f}  "
        f"Acc={results['accuracy']:.4f}",
        fontsize=11
    )
    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")

plt.suptitle(
    "SSM-Embedded HeteroGNN + LSTM — Confusion Matrices",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "confusion_matrices.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("Saved: confusion_matrices.png")

# Also print raw numbers clearly
for results, split_name in zip(
    [val_results, test_results], ["Validation", "Test"]
):
    tn, fp, fn, tp = results["cm"].ravel()
    print(f"\n=== {split_name} ===")
    print(f"True Negatives  (Safe    → Safe)    : {tn:,}")
    print(f"False Positives (Safe    → Conflict): {fp:,}")
    print(f"False Negatives (Conflict → Safe)   : {fn:,}")
    print(f"True Positives  (Conflict → Conflict): {tp:,}")

In [ ]:
# Save results
results = {
    "graph_model": {
        "val" : val_results,
        "test": test_results
    },
    "history"     : history,
    "best_epoch"  : 35,
    "batch_size"  : 16,
    "model_params": 82436
}

with open(checkpoint_path / "graph_model_results.pkl", "wb") as f:
    pickle.dump(results, f)

pd.DataFrame(history).to_csv(
    checkpoint_path / "training_history.csv", index=False
)

print("Saved graph_model_results.pkl")
print("Saved training_history.csv")

print("\n=== Graph Model Final Results Summary ===")
print(f"{'Metric':<25} {'Val':>8} {'Test':>8}")
print("-" * 43)
metrics = ["auc_roc","auc_pr","accuracy","f1",
           "precision","recall","specificity",
           "mcc","kappa","brier","sens_5far","sens_10far"]
for m in metrics:
    print(f"{m:<25} {val_results[m]:>8.4f} {test_results[m]:>8.4f}")

#  ML models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix,
    classification_report, roc_curve,
    matthews_corrcoef, cohen_kappa_score,
    brier_score_loss
)

def extract_ml_features(sequences):
    """
    Flatten each 8-frame sequence into a fixed-size feature vector.
    For each sequence compute:
    - Node-level stats: mean/min/max/std of speed, accel per frame
    - Edge-level stats: mean/min/max/std of distance, rel_speed, drac, heading_diff
    - Temporal stats: trend (last - first) for speed and drac
    - Graph stats: num_nodes, num_edges per frame stats
    - Agent composition: fraction of vehicles vs pedestrians
    """
    all_feats = []

    for seq in sequences:
        feat = []

        # Per-frame node and edge stats
        frame_speeds, frame_accels          = [], []
        frame_distances, frame_rel_speeds   = [], []
        frame_dracs, frame_heading_diffs    = [], []
        frame_rel_vx, frame_rel_vy          = [], []
        frame_num_nodes, frame_num_edges    = [], []
        frame_veh_frac                      = []

        for g in seq:
            nodes = g["nodes"]
            edges = g["edges"]

            # Node stats
            speeds = [n["speed"] for n in nodes]
            accels = [n["accel"] for n in nodes]
            frame_speeds.extend(speeds)
            frame_accels.extend(accels)

            # Edge stats
            if edges:
                frame_distances.extend([e["distance"]     for e in edges])
                frame_rel_speeds.extend([e["rel_speed"]   for e in edges])
                frame_dracs.extend([e["drac"]             for e in edges])
                frame_heading_diffs.extend([e["heading_diff"] for e in edges])
                frame_rel_vx.extend([e["rel_vx"]         for e in edges])
                frame_rel_vy.extend([e["rel_vy"]         for e in edges])

            frame_num_nodes.append(len(nodes))
            frame_num_edges.append(len(edges))

            # Agent composition
            n_veh = sum(1 for n in nodes if n["agent_type"] == "vehicle")
            frame_veh_frac.append(n_veh / max(len(nodes), 1))

        # Aggregate stats helper
        def stats(arr):
            if len(arr) == 0:
                return [0.0, 0.0, 0.0, 0.0]
            a = np.array(arr, dtype=np.float32)
            return [a.mean(), a.min(), a.max(), a.std()]

        # Node features
        feat.extend(stats(frame_speeds))         # 4
        feat.extend(stats(frame_accels))         # 4

        # Edge features
        feat.extend(stats(frame_distances))      # 4
        feat.extend(stats(frame_rel_speeds))     # 4
        feat.extend(stats(frame_dracs))          # 4
        feat.extend(stats(frame_heading_diffs))  # 4
        feat.extend(stats(frame_rel_vx))         # 4
        feat.extend(stats(frame_rel_vy))         # 4

        # Graph structure
        feat.extend(stats(frame_num_nodes))      # 4
        feat.extend(stats(frame_num_edges))      # 4

        # Agent composition
        feat.extend(stats(frame_veh_frac))       # 4

        # Temporal trend: last frame - first frame
        first_g = seq[0]
        last_g  = seq[-1]

        first_speed = np.mean([n["speed"] for n in first_g["nodes"]])
        last_speed  = np.mean([n["speed"] for n in last_g["nodes"]])
        feat.append(last_speed - first_speed)    # 1

        first_drac = np.mean([e["drac"] for e in first_g["edges"]]) if first_g["edges"] else 0.0
        last_drac  = np.mean([e["drac"] for e in last_g["edges"]])  if last_g["edges"]  else 0.0
        feat.append(last_drac - first_drac)      # 1

        first_dist = np.mean([e["distance"] for e in first_g["edges"]]) if first_g["edges"] else 0.0
        last_dist  = np.mean([e["distance"] for e in last_g["edges"]])  if last_g["edges"]  else 0.0
        feat.append(last_dist - first_dist)      # 1

        all_feats.append(feat)

    return np.array(all_feats, dtype=np.float32)


print("Extracting ML features...")
X_train_ml = extract_ml_features(train_seqs)
X_val_ml   = extract_ml_features(val_seqs)
X_test_ml  = extract_ml_features(test_seqs)

y_train_ml = train_labels.astype(int)
y_val_ml   = val_labels.astype(int)
y_test_ml  = test_labels.astype(int)

print(f"Train features : {X_train_ml.shape}")
print(f"Val features   : {X_val_ml.shape}")
print(f"Test features  : {X_test_ml.shape}")

print(f"\nFeature vector size: {X_train_ml.shape[1]}")
print(f"Any NaN in train   : {np.isnan(X_train_ml).sum()}")
print(f"Any NaN in val     : {np.isnan(X_val_ml).sum()}")
print(f"Any NaN in test    : {np.isnan(X_test_ml).sum()}")

# Scale features for SVM baseline
scaler    = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_ml)
X_val_sc   = scaler.transform(X_val_ml)
X_test_sc  = scaler.transform(X_test_ml)

print("\nFeature scaling done (for SVM baseline)")

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb
import time

# 3-fold stratified CV on train set only
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Class imbalance ratio for tree models
scale_pos = int((y_train_ml == 0).sum() / (y_train_ml == 1).sum())
print(f"scale_pos_weight: {scale_pos}")

# ── 1. Logistic Regression ─────────────────────────────────
print("\n[1/5] Logistic Regression grid search...")
t0 = time.time()

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, solver="saga",
                       class_weight="balanced", random_state=42),
    param_grid={"C": [0.01, 0.1, 1.0, 10.0],
                "penalty": ["l1", "l2"]},
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0
)
lr_grid.fit(X_train_sc, y_train_ml)
print(f"  Best params : {lr_grid.best_params_}")
print(f"  Best CV AUC : {lr_grid.best_score_:.4f}")
print(f"  Time        : {time.time()-t0:.1f}s")

# ── 2. Random Forest ───────────────────────────────────────
print("\n[2/5] Random Forest grid search...")
t0 = time.time()

rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42,
                           n_jobs=-1),
    param_grid={"n_estimators" : [100, 300, 500],
                "max_depth"    : [5, 10, 20, None],
                "min_samples_leaf": [1, 5, 10]},
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0
)
rf_grid.fit(X_train_ml, y_train_ml)
print(f"  Best params : {rf_grid.best_params_}")
print(f"  Best CV AUC : {rf_grid.best_score_:.4f}")
print(f"  Time        : {time.time()-t0:.1f}s")

# ── 3. XGBoost ─────────────────────────────────────────────
print("\n[3/5] XGBoost grid search...")
t0 = time.time()

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(scale_pos_weight=scale_pos,
                      use_label_encoder=False,
                      eval_metric="auc",
                      random_state=42, n_jobs=-1),
    param_grid={"n_estimators"  : [100, 300, 500],
                "max_depth"     : [3, 5, 7],
                "learning_rate" : [0.01, 0.05, 0.1],
                "subsample"     : [0.7, 0.9]},
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0
)
xgb_grid.fit(X_train_ml, y_train_ml)
print(f"  Best params : {xgb_grid.best_params_}")
print(f"  Best CV AUC : {xgb_grid.best_score_:.4f}")
print(f"  Time        : {time.time()-t0:.1f}s")

# ── 4. LightGBM ────────────────────────────────────────────
print("\n[4/5] LightGBM grid search...")
t0 = time.time()

lgb_grid = GridSearchCV(
    lgb.LGBMClassifier(class_weight="balanced",
                       random_state=42, n_jobs=-1, verbose=-1),
    param_grid={"n_estimators"  : [100, 300, 500],
                "max_depth"     : [5, 10, 20],
                "learning_rate" : [0.01, 0.05, 0.1],
                "num_leaves"    : [31, 63, 127]},
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0
)
lgb_grid.fit(X_train_ml, y_train_ml)
print(f"  Best params : {lgb_grid.best_params_}")
print(f"  Best CV AUC : {lgb_grid.best_score_:.4f}")
print(f"  Time        : {time.time()-t0:.1f}s")

# ── 5. SVM ─────────────────────────────────────────────────
print("\n[5/5] SVM grid search...")
t0 = time.time()

svm_grid = GridSearchCV(
    SVC(kernel="rbf", class_weight="balanced",
        probability=True, random_state=42),
    param_grid={"C"    : [0.1, 1.0, 10.0],
                "gamma": ["scale", "auto"]},
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0
)
svm_grid.fit(X_train_sc, y_train_ml)
print(f"  Best params : {svm_grid.best_params_}")
print(f"  Best CV AUC : {svm_grid.best_score_:.4f}")
print(f"  Time        : {time.time()-t0:.1f}s")

print("\n=== Grid Search Summary ===")
print(f"{'Model':<20} {'Best CV AUC':>11}  Best Params")
print("-" * 70)
for name, grid in [("Logistic Reg", lr_grid),
                   ("Random Forest", rf_grid),
                   ("XGBoost",       xgb_grid),
                   ("LightGBM",      lgb_grid),
                   ("SVM",           svm_grid)]:
    print(f"{name:<20} {grid.best_score_:>11.4f}  {grid.best_params_}")

In [ ]:
def evaluate_ml(model, X_val, y_val, X_test, y_test,
                model_name, scaled=False,
                X_val_sc=None, X_test_sc=None):

    results = {}
    for split, X_use, y_use in [
        ("val",  X_val_sc  if scaled else X_val,  y_val),
        ("test", X_test_sc if scaled else X_test, y_test)
    ]:
        y_score = model.predict_proba(X_use)[:, 1]
        y_pred  = (y_score >= 0.5).astype(int)
        cm      = confusion_matrix(y_use, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fpr, tpr, _    = roc_curve(y_use, y_score)

        results[split] = {
            "auc_roc"    : roc_auc_score(y_use, y_score),
            "auc_pr"     : average_precision_score(y_use, y_score),
            "accuracy"   : accuracy_score(y_use, y_pred),
            "f1"         : f1_score(y_use, y_pred,        zero_division=0),
            "precision"  : precision_score(y_use, y_pred, zero_division=0),
            "recall"     : recall_score(y_use, y_pred,    zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
            "mcc"        : matthews_corrcoef(y_use, y_pred),
            "kappa"      : cohen_kappa_score(y_use, y_pred),
            "brier"      : brier_score_loss(y_use, y_score),
            "sens_5far"  : float(tpr[np.searchsorted(fpr, 0.05)]),
            "sens_10far" : float(tpr[np.searchsorted(fpr, 0.10)]),
            "y_true"     : y_use,
            "y_score"    : y_score,
            "y_pred"     : y_pred,
            "cm"         : cm
        }

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  {'Metric':<25} {'Val':>8} {'Test':>8}")
    print(f"  {'-'*43}")
    metrics = ["auc_roc","auc_pr","accuracy","f1",
               "precision","recall","specificity",
               "mcc","kappa","brier","sens_5far","sens_10far"]
    for m in metrics:
        print(f"  {m:<25} {results['val'][m]:>8.4f} {results['test'][m]:>8.4f}")

    return results


# Evaluate all models
print("Evaluating all ML models on val and test sets...")

ml_results = {}

ml_results["Logistic Regression"] = evaluate_ml(
    lr_grid.best_estimator_,
    X_val_ml, y_val_ml, X_test_ml, y_test_ml,
    "Logistic Regression", scaled=True,
    X_val_sc=X_val_sc, X_test_sc=X_test_sc
)

ml_results["Random Forest"] = evaluate_ml(
    rf_grid.best_estimator_,
    X_val_ml, y_val_ml, X_test_ml, y_test_ml,
    "Random Forest"
)

ml_results["XGBoost"] = evaluate_ml(
    xgb_grid.best_estimator_,
    X_val_ml, y_val_ml, X_test_ml, y_test_ml,
    "XGBoost"
)

ml_results["LightGBM"] = evaluate_ml(
    lgb_grid.best_estimator_,
    X_val_ml, y_val_ml, X_test_ml, y_test_ml,
    "LightGBM"
)

ml_results["SVM"] = evaluate_ml(
    svm_grid.best_estimator_,
    X_val_ml, y_val_ml, X_test_ml, y_test_ml,
    "SVM", scaled=True,
    X_val_sc=X_val_sc, X_test_sc=X_test_sc
)

In [ ]:
# ── Full comparison table ──────────────────────────────────
print("\n" + "="*95)
print("COMPLETE MODEL COMPARISON — TEST SET")
print("="*95)
print(f"{'Model':<25} {'AUC-ROC':>7} {'AUC-PR':>7} {'Acc':>6} {'F1':>6} "
      f"{'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6} {'Kappa':>6} "
      f"{'Brier':>6} {'S@5%':>6} {'S@10%':>6}")
print("-"*95)

# ML models
for name, res in ml_results.items():
    r = res["test"]
    print(f"{name:<25} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
          f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
          f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
          f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
          f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

# Graph model
r = test_results
print(f"{'HeteroGNN+LSTM':<25} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
      f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
      f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
      f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
      f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

print("="*95)

# Save ML results
with open(checkpoint_path / "ml_results.pkl", "wb") as f:
    pickle.dump(ml_results, f)
print("\nSaved ml_results.pkl")

In [ ]:
import shap
import matplotlib.pyplot as plt

# Feature names for SHAP
feature_names = []
for base in ["speed", "accel"]:
    for stat in ["mean", "min", "max", "std"]:
        feature_names.append(f"node_{base}_{stat}")

for base in ["distance", "rel_speed", "drac", "heading_diff", "rel_vx", "rel_vy"]:
    for stat in ["mean", "min", "max", "std"]:
        feature_names.append(f"edge_{base}_{stat}")

for base in ["num_nodes", "num_edges", "veh_frac"]:
    for stat in ["mean", "min", "max", "std"]:
        feature_names.append(f"graph_{base}_{stat}")

feature_names.extend([
    "trend_speed", "trend_drac", "trend_distance"
])

print(f"Feature names count: {len(feature_names)}")
assert len(feature_names) == X_train_ml.shape[1], \
    f"Mismatch: {len(feature_names)} names vs {X_train_ml.shape[1]} features"

# SHAP on XGBoost — best ML model
print("\nComputing SHAP values on XGBoost (test set)...")
explainer   = shap.TreeExplainer(xgb_grid.best_estimator_)
shap_values = explainer.shap_values(X_test_ml)

print(f"SHAP values shape: {shap_values.shape}")

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_test_ml,
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title("SHAP Feature Importance — XGBoost (Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(checkpoint_path / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: shap_summary.png")

# Bar plot — mean absolute SHAP
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_test_ml,
    feature_names=feature_names,
    max_display=20,
    plot_type="bar",
    show=False
)
plt.title("SHAP Mean |Value| — XGBoost Feature Importance", fontsize=13)
plt.tight_layout()
plt.savefig(checkpoint_path / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: shap_bar.png")

In [ ]:
import pickle

# Save SHAP values
shap_data = {
    "shap_values"  : shap_values,
    "feature_names": feature_names,
    "X_test"       : X_test_ml
}
with open(checkpoint_path / "shap_data.pkl", "wb") as f:
    pickle.dump(shap_data, f)

print("Saved shap_data.pkl")
print("\n=== Top 10 Features by Mean |SHAP| ===")
mean_shap = np.abs(shap_values).mean(axis=0)
top10_idx = np.argsort(mean_shap)[::-1][:10]
for rank, idx in enumerate(top10_idx, 1):
    print(f"  {rank:>2}. {feature_names[idx]:<30} {mean_shap[idx]:.4f}")

# DL

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Flat sequence dataset ──────────────────────────────────
# Per frame: concatenate all node and edge features into fixed vector
# Max nodes = 9, max edges = 36 (9*8/2 padded)
# Node features per agent: 7 continuous (normalized)
# Edge features per pair: 6 (normalized)
# Frame vector = mean/max pooled node features + mean/max pooled edge features

class FlatSequenceDataset(Dataset):
    """
    Flattens each scene graph frame into a fixed-size vector.
    Uses pooled node and edge features — no graph structure.
    This is the DL baseline tier.
    """
    def __init__(self, sequences, labels,
                 node_mean, node_std,
                 edge_mean, edge_std):
        self.sequences = sequences
        self.labels    = labels
        self.node_mean = node_mean
        self.node_std  = node_std
        self.edge_mean = edge_mean
        self.edge_std  = edge_std

    def __len__(self):
        return len(self.sequences)

    def frame_to_vector(self, graph):
        nodes = graph["nodes"]
        edges = graph["edges"]

        # Normalize node features
        node_feats = []
        for n in nodes:
            raw = np.array([
                n["x"], n["y"], n["vx"], n["vy"],
                n["speed"], n["accel"], n["heading"]
            ], dtype=np.float32)
            node_feats.append((raw - self.node_mean) / self.node_std)

        node_arr = np.array(node_feats, dtype=np.float32)  # [N, 7]

        # Pool node features: mean + max → [14]
        node_mean_pool = node_arr.mean(axis=0)
        node_max_pool  = node_arr.max(axis=0)
        node_vec       = np.concatenate([node_mean_pool, node_max_pool])  # [14]

        # Normalize edge features
        if edges:
            edge_feats = []
            for e in edges:
                raw = np.array([
                    e["distance"], e["rel_speed"], e["drac"],
                    e["heading_diff"], e["rel_vx"], e["rel_vy"]
                ], dtype=np.float32)
                edge_feats.append((raw - self.edge_mean) / self.edge_std)

            edge_arr = np.array(edge_feats, dtype=np.float32)  # [E, 6]
            edge_mean_pool = edge_arr.mean(axis=0)
            edge_max_pool  = edge_arr.max(axis=0)
            edge_min_pool  = edge_arr.min(axis=0)
        else:
            edge_mean_pool = np.zeros(6, dtype=np.float32)
            edge_max_pool  = np.zeros(6, dtype=np.float32)
            edge_min_pool  = np.zeros(6, dtype=np.float32)

        edge_vec = np.concatenate([
            edge_mean_pool, edge_max_pool, edge_min_pool
        ])  # [18]

        # Graph-level scalar features
        n_nodes  = len(nodes)
        n_edges  = len(edges)
        veh_frac = sum(1 for n in nodes
                       if n["agent_type"] == "vehicle") / max(n_nodes, 1)
        graph_vec = np.array([n_nodes, n_edges, veh_frac],
                             dtype=np.float32)  # [3]

        # Full frame vector [14 + 18 + 3 = 35]
        return np.concatenate([node_vec, edge_vec, graph_vec])

    def __getitem__(self, idx):
        seq   = self.sequences[idx]
        label = self.labels[idx]

        frames = np.stack([self.frame_to_vector(g) for g in seq])  # [T, 35]

        return (
            torch.tensor(frames, dtype=torch.float32),
            torch.tensor(label,  dtype=torch.float32)
        )


# Instantiate flat datasets
ds_train_flat = FlatSequenceDataset(
    train_seqs, train_labels,
    node_mean, node_std, edge_mean, edge_std
)
ds_val_flat = FlatSequenceDataset(
    val_seqs, val_labels,
    node_mean, node_std, edge_mean, edge_std
)
ds_test_flat = FlatSequenceDataset(
    test_seqs, test_labels,
    node_mean, node_std, edge_mean, edge_std
)

# Sanity check
x, y = ds_train_flat[0]
print(f"Flat frame vector shape : {x.shape}  ← [T=8, F=35]")
print(f"Label                   : {y}")
print(f"\nTrain: {len(ds_train_flat):,}")
print(f"Val  : {len(ds_val_flat):,}")
print(f"Test : {len(ds_test_flat):,}")

FLAT_DIM = x.shape[1]
print(f"\nInput feature dim per frame: {FLAT_DIM}")

In [ ]:
# ── DL Model Definitions ───────────────────────────────────

# 1. Bidirectional LSTM
class BiLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        out, _  = self.lstm(x)
        final   = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# 2. GRU
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru     = nn.GRU(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _  = self.gru(x)
        final   = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# 3. TCN
class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels,
                 kernel_size, dilation, dropout):
        super().__init__()
        padding      = (kernel_size - 1) * dilation
        self.conv1   = nn.Conv1d(in_channels, out_channels,
                                 kernel_size, dilation=dilation,
                                 padding=padding)
        self.conv2   = nn.Conv1d(out_channels, out_channels,
                                 kernel_size, dilation=dilation,
                                 padding=padding)
        self.norm1   = nn.BatchNorm1d(out_channels)
        self.norm2   = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.relu    = nn.ReLU()
        self.residual = nn.Conv1d(in_channels, out_channels, 1) \
                        if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        out = self.conv1(x)
        out = out[:, :, :x.shape[2]]
        out = self.norm1(self.relu(out))
        out = self.dropout(out)
        out = self.conv2(out)
        out = out[:, :, :x.shape[2]]
        out = self.norm2(self.relu(out))
        out = self.dropout(out)
        return self.relu(out + res)


class TCNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64,
                 num_layers=4, kernel_size=3, dropout=0.3):
        super().__init__()
        layers = []
        in_ch  = input_dim
        for i in range(num_layers):
            dilation = 2 ** i
            layers.append(TCNBlock(in_ch, hidden_dim,
                                   kernel_size, dilation, dropout))
            in_ch = hidden_dim
        self.tcn     = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x   = x.permute(0, 2, 1)
        out = self.tcn(x)
        out = out.mean(dim=-1)
        out = self.dropout(out)
        return self.fc(out).squeeze(-1)


# 4. Transformer
class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64,
                 num_heads=4, num_layers=2,
                 dropout=0.3, max_seq_len=8):
        super().__init__()
        self.input_proj  = nn.Linear(input_dim, hidden_dim)
        self.pos_enc     = nn.Embedding(max_seq_len, hidden_dim)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        B, T, _ = x.shape
        x        = self.input_proj(x)
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        x        = x + self.pos_enc(positions)
        out      = self.transformer(x)
        final    = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# Sanity check
x_dummy = torch.randn(2, 8, FLAT_DIM).to(device)

for ModelClass, name, kwargs in [
    (BiLSTMModel,      "BiLSTM",      {"input_dim": FLAT_DIM}),
    (GRUModel,         "GRU",         {"input_dim": FLAT_DIM}),
    (TCNModel,         "TCN",         {"input_dim": FLAT_DIM}),
    (TransformerModel, "Transformer", {"input_dim": FLAT_DIM}),
]:
    m      = ModelClass(**kwargs).to(device)
    out    = m(x_dummy)
    params = sum(p.numel() for p in m.parameters())
    print(f"{name:<15} output: {out.shape}  params: {params:,}")

In [ ]:
import itertools
import time

def quick_dl_test(ModelClass, model_kwargs, n_epochs=5,
                  batch_size=16, device=device):
    """Train for n_epochs and return val AUC."""
    loader_tr = DataLoader(ds_train_flat, batch_size=batch_size,
                           shuffle=True,  num_workers=0)
    loader_vl = DataLoader(ds_val_flat,   batch_size=batch_size,
                           shuffle=False, num_workers=0)

    m   = ModelClass(**model_kwargs).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)

    # Training
    m.train()
    for epoch in range(n_epochs):
        for batch in loader_tr:
            x, y = [b.to(device) for b in batch]
            logits = m(x)
            loss   = criterion(logits, y)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

    # Validation
    m.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader_vl:
            x, y = [b.to(device) for b in batch]
            logits = m(x)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    return roc_auc_score(y_true, y_score), sum(p.numel() for p in m.parameters())


# Search spaces
hidden_dims = [32, 64, 128]
num_layers  = [1, 2, 3]

dl_best_configs = {}

for ModelClass, name, extra_kwargs in [
    (BiLSTMModel,      "BiLSTM",      {}),
    (GRUModel,         "GRU",         {}),
    (TCNModel,         "TCN",         {"kernel_size": 3}),
    (TransformerModel, "Transformer", {"num_heads": 4, "max_seq_len": 8}),
]:
    print(f"\n[{name}] Grid search (hidden_dim x num_layers)...")
    print(f"  {'hidden':>6}  {'layers':>6}  {'val_auc':>8}  {'params':>8}  {'time':>6}")
    print(f"  {'-'*45}")

    best_auc    = 0.0
    best_config = {}

    for hd, nl in itertools.product(hidden_dims, num_layers):
        t0 = time.time()

        kwargs = {"input_dim": FLAT_DIM, "hidden_dim": hd,
                  "num_layers": nl, "dropout": 0.3, **extra_kwargs}

        # Transformer num_heads must divide hidden_dim
        if name == "Transformer" and hd % 4 != 0:
            continue

        auc, params = quick_dl_test(ModelClass, kwargs, n_epochs=5)
        elapsed     = time.time() - t0

        marker = " ←" if auc > best_auc else ""
        print(f"  {hd:>6}  {nl:>6}  {auc:>8.4f}  {params:>8,}  {elapsed:>5.1f}s{marker}")

        if auc > best_auc:
            best_auc    = auc
            best_config = kwargs

    dl_best_configs[name] = {
        "ModelClass": ModelClass,
        "kwargs"    : best_config,
        "best_auc"  : best_auc
    }
    print(f"\n  Best config: {best_config}")
    print(f"  Best val AUC (5 epochs): {best_auc:.4f}")

print("\n=== DL Grid Search Summary ===")
print(f"{'Model':<15} {'Best 5-ep AUC':>13}  Best Config")
print("-" * 70)
for name, info in dl_best_configs.items():
    cfg = {k:v for k,v in info["kwargs"].items()
           if k not in ["input_dim","dropout"]}
    print(f"{name:<15} {info['best_auc']:>13.4f}  {cfg}")

In [ ]:
def train_dl_model(ModelClass, model_kwargs, model_name,
                   n_epochs=50, patience=5, batch_size=16):

    loader_tr = DataLoader(ds_train_flat, batch_size=batch_size,
                           shuffle=True,  num_workers=0)
    loader_vl = DataLoader(ds_val_flat,   batch_size=batch_size,
                           shuffle=False, num_workers=0)

    model     = ModelClass(**model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    best_val_auc  = 0.0
    best_epoch    = 0
    patience_ctr  = 0
    history       = []

    print(f"\n{'='*70}")
    print(f"Training {model_name}")
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*70}")
    print(f"{'Ep':>3}  {'TrLoss':>7}  {'TrAUC':>6}  {'TrF1':>5}  {'TrAcc':>5}  "
          f"{'VaLoss':>7}  {'VaAUC':>6}  {'VaF1':>5}  {'VaAcc':>5}  {'Time':>5}")
    print("-" * 70)

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()

        # Train
        model.train()
        total_loss = 0.0
        y_true_tr, y_score_tr = [], []

        for batch in loader_tr:
            x, y = [b.to(device) for b in batch]
            logits = model(x)
            loss   = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss   += loss.item() * len(y)
            y_true_tr.extend(y.cpu().numpy())
            y_score_tr.extend(torch.sigmoid(logits).detach().cpu().numpy())

        tr_loss = total_loss / len(ds_train_flat)
        tr_preds = [1 if p >= 0.5 else 0 for p in y_score_tr]
        tr_auc   = roc_auc_score(y_true_tr, y_score_tr)
        tr_f1    = f1_score(y_true_tr, tr_preds, zero_division=0)
        tr_acc   = accuracy_score(y_true_tr, tr_preds)

        # Validate
        model.eval()
        total_loss = 0.0
        y_true_vl, y_score_vl = [], []

        with torch.no_grad():
            for batch in loader_vl:
                x, y = [b.to(device) for b in batch]
                logits = model(x)
                loss   = criterion(logits, y)
                total_loss   += loss.item() * len(y)
                y_true_vl.extend(y.cpu().numpy())
                y_score_vl.extend(torch.sigmoid(logits).cpu().numpy())

        va_loss  = total_loss / len(ds_val_flat)
        va_preds = [1 if p >= 0.5 else 0 for p in y_score_vl]
        va_auc   = roc_auc_score(y_true_vl, y_score_vl)
        va_f1    = f1_score(y_true_vl, va_preds, zero_division=0)
        va_acc   = accuracy_score(y_true_vl, va_preds)

        scheduler.step(va_auc)
        elapsed = time.time() - t0

        history.append({
            "epoch"     : epoch,
            "train_loss": tr_loss, "train_auc": tr_auc,
            "train_f1"  : tr_f1,  "train_acc": tr_acc,
            "val_loss"  : va_loss, "val_auc"  : va_auc,
            "val_f1"    : va_f1,  "val_acc"  : va_acc
        })

        print(f"{epoch:>3}  {tr_loss:>7.4f}  {tr_auc:>6.4f}  {tr_f1:>5.4f}  "
              f"{tr_acc:>5.4f}  {va_loss:>7.4f}  {va_auc:>6.4f}  {va_f1:>5.4f}  "
              f"{va_acc:>5.4f}  {elapsed:>4.1f}s")

        if va_auc > best_val_auc:
            best_val_auc = va_auc
            best_epoch   = epoch
            patience_ctr = 0
            torch.save(model.state_dict(),
                       checkpoint_path / f"best_{model_name}.pt")
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"\nEarly stopping at epoch {epoch}.")
                break

    print(f"\nBest val AUC: {best_val_auc:.4f} at epoch {best_epoch}")
    return model, history, best_val_auc, best_epoch


# Train all four models
dl_histories = {}
dl_best_aucs = {}

for name, info in dl_best_configs.items():
    model_trained, hist, best_auc, best_ep = train_dl_model(
        info["ModelClass"],
        info["kwargs"],
        name
    )
    dl_histories[name] = hist
    dl_best_aucs[name] = best_auc

In [ ]:
def evaluate_dl_model(ModelClass, model_kwargs, model_name,
                      loader_val, loader_test, device):
    # Load best checkpoint
    model = ModelClass(**model_kwargs).to(device)
    model.load_state_dict(
        torch.load(checkpoint_path / f"best_{model_name}.pt")
    )
    model.eval()

    results = {}
    for split_name, loader in [("val", loader_val), ("test", loader_test)]:
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader:
                x, y = [b.to(device) for b in batch]
                logits = model(x)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        y_true  = np.array(y_true)
        y_score = np.array(y_score)
        y_pred  = (y_score >= 0.5).astype(int)
        cm      = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fpr, tpr, _    = roc_curve(y_true, y_score)

        results[split_name] = {
            "auc_roc"    : roc_auc_score(y_true, y_score),
            "auc_pr"     : average_precision_score(y_true, y_score),
            "accuracy"   : accuracy_score(y_true, y_pred),
            "f1"         : f1_score(y_true, y_pred,        zero_division=0),
            "precision"  : precision_score(y_true, y_pred, zero_division=0),
            "recall"     : recall_score(y_true, y_pred,    zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
            "mcc"        : matthews_corrcoef(y_true, y_pred),
            "kappa"      : cohen_kappa_score(y_true, y_pred),
            "brier"      : brier_score_loss(y_true, y_score),
            "sens_5far"  : float(tpr[np.searchsorted(fpr, 0.05)]),
            "sens_10far" : float(tpr[np.searchsorted(fpr, 0.10)]),
            "y_true"     : y_true,
            "y_score"    : y_score,
            "y_pred"     : y_pred,
            "cm"         : cm
        }

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  {'Metric':<25} {'Val':>8} {'Test':>8}")
    print(f"  {'-'*43}")
    metrics = ["auc_roc","auc_pr","accuracy","f1",
               "precision","recall","specificity",
               "mcc","kappa","brier","sens_5far","sens_10far"]
    for m in metrics:
        print(f"  {m:<25} {results['val'][m]:>8.4f} "
              f"{results['test'][m]:>8.4f}")

    return results


# DataLoaders for flat datasets
loader_val_flat  = DataLoader(ds_val_flat,  batch_size=16,
                              shuffle=False, num_workers=0)
loader_test_flat = DataLoader(ds_test_flat, batch_size=16,
                              shuffle=False, num_workers=0)

dl_results = {}
for name, info in dl_best_configs.items():
    dl_results[name] = evaluate_dl_model(
        info["ModelClass"],
        info["kwargs"],
        name,
        loader_val_flat,
        loader_test_flat,
        device
    )

# Save DL results
with open(checkpoint_path / "dl_results.pkl", "wb") as f:
    pickle.dump(dl_results, f)
print("\nSaved dl_results.pkl")

# Full comparison table
print("\n" + "="*100)
print("COMPLETE MODEL COMPARISON — TEST SET (ALL TIERS)")
print("="*100)
print(f"{'Model':<25} {'AUC-ROC':>7} {'AUC-PR':>7} {'Acc':>6} {'F1':>6} "
      f"{'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6} {'Kappa':>6} "
      f"{'Brier':>6} {'S@5%':>6} {'S@10%':>6}")
print("-"*100)

print("--- ML Tier ---")
for name, res in ml_results.items():
    r = res["test"]
    print(f"{name:<25} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
          f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
          f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
          f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
          f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

print("--- DL Tier ---")
for name, res in dl_results.items():
    r = res["test"]
    print(f"{name:<25} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
          f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
          f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
          f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
          f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

print("--- Graph Tier ---")
r = test_results
print(f"{'HeteroGNN+LSTM':<25} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
      f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
      f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
      f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
      f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")
print("="*100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compile all results
all_models = {
    # ML Tier
    "LR"        : ml_results["Logistic Regression"]["test"],
    "RF"        : ml_results["Random Forest"]["test"],
    "XGBoost"   : ml_results["XGBoost"]["test"],
    "LightGBM"  : ml_results["LightGBM"]["test"],
    "SVM"       : ml_results["SVM"]["test"],
    # DL Tier
    "BiLSTM"    : dl_results["BiLSTM"]["test"],
    "GRU"       : dl_results["GRU"]["test"],
    "TCN"       : dl_results["TCN"]["test"],
    "Transformer": dl_results["Transformer"]["test"],
    # Graph Tier
    "HeteroGNN+LSTM": test_results
}

tier_colors = {
    "LR": "#AED6F1", "RF": "#AED6F1",
    "XGBoost": "#AED6F1", "LightGBM": "#AED6F1", "SVM": "#AED6F1",
    "BiLSTM": "#A9DFBF", "GRU": "#A9DFBF",
    "TCN": "#A9DFBF", "Transformer": "#A9DFBF",
    "HeteroGNN+LSTM": "#E59866"
}

model_names = list(all_models.keys())
colors      = [tier_colors[m] for m in model_names]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

metrics_to_plot = [
    ("auc_roc",    "AUC-ROC"),
    ("auc_pr",     "AUC-PR"),
    ("f1",         "F1 Score"),
    ("mcc",        "MCC"),
    ("sens_5far",  "Sensitivity @ 5% FAR"),
    ("sens_10far", "Sensitivity @ 10% FAR"),
]

for ax, (metric, title) in zip(axes, metrics_to_plot):
    values = [all_models[m][metric] for m in model_names]
    bars   = ax.barh(model_names, values, color=colors, edgecolor="white")

    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=7.5)

    ax.set_xlabel(title)
    ax.set_title(title, fontweight="bold")
    ax.set_xlim(min(values) * 0.97, max(values) * 1.02)

    # Add tier separator lines
    ax.axhline(y=4.5, color="gray", linestyle="--", alpha=0.5)
    ax.axhline(y=8.5, color="gray", linestyle="--", alpha=0.5)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#AED6F1", label="ML Tier"),
    Patch(facecolor="#A9DFBF", label="DL Tier"),
    Patch(facecolor="#E59866", label="Graph Tier"),
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=3, fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.suptitle("Three-Tier Model Comparison — Test Set Performance",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(checkpoint_path / "three_tier_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: three_tier_comparison.png")

In [ ]:
# ── Ablation Group 1: Edge Feature Subsets ─────────────────

class AblationEdgeDataset(Dataset):
    """
    Same as SceneGraphDataset but with configurable edge feature subset.
    edge_cols: list of edge feature names to include
    """
    EDGE_FEATURE_MAP = {
        "distance"    : 0,
        "rel_speed"   : 1,
        "drac"        : 2,
        "heading_diff": 3,
        "rel_vx"      : 4,
        "rel_vy"      : 5
    }

    def __init__(self, sequences, labels,
                 node_mean, node_std,
                 edge_mean, edge_std,
                 edge_cols,           # list of edge feature names to use
                 max_nodes=MAX_NODES):
        self.sequences = sequences
        self.labels    = labels
        self.max_nodes = max_nodes
        self.node_mean = node_mean
        self.node_std  = node_std
        self.edge_mean = edge_mean
        self.edge_std  = edge_std
        self.edge_cols = edge_cols
        self.edge_idx  = [self.EDGE_FEATURE_MAP[c] for c in edge_cols]
        self.edge_dim  = len(edge_cols)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq   = self.sequences[idx]
        label = self.labels[idx]

        X_seq, Avv_seq, Avp_seq, App_seq, E_seq = [], [], [], [], []

        for g in seq:
            nodes  = g["nodes"]
            edges  = g["edges"]
            N      = len(nodes)
            id2idx = {n["track_id"]: i for i, n in enumerate(nodes)}

            # Node features
            X = []
            for n in nodes:
                raw    = np.array([
                    n["x"], n["y"], n["vx"], n["vy"],
                    n["speed"], n["accel"], n["heading"]
                ], dtype=np.float32)
                normed = (raw - self.node_mean) / self.node_std
                is_veh = 1.0 if n["agent_type"] == "vehicle" else 0.0
                is_ped = 1.0 if n["agent_type"] == "pedestrian" else 0.0
                X.append(np.append(normed, [is_veh, is_ped]))
            X = np.array(X, dtype=np.float32)

            M        = self.max_nodes
            X_pad    = np.zeros((M, NODE_DIM),         dtype=np.float32)
            A_vv_pad = np.zeros((M, M),                dtype=np.float32)
            A_vp_pad = np.zeros((M, M),                dtype=np.float32)
            A_pp_pad = np.zeros((M, M),                dtype=np.float32)
            E_pad    = np.zeros((M, M, self.edge_dim), dtype=np.float32)

            X_pad[:N] = X

            A_vv = np.zeros((N, N), dtype=np.float32)
            A_vp = np.zeros((N, N), dtype=np.float32)
            A_pp = np.zeros((N, N), dtype=np.float32)
            E    = np.zeros((N, N, self.edge_dim), dtype=np.float32)

            for e in edges:
                i = id2idx.get(e["i"])
                j = id2idx.get(e["j"])
                if i is None or j is None:
                    continue

                # Full edge feature vector
                raw_ef = np.array([
                    e["distance"], e["rel_speed"], e["drac"],
                    e["heading_diff"], e["rel_vx"], e["rel_vy"]
                ], dtype=np.float32)
                full_normed = (raw_ef - self.edge_mean) / self.edge_std

                # Select subset
                ef_subset  = full_normed[self.edge_idx]
                E[i, j]    = ef_subset
                E[j, i]    = ef_subset

                itype = e["interaction_type"]
                if itype == "vv":
                    A_vv[i,j] = A_vv[j,i] = 1
                elif itype == "vp":
                    A_vp[i,j] = A_vp[j,i] = 1
                else:
                    A_pp[i,j] = A_pp[j,i] = 1

            A_vv_pad[:N,:N] = A_vv
            A_vp_pad[:N,:N] = A_vp
            A_pp_pad[:N,:N] = A_pp
            E_pad[:N,:N]    = E

            X_seq.append(X_pad)
            Avv_seq.append(A_vv_pad)
            Avp_seq.append(A_vp_pad)
            App_seq.append(A_pp_pad)
            E_seq.append(E_pad)

        return (
            torch.tensor(np.stack(X_seq),   dtype=torch.float32),
            torch.tensor(np.stack(Avv_seq), dtype=torch.float32),
            torch.tensor(np.stack(Avp_seq), dtype=torch.float32),
            torch.tensor(np.stack(App_seq), dtype=torch.float32),
            torch.tensor(np.stack(E_seq),   dtype=torch.float32),
            torch.tensor(label,             dtype=torch.float32)
        )


# Define ablation variants
ablation_A_configs = {
    "A0_nodes_only"   : [],
    "A1_kinematics"   : ["distance", "rel_speed"],
    "A2_+drac"        : ["distance", "rel_speed", "drac"],
    "A3_+heading"     : ["distance", "rel_speed", "drac", "heading_diff"],
    "A4_full"         : ["distance", "rel_speed", "drac",
                         "heading_diff", "rel_vx", "rel_vy"]
}

# Sanity check
for name, cols in ablation_A_configs.items():
    ds = AblationEdgeDataset(
        train_seqs, train_labels,
        node_mean, node_std, edge_mean, edge_std,
        edge_cols=cols if cols else ["distance"]
    )
    x, avv, avp, app, e, y = ds[0]
    print(f"{name:<20} edge_cols={len(cols)}  "
          f"E shape: {e.shape}")

In [ ]:
class SceneRiskModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.edge_dim = edge_dim
        self.gnn      = HeteroSSMGNN(node_dim, hidden_dim, edge_dim)
        self.lstm     = nn.LSTM(hidden_dim, hidden_dim,
                                num_layers=lstm_layers,
                                batch_first=True,
                                dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []

        for t in range(T):
            X   = X_seq[:,t]
            Avv = A_vv_seq[:,t]
            Avp = A_vp_seq[:,t]
            App = A_pp_seq[:,t]
            E   = E_seq[:,t]

            H = self.gnn(X, Avv, Avp, App, E)

            # SSM-weighted pooling
            # Use DRAC (index 2) if available, else use first edge feature
            # If no meaningful edges, fall back to mean pooling
            if self.edge_dim >= 3:
                # DRAC is at index 2
                drac_scores  = E_seq[:,t,:,:,2].max(dim=-1).values
                risk_weights = torch.softmax(drac_scores, dim=-1).unsqueeze(-1)
            elif self.edge_dim >= 1:
                # Use first available edge feature
                drac_scores  = E_seq[:,t,:,:,0].max(dim=-1).values
                risk_weights = torch.softmax(drac_scores, dim=-1).unsqueeze(-1)
            else:
                # No edge features — uniform pooling
                risk_weights = torch.ones(B, N, 1, device=X.device) / N

            H_pool = (H * risk_weights).sum(dim=1)
            frame_embeds.append(H_pool)

        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# Verify fix
model_test = SceneRiskModel(edge_dim=1).to(device)
x_dummy  = torch.randn(2, 8, MAX_NODES, NODE_DIM).to(device)
avv_dummy = torch.zeros(2, 8, MAX_NODES, MAX_NODES).to(device)
e_dummy  = torch.randn(2, 8, MAX_NODES, MAX_NODES, 1).to(device)
out = model_test(x_dummy, avv_dummy, avv_dummy, avv_dummy, e_dummy)
print(f"Edge dim=1 test passed: output shape {out.shape}")

model_test2 = SceneRiskModel(edge_dim=6).to(device)
e_dummy2 = torch.randn(2, 8, MAX_NODES, MAX_NODES, 6).to(device)
out2 = model_test2(x_dummy, avv_dummy, avv_dummy, avv_dummy, e_dummy2)
print(f"Edge dim=6 test passed: output shape {out2.shape}")

In [ ]:
# ── Fix A0: nodes only — zero edge features ────────────────
class NoEdgeDataset(Dataset):
    """Node features only — no edge information passed to GNN."""
    def __init__(self, sequences, labels,
                 node_mean, node_std, max_nodes=MAX_NODES):
        self.sequences = sequences
        self.labels    = labels
        self.max_nodes = max_nodes
        self.node_mean = node_mean
        self.node_std  = node_std

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq   = self.sequences[idx]
        label = self.labels[idx]

        X_seq, Avv_seq, Avp_seq, App_seq, E_seq = [], [], [], [], []

        for g in seq:
            nodes  = g["nodes"]
            edges  = g["edges"]
            N      = len(nodes)
            id2idx = {n["track_id"]: i for i, n in enumerate(nodes)}

            X = []
            for n in nodes:
                raw    = np.array([
                    n["x"], n["y"], n["vx"], n["vy"],
                    n["speed"], n["accel"], n["heading"]
                ], dtype=np.float32)
                normed = (raw - self.node_mean) / self.node_std
                is_veh = 1.0 if n["agent_type"] == "vehicle" else 0.0
                is_ped = 1.0 if n["agent_type"] == "pedestrian" else 0.0
                X.append(np.append(normed, [is_veh, is_ped]))
            X = np.array(X, dtype=np.float32)

            M        = self.max_nodes
            X_pad    = np.zeros((M, NODE_DIM), dtype=np.float32)
            A_vv_pad = np.zeros((M, M),        dtype=np.float32)
            A_vp_pad = np.zeros((M, M),        dtype=np.float32)
            A_pp_pad = np.zeros((M, M),        dtype=np.float32)
            E_pad    = np.zeros((M, M, 1),     dtype=np.float32)  # dummy

            X_pad[:N] = X

            # Still build adjacency for message passing
            # but edge features are all zeros
            A_vv = np.zeros((N, N), dtype=np.float32)
            A_vp = np.zeros((N, N), dtype=np.float32)
            A_pp = np.zeros((N, N), dtype=np.float32)

            for e in edges:
                i = id2idx.get(e["i"])
                j = id2idx.get(e["j"])
                if i is None or j is None:
                    continue
                itype = e["interaction_type"]
                if itype == "vv":   A_vv[i,j] = A_vv[j,i] = 1
                elif itype == "vp": A_vp[i,j] = A_vp[j,i] = 1
                else:               A_pp[i,j] = A_pp[j,i] = 1

            A_vv_pad[:N,:N] = A_vv
            A_vp_pad[:N,:N] = A_vp
            A_pp_pad[:N,:N] = A_pp

            X_seq.append(X_pad)
            Avv_seq.append(A_vv_pad)
            Avp_seq.append(A_vp_pad)
            App_seq.append(A_pp_pad)
            E_seq.append(E_pad)

        return (
            torch.tensor(np.stack(X_seq),   dtype=torch.float32),
            torch.tensor(np.stack(Avv_seq), dtype=torch.float32),
            torch.tensor(np.stack(Avp_seq), dtype=torch.float32),
            torch.tensor(np.stack(App_seq), dtype=torch.float32),
            torch.tensor(np.stack(E_seq),   dtype=torch.float32),
            torch.tensor(label,             dtype=torch.float32)
        )


# ── Ablation training function ─────────────────────────────
def train_ablation_variant(variant_name, ds_train_abl, ds_val_abl,
                           edge_dim, n_epochs=20, batch_size=16):
    """
    Train one ablation variant for fixed n_epochs.
    Returns best val AUC and full metrics.
    """
    loader_tr = DataLoader(ds_train_abl, batch_size=batch_size,
                           shuffle=True,  num_workers=0)
    loader_vl = DataLoader(ds_val_abl,   batch_size=batch_size,
                           shuffle=False, num_workers=0)

    model = SceneRiskModel(
        node_dim=NODE_DIM, hidden_dim=64,
        edge_dim=edge_dim, lstm_layers=2, dropout=0.3
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    best_val_auc = 0.0
    best_state   = None

    for epoch in range(1, n_epochs + 1):
        # Train
        model.train()
        for batch in loader_tr:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = model(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Validate
        model.eval()
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader_vl:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        val_auc = roc_auc_score(y_true, y_score)
        scheduler.step(val_auc)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}

    # Evaluate best model on val
    model.load_state_dict(best_state)
    model.eval()

    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader_vl:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = model(X, Avv, Avp, App, E)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    y_true  = np.array(y_true)
    y_score = np.array(y_score)
    y_pred  = (y_score >= 0.5).astype(int)
    cm      = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr, tpr, _    = roc_curve(y_true, y_score)

    return {
        "variant"    : variant_name,
        "auc_roc"    : roc_auc_score(y_true, y_score),
        "auc_pr"     : average_precision_score(y_true, y_score),
        "f1"         : f1_score(y_true, y_pred,        zero_division=0),
        "accuracy"   : accuracy_score(y_true, y_pred),
        "precision"  : precision_score(y_true, y_pred, zero_division=0),
        "recall"     : recall_score(y_true, y_pred,    zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "mcc"        : matthews_corrcoef(y_true, y_pred),
        "sens_5far"  : float(tpr[np.searchsorted(fpr, 0.05)]),
        "sens_10far" : float(tpr[np.searchsorted(fpr, 0.10)]),
        "best_state" : best_state
    }


# ── Run Ablation Group 1 ───────────────────────────────────
print("="*65)
print("ABLATION GROUP 1 — SSM Edge Feature Contribution")
print("="*65)
print(f"{'Variant':<22} {'AUC-ROC':>7} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

ablation_A_results = {}

# A0: nodes only
ds_a0_tr = NoEdgeDataset(train_seqs, train_labels, node_mean, node_std)
ds_a0_vl = NoEdgeDataset(val_seqs,   val_labels,   node_mean, node_std)
res = train_ablation_variant("A0_nodes_only", ds_a0_tr, ds_a0_vl,
                             edge_dim=1, n_epochs=20)
ablation_A_results["A0_nodes_only"] = res
print(f"{'A0 — nodes only':<22} {res['auc_roc']:>7.4f} {res['auc_pr']:>7.4f} "
      f"{res['f1']:>6.4f} {res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

# A1-A4: increasing edge features
for abl_name, cols in list(ablation_A_configs.items())[1:]:
    ds_tr = AblationEdgeDataset(
        train_seqs, train_labels,
        node_mean, node_std, edge_mean, edge_std,
        edge_cols=cols
    )
    ds_vl = AblationEdgeDataset(
        val_seqs, val_labels,
        node_mean, node_std, edge_mean, edge_std,
        edge_cols=cols
    )
    res = train_ablation_variant(
        abl_name, ds_tr, ds_vl,
        edge_dim=len(cols), n_epochs=20
    )
    ablation_A_results[abl_name] = res
    print(f"{abl_name:<22} {res['auc_roc']:>7.4f} {res['auc_pr']:>7.4f} "
          f"{res['f1']:>6.4f} {res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

print("-"*65)
print("Full model (convergence)  val AUC-ROC=0.9539")

In [ ]:
# ══════════════════════════════════════════════════════════
# ABLATION GROUP 2 — Graph Architecture Components
# ══════════════════════════════════════════════════════════

# B1: No graph structure — mean pool node features only
class MeanPoolModel(nn.Module):
    """
    No GNN — just mean pool node features per frame then LSTM.
    Tests whether graph structure adds value over simple aggregation.
    """
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.proj    = nn.Linear(node_dim, hidden_dim)
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim,
                               num_layers=lstm_layers,
                               batch_first=True,
                               dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []
        for t in range(T):
            X      = X_seq[:,t]              # [B, N, node_dim]
            H      = F.relu(self.proj(X))    # [B, N, hidden]
            H_pool = H.mean(dim=1)           # [B, hidden]
            frame_embeds.append(H_pool)
        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# B2: Simple GNN — uniform adjacency aggregation, no SSM attention
class SimpleGNNModel(nn.Module):
    """
    Basic GNN: A*X aggregation without attention or edge features.
    Tests contribution of SSM-biased attention.
    """
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.W       = nn.Linear(node_dim, hidden_dim)
        self.fusion  = nn.Linear(node_dim + hidden_dim, hidden_dim)
        self.norm    = nn.LayerNorm(hidden_dim)
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim,
                               num_layers=lstm_layers,
                               batch_first=True,
                               dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []

        for t in range(T):
            X = X_seq[:,t]
            # Combined adjacency
            A = (A_vv_seq[:,t] + A_vp_seq[:,t] + A_pp_seq[:,t]).clamp(0, 1)

            H   = self.W(X)                          # [B, N, hidden]
            AX  = torch.bmm(A, H)                    # uniform aggregation
            out = self.norm(F.relu(
                self.fusion(torch.cat([X, AX], dim=-1))
            ))
            H_pool = out.mean(dim=1)
            frame_embeds.append(H_pool)

        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# B3: SSM-biased attention but mean pooling (no DRAC-weighted pooling)
class SSMAttnMeanPoolModel(nn.Module):
    """
    Full SSM-biased hetero GNN but with mean pooling instead of
    DRAC-weighted pooling. Tests contribution of SSM-weighted pooling.
    """
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.gnn     = HeteroSSMGNN(node_dim, hidden_dim, edge_dim)
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim,
                               num_layers=lstm_layers,
                               batch_first=True,
                               dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []

        for t in range(T):
            H      = self.gnn(X_seq[:,t], A_vv_seq[:,t],
                              A_vp_seq[:,t], A_pp_seq[:,t], E_seq[:,t])
            H_pool = H.mean(dim=1)      # mean pooling instead of DRAC-weighted
            frame_embeds.append(H_pool)

        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# B4: Full model (SSM attention + DRAC-weighted pooling) = current best
# Already trained — use existing results


# ── Train Group 2 variants ─────────────────────────────────
def train_ablation_model(model, variant_name, ds_train_abl,
                         ds_val_abl, n_epochs=20, batch_size=16):
    """Generic ablation trainer for any model architecture."""
    loader_tr = DataLoader(ds_train_abl, batch_size=batch_size,
                           shuffle=True,  num_workers=0)
    loader_vl = DataLoader(ds_val_abl,   batch_size=batch_size,
                           shuffle=False, num_workers=0)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    best_val_auc = 0.0
    best_state   = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        for batch in loader_tr:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = model(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader_vl:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        val_auc = roc_auc_score(y_true, y_score)
        scheduler.step(val_auc)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state   = {k: v.clone()
                            for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()

    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader_vl:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = model(X, Avv, Avp, App, E)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    y_true  = np.array(y_true)
    y_score = np.array(y_score)
    y_pred  = (y_score >= 0.5).astype(int)
    cm      = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr, tpr, _    = roc_curve(y_true, y_score)

    return {
        "variant"    : variant_name,
        "auc_roc"    : roc_auc_score(y_true, y_score),
        "auc_pr"     : average_precision_score(y_true, y_score),
        "f1"         : f1_score(y_true, y_pred,        zero_division=0),
        "accuracy"   : accuracy_score(y_true, y_pred),
        "precision"  : precision_score(y_true, y_pred, zero_division=0),
        "recall"     : recall_score(y_true, y_pred,    zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "mcc"        : matthews_corrcoef(y_true, y_pred),
        "sens_5far"  : float(tpr[np.searchsorted(fpr, 0.05)]),
        "sens_10far" : float(tpr[np.searchsorted(fpr, 0.10)]),
        "best_state" : best_state
    }


print("="*65)
print("ABLATION GROUP 2 — Graph Architecture Components")
print("="*65)
print(f"{'Variant':<28} {'AUC-ROC':>7} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

ablation_B_results = {}

# B1: No graph — mean pool
b1 = MeanPoolModel().to(device)
res = train_ablation_model(b1, "B1_no_graph",
                           ds_train, ds_val, n_epochs=20)
ablation_B_results["B1_no_graph"] = res
print(f"{'B1 — no graph (mean pool)':<28} {res['auc_roc']:>7.4f} "
      f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
      f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

# B2: Simple GNN — no attention
b2 = SimpleGNNModel().to(device)
res = train_ablation_model(b2, "B2_simple_gnn",
                           ds_train, ds_val, n_epochs=20)
ablation_B_results["B2_simple_gnn"] = res
print(f"{'B2 — simple GNN':<28} {res['auc_roc']:>7.4f} "
      f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
      f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

# B3: SSM attention + mean pooling
b3 = SSMAttnMeanPoolModel().to(device)
res = train_ablation_model(b3, "B3_ssm_attn_meanpool",
                           ds_train, ds_val, n_epochs=20)
ablation_B_results["B3_ssm_attn_meanpool"] = res
print(f"{'B3 — SSM attn + mean pool':<28} {res['auc_roc']:>7.4f} "
      f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
      f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

# B4: Full model (20 epochs for fair comparison)
b4 = SceneRiskModel().to(device)
res = train_ablation_model(b4, "B4_full_model",
                           ds_train, ds_val, n_epochs=20)
ablation_B_results["B4_full_model"] = res
print(f"{'B4 — full model (SSM+DRAC pool)':<28} {res['auc_roc']:>7.4f} "
      f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
      f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

print("-"*65)
print("Full model (convergence)  val AUC-ROC=0.9539")

In [ ]:
# ══════════════════════════════════════════════════════════
# ABLATION GROUP 3 — Temporal Modeling
# ══════════════════════════════════════════════════════════

# C1: Single frame — no temporal modeling
class SingleFrameModel(nn.Module):
    """
    No temporal modeling — process only the last frame.
    Tests whether sequence modeling adds value.
    """
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, dropout=0.3):
        super().__init__()
        self.gnn     = HeteroSSMGNN(node_dim, hidden_dim, edge_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        # Use only last frame
        t    = X_seq.shape[1] - 1
        H    = self.gnn(X_seq[:,t], A_vv_seq[:,t],
                        A_vp_seq[:,t], A_pp_seq[:,t], E_seq[:,t])
        H_pool = H.mean(dim=1)
        final  = self.dropout(H_pool)
        return self.fc(final).squeeze(-1)


# C2: GNN + GRU
class GNNGRUModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.gnn     = HeteroSSMGNN(node_dim, hidden_dim, edge_dim)
        self.gru     = nn.GRU(hidden_dim, hidden_dim,
                              num_layers=lstm_layers,
                              batch_first=True,
                              dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []
        for t in range(T):
            H = self.gnn(X_seq[:,t], A_vv_seq[:,t],
                         A_vp_seq[:,t], A_pp_seq[:,t], E_seq[:,t])
            if E_seq.shape[-1] >= 3:
                drac_scores  = E_seq[:,t,:,:,2].max(dim=-1).values
                risk_weights = torch.softmax(
                    drac_scores, dim=-1).unsqueeze(-1)
            else:
                risk_weights = torch.ones(
                    B, N, 1, device=X_seq.device) / N
            H_pool = (H * risk_weights).sum(dim=1)
            frame_embeds.append(H_pool)
        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.gru(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# C3: GNN + Transformer temporal
class GNNTransformerModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, num_heads=4,
                 num_layers=2, dropout=0.3, max_seq_len=8):
        super().__init__()
        self.gnn     = HeteroSSMGNN(node_dim, hidden_dim, edge_dim)
        self.pos_enc = nn.Embedding(max_seq_len, hidden_dim)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []
        for t in range(T):
            H = self.gnn(X_seq[:,t], A_vv_seq[:,t],
                         A_vp_seq[:,t], A_pp_seq[:,t], E_seq[:,t])
            if E_seq.shape[-1] >= 3:
                drac_scores  = E_seq[:,t,:,:,2].max(dim=-1).values
                risk_weights = torch.softmax(
                    drac_scores, dim=-1).unsqueeze(-1)
            else:
                risk_weights = torch.ones(
                    B, N, 1, device=X_seq.device) / N
            H_pool = (H * risk_weights).sum(dim=1)
            frame_embeds.append(H_pool)

        H_seq     = torch.stack(frame_embeds, dim=1)
        positions = torch.arange(T, device=H_seq.device).unsqueeze(0)
        H_seq     = H_seq + self.pos_enc(positions)
        out       = self.transformer(H_seq)
        final     = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# C4: Full model GNN+LSTM = existing B4/full model
# Already covered

print("="*65)
print("ABLATION GROUP 3 — Temporal Modeling")
print("="*65)
print(f"{'Variant':<28} {'AUC-ROC':>7} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

ablation_C_results = {}

# C1: Single frame
c1 = SingleFrameModel().to(device)
res = train_ablation_model(c1, "C1_single_frame",
                           ds_train, ds_val, n_epochs=20)
ablation_C_results["C1_single_frame"] = res
print(f"{'C1 — single frame':<28} {res['auc_roc']:>7.4f} "
      f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
      f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

# C2: GNN + GRU
c2 = GNNGRUModel().to(device)
res = train_ablation_model(c2, "C2_gnn_gru",
                           ds_train, ds_val, n_epochs=20)
ablation_C_results["C2_gnn_gru"] = res
print(f"{'C2 — GNN + GRU':<28} {res['auc_roc']:>7.4f} "
      f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
      f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

# C3: GNN + Transformer
c3 = GNNTransformerModel().to(device)
res = train_ablation_model(c3, "C3_gnn_transformer",
                           ds_train, ds_val, n_epochs=20)
ablation_C_results["C3_gnn_transformer"] = res
print(f"{'C3 — GNN + Transformer':<28} {res['auc_roc']:>7.4f} "
      f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
      f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

# C4: GNN + LSTM (full model 20 epochs) — reuse B4 result
res_b4 = ablation_B_results["B4_full_model"]
ablation_C_results["C4_gnn_lstm"] = res_b4
print(f"{'C4 — GNN + LSTM (full)':<28} {res_b4['auc_roc']:>7.4f} "
      f"{res_b4['auc_pr']:>7.4f} {res_b4['f1']:>6.4f} "
      f"{res_b4['mcc']:>6.4f} {res_b4['sens_5far']:>6.4f}")

print("-"*65)
print("Full model (convergence)  val AUC-ROC=0.9539")

In [ ]:
# ══════════════════════════════════════════════════════════
# ABLATION GROUP 4 — Interaction Type Coverage
# ══════════════════════════════════════════════════════════

class InteractionTypeDataset(Dataset):
    """
    Same as SceneGraphDataset but filters edges to specific
    interaction types only.
    """
    def __init__(self, sequences, labels,
                 node_mean, node_std,
                 edge_mean, edge_std,
                 allowed_types,        # e.g. {"vv"} or {"vv","vp"}
                 max_nodes=MAX_NODES):
        self.sequences     = sequences
        self.labels        = labels
        self.max_nodes     = max_nodes
        self.node_mean     = node_mean
        self.node_std      = node_std
        self.edge_mean     = edge_mean
        self.edge_std      = edge_std
        self.allowed_types = allowed_types

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq   = self.sequences[idx]
        label = self.labels[idx]

        X_seq, Avv_seq, Avp_seq, App_seq, E_seq = [], [], [], [], []

        for g in seq:
            nodes  = g["nodes"]
            edges  = g["edges"]
            N      = len(nodes)
            id2idx = {n["track_id"]: i for i, n in enumerate(nodes)}

            X = []
            for n in nodes:
                raw    = np.array([
                    n["x"], n["y"], n["vx"], n["vy"],
                    n["speed"], n["accel"], n["heading"]
                ], dtype=np.float32)
                normed = (raw - self.node_mean) / self.node_std
                is_veh = 1.0 if n["agent_type"] == "vehicle" else 0.0
                is_ped = 1.0 if n["agent_type"] == "pedestrian" else 0.0
                X.append(np.append(normed, [is_veh, is_ped]))
            X = np.array(X, dtype=np.float32)

            M        = self.max_nodes
            X_pad    = np.zeros((M, NODE_DIM),    dtype=np.float32)
            A_vv_pad = np.zeros((M, M),           dtype=np.float32)
            A_vp_pad = np.zeros((M, M),           dtype=np.float32)
            A_pp_pad = np.zeros((M, M),           dtype=np.float32)
            E_pad    = np.zeros((M, M, EDGE_DIM), dtype=np.float32)

            X_pad[:N] = X

            A_vv = np.zeros((N, N), dtype=np.float32)
            A_vp = np.zeros((N, N), dtype=np.float32)
            A_pp = np.zeros((N, N), dtype=np.float32)
            E    = np.zeros((N, N, EDGE_DIM), dtype=np.float32)

            for e in edges:
                # Skip edges not in allowed types
                if e["interaction_type"] not in self.allowed_types:
                    continue

                i = id2idx.get(e["i"])
                j = id2idx.get(e["j"])
                if i is None or j is None:
                    continue

                raw_ef = np.array([
                    e["distance"], e["rel_speed"], e["drac"],
                    e["heading_diff"], e["rel_vx"], e["rel_vy"]
                ], dtype=np.float32)
                normed_ef = (raw_ef - self.edge_mean) / self.edge_std
                E[i,j]    = normed_ef
                E[j,i]    = normed_ef

                itype = e["interaction_type"]
                if itype == "vv":   A_vv[i,j] = A_vv[j,i] = 1
                elif itype == "vp": A_vp[i,j] = A_vp[j,i] = 1
                else:               A_pp[i,j] = A_pp[j,i] = 1

            A_vv_pad[:N,:N] = A_vv
            A_vp_pad[:N,:N] = A_vp
            A_pp_pad[:N,:N] = A_pp
            E_pad[:N,:N]    = E

            X_seq.append(X_pad)
            Avv_seq.append(A_vv_pad)
            Avp_seq.append(A_vp_pad)
            App_seq.append(A_pp_pad)
            E_seq.append(E_pad)

        return (
            torch.tensor(np.stack(X_seq),   dtype=torch.float32),
            torch.tensor(np.stack(Avv_seq), dtype=torch.float32),
            torch.tensor(np.stack(Avp_seq), dtype=torch.float32),
            torch.tensor(np.stack(App_seq), dtype=torch.float32),
            torch.tensor(np.stack(E_seq),   dtype=torch.float32),
            torch.tensor(label,             dtype=torch.float32)
        )


print("="*65)
print("ABLATION GROUP 4 — Interaction Type Coverage")
print("="*65)
print(f"{'Variant':<25} {'AUC-ROC':>7} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

ablation_D_results = {}

for d_name, allowed in [
    ("D1_vv_only",   {"vv"}),
    ("D2_vp_only",   {"vp"}),
    ("D3_vv_vp",     {"vv", "vp"}),
    ("D4_all_types", {"vv", "vp", "pp"}),
]:
    ds_tr = InteractionTypeDataset(
        train_seqs, train_labels,
        node_mean, node_std, edge_mean, edge_std,
        allowed_types=allowed
    )
    ds_vl = InteractionTypeDataset(
        val_seqs, val_labels,
        node_mean, node_std, edge_mean, edge_std,
        allowed_types=allowed
    )
    m   = SceneRiskModel().to(device)
    res = train_ablation_model(m, d_name, ds_tr, ds_vl, n_epochs=20)
    ablation_D_results[d_name] = res
    print(f"{d_name:<25} {res['auc_roc']:>7.4f} {res['auc_pr']:>7.4f} "
          f"{res['f1']:>6.4f} {res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

print("-"*65)
print("Full model (convergence)  val AUC-ROC=0.9539")

In [ ]:
# Save all ablation results
ablation_all = {
    "group1_ssm_features"    : ablation_A_results,
    "group2_architecture"    : ablation_B_results,
    "group3_temporal"        : ablation_C_results,
    "group4_interaction_type": ablation_D_results
}

with open(checkpoint_path / "ablation_results.pkl", "wb") as f:
    pickle.dump(ablation_all, f)

print("Saved ablation_results.pkl")

# Print full ablation summary table
print("\n" + "="*70)
print("COMPLETE ABLATION STUDY SUMMARY (Val Set, 20 epochs)")
print("="*70)

groups = [
    ("Group 1 — SSM Edge Features", ablation_A_results),
    ("Group 2 — Architecture",       ablation_B_results),
    ("Group 3 — Temporal Modeling",  ablation_C_results),
    ("Group 4 — Interaction Types",  ablation_D_results),
]

for group_name, group_results in groups:
    print(f"\n{group_name}")
    print(f"  {'Variant':<30} {'AUC-ROC':>7} {'AUC-PR':>7} "
          f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
    print(f"  {'-'*60}")
    for name, res in group_results.items():
        print(f"  {name:<30} {res['auc_roc']:>7.4f} "
              f"{res['auc_pr']:>7.4f} {res['f1']:>6.4f} "
              f"{res['mcc']:>6.4f} {res['sens_5far']:>6.4f}")

print(f"\nFull model (convergence): val AUC-ROC=0.9539  AUC-PR=0.8457")

# enchanced heterognn

In [ ]:
class MultiHeadSSMAttention(nn.Module):
    def __init__(self, node_dim, hidden_dim,
                 edge_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.W_q       = nn.Linear(node_dim,   hidden_dim)
        self.W_k       = nn.Linear(node_dim,   hidden_dim)
        self.W_v       = nn.Linear(node_dim,   hidden_dim)
        self.edge_enc  = nn.Linear(edge_dim,   num_heads)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.leaky     = nn.LeakyReLU(0.2)

    def forward(self, X, A, E):
        B, N, _ = X.shape
        H = self.num_heads
        D = self.head_dim
        Q = self.W_q(X).view(B,N,H,D).permute(0,2,1,3)
        K = self.W_k(X).view(B,N,H,D).permute(0,2,1,3)
        V = self.W_v(X).view(B,N,H,D).permute(0,2,1,3)

        scale    = D ** -0.5
        scores   = torch.matmul(Q, K.transpose(-2,-1)) * scale
        ssm_bias = self.edge_enc(E).permute(0,3,1,2)
        scores   = scores + ssm_bias
        A_exp    = A.unsqueeze(1).expand(-1,H,-1,-1)
        scores   = scores.masked_fill(A_exp==0, -1e9)
        weights  = torch.softmax(scores, dim=-1)

        out = torch.matmul(weights, V)
        out = out.permute(0,2,1,3).contiguous()
        out = out.view(B, N, -1)
        return self.out_proj(out)


class EdgeUpdateNetwork(nn.Module):
    def __init__(self, hidden_dim, edge_dim):
        super().__init__()
        self.edge_update = nn.Linear(
            hidden_dim*2 + edge_dim, edge_dim)
        self.norm = nn.LayerNorm(edge_dim)
        self.relu = nn.ReLU()

    def forward(self, H, E, A):
        B, N, d = H.shape
        Hi = H.unsqueeze(2).expand(-1,-1,N,-1)
        Hj = H.unsqueeze(1).expand(-1,N,-1,-1)
        inp = torch.cat([Hi, Hj, E], dim=-1)
        dE  = self.relu(self.edge_update(inp))
        mask = (A > 0).float().unsqueeze(-1)
        return self.norm(E + mask * (dE - E))


class EnhancedHeteroGNN(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, num_heads=4):
        super().__init__()
        self.input_proj = nn.Linear(node_dim, hidden_dim)

        for layer in [1, 2]:
            for rel in ["vv","vp","pp"]:
                setattr(self, f"attn_{rel}_{layer}",
                    MultiHeadSSMAttention(
                        hidden_dim, hidden_dim,
                        edge_dim, num_heads))
            setattr(self, f"edge_update_{layer}",
                EdgeUpdateNetwork(hidden_dim, edge_dim))
            setattr(self, f"fusion_{layer}",
                nn.Linear(hidden_dim*4, hidden_dim))
            setattr(self, f"norm_{layer}",
                nn.LayerNorm(hidden_dim))

        self.dropout = nn.Dropout(0.3)

    def forward(self, X_seq, A_vv_seq, A_vp_seq,
                A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []

        for t in range(T):
            X   = X_seq[:,t]
            Avv = A_vv_seq[:,t]
            Avp = A_vp_seq[:,t]
            App = A_pp_seq[:,t]
            E   = E_seq[:,t]

            H = F.relu(self.input_proj(X))

            for layer in [1, 2]:
                msgs = [H]
                A_map = {"vv":Avv,"vp":Avp,"pp":App}
                for rel in ["vv","vp","pp"]:
                    attn = getattr(self,
                        f"attn_{rel}_{layer}")
                    msgs.append(attn(H, A_map[rel], E))

                fusion = getattr(self, f"fusion_{layer}")
                norm   = getattr(self, f"norm_{layer}")
                H_new  = norm(F.relu(
                    fusion(torch.cat(msgs, dim=-1))) + H)

                edge_upd = getattr(self,
                    f"edge_update_{layer}")
                A_any = ((Avv + Avp + App) > 0).float()
                E     = edge_upd(H_new, E, A_any)
                H     = H_new

            # DRAC-weighted pooling
            drac_vals = E[:,:,:,2].max(dim=-1)[0]
            w         = torch.softmax(
                drac_vals, dim=-1).unsqueeze(-1)
            H_pool    = (w * H).sum(dim=1)
            frame_embeds.append(H_pool)

        return torch.stack(frame_embeds, dim=1)


class EnhancedSceneRiskModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, num_heads=4,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.gnn  = EnhancedHeteroGNN(
            node_dim, hidden_dim, edge_dim, num_heads)
        self.lstm = nn.LSTM(
            hidden_dim, hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers>1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq,
                A_pp_seq, E_seq):
        H_seq  = self.gnn(
            X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:,-1,:])
        return self.fc(final).squeeze(-1)

print("Model classes defined.")

In [ ]:
class EdgeUpdateLayer(nn.Module):
    """
    Bidirectional node-edge message passing.
    Updates edge representations — keeps same output dim as input.
    """
    def __init__(self, node_dim, edge_dim):
        super().__init__()
        self.edge_update = nn.Linear(
            node_dim + node_dim + edge_dim, edge_dim  # output same dim as input
        )
        self.norm = nn.LayerNorm(edge_dim)
        self.relu = nn.ReLU()

    def forward(self, X, E, A):
        B, N, _ = X.shape
        Xi = X.unsqueeze(2).expand(-1, -1, N, -1)
        Xj = X.unsqueeze(1).expand(-1, N, -1, -1)

        edge_input = torch.cat([Xi, Xj, E], dim=-1)
        E_new      = self.relu(self.edge_update(edge_input))

        mask      = A.unsqueeze(-1).expand_as(E)
        E_updated = E + mask * (E_new - E)

        return self.norm(E_updated)


class EnhancedHeteroGNN(nn.Module):
    def __init__(self, node_dim, hidden_dim, edge_dim, num_heads=4):
        super().__init__()

        self.input_proj = nn.Linear(node_dim, hidden_dim)

        # Layer 1 — operates on original edge_dim
        self.attn_vv_1    = MultiHeadSSMAttention(
            hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_vp_1    = MultiHeadSSMAttention(
            hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_pp_1    = MultiHeadSSMAttention(
            hidden_dim, hidden_dim, edge_dim, num_heads)
        self.edge_update_1 = EdgeUpdateLayer(hidden_dim, edge_dim)
        self.fusion_1      = nn.Linear(hidden_dim * 4, hidden_dim)
        self.norm_1        = nn.LayerNorm(hidden_dim)

        # Layer 2 — also operates on edge_dim (same dim, updated values)
        self.attn_vv_2    = MultiHeadSSMAttention(
            hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_vp_2    = MultiHeadSSMAttention(
            hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_pp_2    = MultiHeadSSMAttention(
            hidden_dim, hidden_dim, edge_dim, num_heads)
        self.edge_update_2 = EdgeUpdateLayer(hidden_dim, edge_dim)
        self.fusion_2      = nn.Linear(hidden_dim * 4, hidden_dim)
        self.norm_2        = nn.LayerNorm(hidden_dim)

        self.dropout = nn.Dropout(0.3)

    def forward(self, X, A_vv, A_vp, A_pp, E):
        # Input projection
        H = F.relu(self.input_proj(X))   # [B, N, hidden]

        # ── Layer 1 ───────────────────────────────────────
        msg_vv = self.attn_vv_1(H, A_vv, E)
        msg_vp = self.attn_vp_1(H, A_vp, E)
        msg_pp = self.attn_pp_1(H, A_pp, E)

        H1 = self.norm_1(F.relu(
            self.fusion_1(torch.cat([H, msg_vv, msg_vp, msg_pp], dim=-1))
        )) + H   # residual

        A_combined = (A_vv + A_vp + A_pp).clamp(0, 1)
        E1 = self.edge_update_1(H1, E, A_combined)   # still edge_dim

        # ── Layer 2 ───────────────────────────────────────
        msg_vv2 = self.attn_vv_2(H1, A_vv, E1)
        msg_vp2 = self.attn_vp_2(H1, A_vp, E1)
        msg_pp2 = self.attn_pp_2(H1, A_pp, E1)

        H2 = self.norm_2(F.relu(
            self.fusion_2(torch.cat([H1, msg_vv2, msg_vp2, msg_pp2], dim=-1))
        )) + H1  # residual

        return self.dropout(H2), E1


class EnhancedSceneRiskModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, num_heads=4,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.gnn  = EnhancedHeteroGNN(
            node_dim, hidden_dim, edge_dim, num_heads
        )
        self.lstm = nn.LSTM(
            hidden_dim, hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []

        for t in range(T):
            H, E_updated = self.gnn(
                X_seq[:,t], A_vv_seq[:,t],
                A_vp_seq[:,t], A_pp_seq[:,t], E_seq[:,t]
            )

            # DRAC-weighted pooling using updated edge features
            # E_updated is still edge_dim — use DRAC index (2)
            drac_scores  = E_updated[:,:,:,2].max(dim=-1).values
            risk_weights = torch.softmax(
                drac_scores, dim=-1
            ).unsqueeze(-1)
            H_pool = (H * risk_weights).sum(dim=1)
            frame_embeds.append(H_pool)

        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


# Sanity check
enhanced_model = EnhancedSceneRiskModel().to(device)
X, Avv, Avp, App, E, y = ds_train[0]
out = enhanced_model(
    X.unsqueeze(0).to(device),
    Avv.unsqueeze(0).to(device),
    Avp.unsqueeze(0).to(device),
    App.unsqueeze(0).to(device),
    E.unsqueeze(0).to(device)
)
params = sum(p.numel() for p in enhanced_model.parameters())
print(f"Output shape     : {out.shape}")
print(f"Output value     : {out.item():.4f}")
print(f"Total parameters : {params:,}")
print(f"Baseline params  : 82,436")
print(f"Parameter ratio  : {params/82436:.1f}x")

In [ ]:
# Quick batch size test for enhanced model
print("=== Enhanced Model Batch Size Test ===")
print(f"{'Batch':>6}  {'Val AUC (3 epochs)':>18}")
print("-" * 28)

def quick_enhanced_test(batch_size, n_epochs=3):
    loader_tr = DataLoader(ds_train, batch_size=batch_size,
                           shuffle=True,  num_workers=0)
    loader_vl = DataLoader(ds_val,   batch_size=batch_size,
                           shuffle=False, num_workers=0)

    m   = EnhancedSceneRiskModel().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(n_epochs):
        m.train()
        for batch in loader_tr:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = m(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

    m.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader_vl:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = m(X, Avv, Avp, App, E)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    return roc_auc_score(y_true, y_score)

for bs in [8, 16, 32, 64]:
    auc = quick_enhanced_test(bs, n_epochs=3)
    print(f"{bs:>6}  {auc:>18.4f}")

In [ ]:
import time

BATCH_SIZE_ENH = 64
EPOCHS         = 50

loader_train_enh = DataLoader(ds_train, batch_size=BATCH_SIZE_ENH,
                               shuffle=True,  num_workers=0)
loader_val_enh   = DataLoader(ds_val,   batch_size=BATCH_SIZE_ENH,
                               shuffle=False, num_workers=0)
loader_test_enh  = DataLoader(ds_test,  batch_size=BATCH_SIZE_ENH,
                               shuffle=False, num_workers=0)

# Fresh enhanced model
enhanced_model = EnhancedSceneRiskModel(
    node_dim=NODE_DIM, hidden_dim=64,
    edge_dim=EDGE_DIM, num_heads=4,
    lstm_layers=2, dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    enhanced_model.parameters(), lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

best_val_auc = 0.0
best_epoch   = 0
patience     = 5
patience_ctr = 0
history_enh  = []

print(f"Enhanced HeteroGNN Training")
print(f"Batch size    : {BATCH_SIZE_ENH}")
print(f"Max epochs    : {EPOCHS}")
print(f"Parameters    : {sum(p.numel() for p in enhanced_model.parameters()):,}")
print(f"Device        : {device}")
print()
print(f"{'Ep':>3}  {'TrLoss':>7}  {'TrAUC':>6}  {'TrF1':>5}  {'TrAcc':>5}  "
      f"{'VaLoss':>7}  {'VaAUC':>6}  {'VaF1':>5}  {'VaAcc':>5}  "
      f"{'LR':>8}  {'Time':>5}")
print("-" * 85)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # Train
    enhanced_model.train()
    total_loss = 0.0
    y_true_tr, y_score_tr = [], []

    for batch in loader_train_enh:
        X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
        logits = enhanced_model(X, Avv, Avp, App, E)
        loss   = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(enhanced_model.parameters(), 1.0)
        optimizer.step()
        total_loss    += loss.item() * len(y)
        y_true_tr.extend(y.cpu().numpy())
        y_score_tr.extend(torch.sigmoid(logits).detach().cpu().numpy())

    tr_loss  = total_loss / len(ds_train)
    tr_preds = [1 if p >= 0.5 else 0 for p in y_score_tr]
    tr_auc   = roc_auc_score(y_true_tr, y_score_tr)
    tr_f1    = f1_score(y_true_tr, tr_preds,  zero_division=0)
    tr_acc   = accuracy_score(y_true_tr, tr_preds)

    # Validate
    enhanced_model.eval()
    total_loss = 0.0
    y_true_vl, y_score_vl = [], []

    with torch.no_grad():
        for batch in loader_val_enh:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = enhanced_model(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)
            total_loss    += loss.item() * len(y)
            y_true_vl.extend(y.cpu().numpy())
            y_score_vl.extend(torch.sigmoid(logits).cpu().numpy())

    va_loss  = total_loss / len(ds_val)
    va_preds = [1 if p >= 0.5 else 0 for p in y_score_vl]
    va_auc   = roc_auc_score(y_true_vl, y_score_vl)
    va_f1    = f1_score(y_true_vl, va_preds,  zero_division=0)
    va_acc   = accuracy_score(y_true_vl, va_preds)

    scheduler.step(va_auc)
    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]

    history_enh.append({
        "epoch"     : epoch,
        "train_loss": tr_loss, "train_auc": tr_auc,
        "train_f1"  : tr_f1,  "train_acc": tr_acc,
        "val_loss"  : va_loss, "val_auc"  : va_auc,
        "val_f1"    : va_f1,  "val_acc"  : va_acc
    })

    print(f"{epoch:>3}  {tr_loss:>7.4f}  {tr_auc:>6.4f}  {tr_f1:>5.4f}  "
          f"{tr_acc:>5.4f}  {va_loss:>7.4f}  {va_auc:>6.4f}  {va_f1:>5.4f}  "
          f"{va_acc:>5.4f}  {lr_now:>8.6f}  {elapsed:>4.1f}s")

    if va_auc > best_val_auc:
        best_val_auc = va_auc
        best_epoch   = epoch
        patience_ctr = 0
        torch.save(enhanced_model.state_dict(),
                   checkpoint_path / "best_enhanced_model.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nBest val AUC : {best_val_auc:.4f} at epoch {best_epoch}")

In [ ]:
# Load best enhanced model
enhanced_model.load_state_dict(
    torch.load(checkpoint_path / "best_enhanced_model.pt")
)

# Evaluate
enhanced_val_results  = evaluate_model(
    enhanced_model, loader_val_enh, device, split_name="Enhanced — Validation"
)
enhanced_test_results = evaluate_model(
    enhanced_model, loader_test_enh, device, split_name="Enhanced — Test"
)

# Save
torch.save(enhanced_model.state_dict(),
           checkpoint_path / "best_enhanced_model.pt")

with open(checkpoint_path / "enhanced_model_results.pkl", "wb") as f:
    pickle.dump({
        "val" : enhanced_val_results,
        "test": enhanced_test_results,
        "history": history_enh,
        "best_epoch": 39,
        "batch_size": 64,
        "params": 202069
    }, f)

print("\nSaved enhanced_model_results.pkl")

# Quick comparison: baseline vs enhanced
print("\n" + "="*65)
print("BASELINE vs ENHANCED HeteroGNN — Test Set")
print("="*65)
print(f"{'Metric':<25} {'Baseline':>10} {'Enhanced':>10} {'Delta':>8}")
print("-"*55)

metrics = ["auc_roc","auc_pr","accuracy","f1","precision",
           "recall","specificity","mcc","kappa","brier",
           "sens_5far","sens_10far"]

for m in metrics:
    base = test_results[m]
    enh  = enhanced_test_results[m]
    delta = enh - base
    arrow = "↑" if delta > 0 else "↓"
    print(f"{m:<25} {base:>10.4f} {enh:>10.4f} "
          f"{arrow}{abs(delta):>6.4f}")

In [ ]:
# Updated full comparison table with enhanced model
print("\n" + "="*105)
print("FINAL THREE-TIER MODEL COMPARISON — TEST SET")
print("="*105)
print(f"{'Model':<28} {'AUC-ROC':>7} {'AUC-PR':>7} {'Acc':>6} {'F1':>6} "
      f"{'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6} {'Kappa':>6} "
      f"{'Brier':>6} {'S@5%':>6} {'S@10%':>6}")
print("-"*105)

print("─── ML Tier ───")
for name, res in ml_results.items():
    r = res["test"]
    print(f"{name:<28} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
          f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
          f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
          f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
          f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

print("─── DL Tier ───")
for name, res in dl_results.items():
    r = res["test"]
    print(f"{name:<28} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
          f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
          f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
          f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
          f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

print("─── Graph Tier ───")
# Baseline
r = test_results
print(f"{'Baseline HeteroGNN+LSTM':<28} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
      f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
      f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
      f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
      f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

# Enhanced
r = enhanced_test_results
print(f"{'Enhanced HeteroGNN+LSTM':<28} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
      f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
      f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
      f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
      f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

print("="*105)

# Save final comparison
final_comparison = {
    "ml"              : ml_results,
    "dl"              : dl_results,
    "graph_baseline"  : {"test": test_results,          "val": val_results},
    "graph_enhanced"  : {"test": enhanced_test_results, "val": enhanced_val_results},
    "history_baseline": history,
    "history_enhanced": history_enh
}
with open(checkpoint_path / "final_comparison.pkl", "wb") as f:
    pickle.dump(final_comparison, f)
print("\nSaved final_comparison.pkl")

In [ ]:
# ══════════════════════════════════════════════════════════
# MULTI-SEED EVALUATION — 5 seeds for all models
# ══════════════════════════════════════════════════════════

from numpy.random import random


SEEDS = [42, 123, 456, 789, 2024]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


def compute_metrics(y_true, y_score):
    y_pred  = (y_score >= 0.5).astype(int)
    cm      = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr, tpr, _    = roc_curve(y_true, y_score)
    return {
        "auc_roc"    : roc_auc_score(y_true, y_score),
        "auc_pr"     : average_precision_score(y_true, y_score),
        "accuracy"   : accuracy_score(y_true, y_pred),
        "f1"         : f1_score(y_true, y_pred,        zero_division=0),
        "precision"  : precision_score(y_true, y_pred, zero_division=0),
        "recall"     : recall_score(y_true, y_pred,    zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "mcc"        : matthews_corrcoef(y_true, y_pred),
        "kappa"      : cohen_kappa_score(y_true, y_pred),
        "brier"      : brier_score_loss(y_true, y_score),
        "sens_5far"  : float(tpr[np.searchsorted(fpr, 0.05)]),
        "sens_10far" : float(tpr[np.searchsorted(fpr, 0.10)]),
    }


def summarize_seeds(all_seed_results):
    """Compute mean ± std across seeds."""
    metrics = ["auc_roc","auc_pr","accuracy","f1","precision",
               "recall","specificity","mcc","kappa","brier",
               "sens_5far","sens_10far"]
    summary = {}
    for m in metrics:
        vals           = [r[m] for r in all_seed_results]
        summary[f"{m}_mean"] = np.mean(vals)
        summary[f"{m}_std"]  = np.std(vals)
    return summary


def train_graph_multiseed(ModelClass, model_kwargs, model_name,
                          ds_train, ds_val, ds_test,
                          batch_size, seeds=SEEDS,
                          n_epochs=50, patience=5):
    all_results = []
    loader_test = DataLoader(ds_test, batch_size=batch_size,
                             shuffle=False, num_workers=0)

    for seed in seeds:
        print(f"  [{model_name}] Seed {seed}...")
        set_seed(seed)

        model     = ModelClass(**model_kwargs).to(device)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3
        )
        loader_tr = DataLoader(ds_train, batch_size=batch_size,
                               shuffle=True,  num_workers=0)
        loader_vl = DataLoader(ds_val,   batch_size=batch_size,
                               shuffle=False, num_workers=0)

        best_val_auc = 0.0
        best_state   = None
        patience_ctr = 0

        for epoch in range(1, n_epochs + 1):
            model.train()
            for batch in loader_tr:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                loss   = criterion(logits, y)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            y_true, y_score = [], []
            with torch.no_grad():
                for batch in loader_vl:
                    X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                    logits = model(X, Avv, Avp, App, E)
                    y_true.extend(y.cpu().numpy())
                    y_score.extend(torch.sigmoid(logits).cpu().numpy())

            val_auc = roc_auc_score(y_true, y_score)
            scheduler.step(val_auc)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state   = {k: v.clone()
                                for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        # Test evaluation
        model.load_state_dict(best_state)
        model.eval()
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader_test:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        res = compute_metrics(np.array(y_true), np.array(y_score))
        res["seed"]         = seed
        res["best_val_auc"] = best_val_auc
        all_results.append(res)
        print(f"    Test AUC-ROC: {res['auc_roc']:.4f}  "
              f"AUC-PR: {res['auc_pr']:.4f}  "
              f"Val AUC: {best_val_auc:.4f}")

    summary = summarize_seeds(all_results)
    summary["model_name"]   = model_name
    summary["all_results"]  = all_results
    return summary


def train_dl_multiseed(ModelClass, model_kwargs, model_name,
                       ds_train_flat, ds_val_flat, ds_test_flat,
                       batch_size=16, seeds=SEEDS,
                       n_epochs=50, patience=5):
    all_results = []
    loader_test = DataLoader(ds_test_flat, batch_size=batch_size,
                             shuffle=False, num_workers=0)

    for seed in seeds:
        print(f"  [{model_name}] Seed {seed}...")
        set_seed(seed)

        model     = ModelClass(**model_kwargs).to(device)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3
        )
        loader_tr = DataLoader(ds_train_flat, batch_size=batch_size,
                               shuffle=True,  num_workers=0)
        loader_vl = DataLoader(ds_val_flat,   batch_size=batch_size,
                               shuffle=False, num_workers=0)

        best_val_auc = 0.0
        best_state   = None
        patience_ctr = 0

        for epoch in range(1, n_epochs + 1):
            model.train()
            for batch in loader_tr:
                x, y = [b.to(device) for b in batch]
                logits = model(x)
                loss   = criterion(logits, y)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            y_true, y_score = [], []
            with torch.no_grad():
                for batch in loader_vl:
                    x, y = [b.to(device) for b in batch]
                    logits = model(x)
                    y_true.extend(y.cpu().numpy())
                    y_score.extend(torch.sigmoid(logits).cpu().numpy())

            val_auc = roc_auc_score(y_true, y_score)
            scheduler.step(val_auc)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state   = {k: v.clone()
                                for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader_test:
                x, y = [b.to(device) for b in batch]
                logits = model(x)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        res = compute_metrics(np.array(y_true), np.array(y_score))
        res["seed"]         = seed
        res["best_val_auc"] = best_val_auc
        all_results.append(res)
        print(f"    Test AUC-ROC: {res['auc_roc']:.4f}  "
              f"AUC-PR: {res['auc_pr']:.4f}  "
              f"Val AUC: {best_val_auc:.4f}")

    summary = summarize_seeds(all_results)
    summary["model_name"]  = model_name
    summary["all_results"] = all_results
    return summary


def train_ml_multiseed(model_class, model_kwargs, model_name,
                       X_train, y_train, X_test, y_test,
                       seeds=SEEDS):
    all_results = []

    for seed in seeds:
        print(f"  [{model_name}] Seed {seed}...")
        kwargs = {**model_kwargs, "random_state": seed}
        m      = model_class(**kwargs)
        m.fit(X_train, y_train)

        y_score = m.predict_proba(X_test)[:, 1]
        res     = compute_metrics(y_test, y_score)
        res["seed"] = seed
        all_results.append(res)
        print(f"    Test AUC-ROC: {res['auc_roc']:.4f}  "
              f"AUC-PR: {res['auc_pr']:.4f}")

    summary = summarize_seeds(all_results)
    summary["model_name"]  = model_name
    summary["all_results"] = all_results
    return summary


print("Multi-seed evaluation setup complete.")
print(f"Seeds: {SEEDS}")
print(f"Models to evaluate:")
print(f"  ML  : LR, RF, XGBoost, LightGBM, SVM")
print(f"  DL  : BiLSTM, GRU, TCN, Transformer")
print(f"  Graph: Baseline HeteroGNN, Enhanced HeteroGNN")
print(f"\nEstimated time: 15-20 hours total")
print(f"Recommend running overnight.")

In [ ]:
import random
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.svm import SVC

SEEDS = [42, 123, 456, 789, 2024]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def compute_metrics(y_true, y_score):
    y_pred  = (y_score >= 0.5).astype(int)
    cm      = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr, tpr, _    = roc_curve(y_true, y_score)
    return {
        "auc_roc"    : roc_auc_score(y_true, y_score),
        "auc_pr"     : average_precision_score(y_true, y_score),
        "accuracy"   : accuracy_score(y_true, y_pred),
        "f1"         : f1_score(y_true, y_pred,        zero_division=0),
        "precision"  : precision_score(y_true, y_pred, zero_division=0),
        "recall"     : recall_score(y_true, y_pred,    zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "mcc"        : matthews_corrcoef(y_true, y_pred),
        "kappa"      : cohen_kappa_score(y_true, y_pred),
        "brier"      : brier_score_loss(y_true, y_score),
        "sens_5far"  : float(tpr[np.searchsorted(fpr, 0.05)]),
        "sens_10far" : float(tpr[np.searchsorted(fpr, 0.10)]),
    }

def summarize_seeds(all_seed_results):
    metrics = ["auc_roc","auc_pr","accuracy","f1","precision",
               "recall","specificity","mcc","kappa","brier",
               "sens_5far","sens_10far"]
    summary = {}
    for m in metrics:
        vals              = [r[m] for r in all_seed_results]
        summary[f"{m}_mean"] = np.mean(vals)
        summary[f"{m}_std"]  = np.std(vals)
    return summary

def train_ml_multiseed(model_class, model_kwargs, model_name,
                       X_train, y_train, X_test, y_test,
                       seeds=SEEDS):
    all_results = []
    for seed in seeds:
        print(f"  [{model_name}] Seed {seed}...")
        set_seed(seed)
        kwargs = {**model_kwargs, "random_state": seed}
        m      = model_class(**kwargs)
        m.fit(X_train, y_train)
        y_score = m.predict_proba(X_test)[:, 1]
        res     = compute_metrics(y_test, y_score)
        res["seed"] = seed
        all_results.append(res)
        print(f"    Test AUC-ROC: {res['auc_roc']:.4f}  "
              f"AUC-PR: {res['auc_pr']:.4f}")
    summary               = summarize_seeds(all_results)
    summary["model_name"] = model_name
    summary["all_results"]= all_results
    return summary

def train_dl_multiseed(ModelClass, model_kwargs, model_name,
                       ds_train_flat, ds_val_flat, ds_test_flat,
                       batch_size=16, seeds=SEEDS,
                       n_epochs=50, patience=5):
    all_results = []
    loader_test = DataLoader(ds_test_flat, batch_size=batch_size,
                             shuffle=False, num_workers=0)
    for seed in seeds:
        print(f"  [{model_name}] Seed {seed}...")
        set_seed(seed)
        model     = ModelClass(**model_kwargs).to(device)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3
        )
        loader_tr = DataLoader(ds_train_flat, batch_size=batch_size,
                               shuffle=True,  num_workers=0)
        loader_vl = DataLoader(ds_val_flat,   batch_size=batch_size,
                               shuffle=False, num_workers=0)
        best_val_auc = 0.0
        best_state   = None
        patience_ctr = 0

        for epoch in range(1, n_epochs + 1):
            model.train()
            for batch in loader_tr:
                x, y = [b.to(device) for b in batch]
                logits = model(x)
                loss   = criterion(logits, y)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            y_true, y_score = [], []
            with torch.no_grad():
                for batch in loader_vl:
                    x, y = [b.to(device) for b in batch]
                    logits = model(x)
                    y_true.extend(y.cpu().numpy())
                    y_score.extend(torch.sigmoid(logits).cpu().numpy())

            val_auc = roc_auc_score(y_true, y_score)
            scheduler.step(val_auc)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state   = {k: v.clone()
                                for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader_test:
                x, y = [b.to(device) for b in batch]
                logits = model(x)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        res = compute_metrics(np.array(y_true), np.array(y_score))
        res["seed"]         = seed
        res["best_val_auc"] = best_val_auc
        all_results.append(res)
        print(f"    Test AUC-ROC: {res['auc_roc']:.4f}  "
              f"AUC-PR: {res['auc_pr']:.4f}  "
              f"Val AUC: {best_val_auc:.4f}")

    summary               = summarize_seeds(all_results)
    summary["model_name"] = model_name
    summary["all_results"]= all_results
    return summary

def train_graph_multiseed(ModelClass, model_kwargs, model_name,
                          ds_train, ds_val, ds_test,
                          batch_size, seeds=SEEDS,
                          n_epochs=50, patience=5):
    all_results = []
    loader_test = DataLoader(ds_test, batch_size=batch_size,
                             shuffle=False, num_workers=0)
    for seed in seeds:
        print(f"  [{model_name}] Seed {seed}...")
        set_seed(seed)
        model     = ModelClass(**model_kwargs).to(device)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3
        )
        loader_tr = DataLoader(ds_train, batch_size=batch_size,
                               shuffle=True,  num_workers=0)
        loader_vl = DataLoader(ds_val,   batch_size=batch_size,
                               shuffle=False, num_workers=0)
        best_val_auc = 0.0
        best_state   = None
        patience_ctr = 0

        for epoch in range(1, n_epochs + 1):
            model.train()
            for batch in loader_tr:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                loss   = criterion(logits, y)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            y_true, y_score = [], []
            with torch.no_grad():
                for batch in loader_vl:
                    X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                    logits = model(X, Avv, Avp, App, E)
                    y_true.extend(y.cpu().numpy())
                    y_score.extend(torch.sigmoid(logits).cpu().numpy())

            val_auc = roc_auc_score(y_true, y_score)
            scheduler.step(val_auc)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state   = {k: v.clone()
                                for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader_test:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        res = compute_metrics(np.array(y_true), np.array(y_score))
        res["seed"]         = seed
        res["best_val_auc"] = best_val_auc
        all_results.append(res)
        print(f"    Test AUC-ROC: {res['auc_roc']:.4f}  "
              f"AUC-PR: {res['auc_pr']:.4f}  "
              f"Val AUC: {best_val_auc:.4f}")

    summary               = summarize_seeds(all_results)
    summary["model_name"] = model_name
    summary["all_results"]= all_results
    return summary


# ══════════════════════════════════════════════════════════
# RUN ALL — ML → DL → Graph
# ══════════════════════════════════════════════════════════
multiseed_results = {}

# ── ML Tier ───────────────────────────────────────────────
print("="*60)
print("ML TIER — Multi-Seed Evaluation")
print("="*60)

multiseed_results["Logistic Regression"] = train_ml_multiseed(
    LogisticRegression,
    {"C": 0.01, "penalty": "l2", "max_iter": 1000,
     "solver": "saga", "class_weight": "balanced"},
    "Logistic Regression",
    X_train_sc, y_train_ml, X_test_sc, y_test_ml
)

multiseed_results["Random Forest"] = train_ml_multiseed(
    RandomForestClassifier,
    {"n_estimators": 500, "max_depth": None,
     "min_samples_leaf": 1, "class_weight": "balanced",
     "n_jobs": -1},
    "Random Forest",
    X_train_ml, y_train_ml, X_test_ml, y_test_ml
)

multiseed_results["XGBoost"] = train_ml_multiseed(
    xgb.XGBClassifier,
    {"n_estimators": 500, "max_depth": 7,
     "learning_rate": 0.1, "subsample": 0.9,
     "scale_pos_weight": scale_pos,
     "eval_metric": "auc", "n_jobs": -1},
    "XGBoost",
    X_train_ml, y_train_ml, X_test_ml, y_test_ml
)

multiseed_results["LightGBM"] = train_ml_multiseed(
    lgb.LGBMClassifier,
    {"n_estimators": 500, "max_depth": 20,
     "learning_rate": 0.1, "num_leaves": 127,
     "class_weight": "balanced",
     "n_jobs": -1, "verbose": -1},
    "LightGBM",
    X_train_ml, y_train_ml, X_test_ml, y_test_ml
)

multiseed_results["SVM"] = train_ml_multiseed(
    SVC,
    {"C": 10.0, "gamma": "scale", "kernel": "rbf",
     "class_weight": "balanced", "probability": True},
    "SVM",
    X_train_sc, y_train_ml, X_test_sc, y_test_ml
)

with open(checkpoint_path / "multiseed_ml.pkl", "wb") as f:
    pickle.dump({k: multiseed_results[k]
                 for k in ["Logistic Regression","Random Forest",
                           "XGBoost","LightGBM","SVM"]}, f)
print("\nSaved multiseed_ml.pkl")

# ── DL Tier ───────────────────────────────────────────────
print("\n" + "="*60)
print("DL TIER — Multi-Seed Evaluation")
print("="*60)

multiseed_results["BiLSTM"] = train_dl_multiseed(
    BiLSTMModel,
    {"input_dim": FLAT_DIM, "hidden_dim": 128,
     "num_layers": 3, "dropout": 0.3},
    "BiLSTM",
    ds_train_flat, ds_val_flat, ds_test_flat, batch_size=16
)

multiseed_results["GRU"] = train_dl_multiseed(
    GRUModel,
    {"input_dim": FLAT_DIM, "hidden_dim": 64,
     "num_layers": 2, "dropout": 0.3},
    "GRU",
    ds_train_flat, ds_val_flat, ds_test_flat, batch_size=16
)

multiseed_results["TCN"] = train_dl_multiseed(
    TCNModel,
    {"input_dim": FLAT_DIM, "hidden_dim": 64,
     "num_layers": 3, "kernel_size": 3, "dropout": 0.3},
    "TCN",
    ds_train_flat, ds_val_flat, ds_test_flat, batch_size=16
)

multiseed_results["Transformer"] = train_dl_multiseed(
    TransformerModel,
    {"input_dim": FLAT_DIM, "hidden_dim": 32,
     "num_heads": 4, "num_layers": 3,
     "dropout": 0.3, "max_seq_len": 8},
    "Transformer",
    ds_train_flat, ds_val_flat, ds_test_flat, batch_size=16
)

with open(checkpoint_path / "multiseed_dl.pkl", "wb") as f:
    pickle.dump({k: multiseed_results[k]
                 for k in ["BiLSTM","GRU","TCN","Transformer"]}, f)
print("\nSaved multiseed_dl.pkl")

# ── Graph Tier ────────────────────────────────────────────
print("\n" + "="*60)
print("GRAPH TIER — Multi-Seed Evaluation")
print("="*60)

multiseed_results["Baseline HeteroGNN"] = train_graph_multiseed(
    SceneRiskModel,
    {"node_dim": NODE_DIM, "hidden_dim": 64,
     "edge_dim": EDGE_DIM, "lstm_layers": 2, "dropout": 0.3},
    "Baseline HeteroGNN",
    ds_train, ds_val, ds_test, batch_size=16
)

multiseed_results["Enhanced HeteroGNN"] = train_graph_multiseed(
    EnhancedSceneRiskModel,
    {"node_dim": NODE_DIM, "hidden_dim": 64,
     "edge_dim": EDGE_DIM, "num_heads": 4,
     "lstm_layers": 2, "dropout": 0.3},
    "Enhanced HeteroGNN",
    ds_train, ds_val, ds_test, batch_size=64
)

with open(checkpoint_path / "multiseed_graph.pkl", "wb") as f:
    pickle.dump({k: multiseed_results[k]
                 for k in ["Baseline HeteroGNN",
                           "Enhanced HeteroGNN"]}, f)
print("\nSaved multiseed_graph.pkl")

# ══════════════════════════════════════════════════════════
# FINAL SUMMARY TABLE
# ══════════════════════════════════════════════════════════
print("\n" + "="*110)
print("FINAL RESULTS — Mean ± Std over 5 Seeds (Test Set)")
print("="*110)
print(f"{'Model':<28} {'AUC-ROC':>14} {'AUC-PR':>14} "
      f"{'F1':>12} {'MCC':>12} {'S@5%FAR':>12}")
print("-"*110)

tiers = [
    ("ML Tier",    ["Logistic Regression","Random Forest",
                    "XGBoost","LightGBM","SVM"]),
    ("DL Tier",    ["BiLSTM","GRU","TCN","Transformer"]),
    ("Graph Tier", ["Baseline HeteroGNN","Enhanced HeteroGNN"]),
]

for tier_name, model_names in tiers:
    print(f"─── {tier_name} ───")
    for name in model_names:
        r = multiseed_results[name]
        print(f"{name:<28} "
              f"{r['auc_roc_mean']:>7.4f}±{r['auc_roc_std']:.4f}  "
              f"{r['auc_pr_mean']:>7.4f}±{r['auc_pr_std']:.4f}  "
              f"{r['f1_mean']:>6.4f}±{r['f1_std']:.4f}  "
              f"{r['mcc_mean']:>6.4f}±{r['mcc_std']:.4f}  "
              f"{r['sens_5far_mean']:>6.4f}±{r['sens_5far_std']:.4f}")

print("="*110)

with open(checkpoint_path / "multiseed_all.pkl", "wb") as f:
    pickle.dump(multiseed_results, f)
print("\nSaved multiseed_all.pkl")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from matplotlib.collections import LineCollection

# Use busiest video
busiest_video = (
    df.groupby("video_id")
      .size()
      .idxmax()
)

df_plot = df[
    (df["video_id"] == busiest_video) &
    (df["agent_type"] == "vehicle")
].copy()

print(f"Busiest video  : {busiest_video}")
print(f"Rows           : {len(df_plot):,}")
print(f"Unique tracks  : {df_plot['track_id'].nunique():,}")
print(f"Time range     : {df_plot['t'].min():.1f}s → {df_plot['t'].max():.1f}s")
print(f"Distance range : {df_plot['y_local_m'].min():.1f}m → {df_plot['y_local_m'].max():.1f}m")
print(f"Speed range    : {df_plot['speed'].min():.2f} → {df_plot['speed'].max():.2f} m/s")

# Convert speed to km/h for the plot
df_plot["speed_kmh"] = df_plot["speed"] * 3.6

print(f"\nSpeed (km/h)   : {df_plot['speed_kmh'].min():.1f} → {df_plot['speed_kmh'].max():.1f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
import numpy as np

fig, ax = plt.subplots(figsize=(16, 7))

# Color map — blue=fast, red=slow (matching the reference image style)
cmap    = plt.cm.jet
norm    = mcolors.Normalize(vmin=0, vmax=72)

# Plot each track as a colored line segment
tracks = df_plot.groupby("track_id")

plotted = 0
for track_id, track in tracks:
    track = track.sort_values("t")

    if len(track) < 3:
        continue

    t = track["t"].values
    y = track["y_local_m"].values
    s = track["speed_kmh"].values

    # Build line segments: each segment connects two consecutive points
    points  = np.array([t, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    speeds   = (s[:-1] + s[1:]) / 2   # mean speed per segment

    lc = LineCollection(
        segments,
        cmap=cmap,
        norm=norm,
        linewidth=0.6,
        alpha=0.7
    )
    lc.set_array(speeds)
    ax.add_collection(lc)
    plotted += 1

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label("Speed (km/h)", fontsize=11)

# Axes
ax.set_xlim(df_plot["t"].min(), df_plot["t"].max())
ax.set_ylim(df_plot["y_local_m"].min() - 1, df_plot["y_local_m"].max() + 1)
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Distance (m)", fontsize=12)
ax.set_title(
    "Spatial-Temporal Trajectory Diagram — Vehicle Speed over Time\n"
    f"Centerpoint Road / Outlet Mall Intersection  |  {plotted:,} tracks",
    fontsize=13, fontweight="bold"
)

# Add horizontal reference line at y=0 (intersection center)
ax.axhline(y=0, color="black", linestyle="--",
           linewidth=0.8, alpha=0.4, label="Intersection center")
ax.legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.savefig(checkpoint_path / "spatiotemporal_trajectory.png",
            dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: spatiotemporal_trajectory.png")
print(f"Tracks plotted: {plotted:,}")

In [ ]:
# Compute cumulative distance per track
df_plot = df_plot.sort_values(["track_id", "t"]).copy()

df_plot["dx"] = df_plot.groupby("track_id")["x_local_m"].diff().fillna(0)
df_plot["dy"] = df_plot.groupby("track_id")["y_local_m"].diff().fillna(0)
df_plot["ds"] = np.sqrt(df_plot["dx"]**2 + df_plot["dy"]**2)
df_plot["cum_dist"] = df_plot.groupby("track_id")["ds"].cumsum()

fig, ax = plt.subplots(figsize=(16, 7))

cmap = plt.cm.jet
norm = mcolors.Normalize(vmin=0, vmax=72)

tracks  = df_plot.groupby("track_id")
plotted = 0

for track_id, track in tracks:
    track = track.sort_values("t")
    if len(track) < 3:
        continue

    t  = track["t"].values
    cd = track["cum_dist"].values
    s  = track["speed_kmh"].values

    points   = np.array([t, cd]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    speeds   = (s[:-1] + s[1:]) / 2

    lc = LineCollection(
        segments, cmap=cmap, norm=norm,
        linewidth=0.6, alpha=0.7
    )
    lc.set_array(speeds)
    ax.add_collection(lc)
    plotted += 1

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label("Speed (km/h)", fontsize=11)

ax.set_xlim(df_plot["t"].min(), df_plot["t"].max())
ax.set_ylim(0, df_plot["cum_dist"].max())
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Cumulative Distance Traveled (m)", fontsize=12)
ax.set_title(
    "Spatial-Temporal Trajectory Diagram — Vehicle Speed over Time\n"
    f"Centerpoint Road / Outlet Mall Intersection  |  {plotted:,} tracks",
    fontsize=13, fontweight="bold"
)

plt.tight_layout()
plt.savefig(checkpoint_path / "spatiotemporal_trajectory_v2.png",
            dpi=200, bbox_inches="tight")
plt.show()

print(f"Tracks plotted : {plotted:,}")
print(f"Max cum dist   : {df_plot['cum_dist'].max():.1f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

cmap = plt.cm.jet
norm = mcolors.Normalize(vmin=0, vmax=72)

tracks  = df_plot.groupby("track_id")
plotted = 0

for track_id, track in tracks:
    track = track.sort_values("t")
    if len(track) < 5:
        continue

    t = track["t"].values
    y = track["y_local_m"].values
    s = track["speed_kmh"].values

    points   = np.array([t, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    speeds   = (s[:-1] + s[1:]) / 2

    lc = LineCollection(
        segments, cmap=cmap, norm=norm,
        linewidth=0.5, alpha=0.5
    )
    lc.set_array(speeds)
    ax.add_collection(lc)
    plotted += 1

# Overlay conflict events
conflict_times = pairs[
    (pairs["video_id"] == busiest_video) &
    (pairs["pair_conflict_label"] > 0)
]["t"].values

conflict_y_i = pairs[
    (pairs["video_id"] == busiest_video) &
    (pairs["pair_conflict_label"] > 0)
]["y_i"].values

conflict_y_j = pairs[
    (pairs["video_id"] == busiest_video) &
    (pairs["pair_conflict_label"] > 0)
]["y_j"].values

conflict_y_mid = (conflict_y_i + conflict_y_j) / 2

ax.scatter(
    conflict_times, conflict_y_mid,
    c="white", s=8, zorder=5,
    alpha=0.6, label="Conflict event",
    edgecolors="red", linewidths=0.3
)

# Signal cycle visualization — add shaded bands every ~90s
for cycle_start in range(0, 3600, 90):
    ax.axvspan(cycle_start, cycle_start + 45,
               alpha=0.04, color="green")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label("Speed (km/h)", fontsize=11)

ax.set_xlim(df_plot["t"].min(), df_plot["t"].max())
ax.set_ylim(-16, 17)
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Position in Intersection (m)", fontsize=12)
ax.set_title(
    "Spatial-Temporal Trajectory Diagram with Conflict Events\n"
    f"Centerpoint Road / Outlet Mall  |  {plotted:,} vehicle tracks  |  "
    f"{len(conflict_times):,} conflict events overlaid",
    fontsize=12, fontweight="bold"
)

# Intersection boundary lines
ax.axhline(y=16.5,  color="gray", linewidth=1.0,
           linestyle="--", alpha=0.6, label="ROI boundary")
ax.axhline(y=-14.6, color="gray", linewidth=1.0,
           linestyle="--", alpha=0.6)
ax.axhline(y=0, color="white", linewidth=0.8,
           linestyle=":", alpha=0.4, label="Intersection center")

ax.legend(fontsize=9, loc="upper right",
          facecolor="black", labelcolor="white")
ax.set_facecolor("#0a0a0a")
fig.patch.set_facecolor("#0a0a0a")
ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
cbar.set_label("Speed (km/h)", fontsize=11, color="white")

for spine in ax.spines.values():
    spine.set_edgecolor("white")

plt.tight_layout()
plt.savefig(checkpoint_path / "spatiotemporal_trajectory_v3.png",
            dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"Tracks plotted    : {plotted:,}")
print(f"Conflict events   : {len(conflict_times):,}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve
import seaborn as sns

# ── Collect all model results for plotting ─────────────────
# We use the single-run results (not multi-seed) for curves
# Multi-seed gives mean±std for the table
# Single best run gives the actual probability scores for curves

plot_models = {
    # ML Tier
    "LR"           : ml_results["Logistic Regression"]["test"],
    "RF"           : ml_results["Random Forest"]["test"],
    "XGBoost"      : ml_results["XGBoost"]["test"],
    "LightGBM"     : ml_results["LightGBM"]["test"],
    "SVM"          : ml_results["SVM"]["test"],
    # DL Tier
    "BiLSTM"       : dl_results["BiLSTM"]["test"],
    "GRU"          : dl_results["GRU"]["test"],
    "TCN"          : dl_results["TCN"]["test"],
    "Transformer"  : dl_results["Transformer"]["test"],
    # Graph Tier
    "Baseline HERMES" : test_results,
    "Enhanced HERMES" : enhanced_test_results,
}

tier_styles = {
    "LR"              : ("steelblue",  "--",  1.2, "ML"),
    "RF"              : ("steelblue",  "-.",  1.2, "ML"),
    "XGBoost"         : ("dodgerblue", "-",   1.5, "ML"),
    "LightGBM"        : ("royalblue",  ":",   1.5, "ML"),
    "SVM"             : ("lightblue",  "--",  1.2, "ML"),
    "BiLSTM"          : ("mediumseagreen", "--", 1.5, "DL"),
    "GRU"             : ("limegreen",  "-.",  1.5, "DL"),
    "TCN"             : ("darkgreen",  ":",   1.5, "DL"),
    "Transformer"     : ("green",      "-",   2.0, "DL"),
    "Baseline HERMES" : ("orange",     "--",  2.0, "Graph"),
    "Enhanced HERMES" : ("red",        "-",   2.5, "Graph"),
}

# ══════════════════════════════════════════════════════════
# Figure 1: ROC + PR curves side by side
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for name, res in plot_models.items():
    color, ls, lw, tier = tier_styles[name]
    y_true  = res["y_true"]
    y_score = res["y_score"]
    auc_roc = res["auc_roc"]
    auc_pr  = res["auc_pr"]

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    axes[0].plot(fpr, tpr, color=color, linestyle=ls,
                 linewidth=lw, label=f"{name} ({auc_roc:.4f})",
                 alpha=0.85)

    # PR
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    axes[1].plot(rec, prec, color=color, linestyle=ls,
                 linewidth=lw, label=f"{name} ({auc_pr:.4f})",
                 alpha=0.85)

# ROC formatting
axes[0].plot([0,1],[0,1], "k--", alpha=0.3, linewidth=1)
axes[0].axvline(x=0.05, color="gray", linestyle=":",
                alpha=0.6, linewidth=1, label="5% FAR")
axes[0].axvline(x=0.10, color="gray", linestyle="--",
                alpha=0.6, linewidth=1, label="10% FAR")
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate", fontsize=12)
axes[0].set_title("ROC Curves — All Models", fontsize=13,
                  fontweight="bold")
axes[0].legend(fontsize=7.5, loc="lower right",
               ncol=1, framealpha=0.9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

# PR formatting
baseline_rate = np.mean(test_labels)
axes[1].axhline(y=baseline_rate, color="k", linestyle="--",
                alpha=0.3, linewidth=1,
                label=f"Baseline ({baseline_rate:.2f})")
axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("Precision-Recall Curves — All Models",
                  fontsize=13, fontweight="bold")
axes[1].legend(fontsize=7.5, loc="upper right",
               ncol=1, framealpha=0.9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

# Tier legend patches
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
tier_legend = [
    Patch(facecolor="steelblue",      label="ML Tier"),
    Patch(facecolor="mediumseagreen", label="DL Tier"),
    Patch(facecolor="red",            label="Graph Tier"),
]
fig.legend(handles=tier_legend, loc="lower center",
           ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.04))

plt.suptitle("HERMES Framework — Complete Model Comparison",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(checkpoint_path / "roc_pr_curves.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: roc_pr_curves.png")

# ══════════════════════════════════════════════════════════
# Figure 2: Confusion matrices — key models only
# ══════════════════════════════════════════════════════════
key_models = {
    "XGBoost\n(Best ML)"        : ml_results["XGBoost"]["test"],
    "Transformer\n(Best DL)"    : dl_results["Transformer"]["test"],
    "Baseline HERMES\n(Graph)"  : test_results,
    "Enhanced HERMES\n(HERMES)" : enhanced_test_results,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, res) in zip(axes, key_models.items()):
    cm      = res["cm"]
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    annot = np.array([
        [f"{cm[i,j]:,}\n({cm_norm[i,j]:.1%})"
         for j in range(2)]
        for i in range(2)
    ])

    sns.heatmap(
        cm_norm, ax=ax,
        annot=annot, fmt="",
        cmap="Blues",
        xticklabels=["Pred Safe", "Pred Conflict"],
        yticklabels=["True Safe", "True Conflict"],
        vmin=0, vmax=1,
        linewidths=0.5,
        cbar=False
    )

    auc = res["auc_roc"]
    f1  = res["f1"]
    mcc = res["mcc"]
    ax.set_title(
        f"{name}\n"
        f"AUC={auc:.4f}  F1={f1:.4f}  MCC={mcc:.4f}",
        fontsize=10, fontweight="bold"
    )
    ax.set_ylabel("Actual" if ax == axes[0] else "")
    ax.set_xlabel("Predicted", fontsize=10)

plt.suptitle(
    "Confusion Matrices — Best Model Per Tier (Test Set)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "confusion_matrices_key.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices_key.png")

# ══════════════════════════════════════════════════════════
# Figure 3: Bar chart — key metrics comparison
# ══════════════════════════════════════════════════════════
metrics_to_bar = ["auc_roc","auc_pr","f1","mcc","sens_5far","sens_10far"]
metric_labels  = ["AUC-ROC","AUC-PR","F1","MCC","Sens@5%FAR","Sens@10%FAR"]

model_order = list(plot_models.keys())
tier_colors_bar = {
    "LR": "#AED6F1", "RF": "#AED6F1",
    "XGBoost": "#AED6F1", "LightGBM": "#AED6F1", "SVM": "#AED6F1",
    "BiLSTM": "#A9DFBF", "GRU": "#A9DFBF",
    "TCN": "#A9DFBF", "Transformer": "#A9DFBF",
    "Baseline HERMES": "#F0B27A",
    "Enhanced HERMES": "#E74C3C",
}
bar_colors = [tier_colors_bar[m] for m in model_order]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, metric, label in zip(axes, metrics_to_bar, metric_labels):
    values = [plot_models[m][metric] for m in model_order]
    bars   = ax.bar(model_order, values,
                    color=bar_colors, edgecolor="white",
                    linewidth=0.5)

    # Value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f"{val:.3f}", ha="center", va="bottom",
                fontsize=7, rotation=90)

    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_ylim(min(values)*0.95, min(max(values)*1.08, 1.0))
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(axis="y", alpha=0.3)

    # Tier separator lines
    ax.axvline(x=4.5, color="gray", linestyle="--", alpha=0.5)
    ax.axvline(x=8.5, color="gray", linestyle="--", alpha=0.5)

# Tier legend
tier_patches = [
    Patch(facecolor="#AED6F1", label="ML Tier"),
    Patch(facecolor="#A9DFBF", label="DL Tier"),
    Patch(facecolor="#F0B27A", label="Baseline Graph"),
    Patch(facecolor="#E74C3C", label="Enhanced HERMES"),
]
fig.legend(handles=tier_patches, loc="lower center",
           ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle(
    "HERMES Framework — Metric Comparison Across All Models (Test Set)",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "metric_comparison_bars.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: metric_comparison_bars.png")

In [ ]:
print("="*100)
print("COMPLETE METRICS — ALL MODELS — TEST SET (Single Best Run)")
print("="*100)
print(f"{'Model':<25} {'AUC-ROC':>7} {'AUC-PR':>7} {'Acc':>6} {'F1':>6} "
      f"{'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6} {'Kappa':>6} "
      f"{'Brier':>6} {'S@5%':>6} {'S@10%':>6}")
print("-"*100)

tiers = [
    ("ML Tier", {
        "LR"      : ml_results["Logistic Regression"]["test"],
        "RF"      : ml_results["Random Forest"]["test"],
        "XGBoost" : ml_results["XGBoost"]["test"],
        "LightGBM": ml_results["LightGBM"]["test"],
        "SVM"     : ml_results["SVM"]["test"],
    }),
    ("DL Tier", {
        "BiLSTM"     : dl_results["BiLSTM"]["test"],
        "GRU"        : dl_results["GRU"]["test"],
        "TCN"        : dl_results["TCN"]["test"],
        "Transformer": dl_results["Transformer"]["test"],
    }),
    ("Graph Tier", {
        "Baseline HERMES": test_results,
        "Enhanced HERMES": enhanced_test_results,
    }),
]

for tier_name, models in tiers:
    print(f"\n─── {tier_name} ───")
    for name, r in models.items():
        print(f"{name:<25} {r['auc_roc']:>7.4f} {r['auc_pr']:>7.4f} "
              f"{r['accuracy']:>6.4f} {r['f1']:>6.4f} {r['precision']:>6.4f} "
              f"{r['recall']:>6.4f} {r['specificity']:>6.4f} {r['mcc']:>6.4f} "
              f"{r['kappa']:>6.4f} {r['brier']:>6.4f} "
              f"{r['sens_5far']:>6.4f} {r['sens_10far']:>6.4f}")

print("\n" + "="*100)
print("MULTI-SEED RESULTS — Mean ± Std over 5 Seeds (Test Set)")
print("="*100)
print(f"{'Model':<25} {'AUC-ROC':>16} {'AUC-PR':>16} {'F1':>14} "
      f"{'MCC':>14} {'S@5%FAR':>14}")
print("-"*100)

ms_tiers = [
    ("ML Tier",    ["Logistic Regression","Random Forest",
                    "XGBoost","LightGBM","SVM"]),
    ("DL Tier",    ["BiLSTM","GRU","TCN","Transformer"]),
    ("Graph Tier", ["Baseline HeteroGNN","Enhanced HeteroGNN"]),
]

for tier_name, model_names in ms_tiers:
    print(f"\n─── {tier_name} ───")
    for name in model_names:
        r = multiseed_results[name]
        print(f"{name:<25} "
              f"{r['auc_roc_mean']:>7.4f}±{r['auc_roc_std']:.4f}  "
              f"{r['auc_pr_mean']:>7.4f}±{r['auc_pr_std']:.4f}  "
              f"{r['f1_mean']:>6.4f}±{r['f1_std']:.4f}  "
              f"{r['mcc_mean']:>6.4f}±{r['mcc_std']:.4f}  "
              f"{r['sens_5far_mean']:>6.4f}±{r['sens_5far_std']:.4f}")

In [ ]:
# ── Load from disk without overwriting in-memory multiseed_results ───────────
import pickle
with open(checkpoint_path / "multiseed_all.pkl", "rb") as f:
    ms_results = pickle.load(f)

# ── Full multiseed table — all 12 metrics ─────────────────
metrics_ms = [
    ("AUC-ROC",  "auc_roc"),
    ("AUC-PR",   "auc_pr"),
    ("Acc",      "accuracy"),
    ("F1",       "f1"),
    ("Prec",     "precision"),
    ("Rec",      "recall"),
    ("Spec",     "specificity"),
    ("MCC",      "mcc"),
    ("Kappa",    "kappa"),
    ("Brier",    "brier"),
    ("S@5%",     "sens_5far"),
    ("S@10%",    "sens_10far"),
]

col_w  = 14
header = f"{'Model':<28}" + "".join(f"{h:>{col_w}}" for h, _ in metrics_ms)
sep    = "=" * (28 + col_w * len(metrics_ms))

print(sep)
print("MULTI-SEED RESULTS — Mean ± Std over 5 Seeds (Test Set) — ALL METRICS")
print(sep)
print(header)
print("-" * (28 + col_w * len(metrics_ms)))

ms_tiers = [
    ("ML Tier",    ["Logistic Regression", "Random Forest",
                    "XGBoost", "LightGBM", "SVM"]),
    ("DL Tier",    ["BiLSTM", "GRU", "TCN", "Transformer"]),
    ("Graph Tier", ["Baseline HeteroGNN", "Enhanced HeteroGNN"]),
]

for tier_name, model_names in ms_tiers:
    print(f"\n─── {tier_name} ───")
    for name in model_names:
        r    = ms_results[name]
        row  = f"{name:<28}"
        for _, key in metrics_ms:
            mean = r[f"{key}_mean"]
            std  = r[f"{key}_std"]
            cell = f"{mean:.4f}±{std:.4f}"
            row += f"{cell:>{col_w}}"
        print(row)

print(sep)

# ── Export to CSV ─────────────────────────────────────────
import pandas as pd
rows = []
for tier_name, model_names in ms_tiers:
    for name in model_names:
        r   = ms_results[name]
        row = {"Model": name, "Tier": tier_name}
        for h, key in metrics_ms:
            row[f"{h}_mean"] = round(r[f"{key}_mean"], 4)
            row[f"{h}_std"]  = round(r[f"{key}_std"],  4)
            row[f"{h}"]      = f"{r[f'{key}_mean']:.4f}±{r[f'{key}_std']:.4f}"
        rows.append(row)

df_ms = pd.DataFrame(rows)
df_ms.to_csv("table_4_3_multiseed_all_metrics.csv", index=False)
print("\nSaved: table_4_3_multiseed_all_metrics.csv")

In [ ]:
"""
Publication-ready ROC and PR curves — 4 key models
Figure 1: ROC curve
Figure 2: Precision-Recall curve

Requires: ml_results, dl_results, test_results, enhanced_test_results,
          test_labels (in memory)
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
import warnings
warnings.filterwarnings("ignore")

# ── Publication style ─────────────────────────────────────
matplotlib.rcParams.update({
    "font.family"        : "serif",
    "font.serif"         : ["Times New Roman", "DejaVu Serif"],
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "axes.linewidth"     : 1.2,
    "axes.labelsize"     : 18,
    "xtick.labelsize"    : 16,
    "ytick.labelsize"    : 16,
    "legend.fontsize"    : 15,
    "legend.framealpha"  : 0.92,
    "legend.edgecolor"   : "#CCCCCC",
    "figure.dpi"         : 300,
    "savefig.dpi"        : 300,
    "savefig.bbox"       : "tight",
    "savefig.facecolor"  : "white",
})

# ── Model registry — 4 key models only ───────────────────
key_models = {
    "XGBoost"         : ml_results["XGBoost"]["test"],
    "Transformer"     : dl_results["Transformer"]["test"],
    "Baseline HERMES" : test_results,
    "Enhanced HERMES" : enhanced_test_results,
}

# ── Visual style per model ────────────────────────────────
styles = {
    "XGBoost"         : {"color": "#1A5276", "ls": "-.",  "lw": 2.2, "marker": "o",  "ms": 7},
    "Transformer"     : {"color": "#1E8449", "ls": "--",  "lw": 2.2, "marker": "s",  "ms": 7},
    "Baseline HERMES" : {"color": "#E67E22", "ls": ":",   "lw": 2.4, "marker": "^",  "ms": 8},
    "Enhanced HERMES" : {"color": "#C0392B", "ls": "-",   "lw": 3.0, "marker": "D",  "ms": 9},
}

# ── Compute curves ────────────────────────────────────────
curves = {}
for name, res in key_models.items():
    y_true  = res["y_true"]
    y_score = res["y_score"]
    fpr, tpr, _ = roc_curve(y_true, y_score)
    pre, rec, _ = precision_recall_curve(y_true, y_score)
    curves[name] = {
        "fpr": fpr, "tpr": tpr,
        "pre": pre, "rec": rec,
        "auc_roc": res["auc_roc"],
        "auc_pr" : res["auc_pr"],
        "s5"     : res["sens_5far"],
        "s10"    : res["sens_10far"],
    }

baseline_rate = float(np.mean(test_labels))

# ═════════════════════════════════════════════════════════
# FIGURE 1 — ROC CURVE
# ═════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7.5, 7.0), facecolor="white")

# Diagonal reference
ax.plot([0, 1], [0, 1], color="#AAAAAA", linestyle="--",
        linewidth=1.2, label="Random classifier (AUC = 0.500)", zorder=1)

# FAR reference lines
for x_far, label in [(0.05, "5% FAR"), (0.10, "10% FAR")]:
    ax.axvline(x=x_far, color="#000", linestyle=":",
               linewidth=1.0, alpha=0.7, zorder=1)
    ax.text(x_far + 0.005, 0.04, label,
            fontsize=12, color="#000", rotation=90, va="bottom")

# Model curves
for name, c in curves.items():
    s   = styles[name]
    lbl = (f"{name}  "
           f"(AUC = {c['auc_roc']:.4f}, "
           f"S@5% = {c['s5']:.4f})")
    ax.plot(c["fpr"], c["tpr"],
            color=s["color"], linestyle=s["ls"], linewidth=s["lw"],
            label=lbl, alpha=0.92, zorder=3)

    # Mark operating points at 5% and 10% FAR
    for x_op in [0.05, 0.10]:
        idx = np.searchsorted(c["fpr"], x_op)
        if idx < len(c["tpr"]):
            ax.plot(c["fpr"][idx], c["tpr"][idx],
                    marker=s["marker"], color=s["color"],
                    markersize=s["ms"], zorder=4,
                    markeredgecolor="white", markeredgewidth=1.2)

ax.set_xlabel("False Positive Rate (1 − Specificity)", labelpad=10)
ax.set_ylabel("True Positive Rate (Sensitivity)",       labelpad=10)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.set_xticks(np.arange(0, 1.1, 0.2))
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.grid(color="#EBEBEB", linewidth=0.8, zorder=0)
ax.tick_params(axis="both", which="both", length=4)

ax.legend(loc="lower right",
          fontsize=12,
          handlelength=2.2, handleheight=1.0,
          borderpad=0.8, labelspacing=0.6)

plt.tight_layout()
plt.savefig("fig_4_3a_roc_curve.pdf")
plt.savefig("fig_4_3a_roc_curve.png")
plt.show()
print("Saved: fig_4_3a_roc_curve.pdf / .png")


# ═════════════════════════════════════════════════════════
# FIGURE 2 — PRECISION-RECALL CURVE
# ═════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7.5, 7.0), facecolor="white")

# No-skill baseline
ax.axhline(y=baseline_rate, color="#AAAAAA", linestyle="--",
           linewidth=1.2,
           label=f"No-skill classifier (AP = {baseline_rate:.3f})",
           zorder=1)
ax.text(0.72, baseline_rate + 0.008,
        f"Conflict rate = {baseline_rate:.3f}",
        fontsize=12, color="#888888")

# Model curves
for name, c in curves.items():
    s   = styles[name]
    lbl = f"{name}  (AP = {c['auc_pr']:.4f})"
    ax.plot(c["rec"], c["pre"],
            color=s["color"], linestyle=s["ls"], linewidth=s["lw"],
            label=lbl, alpha=0.92, zorder=3)

    # Mark point at recall = 0.95 (high-recall operating point)
    idx = np.searchsorted(c["rec"][::-1], 0.95)
    idx = len(c["rec"]) - 1 - idx
    if 0 <= idx < len(c["rec"]):
        ax.plot(c["rec"][idx], c["pre"][idx],
                marker=s["marker"], color=s["color"],
                markersize=s["ms"], zorder=4,
                markeredgecolor="white", markeredgewidth=1.2)

ax.set_xlabel("Recall (Sensitivity)", labelpad=10)
ax.set_ylabel("Precision",            labelpad=10)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.set_xticks(np.arange(0, 1.1, 0.2))
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.grid(color="#EBEBEB", linewidth=0.8, zorder=0)
ax.tick_params(axis="both", which="both", length=4)

ax.legend(loc="lower left",
          fontsize=12,
          handlelength=2.2, handleheight=1.0,
          borderpad=0.8, labelspacing=0.6)

plt.tight_layout()
plt.savefig("fig_4_3b_pr_curve.pdf")
plt.savefig("fig_4_3b_pr_curve.png")
plt.show()
print("Saved: fig_4_3b_pr_curve.pdf / .png")

print("\nAll Section 4.3 curves generated successfully.")

In [ ]:
# ── Per-seed AUC-ROC for ALL models ──────────────────────
import pickle

with open(checkpoint_path / "multiseed_all.pkl", "rb") as f:
    ms_results = pickle.load(f)

all_models = [
    ("ML Tier",    ["Logistic Regression", "Random Forest",
                    "XGBoost", "LightGBM", "SVM"]),
    ("DL Tier",    ["BiLSTM", "GRU", "TCN", "Transformer"]),
    ("Graph Tier", ["Baseline HeteroGNN", "Enhanced HeteroGNN"]),
]

metrics_to_print = [
    ("AUC-ROC",  "auc_roc"),
    ("AUC-PR",   "auc_pr"),
    ("F1",       "f1"),
    ("MCC",      "mcc"),
    ("S@5%FAR",  "sens_5far"),
]

for metric_label, metric_key in metrics_to_print:
    print("=" * 85)
    print(f"Per-seed {metric_label} — All Models (Seeds: 42, 123, 456, 789, 2024)")
    print("=" * 85)
    print(f"{'Model':<28} {'S1':>8} {'S2':>8} {'S3':>8} {'S4':>8} {'S5':>8}  {'Mean':>8} {'Std':>8}")
    print("-" * 85)

    for tier_name, model_names in all_models:
        print(f"─── {tier_name} ───")
        for name in model_names:
            r        = ms_results[name]
            per_seed = [run[metric_key] for run in r["all_results"]]
            mean     = r[f"{metric_key}_mean"]
            std      = r[f"{metric_key}_std"]
            row      = f"{name:<28}"
            for v in per_seed:
                row += f" {v:>8.4f}"
            row += f"  {mean:>8.4f} {std:>8.4f}"
            print(row)

    print()

# ── Also export full per-seed data to CSV ────────────────
import pandas as pd

rows = []
for tier_name, model_names in all_models:
    for name in model_names:
        r   = ms_results[name]
        row = {"Tier": tier_name, "Model": name}
        for i, run in enumerate(r["all_results"]):
            seed = run.get("seed", [42, 123, 456, 789, 2024][i])
            for metric_label, metric_key in metrics_to_print:
                row[f"{metric_label}_seed{seed}"] = round(run[metric_key], 4)
        for metric_label, metric_key in metrics_to_print:
            row[f"{metric_label}_mean"] = round(r[f"{metric_key}_mean"], 4)
            row[f"{metric_label}_std"]  = round(r[f"{metric_key}_std"],  4)
        rows.append(row)

df_seeds = pd.DataFrame(rows)
df_seeds.to_csv("per_seed_all_models_all_metrics.csv", index=False)
print("Saved: per_seed_all_models_all_metrics.csv")

In [ ]:
# ── Wilcoxon signed-rank test — Enhanced HERMES vs all others ────────────────
from scipy.stats import wilcoxon
import pandas as pd
import pickle

with open(checkpoint_path / "multiseed_all.pkl", "rb") as f:
    ms_results = pickle.load(f)

all_models = [
    ("ML Tier",    ["Logistic Regression", "Random Forest",
                    "XGBoost", "LightGBM", "SVM"]),
    ("DL Tier",    ["BiLSTM", "GRU", "TCN", "Transformer"]),
    ("Graph Tier", ["Baseline HeteroGNN"]),
]

metrics_to_test = [
    ("AUC-ROC",  "auc_roc"),
    ("AUC-PR",   "auc_pr"),
    ("F1",       "f1"),
    ("MCC",      "mcc"),
    ("S@5%FAR",  "sens_5far"),
]

# Get Enhanced HERMES per-seed scores
hermes_scores = {
    metric_key: [run[metric_key]
                 for run in ms_results["Enhanced HeteroGNN"]["all_results"]]
    for _, metric_key in metrics_to_test
}

print("=" * 100)
print("Wilcoxon Signed-Rank Test — Enhanced HERMES vs All Baselines (n=5 seeds, two-sided)")
print("=" * 100)
print(f"{'Model':<28} {'Tier':<12}", end="")
for metric_label, _ in metrics_to_test:
    print(f" {metric_label:>14}", end="")
print()
print("-" * 100)

rows = []
for tier_name, model_names in all_models:
    for name in model_names:
        r           = ms_results[name]
        base_scores = {
            metric_key: [run[metric_key] for run in r["all_results"]]
            for _, metric_key in metrics_to_test
        }

        row = {"Model": name, "Tier": tier_name}
        print(f"{name:<28} {tier_name:<12}", end="")

        for metric_label, metric_key in metrics_to_test:
            h = hermes_scores[metric_key]
            b = base_scores[metric_key]

            # Check if all differences are zero (deterministic models like LR, LightGBM)
            diffs = [hi - bi for hi, bi in zip(h, b)]
            if all(d == 0 for d in diffs):
                p_val  = float("nan")
                sig    = "n/a"
            elif all(d == diffs[0] for d in diffs):
                # All differences identical — wilcoxon will fail, use nan
                p_val  = float("nan")
                sig    = "n/a"
            else:
                stat, p_val = wilcoxon(h, b, alternative="two-sided")
                sig = "***" if p_val < 0.001 else \
                      "**"  if p_val < 0.01  else \
                      "*"   if p_val < 0.05  else \
                      "ns"

            cell = f"{p_val:.4f}{sig}" if not (p_val != p_val) else "n/a"
            print(f" {cell:>14}", end="")
            row[f"{metric_label}_pval"] = round(p_val, 4) if p_val == p_val else None
            row[f"{metric_label}_sig"]  = sig

        print()
        rows.append(row)

print("=" * 100)
print("Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant  n/a = zero variance")
print()
print("Note: ML models with random_state fixed per seed may show zero variance")
print("      across seeds for deterministic models (LR, LightGBM). Wilcoxon")
print("      requires at least one nonzero difference — reported as n/a.")

# ── Export to CSV ─────────────────────────────────────────
df_wilcoxon = pd.DataFrame(rows)
df_wilcoxon.to_csv("table_4_4_wilcoxon_results.csv", index=False)
print("\nSaved: table_4_4_wilcoxon_results.csv")

In [ ]:
"""
Publication-ready AUC-ROC seed distribution — 6 neural models
Horizontal boxplot with individual seed points overlaid.

Requires: ms_results (loaded from multiseed_all.pkl)
          OR run the load block below first.
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings("ignore")

# ── Load if not in memory ─────────────────────────────────
with open(checkpoint_path / "multiseed_all.pkl", "rb") as f:
    ms_results = pickle.load(f)

# ── Publication style ─────────────────────────────────────
matplotlib.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["Times New Roman", "DejaVu Serif"],
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.linewidth"    : 1.2,
    "axes.labelsize"    : 17,
    "xtick.labelsize"   : 15,
    "ytick.labelsize"   : 15,
    "figure.dpi"        : 300,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

# ── Model registry ────────────────────────────────────────
models = [
    ("BiLSTM",           "BiLSTM",           "#A9DFBF", "DL Tier"),
    ("GRU",              "GRU",              "#52BE80", "DL Tier"),
    ("TCN",              "TCN",              "#1E8449", "DL Tier"),
    ("Transformer",      "Transformer",      "#27AE60", "DL Tier"),
    ("Baseline HeteroGNN","Baseline\nHERMES","#F0B27A","Graph Tier"),
    ("Enhanced HeteroGNN","Enhanced\nHERMES","#C0392B","Graph Tier"),
]

# ── Extract per-seed AUC-ROC ──────────────────────────────
data   = []
labels = []
colors = []
tiers  = []
means  = []
stds   = []

for key, label, color, tier in models:
    seeds = [run["auc_roc"] for run in ms_results[key]["all_results"]]
    data.append(seeds)
    labels.append(label)
    colors.append(color)
    tiers.append(tier)
    means.append(np.mean(seeds))
    stds.append(np.std(seeds))

n_models = len(models)
y_pos    = np.arange(n_models)

# ── Figure ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9.0, 6.5), facecolor="white")

# ── Tier background shading ───────────────────────────────
ax.axhspan(-0.5, 3.5,  facecolor="#F0FAF4", alpha=0.5, zorder=0)   # DL
ax.axhspan(3.5,  5.5,  facecolor="#FEF5EB", alpha=0.5, zorder=0)   # Graph

# Tier labels on left margin
ax.text(-0.0005, 1.5, "DL Tier",
        ha="right", va="center", fontsize=12,
        color="#1E8449", style="italic", rotation=90,
        transform=ax.get_yaxis_transform())
ax.text(-0.0005, 4.5, "Graph\nTier",
        ha="right", va="center", fontsize=12,
        color="#E67E22", style="italic", rotation=90,
        transform=ax.get_yaxis_transform())

# ── Horizontal boxplots ───────────────────────────────────
bp = ax.boxplot(
    data,
    positions  = y_pos,
    vert       = False,
    patch_artist = True,
    widths     = 0.38,
    showfliers = False,
    medianprops  = dict(color="black",   linewidth=2.0),
    whiskerprops = dict(color="#555555", linewidth=1.2, linestyle="--"),
    capprops     = dict(color="#555555", linewidth=1.5),
    boxprops     = dict(linewidth=1.2),
    zorder       = 3,
)

for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

# ── Individual seed points ────────────────────────────────
np.random.seed(0)
for i, (seeds, color) in enumerate(zip(data, colors)):
    jitter = np.random.uniform(-0.10, 0.10, len(seeds))
    ax.scatter(seeds, [i] * len(seeds) + jitter,
               color=color, edgecolors="black",
               s=55, linewidths=0.8, zorder=5, alpha=0.95)

# ── Mean ± std annotation on right ───────────────────────
x_ann = max(max(d) for d in data) + 0.0008
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(x_ann, i,
            f"{mean:.4f} ± {std:.4f}",
            va="center", ha="left",
            fontsize=12, color="#333333")

# ── Vertical reference line at best mean ─────────────────
best_mean = max(means)
ax.axvline(x=best_mean, color="#C0392B", linestyle=":",
           linewidth=1.2, alpha=0.6, zorder=1)
# ax.text(best_mean, 1.02, f"Best mean = {best_mean:.4f}",
        # fontsize=10, color="#C0392B", ha="center", va="bottom",
        # transform=ax.get_xaxis_transform())

# ── Axes formatting ───────────────────────────────────────
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=14)
ax.set_xlabel("AUC-ROC", labelpad=10)
ax.set_xlim(
    min(min(d) for d in data) - 0.002,
    x_ann + 0.012
)
ax.set_ylim(-0.7, n_models - 0.3)
ax.grid(axis="x", color="#EBEBEB", linewidth=0.8, zorder=0)
ax.tick_params(axis="both", which="both", length=4)

# ── Separator line between tiers ─────────────────────────
ax.axhline(y=3.5, color="#CCCCCC", linewidth=1.0,
           linestyle="--", zorder=1)

# ── Legend for seed points ────────────────────────────────
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor="#888888", markeredgecolor="black",
           markersize=8, label="Individual seed"),
    matplotlib.patches.Patch(facecolor="#888888", alpha=0.75,
                              edgecolor="black", linewidth=1.0,
                              label="IQR (box) / range (whiskers)"),
    Line2D([0], [0], color="black", linewidth=2.0,
           label="Median"),
]
# ax.legend(handles=legend_elements, loc="lower center",
#           fontsize=11, frameon=True, framealpha=0.92,
#           edgecolor="#CCCCCC", borderpad=0.8)

plt.tight_layout()
plt.savefig("fig_4_4_aucroc_seed_distribution.pdf")
plt.savefig("fig_4_4_aucroc_seed_distribution.png")
plt.show()
print("Saved: fig_4_4_aucroc_seed_distribution.pdf / .png")

In [ ]:
"""
Zoomed ROC curve — FPR in [0, 0.12] — 4 key models
Curve shape from single best run.
Legend annotated with mean ± std from 5-seed multiseed evaluation.
Publication-ready, separate figure.

Requires:
  ml_results, dl_results, test_results, enhanced_test_results  (single run)
  checkpoint_path / "multiseed_all.pkl"                         (multiseed stats)
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
import pickle
import warnings
warnings.filterwarnings("ignore")

# ── Publication style ─────────────────────────────────────
matplotlib.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["Times New Roman", "DejaVu Serif"],
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.linewidth"    : 1.2,
    "axes.labelsize"    : 18,
    "xtick.labelsize"   : 15,
    "ytick.labelsize"   : 15,
    "legend.fontsize"   : 11,
    "legend.framealpha" : 0.93,
    "legend.edgecolor"  : "#CCCCCC",
    "figure.dpi"        : 300,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

# ── Load multiseed stats ──────────────────────────────────
with open(checkpoint_path / "multiseed_all.pkl", "rb") as f:
    ms_data = pickle.load(f)

# ── Single-run results for curve shape ────────────────────
single_run = {
    "XGBoost"         : ml_results["XGBoost"]["test"],
    "Transformer"     : dl_results["Transformer"]["test"],
    "Baseline HERMES" : test_results,
    "Enhanced HERMES" : enhanced_test_results,
}

# ── Multiseed stats for legend ────────────────────────────
ms_keys = {
    "XGBoost"         : "XGBoost",
    "Transformer"     : "Transformer",
    "Baseline HERMES" : "Baseline HeteroGNN",
    "Enhanced HERMES" : "Enhanced HeteroGNN",
}

styles = {
    "XGBoost"         : {"color": "#1A5276", "ls": "-.",  "lw": 2.2, "marker": "o", "ms": 8},
    "Transformer"     : {"color": "#1E8449", "ls": "--",  "lw": 2.2, "marker": "s", "ms": 8},
    "Baseline HERMES" : {"color": "#E67E22", "ls": ":",   "lw": 2.4, "marker": "^", "ms": 9},
    "Enhanced HERMES" : {"color": "#C0392B", "ls": "-",   "lw": 3.0, "marker": "D", "ms": 9},
}

# ── Compute ROC curves ────────────────────────────────────
FPR_MAX = 0.12

curves = {}
for name, res in single_run.items():
    fpr, tpr, _ = roc_curve(res["y_true"], res["y_score"])
    r = ms_data[ms_keys[name]]
    curves[name] = {
        "fpr"     : fpr,
        "tpr"     : tpr,
        "auc_mean": r["auc_roc_mean"],
        "auc_std" : r["auc_roc_std"],
        "s5_mean" : r["sens_5far_mean"],
        "s5_std"  : r["sens_5far_std"],
        "s10_mean": r["sens_10far_mean"],
        "s10_std" : r["sens_10far_std"],
    }

# ── Figure ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.0, 7.0), facecolor="white")

# ── FAR reference shading ─────────────────────────────────
ax.axvspan(0,    0.05, alpha=0.04, color="#C0392B", zorder=0)
ax.axvspan(0.05, 0.10, alpha=0.03, color="#E67E22", zorder=0)

# ── Model curves ──────────────────────────────────────────
for name, c in curves.items():
    s = styles[name]

    # Clip to FPR_MAX
    mask     = c["fpr"] <= FPR_MAX + 0.001
    fpr_clip = c["fpr"][mask]
    tpr_clip = c["tpr"][mask]

    lbl = (f"{name}\n"
           f"  AUC = {c['auc_mean']:.4f} ± {c['auc_std']:.4f}"
           f"   S@5% = {c['s5_mean']:.4f} ± {c['s5_std']:.4f}"
           f"   S@10% = {c['s10_mean']:.4f} ± {c['s10_std']:.4f}")

    ax.plot(fpr_clip, tpr_clip,
            color=s["color"], linestyle=s["ls"], linewidth=s["lw"],
            label=lbl, alpha=0.92, zorder=3)

    # Mark operating points at 5% and 10% FAR
    for x_op in [0.05, 0.10]:
        idx = np.searchsorted(c["fpr"], x_op)
        if idx < len(c["tpr"]) and c["fpr"][idx] <= FPR_MAX:
            ax.plot(c["fpr"][idx], c["tpr"][idx],
                    marker=s["marker"],
                    color=s["color"],
                    markersize=s["ms"],
                    zorder=5,
                    markeredgecolor="white",
                    markeredgewidth=1.5)

# ── Axes formatting ───────────────────────────────────────
ax.set_xlabel("False Positive Rate (1 − Specificity)", labelpad=10)
ax.set_ylabel("True Positive Rate (Sensitivity)",       labelpad=10)
ax.set_xlim(-0.002, FPR_MAX)
ax.set_ylim(0.30,   1.005)
ax.set_xticks(np.arange(0, FPR_MAX + 0.01, 0.02))
ax.set_yticks(np.arange(0.30, 1.01, 0.10))
ax.grid(color="#EBEBEB", linewidth=0.8, zorder=0)
ax.tick_params(axis="both", which="both", length=4)

# ── FAR reference lines ───────────────────────────────────
for x_far, label in [(0.05, "5% FAR"), (0.10, "10% FAR")]:
    ax.axvline(x=x_far, color="#888888", linestyle=":",
               linewidth=1.1, alpha=0.8, zorder=1)
    ax.text(x_far + 0.001, 0.50, label,
            fontsize=12, color="#888888",
            rotation=90, va="center", ha="left",
            transform=ax.get_xaxis_transform(),
            clip_on=True)

# ── Annotations ───────────────────────────────────────────
ax.text(0.97, 0.06,
        "Curve: representative single run\nStats: mean ± SD over 5 seeds",
        transform=ax.transAxes,
        fontsize=10, color="#888888",
        ha="right", va="bottom", style="italic")

ax.text(0.97, 0.02,
        f"Zoomed: FPR ∈ [0, {FPR_MAX}]",
        transform=ax.transAxes,
        fontsize=10, color="#666666",
        ha="right", va="bottom", style="italic")

# ── Legend ────────────────────────────────────────────────
ax.legend(loc="lower right",
          fontsize=10,
          handlelength=2.5,
          handleheight=1.0,
          borderpad=0.9,
          labelspacing=0.9)

plt.tight_layout()
plt.savefig("fig_4_5_roc_zoomed_mean.pdf")
plt.savefig("fig_4_5_roc_zoomed_mean.png")
plt.show()
print("Saved: fig_4_5_roc_zoomed_mean.pdf / .png")

In [ ]:
# ── Check what keys are in each seed result ───────────────
with open(checkpoint_path / "multiseed_all.pkl", "rb") as f:
    ms_data = pickle.load(f)

# Print keys from first seed of Enhanced HERMES
sample = ms_data["Enhanced HeteroGNN"]["all_results"][0]
print("Keys in each seed result:")
print(list(sample.keys()))

In [ ]:
"""
Publication-ready confusion matrices — 2x2 grid
4 key models: XGBoost, Transformer, Baseline HERMES, Enhanced HERMES

Requires: ml_results, dl_results, test_results, enhanced_test_results,
          ms_data (multiseed_all.pkl) for mean±std annotations
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings("ignore")

# ── Publication style ─────────────────────────────────────
matplotlib.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["Times New Roman", "DejaVu Serif"],
    "axes.linewidth"    : 1.2,
    "axes.labelsize"    : 16,
    "xtick.labelsize"   : 15,
    "ytick.labelsize"   : 15,
    "figure.dpi"        : 300,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

# ── Load multiseed stats for title annotation ─────────────
with open(checkpoint_path / "multiseed_all.pkl", "rb") as f:
    ms_data = pickle.load(f)

# ── Model registry ────────────────────────────────────────
key_models = {
    "XGBoost (Best ML)"        : {
        "res"    : ml_results["XGBoost"]["test"],
        "ms_key" : "XGBoost",
        "color"  : "#1A5276",
    },
    "Transformer (Best DL)"    : {
        "res"    : dl_results["Transformer"]["test"],
        "ms_key" : "Transformer",
        "color"  : "#1E8449",
    },
    "Baseline HERMES (Graph)"  : {
        "res"    : test_results,
        "ms_key" : "Baseline HeteroGNN",
        "color"  : "#E67E22",
    },
    "Enhanced HERMES (Proposed)": {
        "res"    : enhanced_test_results,
        "ms_key" : "Enhanced HeteroGNN",
        "color"  : "#C0392B",
    },
}

# ── Figure — 2×2 grid ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 11), facecolor="white")
axes_flat = axes.flatten()

for ax, (title, info) in zip(axes_flat, key_models.items()):
    res    = info["res"]
    ms_key = info["ms_key"]
    color  = info["color"]
    r      = ms_data[ms_key]

    cm      = res["cm"]
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Cell annotations: count + percentage
    annot = np.array([
        [f"{cm[i,j]:,}\n({cm_norm[i,j]:.1%})"
         for j in range(2)]
        for i in range(2)
    ])

    # Custom colormap per model using model color
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list(
        "custom", ["#FFFFFF", color], N=256
    )

    sns.heatmap(
        cm_norm,
        ax          = ax,
        annot       = annot,
        fmt         = "",
        cmap        = cmap,
        xticklabels = ["Predicted\nSafe", "Predicted\nConflict"],
        yticklabels = ["True\nSafe", "True\nConflict"],
        vmin        = 0,
        vmax        = 1,
        linewidths  = 1.0,
        linecolor   = "white",
        cbar        = False,
        annot_kws   = {"size": 15, "fontfamily": "serif",
                       "fontweight": "bold"},
    )

    # Title with mean ± std metrics
    auc_mean = r["auc_roc_mean"]
    auc_std  = r["auc_roc_std"]
    f1_mean  = r["f1_mean"]
    f1_std   = r["f1_std"]
    mcc_mean = r["mcc_mean"]
    mcc_std  = r["mcc_std"]
    s5_mean  = r["sens_5far_mean"]
    s5_std   = r["sens_5far_std"]

    ax.set_title(
        f"{title}\n"
        f"AUC = {auc_mean:.4f}±{auc_std:.4f}   "
        f"F1 = {f1_mean:.4f}±{f1_std:.4f}\n"
        f"MCC = {mcc_mean:.4f}±{mcc_std:.4f}   "
        f"S@5% = {s5_mean:.4f}±{s5_std:.4f}",
        fontsize    = 13,
        fontweight  = "bold",
        pad         = 10,
        color       = color,
        fontfamily  = "serif",
    )

    ax.set_xlabel("Predicted label", fontsize=14, labelpad=8)
    ax.set_ylabel("True label",      fontsize=14, labelpad=8)
    ax.tick_params(axis="both", which="both", length=0)

    # Bold tick labels
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontsize(14)
        tick.set_fontfamily("serif")

# plt.suptitle(
#     "Confusion Matrices — Representative Run, Key Models\n"
#     "Title metrics: mean ± SD over 5 seeds",
#     fontsize   = 15,
#     fontweight = "bold",
#     y          = 1.01,
#     fontfamily = "serif",
# )

plt.tight_layout(h_pad=3.0, w_pad=2.5)
plt.savefig("fig_4_3_confusion_matrices.pdf")
plt.savefig("fig_4_3_confusion_matrices.png")
plt.show()
print("Saved: fig_4_3_confusion_matrices.pdf / .png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve
import seaborn as sns
from matplotlib.patches import Patch

# ── Model registry ─────────────────────────────────────────
plot_models = {
    "LR"              : ml_results["Logistic Regression"]["test"],
    "RF"              : ml_results["Random Forest"]["test"],
    "XGBoost"         : ml_results["XGBoost"]["test"],
    "LightGBM"        : ml_results["LightGBM"]["test"],
    "SVM"             : ml_results["SVM"]["test"],
    "BiLSTM"          : dl_results["BiLSTM"]["test"],
    "GRU"             : dl_results["GRU"]["test"],
    "TCN"             : dl_results["TCN"]["test"],
    "Transformer"     : dl_results["Transformer"]["test"],
    "Baseline HERMES" : test_results,
    "Enhanced HERMES" : enhanced_test_results,
}

tier_styles = {
    "LR"              : ("#AED6F1", "--",  1.2),
    "RF"              : ("#5DADE2", "-.",  1.2),
    "XGBoost"         : ("#2E86C1", "-",   1.8),
    "LightGBM"        : ("#1A5276", ":",   1.5),
    "SVM"             : ("#85C1E9", "--",  1.2),
    "BiLSTM"          : ("#A9DFBF", "--",  1.5),
    "GRU"             : ("#52BE80", "-.",  1.5),
    "TCN"             : ("#1E8449", ":",   1.5),
    "Transformer"     : ("#27AE60", "-",   2.0),
    "Baseline HERMES" : ("#F0B27A", "--",  2.0),
    "Enhanced HERMES" : ("#E74C3C", "-",   2.5),
}

tier_patches = [
    Patch(facecolor="#2E86C1", label="ML Tier"),
    Patch(facecolor="#27AE60", label="DL Tier"),
    Patch(facecolor="#F0B27A", label="Baseline HERMES"),
    Patch(facecolor="#E74C3C", label="Enhanced HERMES"),
]

# ══════════════════════════════════════════════════════════
# Figure 1: ROC + PR Curves
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for name, res in plot_models.items():
    color, ls, lw = tier_styles[name]
    y_true  = res["y_true"]
    y_score = res["y_score"]

    fpr, tpr, _  = roc_curve(y_true, y_score)
    prec, rec, _ = precision_recall_curve(y_true, y_score)

    axes[0].plot(fpr, tpr, color=color, linestyle=ls, linewidth=lw,
                 label=f"{name} ({res['auc_roc']:.4f})", alpha=0.85)
    axes[1].plot(rec, prec, color=color, linestyle=ls, linewidth=lw,
                 label=f"{name} ({res['auc_pr']:.4f})", alpha=0.85)

# ROC
axes[0].plot([0,1],[0,1], "k--", alpha=0.3, linewidth=1)
axes[0].axvline(x=0.05, color="gray", linestyle=":",
                alpha=0.5, linewidth=1)
axes[0].axvline(x=0.10, color="gray", linestyle="--",
                alpha=0.5, linewidth=1)
axes[0].text(0.05, 0.02, "5% FAR",  fontsize=8,
             color="gray", ha="center")
axes[0].text(0.10, 0.02, "10% FAR", fontsize=8,
             color="gray", ha="center")
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate",  fontsize=12)
axes[0].set_title("ROC Curves", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=7.5, loc="lower right", framealpha=0.9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

# PR
baseline_rate = np.mean(test_labels)
axes[1].axhline(y=baseline_rate, color="k", linestyle="--",
                alpha=0.3, linewidth=1,
                label=f"Baseline ({baseline_rate:.2f})")
axes[1].set_xlabel("Recall",    fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("Precision-Recall Curves", fontsize=13,
                  fontweight="bold")
axes[1].legend(fontsize=7.5, loc="upper right", framealpha=0.9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

fig.legend(handles=tier_patches, loc="lower center",
           ncol=4, fontsize=11, bbox_to_anchor=(0.5, -0.04))
plt.suptitle("HERMES Framework — ROC and PR Curves (Test Set)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(checkpoint_path / "roc_pr_curves.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: roc_pr_curves.png")

# ══════════════════════════════════════════════════════════
# Figure 2: Confusion matrices — best per tier + Enhanced
# ══════════════════════════════════════════════════════════
key_models = {
    "XGBoost\n(Best ML)"           : ml_results["XGBoost"]["test"],
    "Transformer\n(Best DL)"       : dl_results["Transformer"]["test"],
    "Baseline HERMES\n(Graph)"     : test_results,
    "Enhanced HERMES\n(HERMES)"    : enhanced_test_results,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, res) in zip(axes, key_models.items()):
    cm      = res["cm"]
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    annot = np.array([
        [f"{cm[i,j]:,}\n({cm_norm[i,j]:.1%})"
         for j in range(2)]
        for i in range(2)
    ])

    sns.heatmap(
        cm_norm, ax=ax,
        annot=annot, fmt="",
        cmap="Blues",
        xticklabels=["Pred Safe", "Pred Conflict"],
        yticklabels=["True Safe", "True Conflict"],
        vmin=0, vmax=1,
        linewidths=0.5,
        cbar=ax == list(key_models.keys())[-1]
    )

    ax.set_title(
        f"{name}\n"
        f"AUC-ROC={res['auc_roc']:.4f}\n"
        f"F1={res['f1']:.4f}  MCC={res['mcc']:.4f}",
        fontsize=10, fontweight="bold"
    )
    ax.set_ylabel("Actual"    if ax == axes[0] else "")
    ax.set_xlabel("Predicted", fontsize=10)

plt.suptitle(
    "Confusion Matrices — Best Model Per Tier (Test Set)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "confusion_matrices_key.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices_key.png")

# ══════════════════════════════════════════════════════════
# Figure 3: Grouped bar chart — 6 key metrics
# ══════════════════════════════════════════════════════════
metrics_to_bar = ["auc_roc","auc_pr","accuracy","f1","mcc","sens_5far"]
metric_labels  = ["AUC-ROC","AUC-PR","Accuracy","F1","MCC","Sens@5%FAR"]

model_order = list(plot_models.keys())
bar_colors  = [
    "#AED6F1","#5DADE2","#2E86C1","#1A5276","#85C1E9",  # ML
    "#A9DFBF","#52BE80","#1E8449","#27AE60",             # DL
    "#F0B27A","#E74C3C",                                  # Graph
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, metric, label in zip(axes, metrics_to_bar, metric_labels):
    values = [plot_models[m][metric] for m in model_order]
    bars   = ax.bar(model_order, values,
                    color=bar_colors,
                    edgecolor="white", linewidth=0.5)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f"{val:.3f}", ha="center", va="bottom",
                fontsize=7, rotation=90)

    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label,  fontsize=12, fontweight="bold")
    ax.set_ylim(min(values)*0.95, min(max(values)*1.10, 1.02))
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(axis="y", alpha=0.3)
    ax.axvline(x=4.5, color="gray", linestyle="--", alpha=0.4)
    ax.axvline(x=8.5, color="gray", linestyle="--", alpha=0.4)

fig.legend(handles=tier_patches, loc="lower center",
           ncol=4, fontsize=11, bbox_to_anchor=(0.5, -0.02))
plt.suptitle(
    "HERMES Framework — Key Metric Comparison Across All Models (Test Set)",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "metric_comparison_bars.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: metric_comparison_bars.png")

# ══════════════════════════════════════════════════════════
# Figure 4: Multi-seed mean±std error bar plot
# ══════════════════════════════════════════════════════════
ms_model_names = [
    "Logistic Regression","Random Forest","XGBoost",
    "LightGBM","SVM",
    "BiLSTM","GRU","TCN","Transformer",
    "Baseline HeteroGNN","Enhanced HeteroGNN"
]
ms_labels = [
    "LR","RF","XGBoost","LightGBM","SVM",
    "BiLSTM","GRU","TCN","Transformer",
    "Base\nHERMES","Enh.\nHERMES"
]
ms_colors = [
    "#AED6F1","#5DADE2","#2E86C1","#1A5276","#85C1E9",
    "#A9DFBF","#52BE80","#1E8449","#27AE60",
    "#F0B27A","#E74C3C"
]

ms_metrics = ["auc_roc","auc_pr","f1","mcc"]
ms_titles  = ["AUC-ROC","AUC-PR","F1","MCC"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, metric, title in zip(axes, ms_metrics, ms_titles):
    means = [multiseed_results[m][f"{metric}_mean"] for m in ms_model_names]
    stds  = [multiseed_results[m][f"{metric}_std"]  for m in ms_model_names]

    x = np.arange(len(ms_model_names))
    bars = ax.bar(x, means, color=ms_colors,
                  edgecolor="white", linewidth=0.5)
    ax.errorbar(x, means, yerr=stds,
                fmt="none", color="black",
                capsize=4, linewidth=1.5)

    ax.set_xticks(x)
    ax.set_xticklabels(ms_labels, fontsize=9)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f"{title} — Mean ± Std (5 Seeds)",
                 fontsize=12, fontweight="bold")
    ax.set_ylim(min(means)*0.95, min(max(means)*1.08, 1.02))
    ax.grid(axis="y", alpha=0.3)
    ax.axvline(x=4.5, color="gray", linestyle="--", alpha=0.4)
    ax.axvline(x=8.5, color="gray", linestyle="--", alpha=0.4)

    # Tier labels
    ax.text(2,   ax.get_ylim()[0], "ML",    ha="center",
            fontsize=9, color="gray", style="italic")
    ax.text(6.5, ax.get_ylim()[0], "DL",    ha="center",
            fontsize=9, color="gray", style="italic")
    ax.text(9.5, ax.get_ylim()[0], "Graph", ha="center",
            fontsize=9, color="gray", style="italic")

fig.legend(handles=tier_patches, loc="lower center",
           ncol=4, fontsize=11, bbox_to_anchor=(0.5, -0.02))
plt.suptitle(
    "HERMES Framework — Multi-Seed Stability Analysis (Test Set)",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "multiseed_stability.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: multiseed_stability.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# Figure 5: Precision, Recall, Specificity comparison
# ══════════════════════════════════════════════════════════
metrics_prs  = ["precision", "recall", "specificity", "f1",
                "accuracy",  "brier"]
labels_prs   = ["Precision", "Recall", "Specificity", "F1",
                "Accuracy",  "Brier Score"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, metric, label in zip(axes, metrics_prs, labels_prs):
    values = [plot_models[m][metric] for m in model_order]

    # Brier score: lower is better — invert color emphasis
    if metric == "brier":
        highlight = [1 - v for v in values]  # invert for visual
        ax.bar(model_order, values, color=bar_colors,
               edgecolor="white", linewidth=0.5)
        ax.set_title(f"{label} (↓ lower is better)",
                     fontsize=12, fontweight="bold")
    else:
        ax.bar(model_order, values, color=bar_colors,
               edgecolor="white", linewidth=0.5)
        ax.set_title(label, fontsize=12, fontweight="bold")

    # Value labels
    for i, (bar_name, val) in enumerate(zip(model_order, values)):
        bar_x = i
        ax.text(bar_x, val + 0.002,
                f"{val:.3f}", ha="center", va="bottom",
                fontsize=7, rotation=90)

    ax.set_ylabel(label, fontsize=11)
    ax.set_ylim(0, min(max(values) * 1.15, 1.05))
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.set_xticks(range(len(model_order)))
    ax.set_xticklabels(model_order, rotation=45,
                       ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    ax.axvline(x=4.5, color="gray", linestyle="--", alpha=0.4)
    ax.axvline(x=8.5, color="gray", linestyle="--", alpha=0.4)

    # Tier labels at bottom
    ymin = ax.get_ylim()[0]
    ax.text(2,    ymin, "ML",    ha="center", fontsize=9,
            color="gray", style="italic")
    ax.text(6.5,  ymin, "DL",    ha="center", fontsize=9,
            color="gray", style="italic")
    ax.text(9.5,  ymin, "Graph", ha="center", fontsize=9,
            color="gray", style="italic")

fig.legend(handles=tier_patches, loc="lower center",
           ncol=4, fontsize=11, bbox_to_anchor=(0.5, -0.02))
plt.suptitle(
    "HERMES Framework — Precision, Recall, Specificity & More (Test Set)",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "precision_recall_specificity.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: precision_recall_specificity.png")

# ══════════════════════════════════════════════════════════
# Figure 6: Radar chart — Enhanced HERMES vs best per tier
# ══════════════════════════════════════════════════════════
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

radar_models = {
    "XGBoost\n(Best ML)"       : ml_results["XGBoost"]["test"],
    "Transformer\n(Best DL)"   : dl_results["Transformer"]["test"],
    "Baseline\nHERMES"         : test_results,
    "Enhanced\nHERMES"         : enhanced_test_results,
}
radar_colors = ["#2E86C1", "#27AE60", "#F0B27A", "#E74C3C"]

radar_metrics = ["auc_roc","auc_pr","f1","mcc",
                 "recall","precision","sens_5far","sens_10far"]
radar_labels  = ["AUC-ROC","AUC-PR","F1","MCC",
                 "Recall","Precision","Sens@5%","Sens@10%"]

N      = len(radar_metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(9, 9),
                        subplot_kw=dict(polar=True))

for (name, res), color in zip(radar_models.items(), radar_colors):
    values = [res[m] for m in radar_metrics]
    values += values[:1]

    ax.plot(angles, values, color=color,
            linewidth=2, linestyle="-", label=name)
    ax.fill(angles, values, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(["0.6","0.7","0.8","0.9","1.0"],
                   fontsize=8, color="gray")
ax.grid(color="gray", linestyle="--", linewidth=0.5, alpha=0.5)
ax.set_title("HERMES vs Best ML/DL — Radar Chart (Test Set)",
             fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15),
          fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.savefig(checkpoint_path / "radar_chart.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: radar_chart.png")

In [ ]:
# Check what ablation results we have in memory
print("=== Ablation results in memory ===")

print("\nGroup 1 — SSM Edge Features:")
for name, res in ablation_A_results.items():
    has_state = "best_state" in res and res["best_state"] is not None
    print(f"  {name:<25} val_AUC={res['auc_roc']:.4f}  state_in_memory={has_state}")

print("\nGroup 2 — Architecture:")
for name, res in ablation_B_results.items():
    has_state = "best_state" in res and res["best_state"] is not None
    print(f"  {name:<25} val_AUC={res['auc_roc']:.4f}  state_in_memory={has_state}")

print("\nGroup 3 — Temporal:")
for name, res in ablation_C_results.items():
    has_state = "best_state" in res and res["best_state"] is not None
    print(f"  {name:<25} val_AUC={res['auc_roc']:.4f}  state_in_memory={has_state}")

print("\nGroup 4 — Interaction Types:")
for name, res in ablation_D_results.items():
    has_state = "best_state" in res and res["best_state"] is not None
    print(f"  {name:<25} val_AUC={res['auc_roc']:.4f}  state_in_memory={has_state}")

In [ ]:
class MeanPoolModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.proj    = nn.Linear(node_dim, hidden_dim)
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim,
                               num_layers=lstm_layers,
                               batch_first=True,
                               dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []
        for t in range(T):
            X      = X_seq[:,t]
            H      = F.relu(self.proj(X))
            H_pool = H.mean(dim=1)
            frame_embeds.append(H_pool)
        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


class SimpleGNNModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.W       = nn.Linear(node_dim, hidden_dim)
        self.fusion  = nn.Linear(node_dim + hidden_dim, hidden_dim)
        self.norm    = nn.LayerNorm(hidden_dim)
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim,
                               num_layers=lstm_layers,
                               batch_first=True,
                               dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []
        for t in range(T):
            X = X_seq[:,t]
            A = (A_vv_seq[:,t] + A_vp_seq[:,t] +
                 A_pp_seq[:,t]).clamp(0, 1)
            H   = self.W(X)
            AX  = torch.bmm(A, H)
            out = self.norm(F.relu(
                self.fusion(torch.cat([X, AX], dim=-1))
            ))
            H_pool = out.mean(dim=1)
            frame_embeds.append(H_pool)
        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:, -1, :])
        return self.fc(final).squeeze(-1)


print("Models redefined. Rerun the ablation test evaluation cell.")

In [ ]:
def evaluate_ablation_on_test(model, state_dict, loader_test, device):
    model.load_state_dict(state_dict)
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader_test:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = model(X, Avv, Avp, App, E)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())
    return compute_metrics(np.array(y_true), np.array(y_score))


# Standard test loader (edge_dim=6)
loader_test_graph = DataLoader(ds_test, batch_size=64,
                               shuffle=False, num_workers=0)

# A0 needs its own test loader (NoEdgeDataset, edge_dim=1)
ds_test_a0    = NoEdgeDataset(test_seqs, test_labels, node_mean, node_std)
loader_test_a0 = DataLoader(ds_test_a0, batch_size=64,
                             shuffle=False, num_workers=0)

ablation_test_results = {}

# ── Group 1 ───────────────────────────────────────────────
print("="*65)
print("ABLATION TEST RESULTS — Group 1: SSM Edge Features")
print("="*65)
print(f"{'Variant':<25} {'Val AUC':>8} {'Test AUC':>9} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

# A0 — use NoEdgeDataset loader
m_a0 = SceneRiskModel(edge_dim=1).to(device)
r    = evaluate_ablation_on_test(
    m_a0, ablation_A_results["A0_nodes_only"]["best_state"],
    loader_test_a0, device
)
ablation_test_results["A0_nodes_only"] = r
print(f"{'A0 — nodes only':<25} "
      f"{ablation_A_results['A0_nodes_only']['auc_roc']:>8.4f} "
      f"{r['auc_roc']:>9.4f} {r['auc_pr']:>7.4f} "
      f"{r['f1']:>6.4f} {r['mcc']:>6.4f} {r['sens_5far']:>6.4f}")

# A1–A4 — use AblationEdgeDataset loaders
edge_col_configs = {
    "A1_kinematics": ["distance", "rel_speed"],
    "A2_+drac"     : ["distance", "rel_speed", "drac"],
    "A3_+heading"  : ["distance", "rel_speed", "drac", "heading_diff"],
    "A4_full"      : ["distance", "rel_speed", "drac",
                      "heading_diff", "rel_vx", "rel_vy"],
}
for abl_name, cols in edge_col_configs.items():
    ds_test_abl = AblationEdgeDataset(
        test_seqs, test_labels,
        node_mean, node_std, edge_mean, edge_std,
        edge_cols=cols
    )
    loader_test_abl = DataLoader(ds_test_abl, batch_size=64,
                                 shuffle=False, num_workers=0)
    m = SceneRiskModel(edge_dim=len(cols)).to(device)
    r = evaluate_ablation_on_test(
        m, ablation_A_results[abl_name]["best_state"],
        loader_test_abl, device
    )
    ablation_test_results[abl_name] = r
    print(f"{abl_name:<25} "
          f"{ablation_A_results[abl_name]['auc_roc']:>8.4f} "
          f"{r['auc_roc']:>9.4f} {r['auc_pr']:>7.4f} "
          f"{r['f1']:>6.4f} {r['mcc']:>6.4f} {r['sens_5far']:>6.4f}")

# ── Group 2 ───────────────────────────────────────────────
print("\n" + "="*65)
print("ABLATION TEST RESULTS — Group 2: Architecture")
print("="*65)
print(f"{'Variant':<28} {'Val AUC':>8} {'Test AUC':>9} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

arch_models = {
    "B1_no_graph"          : MeanPoolModel(),
    "B2_simple_gnn"        : SimpleGNNModel(),
    "B3_ssm_attn_meanpool" : SSMAttnMeanPoolModel(),
    "B4_full_model"        : SceneRiskModel(),
}
for abl_name, m in arch_models.items():
    m = m.to(device)
    r = evaluate_ablation_on_test(
        m, ablation_B_results[abl_name]["best_state"],
        loader_test_graph, device
    )
    ablation_test_results[abl_name] = r
    print(f"{abl_name:<28} "
          f"{ablation_B_results[abl_name]['auc_roc']:>8.4f} "
          f"{r['auc_roc']:>9.4f} {r['auc_pr']:>7.4f} "
          f"{r['f1']:>6.4f} {r['mcc']:>6.4f} {r['sens_5far']:>6.4f}")

# ── Group 3 ───────────────────────────────────────────────
print("\n" + "="*65)
print("ABLATION TEST RESULTS — Group 3: Temporal Modeling")
print("="*65)
print(f"{'Variant':<25} {'Val AUC':>8} {'Test AUC':>9} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

temporal_models = {
    "C1_single_frame"   : SingleFrameModel(),
    "C2_gnn_gru"        : GNNGRUModel(),
    "C3_gnn_transformer": GNNTransformerModel(),
    "C4_gnn_lstm"       : SceneRiskModel(),
}
for abl_name, m in temporal_models.items():
    m = m.to(device)
    r = evaluate_ablation_on_test(
        m, ablation_C_results[abl_name]["best_state"],
        loader_test_graph, device
    )
    ablation_test_results[abl_name] = r
    print(f"{abl_name:<25} "
          f"{ablation_C_results[abl_name]['auc_roc']:>8.4f} "
          f"{r['auc_roc']:>9.4f} {r['auc_pr']:>7.4f} "
          f"{r['f1']:>6.4f} {r['mcc']:>6.4f} {r['sens_5far']:>6.4f}")

# ── Group 4 ───────────────────────────────────────────────
print("\n" + "="*65)
print("ABLATION TEST RESULTS — Group 4: Interaction Types")
print("="*65)
print(f"{'Variant':<25} {'Val AUC':>8} {'Test AUC':>9} {'AUC-PR':>7} "
      f"{'F1':>6} {'MCC':>6} {'S@5%':>6}")
print("-"*65)

for abl_name, allowed in [
    ("D1_vv_only",   {"vv"}),
    ("D2_vp_only",   {"vp"}),
    ("D3_vv_vp",     {"vv","vp"}),
    ("D4_all_types", {"vv","vp","pp"}),
]:
    ds_test_d = InteractionTypeDataset(
        test_seqs, test_labels,
        node_mean, node_std, edge_mean, edge_std,
        allowed_types=allowed
    )
    loader_test_d = DataLoader(ds_test_d, batch_size=64,
                               shuffle=False, num_workers=0)
    m = SceneRiskModel().to(device)
    m.load_state_dict(ablation_D_results[abl_name]["best_state"])
    m.eval()

    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader_test_d:
            X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
            logits = m(X, Avv, Avp, App, E)
            y_true.extend(y.cpu().numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    r = compute_metrics(np.array(y_true), np.array(y_score))
    ablation_test_results[abl_name] = r
    print(f"{abl_name:<25} "
          f"{ablation_D_results[abl_name]['auc_roc']:>8.4f} "
          f"{r['auc_roc']:>9.4f} {r['auc_pr']:>7.4f} "
          f"{r['f1']:>6.4f} {r['mcc']:>6.4f} {r['sens_5far']:>6.4f}")

# Save
with open(checkpoint_path / "ablation_test_results.pkl", "wb") as f:
    pickle.dump(ablation_test_results, f)
print("\nSaved ablation_test_results.pkl")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.subplots_adjust(hspace=0.45, wspace=0.35)

def plot_ablation_group(ax, names, auc_roc, auc_pr, title,
                        colors, ylim_bottom=0.5):
    x = np.arange(len(names))
    w = 0.35

    bars1 = ax.bar(x - w/2, auc_roc, w, label="AUC-ROC",
                   color=colors, edgecolor="white", linewidth=0.8)
    bars2 = ax.bar(x + w/2, auc_pr, w, label="AUC-PR",
                   color=colors, edgecolor="white", linewidth=0.8,
                   alpha=0.55, hatch="///")

    # Value labels inside bars
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h - 0.012,
                f"{h:.3f}", ha="center", va="top",
                fontsize=8, color="white", fontweight="bold")

    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h - 0.012,
                f"{h:.3f}", ha="center", va="top",
                fontsize=8, color="white", fontweight="bold")

    # Enhanced HERMES reference
    ax.axhline(y=0.9903, color="crimson", linestyle="--",
               linewidth=1.8, label="Enhanced HERMES\n(converged, AUC-ROC=0.9903)")
    ax.axhline(y=0.9420, color="crimson", linestyle=":",
               linewidth=1.4, label="Enhanced HERMES\n(converged, AUC-PR=0.9420)")

    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=10)
    ax.set_ylim(ylim_bottom, 1.04)
    ax.set_ylabel("Score", fontsize=11)
    ax.set_title(title, fontweight="bold", fontsize=12, pad=10)
    ax.legend(fontsize=8, loc="lower right", framealpha=0.9,
              ncol=2)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ── Group 1 ───────────────────────────────────────────────
plot_ablation_group(
    axes[0,0],
    names=["A0\nNodes only", "A1\nKinematics\nonly",
           "A2\n+DRAC", "A3\n+Heading", "A4\nFull (6-dim)"],
    auc_roc=[ablation_test_results["A0_nodes_only"]["auc_roc"],
             ablation_test_results["A1_kinematics"]["auc_roc"],
             ablation_test_results["A2_+drac"]["auc_roc"],
             ablation_test_results["A3_+heading"]["auc_roc"],
             ablation_test_results["A4_full"]["auc_roc"]],
    auc_pr=[ablation_test_results["A0_nodes_only"]["auc_pr"],
            ablation_test_results["A1_kinematics"]["auc_pr"],
            ablation_test_results["A2_+drac"]["auc_pr"],
            ablation_test_results["A3_+heading"]["auc_pr"],
            ablation_test_results["A4_full"]["auc_pr"]],
    title="Group 1 — SSM Edge Feature Contribution",
    colors=["#AED6F1","#5DADE2","#2E86C1","#1A5276","#154360"],
    ylim_bottom=0.50
)

# ── Group 2 ───────────────────────────────────────────────
g2_auc = [ablation_test_results["B1_no_graph"]["auc_roc"],
          ablation_test_results["B2_simple_gnn"]["auc_roc"],
          ablation_test_results["B3_ssm_attn_meanpool"]["auc_roc"],
          ablation_test_results["B4_full_model"]["auc_roc"]]
g2_pr  = [ablation_test_results["B1_no_graph"]["auc_pr"],
          ablation_test_results["B2_simple_gnn"]["auc_pr"],
          ablation_test_results["B3_ssm_attn_meanpool"]["auc_pr"],
          ablation_test_results["B4_full_model"]["auc_pr"]]

plot_ablation_group(
    axes[1,0],
    names=["B1\nNo graph\n(mean pool)", "B2\nSimple GNN\n(no attn)",
           "B3\nSSM attn\n+mean pool", "B4\nSSM attn\n+DRAC pool"],
    auc_roc=g2_auc,
    auc_pr=g2_pr,
    title="Group 2 — Graph Architecture Components",
    colors=["#FADBD8","#E59866","#E74C3C","#922B21"],
    ylim_bottom=0.40
)

# Stepwise improvement annotations for Group 2
for i in range(len(g2_auc)-1):
    delta = g2_auc[i+1] - g2_auc[i]
    col   = "darkgreen" if delta > 0 else "red"
    sign  = "+" if delta > 0 else ""
    axes[1,0].annotate(
        f"{sign}{delta:.3f}",
        xy=(i + 0.5, max(g2_auc[i], g2_auc[i+1]) + 0.015),
        ha="center", fontsize=9, color=col, fontweight="bold"
    )

# ── Group 3 ───────────────────────────────────────────────
plot_ablation_group(
    axes[0,1],
    names=["C1\nSingle frame\n(no temporal)",
           "C2\nGNN\n+GRU",
           "C3\nGNN\n+Transformer",
           "C4\nGNN\n+LSTM"],
    auc_roc=[ablation_test_results["C1_single_frame"]["auc_roc"],
             ablation_test_results["C2_gnn_gru"]["auc_roc"],
             ablation_test_results["C3_gnn_transformer"]["auc_roc"],
             ablation_test_results["C4_gnn_lstm"]["auc_roc"]],
    auc_pr=[ablation_test_results["C1_single_frame"]["auc_pr"],
            ablation_test_results["C2_gnn_gru"]["auc_pr"],
            ablation_test_results["C3_gnn_transformer"]["auc_pr"],
            ablation_test_results["C4_gnn_lstm"]["auc_pr"]],
    title="Group 3 — Temporal Modeling",
    colors=["#ABEBC6","#27AE60","#1E8449","#145A32"],
    ylim_bottom=0.45
)

# ── Group 4 ───────────────────────────────────────────────
plot_ablation_group(
    axes[1,1],
    names=["D1\nVV only", "D2\nVP only",
           "D3\nVV+VP\n(optimal)", "D4\nAll types\n(VV+VP+PP)"],
    auc_roc=[ablation_test_results["D1_vv_only"]["auc_roc"],
             ablation_test_results["D2_vp_only"]["auc_roc"],
             ablation_test_results["D3_vv_vp"]["auc_roc"],
             ablation_test_results["D4_all_types"]["auc_roc"]],
    auc_pr=[ablation_test_results["D1_vv_only"]["auc_pr"],
            ablation_test_results["D2_vp_only"]["auc_pr"],
            ablation_test_results["D3_vv_vp"]["auc_pr"],
            ablation_test_results["D4_all_types"]["auc_pr"]],
    title="Group 4 — Interaction Type Coverage",
    colors=["#D2B4DE","#A569BD","#7D3C98","#4A235A"],
    ylim_bottom=0.50
)

# Highlight optimal D3
axes[1,1].get_children()[4].set_edgecolor("gold")
axes[1,1].get_children()[4].set_linewidth(2.5)

plt.suptitle(
    "HERMES Ablation Study — Test Set Results (20 epochs each)",
    fontsize=15, fontweight="bold", y=1.01
)

plt.savefig(checkpoint_path / "ablation_test_results_v2.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: ablation_test_results_v2.png")

In [ ]:
# Already saved as ablation_test_results_v2.png
# Verify file exists
import os
fpath = checkpoint_path / "ablation_test_results_v2.png"
size  = os.path.getsize(fpath) / 1024
print(f"File: {fpath}")
print(f"Size: {size:.1f} KB")

In [ ]:
class PrecomputedDataset(Dataset):
    """
    Pre-tensorizes all sequences at init time.
    __getitem__ is just a list lookup — extremely fast.
    """
    def __init__(self, sequences, labels,
                 node_mean, node_std,
                 edge_mean, edge_std,
                 max_nodes=MAX_NODES):

        print(f"  Pre-computing {len(sequences):,} sequences...")
        self.X_all   = []
        self.Avv_all = []
        self.Avp_all = []
        self.App_all = []
        self.E_all   = []
        self.labels  = torch.tensor(labels, dtype=torch.float32)

        for seq in sequences:
            X_seq, Avv_seq, Avp_seq, App_seq, E_seq = [], [], [], [], []
            for g in seq:
                X, Avv, Avp, App, E = scene_graph_to_tensors(
                    g, max_nodes,
                    node_mean, node_std,
                    edge_mean, edge_std
                )
                X_seq.append(X)
                Avv_seq.append(Avv)
                Avp_seq.append(Avp)
                App_seq.append(App)
                E_seq.append(E)

            self.X_all.append(torch.tensor(np.stack(X_seq),   dtype=torch.float32))
            self.Avv_all.append(torch.tensor(np.stack(Avv_seq), dtype=torch.float32))
            self.Avp_all.append(torch.tensor(np.stack(Avp_seq), dtype=torch.float32))
            self.App_all.append(torch.tensor(np.stack(App_seq), dtype=torch.float32))
            self.E_all.append(torch.tensor(np.stack(E_seq),   dtype=torch.float32))

        print(f"  Pre-computation done.")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.X_all[idx],
            self.Avv_all[idx],
            self.Avp_all[idx],
            self.App_all[idx],
            self.E_all[idx],
            self.labels[idx]
        )


# Pre-compute once — reuse across all seeds and variants
print("Pre-computing train dataset...")
ds_train_pre = PrecomputedDataset(
    train_seqs, train_labels,
    node_mean, node_std, edge_mean, edge_std
)

print("Pre-computing val dataset...")
ds_val_pre = PrecomputedDataset(
    val_seqs, val_labels,
    node_mean, node_std, edge_mean, edge_std
)

print("Pre-computing test dataset...")
ds_test_pre = PrecomputedDataset(
    test_seqs, test_labels,
    node_mean, node_std, edge_mean, edge_std
)

print(f"\nTrain: {len(ds_train_pre):,}")
print(f"Val  : {len(ds_val_pre):,}")
print(f"Test : {len(ds_test_pre):,}")

In [ ]:
ABLATION_SEEDS = [42, 123, 456, 789, 2024]

def train_ablation_multiseed_fast(variant_name,
                                   ds_train_abl, ds_val_abl, ds_test_abl,
                                   edge_dim,
                                   ModelClass=None, model_kwargs=None,
                                   seeds=ABLATION_SEEDS,
                                   n_epochs=10, batch_size=64):
    all_results = []

    loader_test = DataLoader(ds_test_abl, batch_size=batch_size,
                             shuffle=False, num_workers=0,
                             pin_memory=True)

    for seed in seeds:
        set_seed(seed)

        if ModelClass is not None:
            model = ModelClass(**model_kwargs).to(device)
        else:
            model = SceneRiskModel(edge_dim=edge_dim).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3
        )
        loader_tr = DataLoader(ds_train_abl, batch_size=batch_size,
                               shuffle=True,  num_workers=0,
                               pin_memory=True)
        loader_vl = DataLoader(ds_val_abl,   batch_size=batch_size,
                               shuffle=False, num_workers=0,
                               pin_memory=True)

        best_val_auc = 0.0
        best_state   = None

        for epoch in range(1, n_epochs + 1):
            model.train()
            for batch in loader_tr:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                loss   = criterion(logits, y)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            y_true, y_score = [], []
            with torch.no_grad():
                for batch in loader_vl:
                    X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                    logits = model(X, Avv, Avp, App, E)
                    y_true.extend(y.cpu().numpy())
                    y_score.extend(torch.sigmoid(logits).cpu().numpy())

            val_auc = roc_auc_score(y_true, y_score)
            scheduler.step(val_auc)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state   = {k: v.clone()
                                for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        model.eval()
        y_true, y_score = [], []
        with torch.no_grad():
            for batch in loader_test:
                X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
                logits = model(X, Avv, Avp, App, E)
                y_true.extend(y.cpu().numpy())
                y_score.extend(torch.sigmoid(logits).cpu().numpy())

        res = compute_metrics(np.array(y_true), np.array(y_score))
        res["seed"] = seed
        all_results.append(res)

    summary               = summarize_seeds(all_results)
    summary["variant"]    = variant_name
    summary["all_results"]= all_results
    return summary


# ── Pre-compute ablation-specific datasets ─────────────────
print("Pre-computing ablation datasets...")

# Group 1 datasets
ds_a0_tr = NoEdgeDataset(train_seqs, train_labels, node_mean, node_std)
ds_a0_vl = NoEdgeDataset(val_seqs,   val_labels,   node_mean, node_std)
ds_a0_te = NoEdgeDataset(test_seqs,  test_labels,  node_mean, node_std)

edge_col_configs = {
    "A1_kinematics": ["distance","rel_speed"],
    "A2_+drac"     : ["distance","rel_speed","drac"],
    "A3_+heading"  : ["distance","rel_speed","drac","heading_diff"],
    "A4_full"      : ["distance","rel_speed","drac",
                      "heading_diff","rel_vx","rel_vy"],
}
abl_datasets = {}
for name, cols in edge_col_configs.items():
    abl_datasets[name] = {
        "tr": AblationEdgeDataset(train_seqs, train_labels,
                                  node_mean, node_std, edge_mean, edge_std,
                                  edge_cols=cols),
        "vl": AblationEdgeDataset(val_seqs,   val_labels,
                                  node_mean, node_std, edge_mean, edge_std,
                                  edge_cols=cols),
        "te": AblationEdgeDataset(test_seqs,  test_labels,
                                  node_mean, node_std, edge_mean, edge_std,
                                  edge_cols=cols),
    }

# Group 4 datasets
int_type_configs = {
    "D1_vv_only"  : {"vv"},
    "D2_vp_only"  : {"vp"},
    "D3_vv_vp"    : {"vv","vp"},
    "D4_all_types": {"vv","vp","pp"},
}
int_datasets = {}
for name, allowed in int_type_configs.items():
    int_datasets[name] = {
        "tr": InteractionTypeDataset(train_seqs, train_labels,
                                     node_mean, node_std,
                                     edge_mean, edge_std,
                                     allowed_types=allowed),
        "vl": InteractionTypeDataset(val_seqs,   val_labels,
                                     node_mean, node_std,
                                     edge_mean, edge_std,
                                     allowed_types=allowed),
        "te": InteractionTypeDataset(test_seqs,  test_labels,
                                     node_mean, node_std,
                                     edge_mean, edge_std,
                                     allowed_types=allowed),
    }

print("All ablation datasets ready.")

# ══════════════════════════════════════════════════════════
# RUN ALL GROUPS
# ══════════════════════════════════════════════════════════
import time
ablation_ms_results = {}
t_total = time.time()

# ── Group 1 ───────────────────────────────────────────────
print("\n" + "="*60)
print("Group 1 — SSM Edge Features")
print("="*60)

t0 = time.time()
ablation_ms_results["A0_nodes_only"] = train_ablation_multiseed_fast(
    "A0_nodes_only", ds_a0_tr, ds_a0_vl, ds_a0_te, edge_dim=1
)
r = ablation_ms_results["A0_nodes_only"]
print(f"A0  {r['auc_roc_mean']:.4f}±{r['auc_roc_std']:.4f}  "
      f"{r['auc_pr_mean']:.4f}±{r['auc_pr_std']:.4f}  "
      f"{r['f1_mean']:.4f}±{r['f1_std']:.4f}  "
      f"{r['mcc_mean']:.4f}±{r['mcc_std']:.4f}  "
      f"{r['sens_5far_mean']:.4f}±{r['sens_5far_std']:.4f}  "
      f"[{time.time()-t0:.0f}s]")

for abl_name, cols in edge_col_configs.items():
    t0 = time.time()
    ablation_ms_results[abl_name] = train_ablation_multiseed_fast(
        abl_name,
        abl_datasets[abl_name]["tr"],
        abl_datasets[abl_name]["vl"],
        abl_datasets[abl_name]["te"],
        edge_dim=len(cols)
    )
    r = ablation_ms_results[abl_name]
    print(f"{abl_name:<20}  "
          f"{r['auc_roc_mean']:.4f}±{r['auc_roc_std']:.4f}  "
          f"{r['auc_pr_mean']:.4f}±{r['auc_pr_std']:.4f}  "
          f"{r['f1_mean']:.4f}±{r['f1_std']:.4f}  "
          f"{r['mcc_mean']:.4f}±{r['mcc_std']:.4f}  "
          f"{r['sens_5far_mean']:.4f}±{r['sens_5far_std']:.4f}  "
          f"[{time.time()-t0:.0f}s]")

# ── Group 2 ───────────────────────────────────────────────
print("\n" + "="*60)
print("Group 2 — Architecture")
print("="*60)

for abl_name, ModelClass, mkwargs in [
    ("B1_no_graph",
     MeanPoolModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "lstm_layers": 2, "dropout": 0.3}),
    ("B2_simple_gnn",
     SimpleGNNModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "lstm_layers": 2, "dropout": 0.3}),
    ("B3_ssm_attn_meanpool",
     SSMAttnMeanPoolModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "edge_dim": EDGE_DIM, "lstm_layers": 2, "dropout": 0.3}),
    ("B4_full_model",
     SceneRiskModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "edge_dim": EDGE_DIM, "lstm_layers": 2, "dropout": 0.3}),
]:
    t0 = time.time()
    ablation_ms_results[abl_name] = train_ablation_multiseed_fast(
        abl_name, ds_train_pre, ds_val_pre, ds_test_pre,
        edge_dim=EDGE_DIM,
        ModelClass=ModelClass, model_kwargs=mkwargs
    )
    r = ablation_ms_results[abl_name]
    print(f"{abl_name:<25}  "
          f"{r['auc_roc_mean']:.4f}±{r['auc_roc_std']:.4f}  "
          f"{r['auc_pr_mean']:.4f}±{r['auc_pr_std']:.4f}  "
          f"{r['f1_mean']:.4f}±{r['f1_std']:.4f}  "
          f"{r['mcc_mean']:.4f}±{r['mcc_std']:.4f}  "
          f"{r['sens_5far_mean']:.4f}±{r['sens_5far_std']:.4f}  "
          f"[{time.time()-t0:.0f}s]")

# ── Group 3 ───────────────────────────────────────────────
print("\n" + "="*60)
print("Group 3 — Temporal Modeling")
print("="*60)

for abl_name, ModelClass, mkwargs in [
    ("C1_single_frame",
     SingleFrameModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "edge_dim": EDGE_DIM, "dropout": 0.3}),
    ("C2_gnn_gru",
     GNNGRUModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "edge_dim": EDGE_DIM, "lstm_layers": 2, "dropout": 0.3}),
    ("C3_gnn_transformer",
     GNNTransformerModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "edge_dim": EDGE_DIM, "num_heads": 4,
      "num_layers": 2, "dropout": 0.3, "max_seq_len": 8}),
    ("C4_gnn_lstm",
     SceneRiskModel,
     {"node_dim": NODE_DIM, "hidden_dim": 64,
      "edge_dim": EDGE_DIM, "lstm_layers": 2, "dropout": 0.3}),
]:
    t0 = time.time()
    ablation_ms_results[abl_name] = train_ablation_multiseed_fast(
        abl_name, ds_train_pre, ds_val_pre, ds_test_pre,
        edge_dim=EDGE_DIM,
        ModelClass=ModelClass, model_kwargs=mkwargs
    )
    r = ablation_ms_results[abl_name]
    print(f"{abl_name:<25}  "
          f"{r['auc_roc_mean']:.4f}±{r['auc_roc_std']:.4f}  "
          f"{r['auc_pr_mean']:.4f}±{r['auc_pr_std']:.4f}  "
          f"{r['f1_mean']:.4f}±{r['f1_std']:.4f}  "
          f"{r['mcc_mean']:.4f}±{r['mcc_std']:.4f}  "
          f"{r['sens_5far_mean']:.4f}±{r['sens_5far_std']:.4f}  "
          f"[{time.time()-t0:.0f}s]")

# ── Group 4 ───────────────────────────────────────────────
print("\n" + "="*60)
print("Group 4 — Interaction Types")
print("="*60)

for abl_name in ["D1_vv_only","D2_vp_only","D3_vv_vp","D4_all_types"]:
    t0 = time.time()
    ablation_ms_results[abl_name] = train_ablation_multiseed_fast(
        abl_name,
        int_datasets[abl_name]["tr"],
        int_datasets[abl_name]["vl"],
        int_datasets[abl_name]["te"],
        edge_dim=EDGE_DIM
    )
    r = ablation_ms_results[abl_name]
    print(f"{abl_name:<20}  "
          f"{r['auc_roc_mean']:.4f}±{r['auc_roc_std']:.4f}  "
          f"{r['auc_pr_mean']:.4f}±{r['auc_pr_std']:.4f}  "
          f"{r['f1_mean']:.4f}±{r['f1_std']:.4f}  "
          f"{r['mcc_mean']:.4f}±{r['mcc_std']:.4f}  "
          f"{r['sens_5far_mean']:.4f}±{r['sens_5far_std']:.4f}  "
          f"[{time.time()-t0:.0f}s]")

# Save
with open(checkpoint_path / "ablation_multiseed_fast.pkl", "wb") as f:
    pickle.dump(ablation_ms_results, f)

print(f"\nTotal time: {(time.time()-t_total)/60:.1f} min")
print("Saved ablation_multiseed_fast.pkl")

In [ ]:
from scipy.stats import wilcoxon

def wilcoxon_test(results_a, results_b, metric="auc_roc"):
    scores_a = [r[metric] for r in results_a["all_results"]]
    scores_b = [r[metric] for r in results_b["all_results"]]
    diffs    = [a - b for a, b in zip(scores_a, scores_b)]
    if all(d == 0 for d in diffs):
        return 1.0, False
    stat, p = wilcoxon(scores_a, scores_b, alternative="two-sided")
    return p, p < 0.05


# ── Main comparison significance ──────────────────────────
print("="*70)
print("SIGNIFICANCE TESTS — Main Comparison (AUC-ROC, 5 seeds)")
print("="*70)
print(f"{'Comparison':<45} {'p-value':>8} {'Sig':>5}")
print("-"*70)

main_comparisons = [
    ("Enhanced HERMES vs Transformer",
     "Enhanced HeteroGNN", "Transformer"),
    ("Enhanced HERMES vs BiLSTM",
     "Enhanced HeteroGNN", "BiLSTM"),
    ("Enhanced HERMES vs Baseline HERMES",
     "Enhanced HeteroGNN", "Baseline HeteroGNN"),
    ("Enhanced HERMES vs XGBoost",
     "Enhanced HeteroGNN", "XGBoost"),
    ("Baseline HERMES vs Transformer",
     "Baseline HeteroGNN", "Transformer"),
    ("Transformer vs XGBoost",
     "Transformer", "XGBoost"),
]
for label, a, b in main_comparisons:
    p, sig = wilcoxon_test(multiseed_results[a], multiseed_results[b])
    print(f"{label:<45} {p:>8.4f} {'✓' if sig else '✗':>5}")

# ── Ablation Group 2 significance ─────────────────────────
print("\n" + "="*70)
print("SIGNIFICANCE TESTS — Ablation Group 2: Architecture (3 seeds)")
print("="*70)
print(f"{'Comparison':<40} {'p-value':>8} {'Sig':>5}")
print("-"*70)

g2_comparisons = [
    ("B2 vs B1 (graph structure added)",
     "B2_simple_gnn",        "B1_no_graph"),
    ("B3 vs B2 (SSM attention added)",
     "B3_ssm_attn_meanpool", "B2_simple_gnn"),
    ("B4 vs B3 (DRAC pooling vs mean)",
     "B4_full_model",        "B3_ssm_attn_meanpool"),
    ("B3 vs B1 (SSM attn vs no graph)",
     "B3_ssm_attn_meanpool", "B1_no_graph"),
    ("B4 vs B1 (full model vs no graph)",
     "B4_full_model",        "B1_no_graph"),
]
for label, a, b in g2_comparisons:
    p, sig = wilcoxon_test(ablation_ms_results[a], ablation_ms_results[b])
    print(f"{label:<40} {p:>8.4f} {'✓' if sig else '✗':>5}")

# ── Ablation Group 3 significance ─────────────────────────
print("\n" + "="*70)
print("SIGNIFICANCE TESTS — Ablation Group 3: Temporal (3 seeds)")
print("="*70)
print(f"{'Comparison':<40} {'p-value':>8} {'Sig':>5}")
print("-"*70)

g3_comparisons = [
    ("C2 vs C1 (GRU vs single frame)",
     "C2_gnn_gru",          "C1_single_frame"),
    ("C3 vs C1 (Transformer vs single)",
     "C3_gnn_transformer",  "C1_single_frame"),
    ("C4 vs C1 (LSTM vs single frame)",
     "C4_gnn_lstm",         "C1_single_frame"),
    ("C3 vs C4 (Transformer vs LSTM)",
     "C3_gnn_transformer",  "C4_gnn_lstm"),
    ("C2 vs C4 (GRU vs LSTM)",
     "C2_gnn_gru",          "C4_gnn_lstm"),
    ("C2 vs C3 (GRU vs Transformer)",
     "C2_gnn_gru",          "C3_gnn_transformer"),
]
for label, a, b in g3_comparisons:
    p, sig = wilcoxon_test(ablation_ms_results[a], ablation_ms_results[b])
    print(f"{label:<40} {p:>8.4f} {'✓' if sig else '✗':>5}")

# ── Ablation Group 1 significance ─────────────────────────
print("\n" + "="*70)
print("SIGNIFICANCE TESTS — Ablation Group 1: SSM Features (3 seeds)")
print("="*70)
print(f"{'Comparison':<40} {'p-value':>8} {'Sig':>5}")
print("-"*70)

g1_comparisons = [
    ("A1 vs A0 (kinematics vs none)",
     "A1_kinematics",  "A0_nodes_only"),
    ("A2 vs A1 (+DRAC vs kinematics)",
     "A2_+drac",       "A1_kinematics"),
    ("A3 vs A2 (+heading vs +DRAC)",
     "A3_+heading",    "A2_+drac"),
    ("A4 vs A1 (full vs kinematics)",
     "A4_full",        "A1_kinematics"),
    ("A4 vs A0 (full vs no edges)",
     "A4_full",        "A0_nodes_only"),
]
for label, a, b in g1_comparisons:
    p, sig = wilcoxon_test(ablation_ms_results[a], ablation_ms_results[b])
    print(f"{label:<40} {p:>8.4f} {'✓' if sig else '✗':>5}")

# ── Ablation Group 4 significance ─────────────────────────
print("\n" + "="*70)
print("SIGNIFICANCE TESTS — Ablation Group 4: Interaction Types (3 seeds)")
print("="*70)
print(f"{'Comparison':<40} {'p-value':>8} {'Sig':>5}")
print("-"*70)

g4_comparisons = [
    ("D3 vs D1 (VV+VP vs VV only)",
     "D3_vv_vp",    "D1_vv_only"),
    ("D3 vs D2 (VV+VP vs VP only)",
     "D3_vv_vp",    "D2_vp_only"),
    ("D4 vs D3 (all vs VV+VP)",
     "D4_all_types","D3_vv_vp"),
    ("D1 vs D2 (VV vs VP)",
     "D1_vv_only",  "D2_vp_only"),
]
for label, a, b in g4_comparisons:
    p, sig = wilcoxon_test(ablation_ms_results[a], ablation_ms_results[b])
    print(f"{label:<40} {p:>8.4f} {'✓' if sig else '✗':>5}")

# ── Final ablation summary table ──────────────────────────
print("\n" + "="*85)
print("FINAL ABLATION — Mean ± Std (3 Seeds, Test Set)")
print("="*85)
print(f"{'Variant':<25} {'AUC-ROC':>16} {'AUC-PR':>16} "
      f"{'F1':>14} {'MCC':>14} {'S@5%':>14}")
print("-"*85)

groups = [
    ("Group 1 — SSM Edge Features",
     ["A0_nodes_only","A1_kinematics","A2_+drac",
      "A3_+heading","A4_full"]),
    ("Group 2 — Architecture",
     ["B1_no_graph","B2_simple_gnn",
      "B3_ssm_attn_meanpool","B4_full_model"]),
    ("Group 3 — Temporal Modeling",
     ["C1_single_frame","C2_gnn_gru",
      "C3_gnn_transformer","C4_gnn_lstm"]),
    ("Group 4 — Interaction Types",
     ["D1_vv_only","D2_vp_only",
      "D3_vv_vp","D4_all_types"]),
]
for group_name, variants in groups:
    print(f"\n─── {group_name} ───")
    for v in variants:
        r = ablation_ms_results[v]
        print(f"{v:<25} "
              f"{r['auc_roc_mean']:>7.4f}±{r['auc_roc_std']:.4f}  "
              f"{r['auc_pr_mean']:>7.4f}±{r['auc_pr_std']:.4f}  "
              f"{r['f1_mean']:>6.4f}±{r['f1_std']:.4f}  "
              f"{r['mcc_mean']:>6.4f}±{r['mcc_std']:.4f}  "
              f"{r['sens_5far_mean']:>6.4f}±{r['sens_5far_std']:.4f}")

# Save
with open(checkpoint_path / "ablation_multiseed_final.pkl", "wb") as f:
    pickle.dump(ablation_ms_results, f)
print("\nSaved ablation_multiseed_final.pkl")

In [ ]:
"""
Figure 4.6 — Multi-group ablation analysis (4 panels A/B/C/D)
Y-axis: 0.90 to 1.0
Fixed: clean title, no star, bold outline on best bar.

Requires: checkpoint_path / "ablation_multiseed_final.pkl"
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
import matplotlib.colors as mcolors
import pickle
import warnings
warnings.filterwarnings("ignore")

# ── Publication style ─────────────────────────────────────
matplotlib.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["Times New Roman", "DejaVu Serif"],
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.linewidth"    : 1.1,
    "axes.labelsize"    : 14,
    "xtick.labelsize"   : 12,
    "ytick.labelsize"   : 12,
    "figure.dpi"        : 300,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

# ── Load ablation multiseed results ───────────────────────
with open(checkpoint_path / "ablation_multiseed_final.pkl", "rb") as f:
    abl = pickle.load(f)

# ── Group definitions ─────────────────────────────────────
groups = [
    {
        "label"   : "(a) Edge SSM feature enrichment",
        "variants": [
            ("A0\nNodes only",         "A0_nodes_only"),
            ("A1\nKinematics",         "A1_kinematics"),
            ("A2\n+DRAC",              "A2_+drac"),
            ("A3\n+Heading",           "A3_+heading"),
            ("A4\nFull edges",         "A4_full"),
        ],
        "color_base" : "#2166AC",
        "color_last" : "#1A5276",
    },
    {
        "label"   : "(b) Graph architectural enhancements",
        "variants": [
            ("B1\nNo graph",           "B1_no_graph"),
            ("B2\nSimple GNN",         "B2_simple_gnn"),
            ("B3\nSSM attn\nmean pool","B3_ssm_attn_meanpool"),
            ("B4\nFull model",         "B4_full_model"),
        ],
        "color_base" : "#1E8449",
        "color_last" : "#145A32",
    },
    {
        "label"   : "(c) Temporal encoder selection",
        "variants": [
            ("C1\nSingle frame",       "C1_single_frame"),
            ("C2\nGNN+GRU",            "C2_gnn_gru"),
            ("C3\nGNN+Transformer",    "C3_gnn_transformer"),
            ("C4\nGNN+LSTM",           "C4_gnn_lstm"),
        ],
        "color_base" : "#7B241C",
        "color_last" : "#C0392B",
    },
    {
        "label"   : "(d) Interaction-type coverage",
        "variants": [
            ("D1\nVV only",            "D1_vv_only"),
            ("D2\nVP only",            "D2_vp_only"),
            ("D3\nVV+VP",              "D3_vv_vp"),
            ("D4\nAll types",          "D4_all_types"),
        ],
        "color_base" : "#6C3483",
        "color_last" : "#4A235A",
    },
]

METRIC     = "auc_roc"
METRIC_LBL = "AUC-ROC"
Y_MIN      = 0.80
Y_MAX      = 1.005

# ── Figure — 2×2 grid ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor="white")
axes_flat = axes.flatten()

for ax, grp in zip(axes_flat, groups):
    variant_labels = [v[0] for v in grp["variants"]]
    variant_keys   = [v[1] for v in grp["variants"]]
    n              = len(variant_keys)

    means = [abl[k][f"{METRIC}_mean"] for k in variant_keys]
    stds  = [abl[k][f"{METRIC}_std"]  for k in variant_keys]

    # Color gradient: lighter → darker across variants
    base_rgb = mcolors.to_rgb(grp["color_base"])
    last_rgb = mcolors.to_rgb(grp["color_last"])
    colors   = [
        tuple(
            base_rgb[c] + (last_rgb[c] - base_rgb[c]) * i / max(n - 1, 1)
            for c in range(3)
        )
        for i in range(n)
    ]
    colors[-1] = last_rgb

    x    = np.arange(n)
    W    = 0.52
    bars = ax.bar(x, means, width=W, color=colors,
                  edgecolor="white", linewidth=1.0,
                  zorder=3)

    # Bold outline on best bar
    best_idx = int(np.argmax(means))
    bars[best_idx].set_edgecolor("#222222")
    bars[best_idx].set_linewidth(2.2)

    # Error bars
    ax.errorbar(x, means, yerr=stds,
                fmt="none", color="#333333",
                capsize=5, linewidth=1.5, zorder=4)

    # Value labels above bars — tight offset for compressed y scale
    for i, (bar, mean, std) in enumerate(zip(bars, means, stds)):
        ax.text(bar.get_x() + bar.get_width() / 2,
                mean + std + 0.001,
                f"{mean:.4f}",
                ha="center", va="bottom",
                fontsize=11, fontweight="bold",
                color=colors[i])

    # Δ improvement annotation
    delta = means[-1] - means[0]
    sign  = "+" if delta >= 0 else ""
    ax.annotate(
        f"Δ = {sign}{delta:.4f}",
        xy        = (x[-1], means[-1]),
        xytext    = (x[-1] + 0.28, means[-1] - 0.004),
        fontsize  = 10,
        color     = "#555555",
        style     = "italic",
        arrowprops= dict(arrowstyle="-", color="#CCCCCC", lw=0.8),
    )

    # Axes
    ax.set_xticks(x)
    ax.set_xticklabels(variant_labels, fontsize=11, linespacing=1.3)
    ax.set_ylabel(METRIC_LBL, labelpad=8)
    ax.set_title(grp["label"], fontsize=13, fontweight="bold", pad=10)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_yticks(np.arange(Y_MIN, 1.01, 0.02))
    ax.yaxis.set_major_formatter(
        matplotlib.ticker.FormatStrFormatter("%.2f"))
    # ax.grid(axis="y", color="#EBEBEB", linewidth=0.8, zorder=0)
    ax.tick_params(axis="both", which="both", length=4)

# ── Minimal internal heading ──────────────────────────────
# fig.suptitle(
#     "Multi-group Ablation Analysis of HERMES (AUC-ROC)",
#     fontsize   = 14,
#     fontweight = "bold",
#     y          = 1.01,
# )

plt.tight_layout(h_pad=3.5, w_pad=3.0)
plt.savefig("fig_4_6_ablation_panels.pdf")
plt.savefig("fig_4_6_ablation_panels.png")
plt.show()
print("Saved: fig_4_6_ablation_panels.pdf / .png")

In [ ]:
from scipy.stats import wilcoxon, t as t_dist
import numpy as np

def comprehensive_test(results_a, results_b, metric="auc_roc",
                       model_a_name="A", model_b_name="B"):
    scores_a = np.array([r[metric] for r in results_a["all_results"]])
    scores_b = np.array([r[metric] for r in results_b["all_results"]])
    diffs    = scores_a - scores_b
    n        = len(diffs)

    # Paired t-test
    from scipy.stats import ttest_rel
    t_stat, p_ttest = ttest_rel(scores_a, scores_b)

    # One-tailed Wilcoxon (A > B)
    if not all(d == 0 for d in diffs):
        _, p_wilcox_one = wilcoxon(scores_a, scores_b,
                                    alternative="greater")
    else:
        p_wilcox_one = 1.0

    # Cohen's d (effect size)
    cohens_d = diffs.mean() / (diffs.std() + 1e-10)

    # Interpret effect size
    if abs(cohens_d) < 0.2:
        effect = "negligible"
    elif abs(cohens_d) < 0.5:
        effect = "small"
    elif abs(cohens_d) < 0.8:
        effect = "medium"
    else:
        effect = "large"

    return {
        "mean_diff"   : diffs.mean(),
        "std_diff"    : diffs.std(),
        "p_ttest"     : p_ttest,
        "p_wilcox_1t" : p_wilcox_one,
        "cohens_d"    : cohens_d,
        "effect"      : effect,
        "n"           : n
    }


# ── Main comparison ────────────────────────────────────────
print("="*85)
print("STATISTICAL ANALYSIS — Main Comparison (AUC-ROC)")
print(f"Paired t-test + one-tailed Wilcoxon + Cohen's d  (n=5 seeds)")
print("="*85)
print(f"{'Comparison':<42} {'ΔMean':>7} {'p-t':>7} {'p-W(1t)':>8} "
      f"{'d':>6} {'Effect':<12}")
print("-"*85)

for label, a, b in main_comparisons:
    r = comprehensive_test(
        multiseed_results[a], multiseed_results[b]
    )
    sig_t = "✓" if r["p_ttest"]     < 0.05 else "✗"
    sig_w = "✓" if r["p_wilcox_1t"] < 0.05 else "✗"
    print(f"{label:<42} {r['mean_diff']:>+7.4f} "
          f"{r['p_ttest']:>6.4f}{sig_t} "
          f"{r['p_wilcox_1t']:>7.4f}{sig_w} "
          f"{r['cohens_d']:>6.2f} {r['effect']:<12}")

# ── Ablation Group 2 ──────────────────────────────────────
print("\n" + "="*85)
print("STATISTICAL ANALYSIS — Ablation Group 2: Architecture (n=3)")
print("="*85)
print(f"{'Comparison':<40} {'ΔMean':>7} {'p-t':>7} {'p-W(1t)':>8} "
      f"{'d':>6} {'Effect':<12}")
print("-"*85)

for label, a, b in g2_comparisons:
    r = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )
    sig_t = "✓" if r["p_ttest"]     < 0.05 else "✗"
    sig_w = "✓" if r["p_wilcox_1t"] < 0.05 else "✗"
    print(f"{label:<40} {r['mean_diff']:>+7.4f} "
          f"{r['p_ttest']:>6.4f}{sig_t} "
          f"{r['p_wilcox_1t']:>7.4f}{sig_w} "
          f"{r['cohens_d']:>6.2f} {r['effect']:<12}")

# ── Ablation Group 3 ──────────────────────────────────────
print("\n" + "="*85)
print("STATISTICAL ANALYSIS — Ablation Group 3: Temporal (n=3)")
print("="*85)
print(f"{'Comparison':<40} {'ΔMean':>7} {'p-t':>7} {'p-W(1t)':>8} "
      f"{'d':>6} {'Effect':<12}")
print("-"*85)

for label, a, b in g3_comparisons:
    r = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )
    sig_t = "✓" if r["p_ttest"]     < 0.05 else "✗"
    sig_w = "✓" if r["p_wilcox_1t"] < 0.05 else "✗"
    print(f"{label:<40} {r['mean_diff']:>+7.4f} "
          f"{r['p_ttest']:>6.4f}{sig_t} "
          f"{r['p_wilcox_1t']:>7.4f}{sig_w} "
          f"{r['cohens_d']:>6.2f} {r['effect']:<12}")

# ── Ablation Groups 1 and 4 ───────────────────────────────
print("\n" + "="*85)
print("STATISTICAL ANALYSIS — Ablation Group 1: SSM Features (n=3)")
print("="*85)
print(f"{'Comparison':<40} {'ΔMean':>7} {'p-t':>7} {'p-W(1t)':>8} "
      f"{'d':>6} {'Effect':<12}")
print("-"*85)

for label, a, b in g1_comparisons:
    r = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )
    sig_t = "✓" if r["p_ttest"]     < 0.05 else "✗"
    sig_w = "✓" if r["p_wilcox_1t"] < 0.05 else "✗"
    print(f"{label:<40} {r['mean_diff']:>+7.4f} "
          f"{r['p_ttest']:>6.4f}{sig_t} "
          f"{r['p_wilcox_1t']:>7.4f}{sig_w} "
          f"{r['cohens_d']:>6.2f} {r['effect']:<12}")

print("\n" + "="*85)
print("STATISTICAL ANALYSIS — Ablation Group 4: Interaction Types (n=3)")
print("="*85)
print(f"{'Comparison':<40} {'ΔMean':>7} {'p-t':>7} {'p-W(1t)':>8} "
      f"{'d':>6} {'Effect':<12}")
print("-"*85)

for label, a, b in g4_comparisons:
    r = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )
    sig_t = "✓" if r["p_ttest"]     < 0.05 else "✗"
    sig_w = "✓" if r["p_wilcox_1t"] < 0.05 else "✗"
    print(f"{label:<40} {r['mean_diff']:>+7.4f} "
          f"{r['p_ttest']:>6.4f}{sig_t} "
          f"{r['p_wilcox_1t']:>7.4f}{sig_w} "
          f"{r['cohens_d']:>6.2f} {r['effect']:<12}")

In [ ]:
# Check how many seeds are in each ablation result
print("=== Seed count verification ===")
for group_name, variants in [
    ("Group 1", ["A0_nodes_only","A1_kinematics","A2_+drac",
                 "A3_+heading","A4_full"]),
    ("Group 2", ["B1_no_graph","B2_simple_gnn",
                 "B3_ssm_attn_meanpool","B4_full_model"]),
    ("Group 3", ["C1_single_frame","C2_gnn_gru",
                 "C3_gnn_transformer","C4_gnn_lstm"]),
    ("Group 4", ["D1_vv_only","D2_vp_only",
                 "D3_vv_vp","D4_all_types"]),
]:
    print(f"\n{group_name}:")
    for v in variants:
        n_seeds = len(ablation_ms_results[v]["all_results"])
        seeds   = [r["seed"] for r in ablation_ms_results[v]["all_results"]]
        print(f"  {v:<25} n_seeds={n_seeds}  seeds={seeds}")

In [ ]:
# Fix the header labels — reprint with correct n=5
print("="*85)
print("STATISTICAL ANALYSIS — Main Comparison (AUC-ROC, n=5 seeds)")
print("="*85)
print(f"{'Comparison':<42} {'ΔMean':>7} {'p-t':>7} {'p-W(1t)':>8} "
      f"{'d':>6} {'Effect':<12}")
print("-"*85)
for label, a, b in main_comparisons:
    r = comprehensive_test(multiseed_results[a], multiseed_results[b])
    sig_t = "✓" if r["p_ttest"]     < 0.05 else "✗"
    sig_w = "✓" if r["p_wilcox_1t"] < 0.05 else "✗"
    print(f"{label:<42} {r['mean_diff']:>+7.4f} "
          f"{r['p_ttest']:>6.4f}{sig_t} "
          f"{r['p_wilcox_1t']:>7.4f}{sig_w} "
          f"{r['cohens_d']:>6.2f} {r['effect']:<12}")

for group_title, comparisons in [
    ("Ablation Group 1: SSM Features (n=5 seeds)",    g1_comparisons),
    ("Ablation Group 2: Architecture (n=5 seeds)",    g2_comparisons),
    ("Ablation Group 3: Temporal Modeling (n=5 seeds)", g3_comparisons),
    ("Ablation Group 4: Interaction Types (n=5 seeds)", g4_comparisons),
]:
    print("\n" + "="*85)
    print(f"STATISTICAL ANALYSIS — {group_title}")
    print("="*85)
    print(f"{'Comparison':<40} {'ΔMean':>7} {'p-t':>7} {'p-W(1t)':>8} "
          f"{'d':>6} {'Effect':<12}")
    print("-"*85)
    for label, a, b in comparisons:
        r     = comprehensive_test(ablation_ms_results[a],
                                   ablation_ms_results[b])
        sig_t = "✓" if r["p_ttest"]     < 0.05 else "✗"
        sig_w = "✓" if r["p_wilcox_1t"] < 0.05 else "✗"
        print(f"{label:<40} {r['mean_diff']:>+7.4f} "
              f"{r['p_ttest']:>6.4f}{sig_t} "
              f"{r['p_wilcox_1t']:>7.4f}{sig_w} "
              f"{r['cohens_d']:>6.2f} {r['effect']:<12}")

print("\n" + "="*85)
print("NOTE: All comparisons use n=5 random seeds [42, 123, 456, 789, 2024]")
print("      Paired t-test (p-t) and one-tailed Wilcoxon (p-W) reported")
print("      Cohen's d effect size: <0.2 negligible, 0.2-0.5 small,")
print("                             0.5-0.8 medium, >0.8 large")
print("="*85)

In [ ]:
# Save significance results
sig_summary = {
    "main_comparison": {},
    "ablation_g1"    : {},
    "ablation_g2"    : {},
    "ablation_g3"    : {},
    "ablation_g4"    : {},
}

for label, a, b in main_comparisons:
    sig_summary["main_comparison"][label] = comprehensive_test(
        multiseed_results[a], multiseed_results[b]
    )

for label, a, b in g1_comparisons:
    sig_summary["ablation_g1"][label] = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )

for label, a, b in g2_comparisons:
    sig_summary["ablation_g2"][label] = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )

for label, a, b in g3_comparisons:
    sig_summary["ablation_g3"][label] = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )

for label, a, b in g4_comparisons:
    sig_summary["ablation_g4"][label] = comprehensive_test(
        ablation_ms_results[a], ablation_ms_results[b]
    )

with open(checkpoint_path / "significance_results.pkl", "wb") as f:
    pickle.dump(sig_summary, f)
print("Saved significance_results.pkl")

# ── Paper-ready sentences ──────────────────────────────────
print("\n" + "="*70)
print("PAPER-READY STATISTICAL STATEMENTS")
print("="*70)

print("""
Section 4.2 — Main Comparison:
  Enhanced HERMES significantly outperformed all baseline models on
  AUC-ROC (paired t-test, p<0.001; one-tailed Wilcoxon, p=0.031)
  with large effect sizes across all comparisons (Cohen's d=5.14–19.23).
  Baseline HERMES also significantly outperformed the Transformer
  (p=0.017, d=1.98), confirming the graph structure advantage
  independent of architectural enhancements.

Section 4.4.2 — Group 2 Architecture:
  Graph structure (B2 vs B1) contributed significantly (p<0.001, d=11.67).
  SSM-biased attention (B3 vs B2) contributed significantly (p<0.001,
  d=12.14). DRAC-weighted pooling did not show significant improvement
  over mean pooling at 10 training epochs (p=0.018 in the wrong direction),
  consistent with our finding that pooling benefits require full convergence.

Section 4.4.3 — Group 3 Temporal:
  All sequence-based models significantly outperformed the single-frame
  baseline (p<0.001, d>28 for all comparisons), confirming that temporal
  context is critical for conflict prediction. No significant difference
  was found among GRU, Transformer, and LSTM variants
  (p>0.82, d<0.12 for all pairwise comparisons), confirming that
  temporal architecture choice is negligible relative to having
  temporal context.

Section 4.4.1 — Group 1 SSM Features:
  Adding kinematic edge features (A1 vs A0) provided a significant
  improvement (p=0.002, d=3.48). Additional SSM features (DRAC,
  heading_diff) did not achieve significance at 10 epochs (p>0.05),
  suggesting these features require more training to show their benefit.
  The full 6-dim edge feature set significantly outperformed no-edge
  baseline (p=0.004, d=3.06).

Section 4.4.4 — Group 4 Interaction Types:
  VV+VP significantly outperformed VP-only (p=0.0001, d=7.02) and
  VV-only significantly outperformed VP-only (p=0.0001, d=9.08).
  No significant difference was found between VV+VP and VV-only
  (p=0.83, d=0.11) or between all-types and VV+VP (p=0.82, d=0.12),
  confirming that PP interactions contribute negligible information.
""")

In [ ]:
# ── Section 1: Raw Data Overview ──────────────────────────
print("="*65)
print("1. RAW DATA OVERVIEW")
print("="*65)
print(f"  Video segments            : 38")
print(f"  Recording date            : August 9, 2025")
print(f"  Recording period          : 9:51 AM – 7:39 PM")
print(f"  Total duration            : ~10 hours")
print(f"  Frame rate                : 30 fps")
print(f"  Resolution                : 1280 × 720 pixels")
print(f"  Intersection              : Centerpoint Road / Outlet Mall")
print(f"  Location                  : San Marcos, Texas, USA")
print(f"  Intersection type         : Signalized")
print(f"  Raw detections (all)      : ~15.06M rows")
print(f"  After class filter        : 14.03M rows")
print(f"  After ROI filter          : 3.88M rows  (27.7% retention)")
print(f"  After track QC            : 2.45M rows")
print(f"  After motion filter       : {len(df):,} rows  (final)")

In [ ]:
# ── Section 2: Agent Distribution ─────────────────────────
print("="*65)
print("2. AGENT CLASS DISTRIBUTION")
print("="*65)
class_dist = df["class"].value_counts()
total      = len(df)
for cls, cnt in class_dist.items():
    print(f"  {cls:<15} {cnt:>10,}  ({cnt/total*100:.1f}%)")

veh_total = df[df["agent_type"]=="vehicle"].shape[0]
ped_total = df[df["agent_type"]=="pedestrian"].shape[0]
print(f"\n  Vehicle total   : {veh_total:>10,}  ({veh_total/total*100:.1f}%)")
print(f"  Pedestrian total: {ped_total:>10,}  ({ped_total/total*100:.1f}%)")

In [ ]:
# ── Section 3: Track Statistics ────────────────────────────
print("="*65)
print("3. TRACK STATISTICS")
print("="*65)
track_stats = df.groupby(["video_id","track_id"]).size()
veh_tracks  = df[df["agent_type"]=="vehicle"].groupby(
    ["video_id","track_id"]).size()
ped_tracks  = df[df["agent_type"]=="pedestrian"].groupby(
    ["video_id","track_id"]).size()

print(f"  Total unique tracks       : {len(track_stats):,}")
print(f"  Vehicle tracks            : {len(veh_tracks):,}")
print(f"  Pedestrian tracks         : {len(ped_tracks):,}")
print(f"\n  Track length (all tracks):")
print(f"    Mean   : {track_stats.mean():.1f} frames  "
      f"({track_stats.mean()/30:.2f}s)")
print(f"    Median : {track_stats.median():.0f} frames  "
      f"({track_stats.median()/30:.2f}s)")
print(f"    Std    : {track_stats.std():.1f} frames")
print(f"    Min    : {track_stats.min()} frames")
print(f"    Max    : {track_stats.max():,} frames  "
      f"({track_stats.max()/30:.1f}s)")

print(f"\n  Vehicle track length:")
print(f"    Mean   : {veh_tracks.mean():.1f} frames  "
      f"({veh_tracks.mean()/30:.2f}s)")
print(f"    Median : {veh_tracks.median():.0f} frames")

print(f"\n  Pedestrian track length:")
print(f"    Mean   : {ped_tracks.mean():.1f} frames  "
      f"({ped_tracks.mean()/30:.2f}s)")
print(f"    Median : {ped_tracks.median():.0f} frames")

In [ ]:
# ── Section 4: Kinematic Statistics ───────────────────────
print("="*65)
print("4. KINEMATIC STATISTICS")
print("="*65)
for agent in ["vehicle","pedestrian"]:
    sub = df[df["agent_type"]==agent]
    print(f"\n  {agent.capitalize()}:")
    print(f"    Speed (m/s)  "
          f"mean={sub['speed'].mean():.3f}  "
          f"std={sub['speed'].std():.3f}  "
          f"min={sub['speed'].min():.3f}  "
          f"max={sub['speed'].max():.3f}")
    print(f"    Speed (km/h) "
          f"mean={sub['speed'].mean()*3.6:.2f}  "
          f"median={sub['speed'].median()*3.6:.2f}  "
          f"max={sub['speed'].max()*3.6:.2f}")
    print(f"    Accel (m/s²) "
          f"mean={sub['accel'].mean():.3f}  "
          f"std={sub['accel'].std():.3f}  "
          f"max={sub['accel'].max():.3f}")
    print(f"    Heading (°)  "
          f"mean={sub['heading'].mean():.2f}  "
          f"std={sub['heading'].std():.2f}")

print(f"\n  Intersection geometry:")
x_span = df["x_local_m"].max() - df["x_local_m"].min()
y_span = df["y_local_m"].max() - df["y_local_m"].min()
print(f"    ROI X span : {x_span:.1f} m")
print(f"    ROI Y span : {y_span:.1f} m")
print(f"    ROI area   : {x_span*y_span:.0f} m²")
print(f"    Center     : {lat0:.6f}°N, {abs(lon0):.6f}°W")

In [ ]:
from scipy.stats import mannwhitneyu, kruskal

# ── Section 4: Kinematic Statistics + Statistical Tests ───
print("="*65)
print("4. KINEMATIC STATISTICS")
print("="*65)
for agent in ["vehicle","pedestrian"]:
    sub = df[df["agent_type"]==agent]
    print(f"\n  {agent.capitalize()}:")
    print(f"    Speed (m/s)  "
          f"mean={sub['speed'].mean():.3f}  "
          f"std={sub['speed'].std():.3f}  "
          f"min={sub['speed'].min():.3f}  "
          f"max={sub['speed'].max():.3f}")
    print(f"    Speed (km/h) "
          f"mean={sub['speed'].mean()*3.6:.2f}  "
          f"median={sub['speed'].median()*3.6:.2f}  "
          f"max={sub['speed'].max()*3.6:.2f}")
    print(f"    Accel (m/s²) "
          f"mean={sub['accel'].mean():.3f}  "
          f"std={sub['accel'].std():.3f}  "
          f"max={sub['accel'].max():.3f}")
    print(f"    Heading (°)  "
          f"mean={sub['heading'].mean():.2f}  "
          f"std={sub['heading'].std():.2f}")

# Test 1: Vehicle vs Pedestrian speed
print(f"\n  Statistical Tests:")
veh_spd = df[df["agent_type"]=="vehicle"]["speed"].dropna().sample(
    10000, random_state=42)
ped_spd = df[df["agent_type"]=="pedestrian"]["speed"].dropna().sample(
    min(10000, (df["agent_type"]=="pedestrian").sum()),
    random_state=42)
stat, p = mannwhitneyu(veh_spd, ped_spd, alternative="two-sided")
print(f"\n  [1] Mann-Whitney U: Vehicle vs Pedestrian speed")
print(f"    Vehicle mean   : {veh_spd.mean():.3f} m/s  "
      f"({veh_spd.mean()*3.6:.2f} km/h)")
print(f"    Pedestrian mean: {ped_spd.mean():.3f} m/s  "
      f"({ped_spd.mean()*3.6:.2f} km/h)")
print(f"    U={stat:.0f}  p={p:.6f}  "
      f"Sig={'Yes ✓' if p<0.05 else 'No ✗'}")

# Test 2: Vehicle vs Pedestrian acceleration
veh_acc = df[df["agent_type"]=="vehicle"]["accel"].dropna().sample(
    10000, random_state=42)
ped_acc = df[df["agent_type"]=="pedestrian"]["accel"].dropna().sample(
    min(10000, (df["agent_type"]=="pedestrian").sum()),
    random_state=42)
stat, p = mannwhitneyu(veh_acc, ped_acc, alternative="two-sided")
print(f"\n  [2] Mann-Whitney U: Vehicle vs Pedestrian acceleration")
print(f"    Vehicle mean   : {veh_acc.mean():.3f} m/s²")
print(f"    Pedestrian mean: {ped_acc.mean():.3f} m/s²")
print(f"    U={stat:.0f}  p={p:.6f}  "
      f"Sig={'Yes ✓' if p<0.05 else 'No ✗'}")

# Test 3: Kruskal-Wallis across vehicle classes
print(f"\n  [3] Kruskal-Wallis: Speed across vehicle classes")
class_speeds = []
class_names  = []
for c in ["car","truck","bus","motorcycle"]:
    sub = df[df["class"]==c]["speed"].dropna()
    if len(sub) > 10:
        class_speeds.append(
            sub.sample(min(5000, len(sub)), random_state=42).values
        )
        class_names.append(c)

stat, p = kruskal(*class_speeds)
print(f"    H={stat:.4f}  p={p:.6f}  "
      f"Sig={'Yes ✓' if p<0.05 else 'No ✗'}")
for c in class_names:
    sub = df[df["class"]==c]["speed"]
    print(f"    {c:<12} n={len(sub):,}  "
          f"mean={sub.mean():.3f} m/s  "
          f"({sub.mean()*3.6:.2f} km/h)")

print(f"\n  Intersection geometry:")
x_span = df["x_local_m"].max() - df["x_local_m"].min()
y_span = df["y_local_m"].max() - df["y_local_m"].min()
print(f"    ROI X span : {x_span:.1f} m")
print(f"    ROI Y span : {y_span:.1f} m")
print(f"    ROI area   : {x_span*y_span:.0f} m²")
print(f"    Center     : {lat0:.6f}°N, {abs(lon0):.6f}°W")

In [ ]:
# ── Section 4 simplified: Vehicle vs Pedestrian only ──────
print("="*65)
print("4. KINEMATIC STATISTICS (Final Dataset)")
print("="*65)

for agent in ["vehicle", "pedestrian"]:
    sub = df[df["agent_type"]==agent]
    print(f"\n  {agent.capitalize()} (n={len(sub):,} rows, "
          f"{sub.groupby(['video_id','track_id']).ngroups:,} tracks):")
    print(f"    Speed (m/s)  "
          f"mean={sub['speed'].mean():.3f}  "
          f"std={sub['speed'].std():.3f}  "
          f"median={sub['speed'].median():.3f}  "
          f"max={sub['speed'].max():.3f}")
    print(f"    Speed (km/h) "
          f"mean={sub['speed'].mean()*3.6:.2f}  "
          f"median={sub['speed'].median()*3.6:.2f}  "
          f"max={sub['speed'].max()*3.6:.2f}")
    print(f"    Accel (m/s²) "
          f"mean={sub['accel'].mean():.3f}  "
          f"std={sub['accel'].std():.3f}  "
          f"median={sub['accel'].median():.3f}  "
          f"max={sub['accel'].max():.3f}")

# Mann-Whitney U: speed
veh_spd = df[df["agent_type"]=="vehicle"]["speed"].dropna().sample(
    10000, random_state=42)
ped_spd = df[df["agent_type"]=="pedestrian"]["speed"].dropna().sample(
    min(10000, (df["agent_type"]=="pedestrian").sum()),
    random_state=42)
stat_s, p_s = mannwhitneyu(veh_spd, ped_spd, alternative="two-sided")

# Mann-Whitney U: acceleration
veh_acc = df[df["agent_type"]=="vehicle"]["accel"].dropna().sample(
    10000, random_state=42)
ped_acc = df[df["agent_type"]=="pedestrian"]["accel"].dropna().sample(
    min(10000, (df["agent_type"]=="pedestrian").sum()),
    random_state=42)
stat_a, p_a = mannwhitneyu(veh_acc, ped_acc, alternative="two-sided")

print(f"\n  Statistical Tests:")
print(f"\n  [1] Mann-Whitney U: Vehicle vs Pedestrian speed")
print(f"    U={stat_s:.0f}  p={p_s:.6f}  "
      f"Sig={'Yes ✓' if p_s<0.05 else 'No ✗'}")
print(f"    Interpretation: Vehicle speed significantly higher "
      f"than pedestrian (p<0.001)")

print(f"\n  [2] Mann-Whitney U: Vehicle vs Pedestrian acceleration")
print(f"    U={stat_a:.0f}  p={p_a:.6f}  "
      f"Sig={'Yes ✓' if p_a<0.05 else 'No ✗'}")
print(f"    Interpretation: Vehicle acceleration significantly higher "
      f"than pedestrian (p<0.001)")

print(f"\n  Intersection geometry:")
x_span = df["x_local_m"].max() - df["x_local_m"].min()
y_span = df["y_local_m"].max() - df["y_local_m"].min()
print(f"    ROI X span : {x_span:.1f} m")
print(f"    ROI Y span : {y_span:.1f} m")
print(f"    ROI area   : {x_span*y_span:.0f} m²")
print(f"    Center     : {lat0:.6f}°N, {abs(lon0):.6f}°W")

In [ ]:
# ── Section 4: Kinematic Statistics (descriptive only) ────
print("="*65)
print("4. KINEMATIC STATISTICS (Final Dataset)")
print("="*65)

for agent in ["vehicle", "pedestrian"]:
    sub    = df[df["agent_type"]==agent]
    tracks = sub.groupby(["video_id","track_id"]).ngroups

    print(f"\n  {agent.capitalize()}:")
    print(f"    Observations : {len(sub):,} rows")
    print(f"    Tracks       : {tracks:,}")

    print(f"\n    Speed (m/s):")
    print(f"      Mean   : {sub['speed'].mean():.3f}")
    print(f"      Median : {sub['speed'].median():.3f}")
    print(f"      Std    : {sub['speed'].std():.3f}")
    print(f"      Min    : {sub['speed'].min():.3f}")
    print(f"      Max    : {sub['speed'].max():.3f}")
    print(f"      Mean (km/h): {sub['speed'].mean()*3.6:.2f}")

    print(f"\n    Acceleration (m/s²):")
    print(f"      Mean   : {sub['accel'].mean():.3f}")
    print(f"      Median : {sub['accel'].median():.3f}")
    print(f"      Std    : {sub['accel'].std():.3f}")
    print(f"      Min    : {sub['accel'].min():.3f}")
    print(f"      Max    : {sub['accel'].max():.3f}")

    print(f"\n    Heading (degrees):")
    print(f"      Mean   : {sub['heading'].mean():.2f}")
    print(f"      Std    : {sub['heading'].std():.2f}")

print(f"\n  Intersection geometry:")
x_span = df["x_local_m"].max() - df["x_local_m"].min()
y_span = df["y_local_m"].max() - df["y_local_m"].min()
print(f"    ROI X span : {x_span:.1f} m")
print(f"    ROI Y span : {y_span:.1f} m")
print(f"    ROI area   : {x_span*y_span:.0f} m²")
print(f"    Center     : {lat0:.6f}°N, {abs(lon0):.6f}°W")

In [ ]:
# ── Section 5: SSM Statistics ──────────────────────────────
print("="*65)
print("5. SURROGATE SAFETY MEASURE STATISTICS")
print("="*65)
print(f"  Total interaction pairs   : {len(pairs):,}")

print(f"\n  By interaction type:")
for itype, cnt in pairs["interaction_type"].value_counts().items():
    print(f"    {itype}  : {cnt:,}  ({cnt/len(pairs)*100:.1f}%)")

valid_ttc  = pairs["ttc"].dropna()
valid_drac = pairs["drac"].dropna()

print(f"\n  TTC (approaching pairs only):")
print(f"    Valid count : {len(valid_ttc):,}  "
      f"({len(valid_ttc)/len(pairs)*100:.1f}% of all pairs)")
print(f"    Mean        : {valid_ttc.mean():.3f} s")
print(f"    Median      : {valid_ttc.median():.3f} s")
print(f"    Std         : {valid_ttc.std():.3f} s")
print(f"    Min         : {valid_ttc.min():.4f} s")
print(f"    Max         : {valid_ttc.max():.3f} s")
print(f"    TTC < 1.5s  : {(valid_ttc<1.5).sum():,}  "
      f"({(valid_ttc<1.5).mean()*100:.1f}%)  serious conflict threshold")
print(f"    TTC < 3.0s  : {(valid_ttc<3.0).sum():,}  "
      f"({(valid_ttc<3.0).mean()*100:.1f}%)  general conflict threshold")

print(f"\n  DRAC:")
print(f"    Valid count : {len(valid_drac):,}  "
      f"({len(valid_drac)/len(pairs)*100:.1f}% of all pairs)")
print(f"    Mean        : {valid_drac.mean():.3f} m/s²")
print(f"    Median      : {valid_drac.median():.3f} m/s²")
print(f"    Std         : {valid_drac.std():.3f} m/s²")
print(f"    Min         : {valid_drac.min():.4f} m/s²")
print(f"    Max         : {valid_drac.max():.3f} m/s²")

print(f"\n  Distance between pairs:")
print(f"    Mean        : {pairs['distance'].mean():.3f} m")
print(f"    Median      : {pairs['distance'].median():.3f} m")
print(f"    Std         : {pairs['distance'].std():.3f} m")
print(f"    Min         : {pairs['distance'].min():.3f} m")
print(f"    Max         : {pairs['distance'].max():.3f} m")

print(f"\n  Relative speed between pairs:")
print(f"    Mean        : {pairs['rel_speed'].mean():.3f} m/s")
print(f"    Median      : {pairs['rel_speed'].median():.3f} m/s")
print(f"    Std         : {pairs['rel_speed'].std():.3f} m/s")
print(f"    Max         : {pairs['rel_speed'].max():.3f} m/s")

In [ ]:
# ── Section 6: Conflict Statistics ────────────────────────
print("="*65)
print("6. CONFLICT STATISTICS")
print("="*65)

print(f"\n  Pair-level conflict labels:")
print(f"  Total unique pairs        : {len(pair_summary):,}")
for label, name in [(0,"No conflict"),
                    (1,"General conflict"),
                    (2,"Serious conflict")]:
    cnt = (pair_summary["pair_conflict_label"]==label).sum()
    pct = cnt/len(pair_summary)*100
    print(f"    {name:<20}: {cnt:,}  ({pct:.1f}%)")

print(f"\n  Conflict by interaction type:")
ct = pair_summary.groupby(
    ["interaction_type","pair_conflict_label"]
).size().unstack(fill_value=0)
print(ct.to_string())

print(f"\n  Conflict rate by interaction type:")
for itype in ["vv","vp","pp"]:
    sub  = pair_summary[pair_summary["interaction_type"]==itype]
    rate = (sub["pair_conflict_label"]>0).mean()
    print(f"    {itype}  : {rate:.1%}  ({(sub['pair_conflict_label']>0).sum():,} "
          f"of {len(sub):,} pairs)")

print(f"\n  Labeling thresholds used:")
print(f"    General conflict  : min_dist < 5m (VV) or 3m (VP)")
print(f"                        AND min_TTC < 3.0s")
print(f"    Serious conflict  : min_dist < 3m (VV) or 2m (VP)")
print(f"                        AND min_TTC < 1.5s")
print(f"    PP excluded       : pedestrian proximity is normal behavior")
print(f"    Heading filter    : VV pairs hdg_diff>120° AND dist>5m excluded")
print(f"    Min rel speed     : VV≥1.0 m/s, VP≥0.5 m/s")

print(f"\n  SSM validation tests:")
# TTC conflict vs no-conflict
ttc_c  = pairs[pairs["pair_conflict_label"]>0]["ttc"].dropna()
ttc_nc = pairs[pairs["pair_conflict_label"]==0]["ttc"].dropna()
ttc_c_s  = ttc_c.sample(min(10000, len(ttc_c)),   random_state=42)
ttc_nc_s = ttc_nc.sample(min(10000, len(ttc_nc)), random_state=42)
stat, p  = mannwhitneyu(ttc_c_s, ttc_nc_s, alternative="less")
print(f"\n  [1] Mann-Whitney U: TTC conflict vs no-conflict")
print(f"    Conflict TTC mean    : {ttc_c.mean():.3f} s")
print(f"    No-conflict TTC mean : {ttc_nc.mean():.3f} s")
print(f"    U={stat:.0f}  p={p:.6f}  "
      f"Sig={'Yes ✓' if p<0.05 else 'No ✗'}")

# DRAC conflict vs no-conflict
drac_c  = pairs[pairs["pair_conflict_label"]>0]["drac"].dropna()
drac_nc = pairs[pairs["pair_conflict_label"]==0]["drac"].dropna()
drac_c_s  = drac_c.sample(min(10000, len(drac_c)),   random_state=42)
drac_nc_s = drac_nc.sample(min(10000, len(drac_nc)), random_state=42)
stat, p   = mannwhitneyu(drac_c_s, drac_nc_s, alternative="greater")
print(f"\n  [2] Mann-Whitney U: DRAC conflict vs no-conflict")
print(f"    Conflict DRAC mean    : {drac_c.mean():.3f} m/s²")
print(f"    No-conflict DRAC mean : {drac_nc.mean():.3f} m/s²")
print(f"    U={stat:.0f}  p={p:.6f}  "
      f"Sig={'Yes ✓' if p<0.05 else 'No ✗'}")

# Distance conflict vs no-conflict
dist_c  = pairs[pairs["pair_conflict_label"]>0]["distance"]
dist_nc = pairs[pairs["pair_conflict_label"]==0]["distance"]
dist_c_s  = dist_c.sample(min(10000, len(dist_c)),   random_state=42)
dist_nc_s = dist_nc.sample(min(10000, len(dist_nc)), random_state=42)
stat, p   = mannwhitneyu(dist_c_s, dist_nc_s, alternative="less")
print(f"\n  [3] Mann-Whitney U: Distance conflict vs no-conflict")
print(f"    Conflict dist mean    : {dist_c.mean():.3f} m")
print(f"    No-conflict dist mean : {dist_nc.mean():.3f} m")
print(f"    U={stat:.0f}  p={p:.6f}  "
      f"Sig={'Yes ✓' if p<0.05 else 'No ✗'}")

# Chi-square: conflict rate VV vs VP
vv = pair_summary[pair_summary["interaction_type"]=="vv"]
vp = pair_summary[pair_summary["interaction_type"]=="vp"]
contingency = np.array([
    [(vv["pair_conflict_label"]>0).sum(),
     (vv["pair_conflict_label"]==0).sum()],
    [(vp["pair_conflict_label"]>0).sum(),
     (vp["pair_conflict_label"]==0).sum()]
])
from scipy.stats import chi2_contingency
chi2, p, dof, _ = chi2_contingency(contingency)
print(f"\n  [4] Chi-square: Conflict rate VV vs VP")
print(f"    VV conflict rate : {(vv['pair_conflict_label']>0).mean():.1%}")
print(f"    VP conflict rate : {(vp['pair_conflict_label']>0).mean():.1%}")
print(f"    Chi2={chi2:.4f}  p={p:.6f}  "
      f"Sig={'Yes ✓' if p<0.05 else 'No ✗'}")

In [ ]:
# Correct way to display
print("Corrected p-value display:")
print(f"  TTC:      p < 0.001  (U=41,209,451)")
print(f"  DRAC:     p < 0.001  (U=53,471,752)")
print(f"  Distance: p < 0.001  (U=24,664,827)")
print(f"  VV vs VP: p < 0.001  (Chi2=1187.48, df=1)")
print()
print("Paper sentence:")
print("Conflict pairs exhibited significantly lower TTC")
print("(mean=2.158s vs 2.716s, U=41,209,451, p<0.001),")
print("higher DRAC (mean=2.138 vs 1.574 m/s², p<0.001),")
print("and smaller inter-agent distances")
print("(mean=9.001m vs 13.802m, p<0.001),")
print("validating the SSM-based conflict labeling criteria.")

In [ ]:

# ── Section 7: Scene Graph Statistics ─────────────────────
print("="*65)
print("7. SCENE GRAPH STATISTICS")
print("="*65)

# Compute from sequences in memory
all_graphs      = [g for seq in sequences for g in seq]
nodes_per_frame = [len(g["nodes"]) for g in all_graphs]
edges_per_frame = [len(g["edges"]) for g in all_graphs]
conflict_frames = sum(g["label"] for g in all_graphs)
total_frames    = len(all_graphs)

print(f"\n  Total frames (in sequences) : {total_frames:,}")
print(f"  Conflict frames             : {conflict_frames:,}  "
      f"({conflict_frames/total_frames*100:.1f}%)")
print(f"  Safe frames                 : {total_frames-conflict_frames:,}  "
      f"({(1-conflict_frames/total_frames)*100:.1f}%)")

print(f"\n  Nodes (agents) per frame:")
print(f"    Mean   : {np.mean(nodes_per_frame):.2f}")
print(f"    Median : {np.median(nodes_per_frame):.0f}")
print(f"    Std    : {np.std(nodes_per_frame):.2f}")
print(f"    Min    : {min(nodes_per_frame)}")
print(f"    Max    : {max(nodes_per_frame)}")

print(f"\n  Edges (interactions) per frame:")
print(f"    Mean   : {np.mean(edges_per_frame):.2f}")
print(f"    Median : {np.median(edges_per_frame):.0f}")
print(f"    Std    : {np.std(edges_per_frame):.2f}")
print(f"    Min    : {min(edges_per_frame)}")
print(f"    Max    : {max(edges_per_frame)}")

# Node count distribution
print(f"\n  Node count distribution:")
node_series = pd.Series(nodes_per_frame).value_counts().sort_index()
for k, v in node_series.items():
    print(f"    {k} agent(s)  : {v:,}  ({v/total_frames*100:.1f}%)")

# Edge type breakdown from a sample
vv_edges = sum(
    sum(1 for e in g["edges"] if e["interaction_type"]=="vv")
    for g in all_graphs[:10000]
)
vp_edges = sum(
    sum(1 for e in g["edges"] if e["interaction_type"]=="vp")
    for g in all_graphs[:10000]
)
pp_edges = sum(
    sum(1 for e in g["edges"] if e["interaction_type"]=="pp")
    for g in all_graphs[:10000]
)
total_edges_sample = vv_edges + vp_edges + pp_edges

print(f"\n  Edge type distribution (sample of 10,000 frames):")
print(f"    VV edges : {vv_edges:,}  ({vv_edges/total_edges_sample*100:.1f}%)")
print(f"    VP edges : {vp_edges:,}  ({vp_edges/total_edges_sample*100:.1f}%)")
print(f"    PP edges : {pp_edges:,}  ({pp_edges/total_edges_sample*100:.1f}%)")


In [ ]:
# ── Section 8: Sequence Statistics ────────────────────────
print("="*65)
print("8. SEQUENCE STATISTICS")
print("="*65)

# Convert all label arrays
seq_labels_np   = np.array(seq_labels)
train_labels_np = np.array(train_labels)
val_labels_np   = np.array(val_labels)
test_labels_np  = np.array(test_labels)

print(f"\n  Sequence construction:")
print(f"    Sequence length   : {SEQ_LEN} frames  "
      f"({SEQ_LEN/30:.3f}s at 30fps)")
print(f"    Stride            : 4 frames  (50% overlap)")
print(f"    Min nodes/frame   : 2 (frames with <2 agents excluded)")

print(f"\n  Total sequences     : {len(sequences):,}")
print(f"  Conflict sequences  : {seq_labels_np.sum():,}  "
      f"({seq_labels_np.mean()*100:.1f}%)")
print(f"  Safe sequences      : {(seq_labels_np==0).sum():,}  "
      f"({(1-seq_labels_np.mean())*100:.1f}%)")
print(f"  Class imbalance     : "
      f"{(seq_labels_np==0).sum()/seq_labels_np.sum():.1f}:1  "
      f"(safe:conflict)")

print(f"\n  Train/Val/Test split (chronological by video):")
print(f"    Train : {len(train_seqs):,}  "
      f"({len(train_seqs)/len(sequences)*100:.1f}%)  "
      f"conflict={train_labels_np.mean()*100:.1f}%  "
      f"9:51AM–2:38PM  (31 videos)")
print(f"    Val   : {len(val_seqs):,}  "
      f"({len(val_seqs)/len(sequences)*100:.1f}%)  "
      f"conflict={val_labels_np.mean()*100:.1f}%  "
      f"2:38PM–4:38PM  (2 videos)")
print(f"    Test  : {len(test_seqs):,}  "
      f"({len(test_seqs)/len(sequences)*100:.1f}%)  "
      f"conflict={test_labels_np.mean()*100:.1f}%  "
      f"5:38PM–7:39PM  (3 videos)")

print(f"\n  Statistical test: Conflict rate by time period")
from scipy.stats import chi2_contingency
contingency = np.array([
    [train_labels_np.sum(), (train_labels_np==0).sum()],
    [val_labels_np.sum(),   (val_labels_np==0).sum()],
    [test_labels_np.sum(),  (test_labels_np==0).sum()]
])
chi2, p, dof, _ = chi2_contingency(contingency)
print(f"    Chi2={chi2:.4f}  df={dof}  p<0.001  Sig=Yes ✓")
print(f"    Interpretation: Conflict rate differs significantly")
print(f"    across time periods, reflecting natural traffic")
print(f"    volume variation throughout the day.")

print(f"\n  Feature dimensions:")
print(f"    Node features     : 9-dim  "
      f"(x,y,vx,vy,speed,accel,heading,is_veh,is_ped)")
print(f"    Edge features     : 6-dim  "
      f"(distance,rel_speed,DRAC,heading_diff,rel_vx,rel_vy)")
print(f"    TTC excluded from features — used for labeling only")

In [ ]:
# ── Section 9: Normalization Statistics ───────────────────
print("="*65)
print("9. FEATURE NORMALIZATION STATISTICS")
print("="*65)
print("  (Computed from training set only, applied to val/test)")

node_feat_names = ["x","y","vx","vy","speed","accel","heading"]
edge_feat_names = ["distance","rel_speed","drac",
                   "heading_diff","rel_vx","rel_vy"]

print(f"\n  Node features (7 continuous dims, 2 one-hot excluded):")
print(f"  {'Feature':<14} {'Mean':>10} {'Std':>10}")
print(f"  {'-'*36}")
for i, name in enumerate(node_feat_names):
    print(f"  {name:<14} {node_mean[i]:>10.3f} {node_std[i]:>10.3f}")

print(f"\n  Edge features (6-dim):")
print(f"  {'Feature':<14} {'Mean':>10} {'Std':>10}")
print(f"  {'-'*36}")
for i, name in enumerate(edge_feat_names):
    print(f"  {name:<14} {edge_mean[i]:>10.3f} {edge_std[i]:>10.3f}")

print(f"\n  Note: All features normalized using z-score")
print(f"  (x - mean) / std computed from train set only")
print(f"  Same stats applied to val and test sets")
print(f"  to prevent data leakage")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── ROI corners in pixel space ─────────────────────────────
roi_pixels = np.array([
    [895,  303],   # TL
    [1275, 437],   # TR
    [74,   715],   # BR
    [70,   360],   # BL
    [895,  303],   # close polygon
])

# ══════════════════════════════════════════════════════════
# Figure 1 — All trajectories colored by agent type
# ══════════════════════════════════════════════════════════
# We need pixel coordinates from raw tracking data
# Use df_raw if available, otherwise reconstruct from df
# Check what pixel columns are available
print("Available columns:", df.columns.tolist())

In [ ]:
# ══════════════════════════════════════════════════════════
# Figure 1 — All trajectories in pixel space
# ══════════════════════════════════════════════════════════

# Use busiest video for clarity
busiest = (df.groupby("video_id").size().idxmax())
df_fig  = df[df["video_id"] == busiest].copy()
print(f"Using video: {busiest}")
print(f"Rows: {len(df_fig):,}")
print(f"Tracks: {df_fig['track_id'].nunique():,}")

fig, ax = plt.subplots(figsize=(14, 9))

# Plot outside ROI first (background)
df_out = df_fig[df_fig["inside_roi"] == False]
df_in  = df_fig[df_fig["inside_roi"] == True]

# Outside ROI — by agent type
veh_out = df_out[df_out["agent_type"]=="vehicle"]
ped_out = df_out[df_out["agent_type"]=="pedestrian"]
veh_in  = df_in[df_in["agent_type"]=="vehicle"]
ped_in  = df_in[df_in["agent_type"]=="pedestrian"]

ax.scatter(veh_out["x_anchor_px"], veh_out["y_anchor_px"],
           s=0.1, c="#E74C3C", alpha=0.15, rasterized=True)
ax.scatter(ped_out["x_anchor_px"], ped_out["y_anchor_px"],
           s=0.1, c="#F0B27A", alpha=0.15, rasterized=True)

# Inside ROI — by agent type (more visible)
ax.scatter(veh_in["x_anchor_px"], veh_in["y_anchor_px"],
           s=0.3, c="#2E86C1", alpha=0.5, rasterized=True)
ax.scatter(ped_in["x_anchor_px"], ped_in["y_anchor_px"],
           s=0.5, c="#27AE60", alpha=0.7, rasterized=True)

# ROI boundary
ax.plot(roi_pixels[:,0], roi_pixels[:,1],
        color="darkgreen", linewidth=2.5,
        label="ROI boundary", zorder=5)

# Legend
legend_handles = [
    mpatches.Patch(color="#E74C3C", alpha=0.6,
                   label=f"Vehicle outside ROI ({len(veh_out):,})"),
    mpatches.Patch(color="#2E86C1", alpha=0.8,
                   label=f"Vehicle inside ROI ({len(veh_in):,})"),
    mpatches.Patch(color="#F0B27A", alpha=0.6,
                   label=f"Pedestrian outside ROI ({len(ped_out):,})"),
    mpatches.Patch(color="#27AE60", alpha=0.8,
                   label=f"Pedestrian inside ROI ({len(ped_in):,})"),
    plt.Line2D([0],[0], color="darkgreen", linewidth=2.5,
               label="ROI boundary"),
]
ax.legend(handles=legend_handles, fontsize=9,
          loc="upper right", framealpha=0.9)

ax.set_xlim(0, 1280)
ax.set_ylim(720, 0)   # flip y axis (pixel convention)
ax.set_xlabel("x pixel", fontsize=12)
ax.set_ylabel("y pixel", fontsize=12)
ax.set_title(
    f"Trajectory Map — All Agents\n"
    f"{busiest}\n"
    f"Total tracks: {df_fig['track_id'].nunique():,}  |  "
    f"Inside ROI: {df_in['track_id'].nunique():,}",
    fontsize=12, fontweight="bold"
)
ax.set_facecolor("#f8f8f8")

plt.tight_layout()
plt.savefig(checkpoint_path / "trajectory_map_all.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: trajectory_map_all.png")

# ══════════════════════════════════════════════════════════
# Figure 2 — Conflict vs No-conflict trajectories
# ══════════════════════════════════════════════════════════

# Get conflict track IDs from pair_summary
conflict_track_ids = set(
    pair_summary[pair_summary["pair_conflict_label"]>0]["track_id_i"].tolist() +
    pair_summary[pair_summary["pair_conflict_label"]>0]["track_id_j"].tolist()
)
serious_track_ids = set(
    pair_summary[pair_summary["pair_conflict_label"]==2]["track_id_i"].tolist() +
    pair_summary[pair_summary["pair_conflict_label"]==2]["track_id_j"].tolist()
)

print(f"\nConflict tracks    : {len(conflict_track_ids):,}")
print(f"Serious tracks     : {len(serious_track_ids):,}")

# Filter to busiest video + inside ROI only
df_roi = df_fig[df_fig["inside_roi"]==True].copy()
df_roi["conflict_type"] = "no conflict"
df_roi.loc[
    df_roi["track_id"].isin(conflict_track_ids),
    "conflict_type"
] = "general conflict"
df_roi.loc[
    df_roi["track_id"].isin(serious_track_ids),
    "conflict_type"
] = "serious conflict"

print(f"\nInside ROI rows by conflict type:")
print(df_roi["conflict_type"].value_counts())

fig, ax = plt.subplots(figsize=(14, 9))

# Plot by conflict type
colors = {
    "no conflict"      : ("#2E86C1", 0.25, 0.2),
    "general conflict" : ("#F39C12", 0.7,  0.8),
    "serious conflict" : ("#E74C3C", 0.9,  1.5),
}
for ctype, (color, alpha, size) in colors.items():
    sub = df_roi[df_roi["conflict_type"]==ctype]
    ax.scatter(sub["x_anchor_px"], sub["y_anchor_px"],
               s=size, c=color, alpha=alpha,
               rasterized=True,
               label=f"{ctype.capitalize()} ({sub['track_id'].nunique():,} tracks)")

# ROI boundary
ax.plot(roi_pixels[:,0], roi_pixels[:,1],
        color="darkgreen", linewidth=2.5,
        label="ROI boundary", zorder=5)

ax.legend(fontsize=10, loc="upper right", framealpha=0.9)
ax.set_xlim(0, 1280)
ax.set_ylim(720, 0)
ax.set_xlabel("x pixel", fontsize=12)
ax.set_ylabel("y pixel", fontsize=12)
ax.set_title(
    f"Conflict vs No-Conflict Trajectory Map (Inside ROI)\n"
    f"{busiest}\n"
    f"General conflict: orange  |  Serious conflict: red  |  "
    f"Safe: blue",
    fontsize=12, fontweight="bold"
)
ax.set_facecolor("#f8f8f8")

plt.tight_layout()
plt.savefig(checkpoint_path / "trajectory_map_conflict.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: trajectory_map_conflict.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Get conflict track IDs from pair_summary ───────────────
conflict_track_ids = set(
    pair_summary[pair_summary["pair_conflict_label"]>0]["track_id_i"].tolist() +
    pair_summary[pair_summary["pair_conflict_label"]>0]["track_id_j"].tolist()
)
serious_track_ids = set(
    pair_summary[pair_summary["pair_conflict_label"]==2]["track_id_i"].tolist() +
    pair_summary[pair_summary["pair_conflict_label"]==2]["track_id_j"].tolist()
)

# Label df
df["conflict_type"] = "no conflict"
df.loc[df["track_id"].isin(conflict_track_ids),
       "conflict_type"] = "general conflict"
df.loc[df["track_id"].isin(serious_track_ids),
       "conflict_type"] = "serious conflict"

print(f"No conflict      : {(df['conflict_type']=='no conflict').sum():,}")
print(f"General conflict : {(df['conflict_type']=='general conflict').sum():,}")
print(f"Serious conflict : {(df['conflict_type']=='serious conflict').sum():,}")

# ══════════════════════════════════════════════════════════
# Figure 1 — All vs Conflict interactions
# ══════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 8))

no_conf = df[df["conflict_type"]=="no conflict"]
conf    = df[df["conflict_type"]!="no conflict"]

ax.scatter(no_conf["x_local_m"], no_conf["y_local_m"],
           s=0.1, c="#2E86C1", alpha=0.15,
           rasterized=True, label=f"No conflict ({no_conf['track_id'].nunique():,} tracks)")
ax.scatter(conf["x_local_m"], conf["y_local_m"],
           s=0.3, c="#E74C3C", alpha=0.4,
           rasterized=True, label=f"Conflict ({conf['track_id'].nunique():,} tracks)")

ax.set_xlabel("x_local_m", fontsize=12)
ax.set_ylabel("y_local_m", fontsize=12)
ax.set_title("All vs Conflict Interactions\n(Final Dataset — All Videos)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper right", framealpha=0.9,
          markerscale=15)
ax.set_facecolor("#f8f8f8")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(checkpoint_path / "fig1_all_vs_conflict.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig1_all_vs_conflict.png")

# ══════════════════════════════════════════════════════════
# Figure 2 — Conflict by interaction type
# ══════════════════════════════════════════════════════════
# Get interaction type for each row from pairs
vv_tracks = set(
    pairs[pairs["interaction_type"]=="vv"]["track_id_i"].tolist() +
    pairs[pairs["interaction_type"]=="vv"]["track_id_j"].tolist()
)
vp_tracks = set(
    pairs[pairs["interaction_type"]=="vp"]["track_id_i"].tolist() +
    pairs[pairs["interaction_type"]=="vp"]["track_id_j"].tolist()
)
pp_tracks = set(
    pairs[pairs["interaction_type"]=="pp"]["track_id_i"].tolist() +
    pairs[pairs["interaction_type"]=="pp"]["track_id_j"].tolist()
)

df["itype"] = "other"
df.loc[df["track_id"].isin(vv_tracks), "itype"] = "vv"
df.loc[df["track_id"].isin(vp_tracks), "itype"] = "vp"
df.loc[df["track_id"].isin(pp_tracks), "itype"] = "pp"

fig, ax = plt.subplots(figsize=(10, 8))

itype_styles = {
    "vv": ("#2E86C1", 0.25, 0.2, "VV — Vehicle↔Vehicle"),
    "vp": ("#E74C3C", 0.60, 0.5, "VP — Vehicle↔Pedestrian"),
    "pp": ("#27AE60", 0.90, 2.0, "PP — Pedestrian↔Pedestrian"),
}
for itype, (color, alpha, size, label) in itype_styles.items():
    sub = df[df["itype"]==itype]
    ax.scatter(sub["x_local_m"], sub["y_local_m"],
               s=size, c=color, alpha=alpha,
               rasterized=True,
               label=f"{label} ({sub['track_id'].nunique():,} tracks)")

ax.set_xlabel("x_local_m", fontsize=12)
ax.set_ylabel("y_local_m", fontsize=12)
ax.set_title("Trajectory Distribution by Interaction Type\n"
             "(Final Dataset — All Videos)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper right", framealpha=0.9,
          markerscale=8)
ax.set_facecolor("#f8f8f8")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(checkpoint_path / "fig2_interaction_types.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig2_interaction_types.png")

# ══════════════════════════════════════════════════════════
# Figure 3 — Serious conflict hotspot density map
# ══════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 8))

# Background — all trajectories
ax.scatter(df["x_local_m"], df["y_local_m"],
           s=0.05, c="#CCCCCC", alpha=0.1,
           rasterized=True)

# Serious conflict density
serious = df[df["conflict_type"]=="serious conflict"]
h = ax.hexbin(
    serious["x_local_m"], serious["y_local_m"],
    gridsize=40, cmap="YlOrRd",
    mincnt=1, alpha=0.85,
    linewidths=0.2
)
cb = plt.colorbar(h, ax=ax)
cb.set_label("Serious conflict density", fontsize=11)

ax.set_xlabel("x_local_m", fontsize=12)
ax.set_ylabel("y_local_m", fontsize=12)
ax.set_title("Serious Conflict Hotspot Map\n"
             "(Final Dataset — All Videos)",
             fontsize=13, fontweight="bold")
ax.set_facecolor("#f0f0f0")
ax.grid(alpha=0.2)

# Add intersection center marker
ax.axhline(y=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.axvline(x=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.scatter([0], [0], c="black", s=60, zorder=5,
           marker="+", linewidths=2,
           label="Intersection center")
ax.legend(fontsize=10, loc="upper right")

plt.tight_layout()
plt.savefig(checkpoint_path / "fig3_serious_conflict_hotspot.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig3_serious_conflict_hotspot.png")

In [ ]:
# Check scene graph structure
sample_seq = sequences[0]
sample_graph = sample_seq[0]

print("Graph keys:", sample_graph.keys())
print("\nFirst node keys:", sample_graph["nodes"][0].keys())
print("First node:", sample_graph["nodes"][0])
print("\nFirst edge keys:", sample_graph["edges"][0].keys())
print("First edge:", sample_graph["edges"][0])

In [ ]:
import matplotlib.cm as cm
cm.get_cmap = plt.colormaps.get_cmap  # patch for newer matplotlib

In [ ]:
# ══════════════════════════════════════════════════════════
# Find good conflict and safe frames
# ══════════════════════════════════════════════════════════
conflict_frame = None
safe_frame     = None

for seq, label in zip(sequences, seq_labels):
    for g in seq:
        if (conflict_frame is None and
            g["label"] == 1 and
            len(g["nodes"]) >= 3 and
            len(g["edges"]) >= 2):
            dracs = [e["drac"] for e in g["edges"]]
            if dracs and max(dracs) > 2.0:
                conflict_frame = g

        if (safe_frame is None and
            g["label"] == 0 and
            len(g["nodes"]) >= 3 and
            len(g["edges"]) >= 2):
            safe_frame = g

    if conflict_frame is not None and safe_frame is not None:
        break

print(f"Conflict frame: {len(conflict_frame['nodes'])} nodes, "
      f"{len(conflict_frame['edges'])} edges  "
      f"max_drac={max(e['drac'] for e in conflict_frame['edges']):.2f}")
print(f"Safe frame    : {len(safe_frame['nodes'])} nodes, "
      f"{len(safe_frame['edges'])} edges  "
      f"max_drac={max(e['drac'] for e in safe_frame['edges']):.2f}")


# ══════════════════════════════════════════════════════════
# Draw function with correct keys
# ══════════════════════════════════════════════════════════
def draw_scene_graph(ax, graph, title):
    nodes = graph["nodes"]
    edges = graph["edges"]

    # Build position lookup by track_id
    positions = {}
    for node in nodes:
        positions[node["track_id"]] = node

    # Max DRAC for color scaling
    dracs    = [e["drac"] for e in edges]
    max_drac = max(dracs) if dracs else 1.0
    cmap     = cm.get_cmap("YlOrRd")

    # ── Draw edges ─────────────────────────────────────────
    for edge in edges:
        i_id  = edge["i"]
        j_id  = edge["j"]
        if i_id not in positions or j_id not in positions:
            continue

        ni = positions[i_id]
        nj = positions[j_id]
        xi, yi = ni["x"], ni["y"]
        xj, yj = nj["x"], nj["y"]

        drac  = edge["drac"]
        dist  = edge["distance"]
        itype = edge["interaction_type"]
        color = cmap(drac / (max_drac + 1e-6))
        lw    = 1.0 + 3.0 * (drac / (max_drac + 1e-6))

        ax.plot([xi, xj], [yi, yj],
                color=color, linewidth=lw,
                alpha=0.85, zorder=1)

        # Edge annotation
        mx = (xi + xj) / 2
        my = (yi + yj) / 2
        ax.text(mx, my,
                f"{itype.upper()}\nd={dist:.1f}m\nDRAC={drac:.2f}",
                fontsize=6.5, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.2",
                          facecolor="white", alpha=0.75,
                          edgecolor="gray", linewidth=0.5),
                zorder=3)

    # ── Draw nodes ─────────────────────────────────────────
    for node in nodes:
        tid   = node["track_id"]
        x     = node["x"]
        y     = node["y"]
        vx    = node["vx"]
        vy    = node["vy"]
        speed = node["speed"]
        atype = node["agent_type"]
        cls   = node["class"]

        color  = "#2E86C1" if atype == "vehicle"    else "#27AE60"
        marker = "s"       if atype == "vehicle"    else "o"
        size   = 250       if atype == "vehicle"    else 180

        ax.scatter(x, y, s=size, c=color,
                   marker=marker, zorder=4,
                   edgecolors="white", linewidths=1.5)

        # Velocity arrow
        spd = np.sqrt(vx**2 + vy**2)
        if spd > 0.3:
            scale = min(1.5, spd * 0.2)
            ax.annotate("",
                xy=(x + vx/spd*scale, y + vy/spd*scale),
                xytext=(x, y),
                arrowprops=dict(
                    arrowstyle="->",
                    color=color,
                    lw=2.0
                ),
                zorder=5
            )

        # Node label
        ax.text(x, y + 0.9,
                f"{cls}\n{speed:.1f}m/s",
                fontsize=7, ha="center", va="bottom",
                fontweight="bold", color=color, zorder=6)

    # Colorbar
    sm = cm.ScalarMappable(
        cmap=cmap,
        norm=plt.Normalize(0, max_drac)
    )
    sm.set_array([])
    cb = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cb.set_label("DRAC (m/s²)", fontsize=9)

    # Legend
    legend_handles = [
        mpatches.Patch(color="#2E86C1", label="Vehicle"),
        mpatches.Patch(color="#27AE60", label="Pedestrian"),
        plt.Line2D([0],[0], color="#E74C3C", linewidth=2.5,
                   label="High DRAC (conflict risk)"),
        plt.Line2D([0],[0], color="#FDEBD0", linewidth=1.5,
                   label="Low DRAC (safe)"),
    ]
    ax.legend(handles=legend_handles, fontsize=8,
              loc="lower right", framealpha=0.9)

    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_xlabel("x_local (m)", fontsize=10)
    ax.set_ylabel("y_local (m)", fontsize=10)
    ax.set_facecolor("#f5f5f5")
    ax.grid(alpha=0.3, linewidth=0.5)


# ══════════════════════════════════════════════════════════
# Plot side by side
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

draw_scene_graph(
    axes[0], conflict_frame,
    f"Conflict Scene Graph\n"
    f"({len(conflict_frame['nodes'])} agents  |  "
    f"{len(conflict_frame['edges'])} interactions  |  "
    f"max DRAC={max(e['drac'] for e in conflict_frame['edges']):.2f} m/s²)"
)
draw_scene_graph(
    axes[1], safe_frame,
    f"Safe Scene Graph\n"
    f"({len(safe_frame['nodes'])} agents  |  "
    f"{len(safe_frame['edges'])} interactions  |  "
    f"max DRAC={max(e['drac'] for e in safe_frame['edges']):.2f} m/s²)"
)

plt.suptitle(
    "Heterogeneous Scene Graph Examples — HERMES Framework\n"
    "Nodes: agents (■=vehicle, ●=pedestrian)  |  "
    "Edges: SSM features colored by DRAC  |  "
    "Arrows: velocity direction",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "scene_graph_examples.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: scene_graph_examples.png")

In [ ]:
conflict_frame = None
safe_frame     = None

for seq, label in zip(sequences, seq_labels):
    for g in seq:
        # Conflict frame: must have actual conflict edge
        # with high DRAC and close distance
        if conflict_frame is None and g["label"] == 1:
            conflict_edges = [
                e for e in g["edges"]
                if e["conflict_label"] == 1
                and e["drac"] > 3.0
                and e["distance"] < 8.0
            ]
            if (len(conflict_edges) >= 1 and
                len(g["nodes"]) >= 2 and
                len(g["nodes"]) <= 5):
                conflict_frame = g

        # Safe frame: NO conflict edges, low DRAC
        if safe_frame is None and g["label"] == 0:
            max_drac = max((e["drac"] for e in g["edges"]),
                           default=0)
            has_conflict_edge = any(
                e["conflict_label"] == 1
                for e in g["edges"]
            )
            if (not has_conflict_edge and
                max_drac < 1.5 and
                len(g["nodes"]) >= 2 and
                len(g["nodes"]) <= 5 and
                len(g["edges"]) >= 1):
                safe_frame = g

    if conflict_frame is not None and safe_frame is not None:
        break

print(f"Conflict frame:")
print(f"  nodes={len(conflict_frame['nodes'])}  "
      f"edges={len(conflict_frame['edges'])}")
for e in conflict_frame["edges"]:
    print(f"  {e['interaction_type']}  "
          f"dist={e['distance']:.2f}m  "
          f"drac={e['drac']:.2f}  "
          f"conflict={e['conflict_label']}")

print(f"\nSafe frame:")
print(f"  nodes={len(safe_frame['nodes'])}  "
      f"edges={len(safe_frame['edges'])}")
for e in safe_frame["edges"]:
    print(f"  {e['interaction_type']}  "
          f"dist={e['distance']:.2f}m  "
          f"drac={e['drac']:.2f}  "
          f"conflict={e['conflict_label']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

draw_scene_graph(
    axes[0], conflict_frame,
    f"Conflict Scene Graph\n"
    f"({len(conflict_frame['nodes'])} agents  |  "
    f"{len(conflict_frame['edges'])} interaction  |  "
    f"dist={conflict_frame['edges'][0]['distance']:.2f}m  |  "
    f"DRAC={conflict_frame['edges'][0]['drac']:.2f} m/s²)"
)
draw_scene_graph(
    axes[1], safe_frame,
    f"Safe Scene Graph\n"
    f"({len(safe_frame['nodes'])} agents  |  "
    f"{len(safe_frame['edges'])} interaction  |  "
    f"dist={safe_frame['edges'][0]['distance']:.2f}m  |  "
    f"DRAC={safe_frame['edges'][0]['drac']:.2f} m/s²)"
)

plt.suptitle(
    "Heterogeneous Scene Graph Examples — HERMES Framework\n"
    "Nodes: agents (■=vehicle, ●=pedestrian)  |  "
    "Edges: SSM features colored by DRAC  |  "
    "Arrows: velocity direction",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "scene_graph_examples.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: scene_graph_examples.png")

In [ ]:
# Find better frames with both vehicle and pedestrian
conflict_frame = None
safe_frame     = None

for seq, label in zip(sequences, seq_labels):
    for g in seq:
        # Conflict: must have VP conflict edge
        # with high DRAC and close distance
        if conflict_frame is None and g["label"] == 1:
            vp_conflict_edges = [
                e for e in g["edges"]
                if e["interaction_type"] == "vp"
                and e["conflict_label"] == 1
                and e["drac"] > 1.5
                and e["distance"] < 5.0
            ]
            has_veh = any(n["agent_type"]=="vehicle"
                         for n in g["nodes"])
            has_ped = any(n["agent_type"]=="pedestrian"
                         for n in g["nodes"])
            if (len(vp_conflict_edges) >= 1
                    and has_veh and has_ped
                    and len(g["nodes"]) >= 2
                    and len(g["nodes"]) <= 6):
                conflict_frame = g

        # Safe: has both veh and ped but NO conflict edges
        # and low DRAC
        if safe_frame is None and g["label"] == 0:
            has_veh = any(n["agent_type"]=="vehicle"
                         for n in g["nodes"])
            has_ped = any(n["agent_type"]=="pedestrian"
                         for n in g["nodes"])
            max_drac = max(
                (e["drac"] for e in g["edges"]), default=0
            )
            has_conflict_edge = any(
                e["conflict_label"]==1 for e in g["edges"]
            )
            if (has_veh and has_ped
                    and not has_conflict_edge
                    and max_drac < 1.0
                    and len(g["nodes"]) >= 2
                    and len(g["nodes"]) <= 6
                    and len(g["edges"]) >= 1):
                safe_frame = g

    if conflict_frame is not None and safe_frame is not None:
        break

print("Conflict frame:")
print(f"  nodes={len(conflict_frame['nodes'])}  "
      f"edges={len(conflict_frame['edges'])}")
for n in conflict_frame["nodes"]:
    print(f"  NODE  {n['agent_type']:<12} "
          f"speed={n['speed']:.2f} m/s")
for e in conflict_frame["edges"]:
    print(f"  EDGE  {e['interaction_type']}  "
          f"dist={e['distance']:.2f}m  "
          f"drac={e['drac']:.2f}  "
          f"conflict={e['conflict_label']}")

print("\nSafe frame:")
print(f"  nodes={len(safe_frame['nodes'])}  "
      f"edges={len(safe_frame['edges'])}")
for n in safe_frame["nodes"]:
    print(f"  NODE  {n['agent_type']:<12} "
          f"speed={n['speed']:.2f} m/s")
for e in safe_frame["edges"]:
    print(f"  EDGE  {e['interaction_type']}  "
          f"dist={e['distance']:.2f}m  "
          f"drac={e['drac']:.2f}  "
          f"conflict={e['conflict_label']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

draw_scene_graph(
    axes[0], conflict_frame,
    f"Conflict Scene Graph\n"
    f"2 vehicles + 1 pedestrian  |  "
    f"VP conflict: dist=2.03m  DRAC=1.51 m/s²"
)
draw_scene_graph(
    axes[1], safe_frame,
    f"Safe Scene Graph\n"
    f"2 vehicles + 1 pedestrian  |  "
    f"max DRAC=0.40 m/s²  all distances > 4m"
)

plt.suptitle(
    "Heterogeneous Scene Graph Examples — HERMES Framework\n"
    "■ = Vehicle  ● = Pedestrian  |  "
    "Edge color = DRAC intensity  |  "
    "Arrows = velocity direction  |  "
    "Edge label = interaction type, distance, DRAC",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "scene_graph_examples.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: scene_graph_examples.png")

In [ ]:
def draw_scene_graph_v2(ax, graph, title, subtitle):
    nodes = graph["nodes"]
    edges = graph["edges"]

    # Position lookup
    positions = {}
    for node in nodes:
        positions[node["track_id"]] = node

    # DRAC range for color scaling
    dracs    = [e["drac"] for e in edges]
    max_drac = max(dracs) if dracs else 1.0
    min_drac = min(dracs) if dracs else 0.0
    cmap     = cm.get_cmap("RdYlGn_r")  # green=safe red=conflict

    # ── Draw edges ─────────────────────────────────────────
    for edge in edges:
        ni = positions.get(edge["i"])
        nj = positions.get(edge["j"])
        if ni is None or nj is None:
            continue

        xi, yi = ni["x"], ni["y"]
        xj, yj = nj["x"], nj["y"]
        drac    = edge["drac"]
        dist    = edge["distance"]
        itype   = edge["interaction_type"].upper()
        clabel  = edge["conflict_label"]

        norm_drac = (drac - min_drac) / (max_drac - min_drac + 1e-6)
        color     = cmap(norm_drac)
        lw        = 1.5 + 3.5 * norm_drac

        # Dashed if conflict edge
        ls = "-" if clabel == 0 else "--"

        ax.plot([xi, xj], [yi, yj],
                color=color, linewidth=lw,
                linestyle=ls, alpha=0.9, zorder=1,
                solid_capstyle="round")

        # Edge label — midpoint
        mx = (xi + xj) / 2
        my = (yi + yj) / 2
        conf_str = " ⚠" if clabel == 1 else ""
        ax.annotate(
            f"{itype}{conf_str}\n{dist:.1f}m | {drac:.2f} m/s²",
            xy=(mx, my),
            fontsize=7.5, ha="center", va="center",
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white", alpha=0.85,
                edgecolor="gray" if clabel==0 else "red",
                linewidth=0.8 if clabel==0 else 1.5
            ),
            zorder=4
        )

    # ── Draw nodes ─────────────────────────────────────────
    for node in nodes:
        x     = node["x"]
        y     = node["y"]
        vx    = node["vx"]
        vy    = node["vy"]
        speed = node["speed"]
        atype = node["agent_type"]
        cls   = node["class"]

        if atype == "vehicle":
            color  = "#1A5276"
            marker = "s"
            size   = 300
            label  = f"{cls}\n{speed:.1f} m/s"
        else:
            color  = "#1E8449"
            marker = "o"
            size   = 200
            label  = f"{cls}\n{speed:.1f} m/s"

        ax.scatter(x, y, s=size, c=color,
                   marker=marker, zorder=5,
                   edgecolors="white", linewidths=2.0)

        # Velocity arrow
        spd = np.sqrt(vx**2 + vy**2)
        if spd > 0.2:
            scale = min(1.2, spd * 0.15)
            ax.annotate("",
                xy=(x + vx/spd*scale, y + vy/spd*scale),
                xytext=(x, y),
                arrowprops=dict(
                    arrowstyle="->,head_width=0.3,head_length=0.2",
                    color=color, lw=2.0
                ),
                zorder=6
            )

        # Node label — offset to avoid overlap
        offset_x = 0.3
        offset_y = 0.5
        ax.text(x + offset_x, y + offset_y, label,
                fontsize=8, ha="left", va="bottom",
                fontweight="bold", color=color,
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    facecolor="white", alpha=0.7,
                    edgecolor=color, linewidth=0.8
                ),
                zorder=7)

    # Colorbar
    sm = cm.ScalarMappable(
        cmap=cmap,
        norm=plt.Normalize(min_drac, max_drac)
    )
    sm.set_array([])
    cb = plt.colorbar(sm, ax=ax, shrink=0.55,
                      pad=0.03, aspect=20)
    cb.set_label("DRAC (m/s²)", fontsize=9)
    cb.ax.tick_params(labelsize=8)

    # Legend
    legend_handles = [
        mpatches.Patch(color="#1A5276",
                       label="Vehicle"),
        mpatches.Patch(color="#1E8449",
                       label="Pedestrian"),
        plt.Line2D([0],[0], color="#E74C3C",
                   linewidth=2.5, linestyle="--",
                   label="Conflict edge (⚠)"),
        plt.Line2D([0],[0], color="#27AE60",
                   linewidth=1.5, linestyle="-",
                   label="Safe edge"),
    ]
    ax.legend(handles=legend_handles, fontsize=8.5,
              loc="lower left", framealpha=0.9,
              edgecolor="gray")

    ax.set_title(f"{title}\n{subtitle}",
                 fontweight="bold", fontsize=11, pad=10)
    ax.set_xlabel("x_local (m)", fontsize=10)
    ax.set_ylabel("y_local (m)", fontsize=10)
    ax.set_facecolor("#f9f9f9")
    ax.grid(alpha=0.25, linewidth=0.5, linestyle="--")

    # Tight axis limits with padding
    all_x = [n["x"] for n in nodes]
    all_y = [n["y"] for n in nodes]
    pad   = 2.5
    ax.set_xlim(min(all_x) - pad, max(all_x) + pad)
    ax.set_ylim(min(all_y) - pad, max(all_y) + pad)
    ax.set_aspect("equal")


# ── Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

draw_scene_graph_v2(
    axes[0], conflict_frame,
    "Conflict Scene Graph",
    "2 vehicles + 1 pedestrian  |  "
    "VP edge: dist=2.03m  DRAC=1.51 m/s²  ⚠"
)
draw_scene_graph_v2(
    axes[1], safe_frame,
    "Safe Scene Graph",
    "2 vehicles + 1 pedestrian  |  "
    "max DRAC=0.40 m/s²  |  all distances > 4m"
)

plt.suptitle(
    "Heterogeneous Scene Graph Visualization — HERMES\n"
    "■ Vehicle  ●  Pedestrian  |  "
    "Edge color: green=safe → red=conflict risk  |  "
    "Dashed edge = conflict  |  Arrow = velocity",
    fontsize=12, fontweight="bold", y=1.02
)

plt.tight_layout()
plt.savefig(checkpoint_path / "scene_graph_examples_v2.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: scene_graph_examples_v2.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# Figure 2 — TTC and DRAC evolution across T=8 frames
# Conflict vs Safe sequences
# ══════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import numpy as np

# Extract TTC and DRAC per frame position for conflict
# and safe sequences
# We look at the minimum TTC and maximum DRAC per frame
# (most critical edge in each frame)

print("Extracting SSM evolution across T=8 frames...")

conflict_ttc_by_frame  = [[] for _ in range(SEQ_LEN)]
conflict_drac_by_frame = [[] for _ in range(SEQ_LEN)]
safe_ttc_by_frame      = [[] for _ in range(SEQ_LEN)]
safe_drac_by_frame     = [[] for _ in range(SEQ_LEN)]

# Sample for speed — use first 5000 of each type
conflict_count = 0
safe_count     = 0
MAX_SAMPLES    = 5000

for seq, label in zip(sequences, seq_labels):
    if label == 1 and conflict_count < MAX_SAMPLES:
        for t, g in enumerate(seq):
            edges = [e for e in g["edges"]
                     if e["interaction_type"] != "pp"]
            if edges:
                # Min distance edge (most critical)
                critical = min(edges, key=lambda e: e["distance"])
                conflict_drac_by_frame[t].append(critical["drac"])
                # TTC: use distance as proxy for risk
                conflict_ttc_by_frame[t].append(critical["distance"])
        conflict_count += 1

    elif label == 0 and safe_count < MAX_SAMPLES:
        for t, g in enumerate(seq):
            edges = [e for e in g["edges"]
                     if e["interaction_type"] != "pp"]
            if edges:
                critical = min(edges, key=lambda e: e["distance"])
                safe_drac_by_frame[t].append(critical["drac"])
                safe_ttc_by_frame[t].append(critical["distance"])
        safe_count += 1

    if conflict_count >= MAX_SAMPLES and safe_count >= MAX_SAMPLES:
        break

print(f"Conflict sequences sampled : {conflict_count:,}")
print(f"Safe sequences sampled     : {safe_count:,}")

# Compute mean and std per frame
frames = np.arange(1, SEQ_LEN + 1)
time_s = frames / 30  # convert to seconds

c_drac_mean = np.array([np.mean(x) for x in conflict_drac_by_frame])
c_drac_std  = np.array([np.std(x)  for x in conflict_drac_by_frame])
s_drac_mean = np.array([np.mean(x) for x in safe_drac_by_frame])
s_drac_std  = np.array([np.std(x)  for x in safe_drac_by_frame])

c_dist_mean = np.array([np.mean(x) for x in conflict_ttc_by_frame])
c_dist_std  = np.array([np.std(x)  for x in conflict_ttc_by_frame])
s_dist_mean = np.array([np.mean(x) for x in safe_ttc_by_frame])
s_dist_std  = np.array([np.std(x)  for x in safe_ttc_by_frame])

# ── Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

time_labels = [f"t{i}\n({i/30:.2f}s)" for i in frames]

# ── Panel 1: DRAC evolution ────────────────────────────────
ax = axes[0]
ax.plot(frames, c_drac_mean, color="#E74C3C", linewidth=2.5,
        marker="o", markersize=6, label="Conflict sequences")
ax.fill_between(frames,
                c_drac_mean - c_drac_std,
                c_drac_mean + c_drac_std,
                color="#E74C3C", alpha=0.15)

ax.plot(frames, s_drac_mean, color="#2E86C1", linewidth=2.5,
        marker="s", markersize=6, label="Safe sequences",
        linestyle="--")
ax.fill_between(frames,
                s_drac_mean - s_drac_std,
                s_drac_mean + s_drac_std,
                color="#2E86C1", alpha=0.15)

ax.set_xticks(frames)
ax.set_xticklabels(time_labels, fontsize=8)
ax.set_xlabel("Frame position in sequence", fontsize=12)
ax.set_ylabel("DRAC — most critical edge (m/s²)", fontsize=12)
ax.set_title("DRAC Evolution Across T=8 Frames\n"
             "(Mean ± Std, most critical non-PP edge per frame)",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_facecolor("#f9f9f9")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Annotation
ax.annotate("Conflict sequences\nshow higher DRAC\nthroughout",
            xy=(4, c_drac_mean[3]),
            xytext=(5.5, c_drac_mean[3] + 0.3),
            fontsize=9, color="#E74C3C",
            arrowprops=dict(arrowstyle="->",
                           color="#E74C3C",
                           lw=1.2))

# ── Panel 2: Distance evolution ────────────────────────────
ax = axes[1]
ax.plot(frames, c_dist_mean, color="#E74C3C", linewidth=2.5,
        marker="o", markersize=6, label="Conflict sequences")
ax.fill_between(frames,
                c_dist_mean - c_dist_std,
                c_dist_mean + c_dist_std,
                color="#E74C3C", alpha=0.15)

ax.plot(frames, s_dist_mean, color="#2E86C1", linewidth=2.5,
        marker="s", markersize=6, label="Safe sequences",
        linestyle="--")
ax.fill_between(frames,
                s_dist_mean - s_dist_std,
                s_dist_mean + s_dist_std,
                color="#2E86C1", alpha=0.15)

# Add conflict threshold lines
ax.axhline(y=5.0, color="#F39C12", linestyle=":",
           linewidth=1.5, alpha=0.8,
           label="General conflict threshold (5m VV)")
ax.axhline(y=3.0, color="#E74C3C", linestyle=":",
           linewidth=1.5, alpha=0.8,
           label="Serious conflict threshold (3m VV)")

ax.set_xticks(frames)
ax.set_xticklabels(time_labels, fontsize=8)
ax.set_xlabel("Frame position in sequence", fontsize=12)
ax.set_ylabel("Min inter-agent distance (m)", fontsize=12)
ax.set_title("Inter-Agent Distance Evolution Across T=8 Frames\n"
             "(Mean ± Std, closest non-PP agent pair per frame)",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=9, framealpha=0.9, loc="upper right")
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_facecolor("#f9f9f9")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle(
    "Temporal SSM Dynamics — Conflict vs Safe Sequences\n"
    "Justification for LSTM temporal encoder in HERMES "
    "(T=8 frames, 0.267s window)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "ssm_temporal_evolution.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: ssm_temporal_evolution.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

time_labels = [f"t{i}\n({i/30:.2f}s)" for i in frames]

# ── Panel 1: DRAC — clip std to non-negative ───────────────
ax = axes[0]
c_drac_lo = np.maximum(0, c_drac_mean - c_drac_std)
s_drac_lo = np.maximum(0, s_drac_mean - s_drac_std)

ax.plot(frames, c_drac_mean, color="#E74C3C", linewidth=2.5,
        marker="o", markersize=7,
        label=f"Conflict (n={conflict_count:,})")
ax.fill_between(frames, c_drac_lo,
                c_drac_mean + c_drac_std,
                color="#E74C3C", alpha=0.15)

ax.plot(frames, s_drac_mean, color="#2E86C1", linewidth=2.5,
        marker="s", markersize=7, linestyle="--",
        label=f"Safe (n={safe_count:,})")
ax.fill_between(frames, s_drac_lo,
                s_drac_mean + s_drac_std,
                color="#2E86C1", alpha=0.15)

# Difference annotation
delta = c_drac_mean.mean() - s_drac_mean.mean()
ax.annotate(
    f"Conflict DRAC consistently\n"
    f"{delta:.2f} m/s² higher than safe\nacross all 8 frames",
    xy=(4, c_drac_mean[3]),
    xytext=(5.5, c_drac_mean[3] + 0.8),
    fontsize=9, color="#E74C3C", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#E74C3C", lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3",
              facecolor="white", alpha=0.8,
              edgecolor="#E74C3C", linewidth=0.8)
)

ax.set_xticks(frames)
ax.set_xticklabels(time_labels, fontsize=9)
ax.set_ylim(bottom=0)
ax.set_xlabel("Frame position in sequence", fontsize=12)
ax.set_ylabel("DRAC — most critical edge (m/s²)", fontsize=12)
ax.set_title("DRAC Evolution Across T=8 Frames\n"
             "Mean ± Std (most critical non-PP edge per frame)",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_facecolor("#f9f9f9")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ── Panel 2: Distance — clip std ──────────────────────────
ax = axes[1]
c_dist_lo = np.maximum(0, c_dist_mean - c_dist_std)
s_dist_lo = np.maximum(0, s_dist_mean - s_dist_std)

ax.plot(frames, c_dist_mean, color="#E74C3C", linewidth=2.5,
        marker="o", markersize=7,
        label=f"Conflict (n={conflict_count:,})")
ax.fill_between(frames, c_dist_lo,
                c_dist_mean + c_dist_std,
                color="#E74C3C", alpha=0.15)

ax.plot(frames, s_dist_mean, color="#2E86C1", linewidth=2.5,
        marker="s", markersize=7, linestyle="--",
        label=f"Safe (n={safe_count:,})")
ax.fill_between(frames, s_dist_lo,
                s_dist_mean + s_dist_std,
                color="#2E86C1", alpha=0.15)

ax.axhline(y=5.0, color="#F39C12", linestyle=":",
           linewidth=1.8, alpha=0.9,
           label="General conflict threshold VV (5m)")
ax.axhline(y=3.0, color="#E74C3C", linestyle=":",
           linewidth=1.8, alpha=0.6,
           label="Serious conflict threshold VV (3m)")

# Gap annotation
gap = s_dist_mean.mean() - c_dist_mean.mean()
ax.annotate(
    f"Safe sequences\n{gap:.1f}m farther apart\non average",
    xy=(4, s_dist_mean[3]),
    xytext=(5.5, s_dist_mean[3] + 1.5),
    fontsize=9, color="#2E86C1", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#2E86C1", lw=1.2),
    bbox=dict(boxstyle="round,pad=0.3",
              facecolor="white", alpha=0.8,
              edgecolor="#2E86C1", linewidth=0.8)
)

ax.set_xticks(frames)
ax.set_xticklabels(time_labels, fontsize=9)
ax.set_ylim(bottom=0)
ax.set_xlabel("Frame position in sequence", fontsize=12)
ax.set_ylabel("Min inter-agent distance (m)", fontsize=12)
ax.set_title("Inter-Agent Distance Evolution Across T=8 Frames\n"
             "Mean ± Std (closest non-PP agent pair per frame)",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=9, framealpha=0.9, loc="upper right")
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_facecolor("#f9f9f9")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle(
    "Temporal SSM Dynamics — Conflict vs Safe Sequences\n"
    "The persistent separation across all 8 frames justifies "
    "the LSTM temporal encoder in HERMES (T=8, 0.267s window)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "ssm_temporal_evolution_v2.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: ssm_temporal_evolution_v2.png")

# Print key numbers
print(f"\nKey findings:")
print(f"  Conflict DRAC mean across frames : "
      f"{c_drac_mean.mean():.3f} m/s²")
print(f"  Safe DRAC mean across frames     : "
      f"{s_drac_mean.mean():.3f} m/s²")
print(f"  Difference                       : "
      f"{c_drac_mean.mean()-s_drac_mean.mean():.3f} m/s²")
print(f"\n  Conflict distance mean           : "
      f"{c_dist_mean.mean():.2f} m")
print(f"  Safe distance mean               : "
      f"{s_dist_mean.mean():.2f} m")
print(f"  Difference                       : "
      f"{s_dist_mean.mean()-c_dist_mean.mean():.2f} m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

time_labels = [f"t{i}\n({i/30:.2f}s)" for i in frames]

# ── Panel 1: Distance (the real signal) ───────────────────
ax = axes[0]
c_dist_lo = np.maximum(0, c_dist_mean - c_dist_std*0.5)
s_dist_lo = np.maximum(0, s_dist_mean - s_dist_std*0.5)

ax.plot(frames, c_dist_mean, color="#E74C3C", linewidth=2.5,
        marker="o", markersize=7,
        label=f"Conflict (n={conflict_count:,})")
ax.fill_between(frames, c_dist_lo,
                c_dist_mean + c_dist_std*0.5,
                color="#E74C3C", alpha=0.2)

ax.plot(frames, s_dist_mean, color="#2E86C1", linewidth=2.5,
        marker="s", markersize=7, linestyle="--",
        label=f"Safe (n={safe_count:,})")
ax.fill_between(frames, s_dist_lo,
                s_dist_mean + s_dist_std*0.5,
                color="#2E86C1", alpha=0.2)

ax.axhline(y=5.0, color="#F39C12", linestyle=":",
           linewidth=1.8, label="General conflict threshold (5m)")
ax.axhline(y=3.0, color="#922B21", linestyle=":",
           linewidth=1.8, label="Serious conflict threshold (3m)")

# Annotate gap
for t in [1, 4, 8]:
    gap = s_dist_mean[t-1] - c_dist_mean[t-1]
    ax.annotate(f"{gap:.1f}m",
                xy=(t, (c_dist_mean[t-1]+s_dist_mean[t-1])/2),
                ha="center", fontsize=8, color="gray",
                fontweight="bold")

ax.set_xticks(frames)
ax.set_xticklabels(time_labels, fontsize=9)
ax.set_ylim(bottom=0)
ax.set_xlabel("Frame position in sequence", fontsize=12)
ax.set_ylabel("Min inter-agent distance (m)", fontsize=12)
ax.set_title("Inter-Agent Distance — Conflict vs Safe\n"
             f"Mean gap = {s_dist_mean.mean()-c_dist_mean.mean():.1f}m "
             f"(consistent across all 8 frames)",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_facecolor("#f9f9f9")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ── Panel 2: Conflict edge count per frame ─────────────────
print("Computing conflict edge counts per frame...")
conflict_edge_count_by_frame = [[] for _ in range(SEQ_LEN)]
safe_edge_count_by_frame     = [[] for _ in range(SEQ_LEN)]

cnt = 0
for seq, label in zip(sequences, seq_labels):
    if cnt >= 10000:
        break
    for t, g in enumerate(seq):
        n_conflict = sum(
            1 for e in g["edges"]
            if e["conflict_label"] == 1
        )
        n_total = len(g["edges"])
        if label == 1:
            conflict_edge_count_by_frame[t].append(n_conflict)
        else:
            safe_edge_count_by_frame[t].append(n_conflict)
    cnt += 1

c_ec_mean = np.array([np.mean(x) if x else 0
                      for x in conflict_edge_count_by_frame])
s_ec_mean = np.array([np.mean(x) if x else 0
                      for x in safe_edge_count_by_frame])

ax = axes[1]
ax.plot(frames, c_ec_mean, color="#E74C3C", linewidth=2.5,
        marker="o", markersize=7,
        label=f"Conflict sequences")
ax.plot(frames, s_ec_mean, color="#2E86C1", linewidth=2.5,
        marker="s", markersize=7, linestyle="--",
        label=f"Safe sequences")
ax.fill_between(frames, s_ec_mean, c_ec_mean,
                alpha=0.15, color="#E74C3C",
                label="Gap (conflict − safe)")

ax.set_xticks(frames)
ax.set_xticklabels(time_labels, fontsize=9)
ax.set_ylim(bottom=0)
ax.set_xlabel("Frame position in sequence", fontsize=12)
ax.set_ylabel("Mean conflict edges per frame", fontsize=12)
ax.set_title("Conflict Edge Count — Conflict vs Safe\n"
             "Number of edges with conflict_label=1 per frame",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3, linewidth=0.5)
ax.set_facecolor("#f9f9f9")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle(
    "Temporal Scene Dynamics — Conflict vs Safe Sequences (T=8 frames)\n"
    "Distance and conflict edge count are persistent and discriminative "
    "across all frames — justifying the LSTM temporal encoder",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "ssm_temporal_evolution_v3.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: ssm_temporal_evolution_v3.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# Conflict vs No-conflict spatial distribution
# Using pair-level data with coordinates
# ══════════════════════════════════════════════════════════

print("Extracting conflict pair coordinates...")

# Get midpoint coordinates for each conflict pair
# Use pairs dataframe which has x_i, y_i, x_j, y_j

print("Pairs columns:", pairs.columns.tolist())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute midpoint of each pair interaction
pairs["x_mid"] = (pairs["x_i"] + pairs["x_j"]) / 2
pairs["y_mid"] = (pairs["y_i"] + pairs["y_j"]) / 2

# Split by conflict type
no_conflict = pairs[pairs["pair_conflict_label"] == 0]
general     = pairs[pairs["pair_conflict_label"] == 1]
serious     = pairs[pairs["pair_conflict_label"] == 2]

print(f"No conflict : {len(no_conflict):,}")
print(f"General     : {len(general):,}")
print(f"Serious     : {len(serious):,}")

# ══════════════════════════════════════════════════════════
# Figure 1 — Conflict vs No-conflict midpoints
# ══════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 9))

# No conflict — sample for speed
nc_sample = no_conflict.sample(
    min(50000, len(no_conflict)), random_state=42)
ax.scatter(nc_sample["x_mid"], nc_sample["y_mid"],
           s=0.5, c="#2E86C1", alpha=0.15,
           rasterized=True,
           label=f"No conflict ({len(no_conflict):,} pairs)")

# General conflict
ax.scatter(general["x_mid"], general["y_mid"],
           s=3, c="#F39C12", alpha=0.6,
           rasterized=True,
           label=f"General conflict ({len(general):,} pairs)")

# Serious conflict — on top, most visible
ax.scatter(serious["x_mid"], serious["y_mid"],
           s=8, c="#E74C3C", alpha=0.85,
           rasterized=True,
           label=f"Serious conflict ({len(serious):,} pairs)")

# Intersection center
ax.axhline(y=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.axvline(x=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.scatter([0], [0], c="black", s=80, zorder=6,
           marker="+", linewidths=2.5,
           label="Intersection center")

ax.set_xlabel("x_local (m)", fontsize=12)
ax.set_ylabel("y_local (m)", fontsize=12)
ax.set_title(
    "Spatial Distribution of Conflict vs No-Conflict Pairs\n"
    "Midpoint coordinates of each agent pair interaction",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=10, loc="upper right",
          framealpha=0.9, markerscale=5)
ax.set_facecolor("#f8f8f8")
ax.grid(alpha=0.2)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(checkpoint_path / "conflict_spatial_midpoint.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: conflict_spatial_midpoint.png")

# ══════════════════════════════════════════════════════════
# Figure 2 — Individual agent positions (not midpoint)
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, label_val, label_name, color in [
    (axes[0], 0, "No conflict", "#2E86C1"),
    (axes[1], None, "All conflict", "#E74C3C"),
]:
    if label_val == 0:
        sub = pairs[pairs["pair_conflict_label"]==0].sample(
            min(30000, (pairs["pair_conflict_label"]==0).sum()),
            random_state=42
        )
    else:
        sub = pairs[pairs["pair_conflict_label"]>0]

    # Plot agent i positions
    ax.scatter(sub["x_i"], sub["y_i"],
               s=0.5, c=color, alpha=0.2,
               rasterized=True, label="Agent i position")
    # Plot agent j positions
    ax.scatter(sub["x_j"], sub["y_j"],
               s=0.5, c=color, alpha=0.2,
               rasterized=True, label="Agent j position")

    ax.axhline(y=0, color="black", linestyle="--",
               alpha=0.3, linewidth=0.8)
    ax.axvline(x=0, color="black", linestyle="--",
               alpha=0.3, linewidth=0.8)
    ax.scatter([0],[0], c="black", s=80,
               marker="+", linewidths=2.5, zorder=5)

    ax.set_xlabel("x_local (m)", fontsize=11)
    ax.set_ylabel("y_local (m)", fontsize=11)
    ax.set_title(f"{label_name} — Agent Positions\n"
                 f"({len(sub):,} pair observations)",
                 fontsize=12, fontweight="bold")
    ax.set_facecolor("#f8f8f8")
    ax.grid(alpha=0.2)
    ax.set_aspect("equal")

plt.suptitle(
    "Agent Position Distribution — No Conflict vs Conflict\n"
    "Individual positions of both agents in each pair",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "conflict_agent_positions.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: conflict_agent_positions.png")

# ══════════════════════════════════════════════════════════
# Figure 3 — Conflict by interaction type + hotspot
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# VV conflicts
vv_conf = pairs[
    (pairs["pair_conflict_label"]>0) &
    (pairs["interaction_type"]=="vv")
]
vp_conf = pairs[
    (pairs["pair_conflict_label"]>0) &
    (pairs["interaction_type"]=="vp")
]

ax = axes[0]
ax.scatter(vv_conf["x_mid"], vv_conf["y_mid"],
           s=2, c="#2E86C1", alpha=0.5,
           rasterized=True,
           label=f"VV conflicts ({len(vv_conf):,})")
ax.scatter(vp_conf["x_mid"], vp_conf["y_mid"],
           s=8, c="#E74C3C", alpha=0.7,
           rasterized=True,
           label=f"VP conflicts ({len(vp_conf):,})")
ax.axhline(y=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.axvline(x=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.scatter([0],[0], c="black", s=80,
           marker="+", linewidths=2.5, zorder=5)
ax.set_xlabel("x_local (m)", fontsize=11)
ax.set_ylabel("y_local (m)", fontsize=11)
ax.set_title("Conflict Locations by Interaction Type\n"
             "VV (blue) vs VP (red) midpoint coordinates",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10, framealpha=0.9, markerscale=5)
ax.set_facecolor("#f8f8f8")
ax.grid(alpha=0.2)
ax.set_aspect("equal")

# Serious conflict hexbin hotspot
ax = axes[1]
all_conf = pairs[pairs["pair_conflict_label"]>0]
ax.scatter(no_conflict.sample(30000, random_state=42)["x_mid"],
           no_conflict.sample(30000, random_state=42)["y_mid"],
           s=0.3, c="#CCCCCC", alpha=0.2,
           rasterized=True, label="No conflict (bg)")

h = ax.hexbin(all_conf["x_mid"], all_conf["y_mid"],
              gridsize=35, cmap="YlOrRd",
              mincnt=1, alpha=0.85, linewidths=0.2)
cb = plt.colorbar(h, ax=ax)
cb.set_label("Conflict pair density", fontsize=10)

ax.axhline(y=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.axvline(x=0, color="black", linestyle="--",
           alpha=0.3, linewidth=0.8)
ax.scatter([0],[0], c="black", s=80,
           marker="+", linewidths=2.5, zorder=5,
           label="Intersection center")
ax.set_xlabel("x_local (m)", fontsize=11)
ax.set_ylabel("y_local (m)", fontsize=11)
ax.set_title("Conflict Hotspot Density Map\n"
             "All conflict pairs (general + serious)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9, markerscale=5)
ax.set_facecolor("#f0f0f0")
ax.grid(alpha=0.2)
ax.set_aspect("equal")

plt.suptitle(
    "Spatial Conflict Analysis — "
    "Centerpoint Road / Outlet Mall Intersection",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "conflict_hotspot_by_type.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: conflict_hotspot_by_type.png")

In [ ]:
# ══════════════════════════════════════════════════════════
# Step 1 — Run Enhanced HERMES on test set
# Extract predictions + coordinates per sequence
# ══════════════════════════════════════════════════════════

import torch
import numpy as np

print("Running Enhanced HERMES on test set...")

enhanced_model.eval()
loader_test_graph = DataLoader(
    ds_test, batch_size=64,
    shuffle=False, num_workers=0
)

all_probs  = []
all_labels = []

with torch.no_grad():
    for batch in loader_test_graph:
        X, Avv, Avp, App, E, y = [b.to(device) for b in batch]
        logits = enhanced_model(X, Avv, Avp, App, E)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y.cpu().numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# Best threshold from single run
threshold = 0.5
all_preds = (all_probs >= threshold).astype(int)

print(f"Test sequences : {len(all_labels):,}")
print(f"Conflict true  : {all_labels.sum():,}")
print(f"Conflict pred  : {all_preds.sum():,}")

# Categorize each sequence
# TP: predicted conflict, actually conflict
# FN: predicted safe,     actually conflict
# FP: predicted conflict, actually safe
# TN: predicted safe,     actually safe
categories = []
for true, pred in zip(all_labels, all_preds):
    if true == 1 and pred == 1:
        categories.append("TP")
    elif true == 1 and pred == 0:
        categories.append("FN")
    elif true == 0 and pred == 1:
        categories.append("FP")
    else:
        categories.append("TN")

from collections import Counter
cat_counts = Counter(categories)
print(f"\nTP (correct conflict)  : {cat_counts['TP']:,}")
print(f"FN (missed conflict)   : {cat_counts['FN']:,}")
print(f"FP (false alarm)       : {cat_counts['FP']:,}")
print(f"TN (correct safe)      : {cat_counts['TN']:,}")

# ── Step 2 — Extract mean agent coordinates per sequence ──
print("\nExtracting coordinates from test sequences...")

seq_coords = []
for seq in test_seqs:
    # Mean x and y of all agents across all 8 frames
    all_x = [n["x"] for g in seq for n in g["nodes"]]
    all_y = [n["y"] for g in seq for n in g["nodes"]]
    seq_coords.append({
        "x_mean": np.mean(all_x) if all_x else 0,
        "y_mean": np.mean(all_y) if all_y else 0,
        "x_min" : np.min(all_x)  if all_x else 0,
        "x_max" : np.max(all_x)  if all_x else 0,
        "y_min" : np.min(all_y)  if all_y else 0,
        "y_max" : np.max(all_y)  if all_y else 0,
    })

print(f"Coordinates extracted for {len(seq_coords):,} sequences")

# Build arrays per category
tp_x = [seq_coords[i]["x_mean"] for i,c in enumerate(categories) if c=="TP"]
tp_y = [seq_coords[i]["y_mean"] for i,c in enumerate(categories) if c=="TP"]
fn_x = [seq_coords[i]["x_mean"] for i,c in enumerate(categories) if c=="FN"]
fn_y = [seq_coords[i]["y_mean"] for i,c in enumerate(categories) if c=="FN"]
fp_x = [seq_coords[i]["x_mean"] for i,c in enumerate(categories) if c=="FP"]
fp_y = [seq_coords[i]["y_mean"] for i,c in enumerate(categories) if c=="FP"]
tn_x = [seq_coords[i]["x_mean"] for i,c in enumerate(categories) if c=="TN"]
tn_y = [seq_coords[i]["y_mean"] for i,c in enumerate(categories) if c=="TN"]

print(f"\nCoordinates per category:")
print(f"  TP : {len(tp_x):,}")
print(f"  FN : {len(fn_x):,}")
print(f"  FP : {len(fp_x):,}")
print(f"  TN : {len(tn_x):,}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# ── Common settings ────────────────────────────────────────
xlim = (min(tn_x + tp_x + fn_x + fp_x) - 1,
        max(tn_x + tp_x + fn_x + fp_x) + 1)
ylim = (min(tn_y + tp_y + fn_y + fp_y) - 1,
        max(tn_y + tp_y + fn_y + fp_y) + 1)

def add_intersection_center(ax):
    ax.axhline(y=0, color="black", linestyle="--",
               alpha=0.25, linewidth=0.8)
    ax.axvline(x=0, color="black", linestyle="--",
               alpha=0.25, linewidth=0.8)
    ax.scatter([0],[0], c="black", s=100,
               marker="+", linewidths=2.5,
               zorder=10)

# ══════════════════════════════════════════════════════════
# Panel 1 — Ground truth (conflict vs safe)
# ══════════════════════════════════════════════════════════
ax = axes[0]

tn_sample_x = tn_x[:5000]
tn_sample_y = tn_y[:5000]

ax.scatter(tn_sample_x, tn_sample_y,
           s=1, c="#AED6F1", alpha=0.3,
           rasterized=True, label=f"True safe (n={len(tn_x):,})")
ax.scatter(tp_x + fn_x, tp_y + fn_y,
           s=4, c="#E74C3C", alpha=0.6,
           rasterized=True,
           label=f"True conflict (n={len(tp_x)+len(fn_x):,})")

add_intersection_center(ax)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel("x_local (m)", fontsize=11)
ax.set_ylabel("y_local (m)", fontsize=11)
ax.set_title("Ground Truth\nConflict vs Safe Sequences",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9,
          markerscale=4, loc="upper right")
ax.set_facecolor("#f8f8f8")
ax.grid(alpha=0.2)
ax.set_aspect("equal")

# ══════════════════════════════════════════════════════════
# Panel 2 — Model predictions (predicted conflict vs safe)
# ══════════════════════════════════════════════════════════
ax = axes[1]

pred_safe_x    = tn_x + fn_x
pred_safe_y    = tn_y + fn_y
pred_conf_x    = tp_x + fp_x
pred_conf_y    = tp_y + fp_y

ps_sample_x = pred_safe_x[:5000]
ps_sample_y = pred_safe_y[:5000]

ax.scatter(ps_sample_x, ps_sample_y,
           s=1, c="#AED6F1", alpha=0.3,
           rasterized=True,
           label=f"Predicted safe (n={len(pred_safe_x):,})")
ax.scatter(pred_conf_x, pred_conf_y,
           s=4, c="#E74C3C", alpha=0.6,
           rasterized=True,
           label=f"Predicted conflict (n={len(pred_conf_x):,})")

add_intersection_center(ax)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel("x_local (m)", fontsize=11)
ax.set_ylabel("y_local (m)", fontsize=11)
ax.set_title("HERMES Predictions\nPredicted Conflict vs Safe",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9,
          markerscale=4, loc="upper right")
ax.set_facecolor("#f8f8f8")
ax.grid(alpha=0.2)
ax.set_aspect("equal")

# ══════════════════════════════════════════════════════════
# Panel 3 — Error analysis (TP, FN, FP)
# ══════════════════════════════════════════════════════════
ax = axes[2]

# TN background
ax.scatter(tn_sample_x, tn_sample_y,
           s=0.5, c="#D5D8DC", alpha=0.2,
           rasterized=True, label=f"TN safe (n={len(tn_x):,})")

# TP — correctly detected conflicts
ax.scatter(tp_x, tp_y,
           s=6, c="#27AE60", alpha=0.7,
           rasterized=True,
           label=f"TP correct conflict (n={len(tp_x):,})")

# FP — false alarms
ax.scatter(fp_x, fp_y,
           s=6, c="#F39C12", alpha=0.7,
           rasterized=True,
           label=f"FP false alarm (n={len(fp_x):,})")

# FN — missed conflicts (most critical — show on top)
ax.scatter(fn_x, fn_y,
           s=25, c="#E74C3C", alpha=0.9,
           marker="x", linewidths=1.5,
           rasterized=True,
           label=f"FN missed conflict (n={len(fn_x):,})")

add_intersection_center(ax)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel("x_local (m)", fontsize=11)
ax.set_ylabel("y_local (m)", fontsize=11)
ax.set_title(
    f"Spatial Error Analysis\n"
    f"TP={len(tp_x):,}  FP={len(fp_x):,}  "
    f"FN={len(fn_x):,}  TN={len(tn_x):,}",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=9, framealpha=0.9,
          markerscale=3, loc="upper right")
ax.set_facecolor("#f8f8f8")
ax.grid(alpha=0.2)
ax.set_aspect("equal")

# ── Stats annotation ───────────────────────────────────────
precision = len(tp_x) / (len(tp_x) + len(fp_x))
recall    = len(tp_x) / (len(tp_x) + len(fn_x))
ax.text(0.02, 0.02,
        f"Precision = {precision:.3f}\n"
        f"Recall    = {recall:.3f}\n"
        f"Miss rate = {len(fn_x)/(len(tp_x)+len(fn_x))*100:.1f}%",
        transform=ax.transAxes,
        fontsize=9, verticalalignment="bottom",
        bbox=dict(boxstyle="round,pad=0.4",
                  facecolor="white", alpha=0.85,
                  edgecolor="gray", linewidth=0.8))

plt.suptitle(
    "HERMES Spatial Prediction Analysis — Test Set\n"
    "Centerpoint Road / Outlet Mall Intersection  |  "
    "Evening hours (5:38PM–7:39PM)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(checkpoint_path / "spatial_prediction_analysis.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: spatial_prediction_analysis.png")

# ── Hexbin version for Panel 3 separately ─────────────────
fig, ax = plt.subplots(figsize=(10, 9))

# Background density of all test sequences
h_bg = ax.hexbin(
    [c["x_mean"] for c in seq_coords],
    [c["y_mean"] for c in seq_coords],
    gridsize=35, cmap="Greys",
    mincnt=1, alpha=0.3, linewidths=0.1
)

# Overlay FN as red X
ax.scatter(fn_x, fn_y,
           s=60, c="#E74C3C", alpha=0.9,
           marker="x", linewidths=2.0,
           zorder=5, label=f"Missed conflicts FN ({len(fn_x):,})")

# Overlay FP as orange triangles
ax.scatter(fp_x, fp_y,
           s=15, c="#F39C12", alpha=0.6,
           marker="^", zorder=4,
           label=f"False alarms FP ({len(fp_x):,})")

add_intersection_center(ax)
ax.set_xlabel("x_local (m)", fontsize=12)
ax.set_ylabel("y_local (m)", fontsize=12)
ax.set_title(
    "HERMES Error Locations — Test Set\n"
    "Where does the model miss conflicts "
    "and raise false alarms?",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11, framealpha=0.9,
          markerscale=2, loc="upper right")
ax.set_facecolor("#f0f0f0")
ax.grid(alpha=0.2)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(checkpoint_path / "spatial_error_map.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: spatial_error_map.png")

In [ ]:
"""
Section 4.1 — Experimental Dataset Summary and Class Distribution
Run this in your Jupyter environment after the sequence construction cell.
Requires: train_labels, val_labels, test_labels, scene_graphs (list of frame dicts)
Outputs : Table 4.1 (printed) + Figure 4.1 (saved as section_4_1_class_dist.pdf)
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings("ignore")

# ── 1. Sequence counts ────────────────────────────────────────────────────────
n_train      = len(train_labels)
n_val        = len(val_labels)
n_test       = len(test_labels)
n_total      = n_train + n_val + n_test

all_labels   = np.concatenate([train_labels, val_labels, test_labels])
n_conflict   = int((all_labels == 1).sum())
n_safe       = int((all_labels == 0).sum())
conflict_pct = n_conflict / n_total * 100
imbalance    = n_safe / n_conflict

train_conf   = int((train_labels == 1).sum())
val_conf     = int((val_labels   == 1).sum())
test_conf    = int((test_labels  == 1).sum())

# ── 2. Graph statistics (nodes / edges per frame) ────────────────────────────
num_nodes, num_edges, num_conf_edges = [], [], []
for g in scene_graphs:
    num_nodes.append(len(g["nodes"]))
    num_edges.append(len(g["edges"]))
    num_conf_edges.append(
        sum(1 for e in g["edges"]
            if e.get("conflict_label", 0) > 0)
    )

avg_nodes      = np.mean(num_nodes)
avg_edges      = np.mean(num_edges)
max_nodes      = max(num_nodes)
max_edges      = max(num_edges)

# ── 3. Print Table 4.1 ───────────────────────────────────────────────────────
print("=" * 60)
print("TABLE 4.1  Experimental dataset summary")
print("=" * 60)
print(f"{'Metric':<38} {'Value':>10}")
print("-" * 60)
print(f"{'Total sequences':<38} {n_total:>10,}")
print(f"{'  Training sequences':<38} {n_train:>10,}  ({n_train/n_total*100:.1f}%)")
print(f"{'  Validation sequences':<38} {n_val:>10,}  ({n_val/n_total*100:.1f}%)")
print(f"{'  Test sequences':<38} {n_test:>10,}  ({n_test/n_total*100:.1f}%)")
print("-" * 60)
print(f"{'Conflict sequences (label = 1)':<38} {n_conflict:>10,}  ({conflict_pct:.1f}%)")
print(f"{'Safe sequences     (label = 0)':<38} {n_safe:>10,}  ({100-conflict_pct:.1f}%)")
print(f"{'Class imbalance ratio (safe:conflict)':<38} {imbalance:>9.2f}:1")
print("-" * 60)
print(f"{'Train conflict rate':<38} {train_conf/n_train*100:>9.1f}%")
print(f"{'Validation conflict rate':<38} {val_conf/n_val*100:>9.1f}%")
print(f"{'Test conflict rate':<38} {test_conf/n_test*100:>9.1f}%")
print("-" * 60)
print(f"{'Sequence length (frames)':<38} {'8':>10}")
print(f"{'Sliding window stride (frames)':<38} {'4':>10}")
print(f"{'Frame rate (fps)':<38} {'30':>10}")
print(f"{'Sequence duration (seconds)':<38} {'0.267':>10}")
print("-" * 60)
print(f"{'Avg. nodes per frame':<38} {avg_nodes:>10.1f}")
print(f"{'Avg. edges per frame':<38} {avg_edges:>10.1f}")
print(f"{'Max nodes per frame':<38} {max_nodes:>10}")
print(f"{'Max edges per frame':<38} {max_edges:>10}")
print("=" * 60)

# ── 4. Figure 4.1 ────────────────────────────────────────────────────────────
BLUE    = "#2563EB"
RED     = "#DC2626"
GRAY    = "#6B7280"
LGRAY   = "#F3F4F6"
BLACK   = "#111827"

fig = plt.figure(figsize=(14, 5.5), facecolor="white")
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# ── Panel A: stacked bar — sequences per split ────────────────────────────────
ax1 = fig.add_subplot(gs[0])
splits      = ["Train", "Validation", "Test"]
safe_counts = [n_train - train_conf, n_val - val_conf, n_test - test_conf]
conf_counts = [train_conf, val_conf, test_conf]
totals      = [n_train, n_val, n_test]

x = np.arange(3)
b1 = ax1.bar(x, safe_counts, color=BLUE,  alpha=0.85, label="Safe",     width=0.55, zorder=3)
b2 = ax1.bar(x, conf_counts, bottom=safe_counts, color=RED, alpha=0.85,
             label="Conflict", width=0.55, zorder=3)

for i, (s, c, t) in enumerate(zip(safe_counts, conf_counts, totals)):
    ax1.text(i, t + 400, f"{t:,}", ha="center", va="bottom",
             fontsize=8.5, color=BLACK, fontweight="semibold")
    cr = c / t * 100
    ax1.text(i, s + c / 2, f"{cr:.1f}%", ha="center", va="center",
             fontsize=7.5, color="white", fontweight="bold")

ax1.set_xticks(x)
ax1.set_xticklabels(splits, fontsize=10)
ax1.set_ylabel("Sequences", fontsize=9, color=GRAY)
ax1.set_title("A  Sequence distribution by split", fontsize=10,
              fontweight="bold", loc="left", pad=8, color=BLACK)
ax1.set_ylim(0, max(totals) * 1.12)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v/1000)}k"))
ax1.legend(fontsize=8.5, frameon=False, loc="upper right")
ax1.spines[["top","right"]].set_visible(False)
ax1.tick_params(axis="y", labelcolor=GRAY, labelsize=8)
ax1.grid(axis="y", color=LGRAY, zorder=0)

# ── Panel B: overall class distribution donut ─────────────────────────────────
ax2 = fig.add_subplot(gs[1])
sizes  = [n_safe, n_conflict]
colors = [BLUE, RED]
wedges, _ = ax2.pie(
    sizes, colors=colors, startangle=90,
    wedgeprops=dict(width=0.48, edgecolor="white", linewidth=2.5),
    counterclock=False
)
ax2.text(0, 0.08, f"{conflict_pct:.1f}%", ha="center", va="center",
         fontsize=17, fontweight="bold", color=RED)
ax2.text(0, -0.22, "conflict", ha="center", va="center",
         fontsize=8.5, color=GRAY)
ax2.set_title("B  Overall class distribution", fontsize=10,
              fontweight="bold", loc="left", pad=8, color=BLACK)
legend_labels = [f"Safe  ({n_safe:,})", f"Conflict  ({n_conflict:,})"]
ax2.legend(wedges, legend_labels, loc="lower center",
           bbox_to_anchor=(0.5, -0.08), fontsize=8.5, frameon=False,
           ncol=2)

# ── Panel C: graph density — avg nodes & edges per frame ─────────────────────
ax3 = fig.add_subplot(gs[2])
metrics = ["Avg nodes\nper frame", "Avg edges\nper frame"]
vals    = [avg_nodes, avg_edges]
bar_c   = [BLUE, "#7C3AED"]
bars    = ax3.bar([0, 1], vals, color=bar_c, alpha=0.85, width=0.5, zorder=3)

for bar, v in zip(bars, vals):
    ax3.text(bar.get_x() + bar.get_width() / 2, v + 0.3,
             f"{v:.1f}", ha="center", va="bottom",
             fontsize=11, fontweight="bold", color=BLACK)

ax3.set_xticks([0, 1])
ax3.set_xticklabels(metrics, fontsize=9.5)
ax3.set_ylabel("Count", fontsize=9, color=GRAY)
ax3.set_title("C  Scene graph density", fontsize=10,
              fontweight="bold", loc="left", pad=8, color=BLACK)
ax3.set_ylim(0, max(vals) * 1.22)
ax3.spines[["top","right"]].set_visible(False)
ax3.tick_params(axis="y", labelcolor=GRAY, labelsize=8)
ax3.grid(axis="y", color=LGRAY, zorder=0)

# Add max annotations
ax3.annotate(f"max={max_nodes}", xy=(0, avg_nodes), xytext=(0.25, avg_nodes + max(vals)*0.08),
             fontsize=7.5, color=GRAY,
             arrowprops=dict(arrowstyle="-", color=LGRAY))
ax3.annotate(f"max={max_edges}", xy=(1, avg_edges), xytext=(1.25, avg_edges + max(vals)*0.08),
             fontsize=7.5, color=GRAY,
             arrowprops=dict(arrowstyle="-", color=LGRAY))

fig.suptitle(
    "Figure 4.1   Experimental dataset summary — sequence counts, class distribution, and scene graph density",
    fontsize=9, color=GRAY, y=0.02, ha="center"
)

plt.savefig("section_4_1_class_dist.pdf", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.savefig("section_4_1_class_dist.png", dpi=200, bbox_inches="tight",
            facecolor="white")
plt.show()
print("\nFigure saved: section_4_1_class_dist.pdf / .png")

In [ ]:
"""
Section 4.1 — Publication-quality figures (separate files, all bar charts)
Figure 4.1a : Sequence distribution by split (stacked bar, % labels)
Figure 4.1b : Overall class distribution (grouped bar)
Figure 4.1c : Scene graph density (bar)

Run after sequence construction. Requires:
  train_labels, val_labels, test_labels  (numpy arrays)
  scene_graphs                            (list of frame dicts)
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── Publication style ─────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["Times New Roman", "DejaVu Serif"],
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.linewidth"    : 1.1,
    "axes.labelsize"    : 16,
    "xtick.labelsize"   : 15,
    "ytick.labelsize"   : 15,
    "legend.fontsize"   : 14,
    "figure.dpi"        : 300,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

# ── Palette ───────────────────────────────────────────────────────────────────
C_SAFE     = "#2166AC"
C_CONFLICT = "#D6604D"
C_NODES    = "#4393C3"
C_EDGES    = "#762A83"
C_ANNOT    = "#222222"
C_GRID     = "#EBEBEB"

# ── Compute stats ─────────────────────────────────────────────────────────────
n_train    = len(train_labels)
n_val      = len(val_labels)
n_test     = len(test_labels)
n_total    = n_train + n_val + n_test

train_conf = int((train_labels == 1).sum())
val_conf   = int((val_labels   == 1).sum())
test_conf  = int((test_labels  == 1).sum())
n_conflict = train_conf + val_conf + test_conf
n_safe     = n_total - n_conflict
conflict_pct = n_conflict / n_total * 100
safe_pct     = 100 - conflict_pct
imbalance    = n_safe / n_conflict

num_nodes, num_edges = [], []
for g in scene_graphs:
    num_nodes.append(len(g["nodes"]))
    num_edges.append(len(g["edges"]))
avg_nodes = np.mean(num_nodes)
avg_edges = np.mean(num_edges)
max_nodes = max(num_nodes)
max_edges = max(num_edges)

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 4.1a — Sequence distribution by split (stacked bar, % only)
# ═════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7.5, 5.8), facecolor="white")

splits      = ["Training", "Validation", "Test"]
totals      = [n_train, n_val, n_test]
conf_counts = [train_conf, val_conf, test_conf]
safe_counts = [t - c for t, c in zip(totals, conf_counts)]

x = np.arange(3)
W = 0.52

b1 = ax.bar(x, safe_counts, width=W, color=C_SAFE,
            label="Safe (label = 0)", zorder=3)
b2 = ax.bar(x, conf_counts, width=W, bottom=safe_counts,
            color=C_CONFLICT, label="Conflict (label = 1)", zorder=3)

# Minimum bar height (in data units) to fit a label inside
MIN_HEIGHT_INSIDE = max(totals) * 0.06

for i, (s, c, t) in enumerate(zip(safe_counts, conf_counts, totals)):
    safe_pct_split = s / t * 100
    conf_pct_split = c / t * 100

    # ── Safe label ──────────────────────────────────────────────────────────
    if s >= MIN_HEIGHT_INSIDE:
        ax.text(i, s / 2,
                f"{safe_pct_split:.1f}%",
                ha="center", va="center",
                fontsize=14, fontweight="bold", color="white", zorder=4)
    else:
        ax.text(i, s / 2,
                f"{safe_pct_split:.1f}%",
                ha="center", va="center",
                fontsize=13, fontweight="bold", color=C_SAFE, zorder=4)

    # ── Conflict label ───────────────────────────────────────────────────────
    if c >= MIN_HEIGHT_INSIDE:
        ax.text(i, s + c / 2,
                f"{conf_pct_split:.1f}%",
                ha="center", va="center",
                fontsize=14, fontweight="bold", color="white", zorder=4)
    else:
        # Place label just above the bar top
        ax.text(i, t + max(totals) * 0.012,
                f"{conf_pct_split:.1f}%",
                ha="center", va="bottom",
                fontsize=13, fontweight="bold", color=C_CONFLICT, zorder=4)

    # ── Total count above bar (always outside) ───────────────────────────────
    offset = max(totals) * 0.048 if c < MIN_HEIGHT_INSIDE else max(totals) * 0.015
    ax.text(i, t + offset + max(totals) * 0.02,
            f"n = {t:,}",
            ha="center", va="bottom",
            fontsize=12, color="#555555", style="italic", zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel("Number of sequences", labelpad=10)
ax.set_xlabel("Data partition", labelpad=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f"{int(v/1000)}k" if v >= 1000 else str(int(v))))
ax.set_ylim(0, max(totals) * 1.22)
# ax.grid(axis="y", color=C_GRID, zorder=0, linewidth=0.9)
ax.legend(loc="upper right", frameon=False,
          handlelength=1.2, handleheight=1.0)
ax.tick_params(axis="both", which="both", length=4)

plt.tight_layout()
plt.savefig("fig_4_1a_split_distribution.pdf")
plt.savefig("fig_4_1a_split_distribution.png")
plt.show()
print("Saved: fig_4_1a_split_distribution.pdf / .png")


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 4.1b — Overall class distribution (horizontal bar chart)
# ═════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7.5, 4.0), facecolor="white")

classes  = ["Safe\n(label = 0)", "Conflict\n(label = 1)"]
counts   = [n_safe, n_conflict]
pcts     = [safe_pct, conflict_pct]
colors   = [C_SAFE, C_CONFLICT]

y = np.arange(2)
W = 0.42

bars = ax.barh(y, counts, height=W, color=colors, zorder=3,
               edgecolor="white", linewidth=1.0)

MIN_WIDTH_INSIDE = max(counts) * 0.15

for i, (bar, cnt, pct, col) in enumerate(zip(bars, counts, pcts, colors)):
    w = bar.get_width()
    label = f"{cnt:,}  ({pct:.1f}%)"
    if w >= MIN_WIDTH_INSIDE:
        ax.text(w / 2, bar.get_y() + bar.get_height() / 2,
                label,
                ha="center", va="center",
                fontsize=15, fontweight="bold", color="white", zorder=4)
    else:
        ax.text(w + max(counts) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                label,
                ha="left", va="center",
                fontsize=14, fontweight="bold", color=col, zorder=4)

ax.set_yticks(y)
ax.set_yticklabels(classes)
ax.set_xlabel("Number of sequences", labelpad=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f"{int(v/1000)}k" if v >= 1000 else str(int(v))))
ax.set_xlim(0, max(counts) * 1.35)
ax.grid(axis="x", color=C_GRID, zorder=0, linewidth=0.9)
ax.tick_params(axis="both", which="both", length=4)

# Imbalance annotation
ax.text(max(counts) * 1.30, 0.5,
        f"Imbalance\n{imbalance:.2f} : 1",
        ha="right", va="center",
        fontsize=12, color="#777777", style="italic")

plt.tight_layout()
plt.savefig("fig_4_1b_class_distribution.pdf")
plt.savefig("fig_4_1b_class_distribution.png")
plt.show()
print("Saved: fig_4_1b_class_distribution.pdf / .png")


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 4.1c — Scene graph density (vertical bar)
# ═════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(6.5, 5.5), facecolor="white")

metrics = ["Avg. nodes\nper frame", "Avg. edges\nper frame"]
avgs    = [avg_nodes, avg_edges]
maxs    = [max_nodes, max_edges]
colors  = [C_NODES, C_EDGES]
x       = np.arange(2)
W       = 0.48

bars = ax.bar(x, avgs, width=W, color=colors,
              zorder=3, edgecolor="white", linewidth=1.2)

for bar, v, mx in zip(bars, avgs, maxs):
    # Avg value label
    ax.text(bar.get_x() + bar.get_width() / 2,
            v + max(avgs) * 0.022,
            f"{v:.1f}",
            ha="center", va="bottom",
            fontsize=18, fontweight="bold", color=C_ANNOT)
    # Max label
    ax.text(bar.get_x() + bar.get_width() / 2,
            v + max(avgs) * 0.11,
            f"max = {mx}",
            ha="center", va="bottom",
            fontsize=13, color="#777777", style="italic")

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Count", labelpad=10)
ax.set_xlabel("Graph statistic", labelpad=10)
ax.set_ylim(0, max(avgs) * 1.35)
ax.grid(axis="y", color=C_GRID, zorder=0, linewidth=0.9)
ax.tick_params(axis="both", which="both", length=4)

for tick, c in zip(ax.get_xticklabels(), colors):
    tick.set_color(c)
    tick.set_fontweight("bold")

plt.tight_layout()
plt.savefig("fig_4_1c_graph_density.pdf")
plt.savefig("fig_4_1c_graph_density.png")
plt.show()
print("Saved: fig_4_1c_graph_density.pdf / .png")

print("\nAll Section 4.1 figures generated successfully.")

In [ ]:
import pandas as pd

# ── ML model best hyperparameters ────────────────────────────────────────────
ml_grids = {
    "Logistic Regression" : lr_grid,
    "Random Forest"       : rf_grid,
    "XGBoost"             : xgb_grid,
    "LightGBM"            : lgb_grid,
    "SVM"                 : svm_grid,
}

print("=" * 72)
print("TABLE 4.2  Best hyperparameters — ML models (3-fold stratified CV)")
print("=" * 72)
print(f"{'Model':<22} {'Parameter':<22} {'Search space':<28} {'Best value'}")
print("-" * 72)

search_spaces = {
    "Logistic Regression": {
        "C"       : "[0.01, 0.1, 1.0, 10.0]",
        "penalty" : "[l1, l2]",
    },
    "Random Forest": {
        "n_estimators"    : "[100, 300, 500]",
        "max_depth"       : "[5, 10, 20, None]",
        "min_samples_leaf": "[1, 5, 10]",
    },
    "XGBoost": {
        "n_estimators" : "[100, 300, 500]",
        "max_depth"    : "[3, 5, 7]",
        "learning_rate": "[0.01, 0.05, 0.1]",
        "subsample"    : "[0.7, 0.9]",
    },
    "LightGBM": {
        "n_estimators" : "[100, 300, 500]",
        "max_depth"    : "[5, 10, 20]",
        "learning_rate": "[0.01, 0.05, 0.1]",
        "num_leaves"   : "[31, 63, 127]",
    },
    "SVM": {
        "C"    : "[0.1, 1.0, 10.0]",
        "gamma": "[scale, auto]",
    },
}

fixed_params = {
    "Logistic Regression" : {"solver": "saga", "class_weight": "balanced", "max_iter": 1000},
    "Random Forest"       : {"class_weight": "balanced"},
    "XGBoost"             : {"scale_pos_weight": scale_pos, "eval_metric": "auc"},
    "LightGBM"            : {"class_weight": "balanced"},
    "SVM"                 : {"kernel": "rbf", "class_weight": "balanced", "probability": True},
}

rows = []
for model_name, grid in ml_grids.items():
    best  = grid.best_params_
    space = search_spaces[model_name]
    fixed = fixed_params[model_name]
    first = True
    for param, search in space.items():
        best_val = str(best.get(param, "—"))
        label    = model_name if first else ""
        print(f"{label:<22} {param:<22} {search:<28} {best_val}")
        rows.append({"Model": model_name, "Parameter": param,
                     "Search space": search, "Best value": best_val,
                     "CV AUC": f"{grid.best_score_:.4f}" if first else ""})
        first = False
    # Fixed params
    for param, val in fixed.items():
        print(f"{'':22} {param:<22} {'fixed':<28} {val}")
        rows.append({"Model": model_name, "Parameter": param,
                     "Search space": "fixed", "Best value": str(val),
                     "CV AUC": ""})
    print(f"{'':22} {'— CV AUC (3-fold)':<22} {'':28} {grid.best_score_:.4f}")
    print("-" * 72)

# ── DL and graph model hyperparameters ───────────────────────────────────────
print("\n" + "=" * 72)
print("TABLE 4.2 (cont.)  Fixed hyperparameters — DL and graph models")
print("=" * 72)
print(f"{'Parameter':<30} {'All DL models':<22} {'Graph models'}")
print("-" * 72)

shared_params = [
    ("Hidden dimension",       "64",           "64"),
    ("Number of layers",       "2",            "2 GNN + 2 LSTM"),
    ("Dropout rate",           "0.3",          "0.3"),
    ("Optimizer",              "Adam",         "Adam"),
    ("Learning rate",          "1e-3",         "1e-3"),
    ("Weight decay",           "1e-4",         "1e-4"),
    ("LR scheduler",           "ReduceLROnPlateau", "ReduceLROnPlateau"),
    ("Scheduler factor",       "0.5",          "0.5"),
    ("Scheduler patience",     "3 epochs",     "3 epochs"),
    ("Early stopping patience","5 epochs",     "5 epochs"),
    ("Max epochs",             "100",          "50"),
    ("Batch size",             "16",           "16"),
    ("Gradient clip norm",     "1.0",          "1.0"),
    ("pos_weight",             f"{scale_pos}",  f"{scale_pos}"),
    ("Random seeds",           "42,123,456,789,2024", "42,123,456,789,2024"),
]

for param, dl_val, graph_val in shared_params:
    print(f"{param:<30} {dl_val:<22} {graph_val}")

print("=" * 72)

# ── Save to CSV ───────────────────────────────────────────────────────────────
df_ml = pd.DataFrame(rows)
df_ml.to_csv("table_4_2_hyperparameters_ml.csv", index=False)
print("\nSaved: table_4_2_hyperparameters_ml.csv")